In [ ]:
# %% [code] {"execution":{"iopub.status.busy":"2026-09-09T12:04:10.807307Z","iopub.execute_input":"2026-09-09T12:04:10.807875Z","iopub.status.idle":"2026-09-09T12:04:43.568916Z","shell.execute_reply.started":"2026-09-09T12:04:10.807790Z","shell.execute_reply":"2026-09-09T12:04:43.566326Z"},"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 1 — NASA FIRMS 2025 MASTER DATA
# Load + Validate
# ============================================================

import pandas as pd
import numpy as np
import os

print("=" * 80)
print("STEP 1 — NASA FIRMS 2025 MASTER DATA")
print("=" * 80)

# ------------------------------------------------------------
# 1. DATA PATH
# ------------------------------------------------------------

FIRMS_PATH = (
    "/kaggle/input/datasets/"
    "gautamkumar0036/"
    "nasafirms-osm-dw-2025/"
    "firms_osm_dynamicworld_full_2025.csv"
)

print("\nDataset path:")
print(FIRMS_PATH)

if not os.path.exists(FIRMS_PATH):
    raise FileNotFoundError("FIRMS dataset not found.")

print("File exists: YES")

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

print("\nLoading dataset...")

df = pd.read_csv(
    FIRMS_PATH,
    low_memory=False
)

print("Dataset loaded successfully.")

# ------------------------------------------------------------
# 3. BASIC INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BASIC DATASET INFORMATION")
print("=" * 80)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns):,}")

# ------------------------------------------------------------
# 4. DISPLAY ALL COLUMNS
# ------------------------------------------------------------

print("\nAvailable columns:")

for i, col in enumerate(df.columns, start=1):
    print(f"{i:3d}. {col}")

# ------------------------------------------------------------
# 5. REQUIRED FIRMS COLUMNS
# ------------------------------------------------------------

required_columns = [
    "latitude",
    "longitude",
    "brightness",
    "scan",
    "track",
    "acq_date",
    "acq_time",
    "satellite",
    "instrument",
    "confidence",
    "version",
    "bright_t31",
    "frp",
    "daynight"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("\n" + "=" * 80)
print("FIRMS CORE COLUMN CHECK")
print("=" * 80)

if missing_columns:
    print("Missing columns:")
    for col in missing_columns:
        print(" -", col)
    raise ValueError("Required FIRMS columns are missing.")
else:
    print("All required FIRMS columns are present.")

# ------------------------------------------------------------
# 6. DATE CONVERSION
# ------------------------------------------------------------

df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

invalid_dates = df["acq_date"].isna().sum()

print("\nInvalid acquisition dates:", f"{invalid_dates:,}")

# ------------------------------------------------------------
# 7. DATE RANGE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATE COVERAGE")
print("=" * 80)

print("Minimum date:", df["acq_date"].min())
print("Maximum date:", df["acq_date"].max())

# ------------------------------------------------------------
# 8. YEAR CHECK
# ------------------------------------------------------------

df["_year"] = df["acq_date"].dt.year

print("\nRows by year:")
print(df["_year"].value_counts().sort_index())

non_2025 = (df["_year"] != 2025).sum()

print("\nRows outside 2025:", f"{non_2025:,}")

# ------------------------------------------------------------
# 9. MONTH CHECK
# ------------------------------------------------------------

df["_month"] = df["acq_date"].dt.month

monthly_counts = (
    df["_month"]
    .value_counts()
    .sort_index()
)

month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}

print("\n" + "=" * 80)
print("FIRMS DETECTIONS BY MONTH — 2025")
print("=" * 80)

for month in range(1, 13):

    count = monthly_counts.get(month, 0)

    print(
        f"{month:02d} | "
        f"{month_names[month]:10s} | "
        f"{count:>10,}"
    )

missing_months = [
    month
    for month in range(1, 13)
    if monthly_counts.get(month, 0) == 0
]

if missing_months:
    print("\nWARNING — Missing months:", missing_months)
else:
    print("\nAll 12 months are present.")

# ------------------------------------------------------------
# 10. COORDINATE VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COORDINATE VALIDATION")
print("=" * 80)

df["latitude"] = pd.to_numeric(
    df["latitude"],
    errors="coerce"
)

df["longitude"] = pd.to_numeric(
    df["longitude"],
    errors="coerce"
)

invalid_lat = (
    df["latitude"].isna() |
    ~df["latitude"].between(-90, 90)
)

invalid_lon = (
    df["longitude"].isna() |
    ~df["longitude"].between(-180, 180)
)

print("Invalid latitude :", f"{invalid_lat.sum():,}")
print("Invalid longitude:", f"{invalid_lon.sum():,}")

# Approximate India bounding box
india_mask = (
    df["latitude"].between(6, 38) &
    df["longitude"].between(67, 98)
)

print(
    "\nObservations inside approximate India bounding box:",
    f"{india_mask.sum():,}"
)

print(
    "Observations outside approximate India bounding box:",
    f"{(~india_mask).sum():,}"
)

# ------------------------------------------------------------
# 11. EXACT DUPLICATES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DUPLICATE ANALYSIS")
print("=" * 80)

exact_duplicates = df.duplicated().sum()

print(
    "Exact duplicate rows:",
    f"{exact_duplicates:,}"
)

# ------------------------------------------------------------
# 12. FIRMS OBSERVATION IDENTITY
# ------------------------------------------------------------

identity_columns = [
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
    "satellite",
    "instrument"
]

available_identity_columns = [
    c for c in identity_columns
    if c in df.columns
]

identity_duplicates = df.duplicated(
    subset=available_identity_columns
).sum()

print(
    "Repeated observation identities:",
    f"{identity_duplicates:,}"
)

# ------------------------------------------------------------
# 13. REPEATED LOCATION DIAGNOSTIC
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("REPEATED LOCATION DIAGNOSTIC")
print("=" * 80)

# Diagnostic only.
# These rounded coordinates are NOT used to remove observations.

df["_lat_round"] = df["latitude"].round(3)
df["_lon_round"] = df["longitude"].round(3)

location_counts = (
    df.groupby(
        ["_lat_round", "_lon_round"]
    )
    .size()
    .sort_values(ascending=False)
)

print(
    "Approximate unique locations:",
    f"{len(location_counts):,}"
)

print("\nTop 20 repeated locations:")

print(location_counts.head(20))

# ------------------------------------------------------------
# 14. NUMBER OF OBSERVATIONS PER LOCATION
# ------------------------------------------------------------

print("\nObservation-frequency distribution:")

print(
    location_counts.describe()
)

# ------------------------------------------------------------
# 15. IMPORTANT PROJECT RULE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PROJECT RULE")
print("=" * 80)

print("""
IMPORTANT:

The FIRMS rows are observations, NOT independent fires.

We will NOT remove repeated detections simply because they
occur at the same or nearby coordinates.

A location may be detected repeatedly throughout 2025.

Later we will separately model:

    Detection
        ↓
    Spatial block
        ↓
    Activity / fire event
        ↓
    Source behavior

The complete 12-month behavior will be retained.
""")

# ------------------------------------------------------------
# 16. REMOVE TEMPORARY DIAGNOSTIC COLUMNS
# ------------------------------------------------------------

df.drop(
    columns=[
        "_year",
        "_month",
        "_lat_round",
        "_lon_round"
    ],
    inplace=True
)

# ------------------------------------------------------------
# 17. FINAL CHECK
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 1 FINAL CHECK")
print("=" * 80)

print(f"Final rows    : {len(df):,}")
print(f"Final columns : {len(df.columns):,}")

print(
    "Date range:",
    df["acq_date"].min(),
    "→",
    df["acq_date"].max()
)

print(
    "Unique dates:",
    df["acq_date"].nunique()
)

print(
    "Exact duplicates:",
    df.duplicated().sum()
)

print(
    "Missing latitude:",
    df["latitude"].isna().sum()
)

print(
    "Missing longitude:",
    df["longitude"].isna().sum()
)

print("\n" + "=" * 80)
print("STEP 1 COMPLETE")
print("=" * 80)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:04:43.574583Z","iopub.execute_input":"2026-09-09T12:04:43.574947Z","iopub.status.idle":"2026-09-09T12:06:22.801681Z","shell.execute_reply.started":"2026-09-09T12:04:43.574918Z","shell.execute_reply":"2026-09-09T12:06:22.799978Z"}}
# ============================================================
# STEP 2A — CREATE 500m × 500m SPATIAL GRID
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 2A — 500m × 500m SPATIAL GRID")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD STEP 1 DATA
# ------------------------------------------------------------

FIRMS_PATH = (
    "/kaggle/input/datasets/"
    "gautamkumar0036/"
    "nasafirms-osm-dw-2025/"
    "firms_osm_dynamicworld_full_2025.csv"
)

print("\nLoading master FIRMS dataset...")

df = pd.read_csv(
    FIRMS_PATH,
    low_memory=False
)

df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

print(f"Rows loaded: {len(df):,}")

# ------------------------------------------------------------
# 2. CHECK COORDINATES
# ------------------------------------------------------------

if df["latitude"].isna().any():
    raise ValueError("Missing latitude values found.")

if df["longitude"].isna().any():
    raise ValueError("Missing longitude values found.")

print("Coordinates validated.")

# ------------------------------------------------------------
# 3. APPROXIMATE 500m GRID
# ------------------------------------------------------------
#
# India spans a large geographic area.
# We therefore calculate longitude distance according to latitude.
#
# This produces an approximately 500m geographic grid.
#
# IMPORTANT:
# This is a spatial CONTEXT grid.
# It does NOT mean all detections inside a cell are one fire.
# ------------------------------------------------------------

LAT_STEP_KM = 0.5
KM_PER_DEG_LAT = 111.32

# Latitude grid
df["block_500m_lat"] = np.floor(
    df["latitude"] * KM_PER_DEG_LAT / LAT_STEP_KM
).astype(np.int32)

# Longitude distance varies with latitude.
# Use a latitude-dependent longitude scale.
lon_km = (
    KM_PER_DEG_LAT *
    np.cos(np.radians(df["latitude"]))
)

# Avoid numerical problems near the poles.
lon_km = np.maximum(lon_km, 1e-6)

df["block_500m_lon"] = np.floor(
    df["longitude"] * lon_km / LAT_STEP_KM
).astype(np.int32)

# ------------------------------------------------------------
# 4. CREATE UNIQUE BLOCK ID
# ------------------------------------------------------------

df["block_500m_id"] = (
    df["block_500m_lat"].astype(str)
    + "_"
    + df["block_500m_lon"].astype(str)
)

# ------------------------------------------------------------
# 5. BASIC GRID STATISTICS
# ------------------------------------------------------------

n_blocks = df["block_500m_id"].nunique()

print("\n" + "=" * 80)
print("500m GRID SUMMARY")
print("=" * 80)

print(f"Total FIRMS observations : {len(df):,}")
print(f"Unique 500m blocks       : {n_blocks:,}")

print(
    f"Average observations/block: "
    f"{len(df) / n_blocks:.2f}"
)

# ------------------------------------------------------------
# 6. OBSERVATIONS PER BLOCK
# ------------------------------------------------------------

block_counts = (
    df.groupby("block_500m_id")
      .size()
      .sort_values(ascending=False)
)

print("\n" + "=" * 80)
print("OBSERVATIONS PER 500m BLOCK")
print("=" * 80)

print(block_counts.describe())

print("\nTop 20 blocks by number of FIRMS observations:")

print(block_counts.head(20))

# ------------------------------------------------------------
# 7. BLOCK DENSITY DISTRIBUTION
# ------------------------------------------------------------

print("\nBlock observation frequency:")

for threshold in [1, 2, 5, 10, 20, 50, 100, 500, 1000]:

    count = (block_counts >= threshold).sum()

    print(
        f"Blocks with >= {threshold:4d} observations: "
        f"{count:,}"
    )

# ------------------------------------------------------------
# 8. CHECK SINGLE-OBSERVATION BLOCKS
# ------------------------------------------------------------

single_blocks = (block_counts == 1).sum()

print(
    "\nBlocks containing exactly one observation:",
    f"{single_blocks:,}"
)

print(
    "Percentage:",
    f"{single_blocks / n_blocks * 100:.2f}%"
)

# ------------------------------------------------------------
# 9. CHECK HIGH-DENSITY BLOCKS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("HIGH-DENSITY BLOCK CHECK")
print("=" * 80)

high_density = block_counts[block_counts >= 100]

print(
    "Blocks with >=100 observations:",
    f"{len(high_density):,}"
)

if len(high_density) > 0:
    print("\nHighest-density blocks:")
    print(high_density.head(20))

# ------------------------------------------------------------
# 10. CHECK MONTHLY BEHAVIOR WITHIN BLOCKS
# ------------------------------------------------------------
#
# This is only a first diagnostic.
# Full block behavior will be constructed in Step 2B.
# ------------------------------------------------------------

df["_month"] = df["acq_date"].dt.month

monthly_block_counts = (
    df.groupby(
        ["block_500m_id", "_month"]
    )
    .size()
)

print("\n" + "=" * 80)
print("MONTHLY BLOCK CHECK")
print("=" * 80)

print(
    "Block-month combinations:",
    f"{len(monthly_block_counts):,}"
)

active_months_per_block = (
    df.groupby("block_500m_id")["_month"]
      .nunique()
)

print("\nActive months per block:")

print(active_months_per_block.describe())

# ------------------------------------------------------------
# 11. IMPORTANT INTERPRETATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)

print("""
500m blocks are being used as SPATIAL CONTEXT.

A 500m block is NOT a fire.
A 500m block is NOT a source.
A 500m block is NOT assigned a class.

A block may contain multiple independent activities.

Later we will calculate the complete 2025 behavior
of each block and attach those behavioral features
to individual FIRMS observations/events.
""")

# ------------------------------------------------------------
# 12. REMOVE TEMPORARY COLUMN
# ------------------------------------------------------------

df.drop(
    columns=["_month"],
    inplace=True
)

# ------------------------------------------------------------
# 13. FINAL VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 2A FINAL VALIDATION")
print("=" * 80)

print(f"Rows              : {len(df):,}")
print(f"Unique 500m blocks: {df['block_500m_id'].nunique():,}")

print(
    "Missing block IDs:",
    df["block_500m_id"].isna().sum()
)

print(
    "Missing latitude:",
    df["latitude"].isna().sum()
)

print(
    "Missing longitude:",
    df["longitude"].isna().sum()
)

# ------------------------------------------------------------
# 14. SAVE CHECKPOINT
# ------------------------------------------------------------

OUTPUT_PATH = "/kaggle/working/firms_2025_500m_grid.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved:")
print(OUTPUT_PATH)

print("\n" + "=" * 80)
print("STEP 2A COMPLETE")
print("=" * 80)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:06:22.804125Z","iopub.execute_input":"2026-09-09T12:06:22.805503Z","iopub.status.idle":"2026-09-09T12:08:15.559209Z","shell.execute_reply.started":"2026-09-09T12:06:22.805448Z","shell.execute_reply":"2026-09-09T12:08:15.557470Z"}}
# ============================================================
# STEP 2B — 500m BLOCK: FULL-YEAR 2025 BEHAVIOR
# CORRECTED VERSION
# ============================================================

import pandas as pd
import numpy as np
import os

print("=" * 80)
print("STEP 2B — 500m BLOCK FULL-YEAR BEHAVIOR")
print("=" * 80)

# ------------------------------------------------------------
# 1. LOAD STEP 2A
# ------------------------------------------------------------

INPUT_PATH = "/kaggle/working/firms_2025_500m_grid.csv"

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"Step 2A output not found:\n{INPUT_PATH}"
    )

print("\nLoading Step 2A dataset...")

df = pd.read_csv(
    INPUT_PATH,
    low_memory=False
)

df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

print(f"Rows loaded: {len(df):,}")

# ------------------------------------------------------------
# 2. VALIDATION
# ------------------------------------------------------------

required_columns = [
    "block_500m_id",
    "acq_date",
    "frp",
    "brightness",
    "latitude",
    "longitude"
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

if df["block_500m_id"].isna().any():
    raise ValueError(
        "Missing 500m block IDs found."
    )

print("Required columns validated.")

# ------------------------------------------------------------
# 3. TEMPORAL FIELDS
# ------------------------------------------------------------

df["_month"] = df["acq_date"].dt.month

df["_day"] = (
    df["acq_date"]
    .dt.normalize()
)

# ------------------------------------------------------------
# 4. BASIC BLOCK ACTIVITY
# ------------------------------------------------------------

print("\nCalculating block activity...")

block_basic = (
    df.groupby("block_500m_id")
      .agg(
          block_500m_total_detections=(
              "block_500m_id",
              "size"
          ),

          block_500m_active_days=(
              "_day",
              "nunique"
          ),

          block_500m_active_months=(
              "_month",
              "nunique"
          ),

          block_500m_mean_frp=(
              "frp",
              "mean"
          ),

          block_500m_median_frp=(
              "frp",
              "median"
          ),

          block_500m_max_frp=(
              "frp",
              "max"
          ),

          block_500m_std_frp=(
              "frp",
              "std"
          ),

          block_500m_mean_brightness=(
              "brightness",
              "mean"
          ),

          block_500m_max_brightness=(
              "brightness",
              "max"
          )
      )
)

# ------------------------------------------------------------
# 5. HANDLE SINGLE-DETECTION BLOCKS
# ------------------------------------------------------------

block_basic[
    "block_500m_std_frp"
] = (
    block_basic[
        "block_500m_std_frp"
    ]
    .fillna(0)
)

print(
    f"Blocks summarized: {len(block_basic):,}"
)

# ------------------------------------------------------------
# 6. MONTHLY DETECTION COUNTS
# ------------------------------------------------------------

print("\nCalculating monthly behavior...")

monthly_counts = (
    df.groupby(
        ["block_500m_id", "_month"]
    )
    .size()
    .unstack(fill_value=0)
)

# Make sure all 12 months exist.
monthly_counts = monthly_counts.reindex(
    columns=range(1, 13),
    fill_value=0
)

monthly_counts.columns = [
    f"block_500m_detections_month_{m:02d}"
    for m in range(1, 13)
]

# ------------------------------------------------------------
# 7. PEAK MONTH
# ------------------------------------------------------------

peak_month = monthly_counts.idxmax(axis=1)

peak_count = monthly_counts.max(axis=1)

peak_features = pd.DataFrame({
    "block_500m_peak_month": peak_month,
    "block_500m_peak_month_detections": peak_count
})

# ------------------------------------------------------------
# 8. MONTHLY CONCENTRATION
# ------------------------------------------------------------
#
# Herfindahl-style concentration:
#
# 1.0  = all activity concentrated in one month
# lower = activity distributed across months
# ------------------------------------------------------------

monthly_proportions = (
    monthly_counts
    .div(
        monthly_counts.sum(axis=1),
        axis=0
    )
)

monthly_concentration = (
    monthly_proportions ** 2
).sum(axis=1)

concentration_features = pd.DataFrame({
    "block_500m_monthly_concentration":
        monthly_concentration
})

# ------------------------------------------------------------
# 9. DETECTIONS PER ACTIVE DAY
# ------------------------------------------------------------

block_basic[
    "block_500m_detections_per_active_day"
] = (
    block_basic[
        "block_500m_total_detections"
    ]
    /
    block_basic[
        "block_500m_active_days"
    ]
)

# ------------------------------------------------------------
# 10. MONTH PERSISTENCE
# ------------------------------------------------------------

block_basic[
    "block_500m_month_persistence_ratio"
] = (
    block_basic[
        "block_500m_active_months"
    ] / 12.0
)

# ------------------------------------------------------------
# 11. DAY PERSISTENCE
# ------------------------------------------------------------

block_basic[
    "block_500m_day_persistence_ratio"
] = (
    block_basic[
        "block_500m_active_days"
    ] / 365.0
)

# ------------------------------------------------------------
# 12. FIRST AND LAST ACTIVE DATE
# ------------------------------------------------------------

date_features = (
    df.groupby("block_500m_id")
      .agg(
          block_500m_first_active_date=(
              "_day",
              "min"
          ),

          block_500m_last_active_date=(
              "_day",
              "max"
          )
      )
)

date_features[
    "block_500m_activity_span_days"
] = (
    date_features[
        "block_500m_last_active_date"
    ]
    -
    date_features[
        "block_500m_first_active_date"
    ]
).dt.days

# ------------------------------------------------------------
# 13. ACTIVE-DAY GAP STATISTICS
# ------------------------------------------------------------

print("\nCalculating active-day gaps...")

active_dates = (
    df[
        ["block_500m_id", "_day"]
    ]
    .drop_duplicates()
    .sort_values(
        ["block_500m_id", "_day"]
    )
)

active_dates["_gap_days"] = (
    active_dates
    .groupby("block_500m_id")["_day"]
    .diff()
    .dt.days
)

gap_features = (
    active_dates
    .groupby("block_500m_id")
    .agg(
        block_500m_mean_active_day_gap=(
            "_gap_days",
            "mean"
        ),

        block_500m_median_active_day_gap=(
            "_gap_days",
            "median"
        ),

        block_500m_max_active_day_gap=(
            "_gap_days",
            "max"
        )
    )
)

gap_features = gap_features.fillna(0)

# ------------------------------------------------------------
# 14. COMBINE BLOCK FEATURES
# ------------------------------------------------------------

print("\nCombining block-level features...")

block_features = (
    block_basic
    .join(peak_features)
    .join(concentration_features)
    .join(date_features)
    .join(gap_features)
    .join(monthly_counts)
)

print(
    f"Block feature table:"
    f" {block_features.shape[0]:,} blocks × "
    f"{block_features.shape[1]:,} features"
)

# ------------------------------------------------------------
# 15. BLOCK BEHAVIOR SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("500m BLOCK BEHAVIOR SUMMARY")
print("=" * 80)

summary_columns = [
    "block_500m_total_detections",
    "block_500m_active_days",
    "block_500m_active_months",
    "block_500m_month_persistence_ratio",
    "block_500m_day_persistence_ratio",
    "block_500m_detections_per_active_day",
    "block_500m_monthly_concentration",
    "block_500m_activity_span_days",
    "block_500m_mean_frp",
    "block_500m_max_frp"
]

print(
    block_features[
        summary_columns
    ].describe()
)

# ------------------------------------------------------------
# 16. TOP BLOCKS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TOP 20 BLOCKS BY 2025 DETECTION COUNT")
print("=" * 80)

top_blocks = (
    block_features
    .sort_values(
        "block_500m_total_detections",
        ascending=False
    )
    .head(20)
)

print(
    top_blocks[
        [
            "block_500m_total_detections",
            "block_500m_active_days",
            "block_500m_active_months",
            "block_500m_peak_month",
            "block_500m_monthly_concentration",
            "block_500m_mean_frp",
            "block_500m_max_frp"
        ]
    ]
)

# ------------------------------------------------------------
# 17. ACTIVE-MONTH DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NUMBER OF ACTIVE MONTHS PER 500m BLOCK")
print("=" * 80)

active_month_distribution = (
    block_features[
        "block_500m_active_months"
    ]
    .value_counts()
    .sort_index()
)

for months, count in active_month_distribution.items():

    print(
        f"{int(months):2d} active month(s): "
        f"{count:,} blocks"
    )

# ------------------------------------------------------------
# 18. PERSISTENT BLOCKS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PERSISTENT BLOCKS")
print("=" * 80)

for threshold in [2, 3, 4, 6, 9, 12]:

    count = (
        block_features[
            "block_500m_active_months"
        ] >= threshold
    ).sum()

    print(
        f"Blocks active in >= {threshold:2d} months: "
        f"{count:,}"
    )

# ------------------------------------------------------------
# 19. VALIDATE BLOCK FEATURE TABLE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BLOCK FEATURE VALIDATION")
print("=" * 80)

unique_blocks = (
    df["block_500m_id"]
    .nunique()
)

print(
    "Unique 500m blocks:",
    f"{unique_blocks:,}"
)

print(
    "Block feature rows:",
    f"{len(block_features):,}"
)

if len(block_features) != unique_blocks:

    raise ValueError(
        "Block feature count does not match "
        "unique 500m blocks."
    )

if (
    block_features[
        "block_500m_active_months"
    ].min() < 1
):

    raise ValueError(
        "Invalid active-month count."
    )

if (
    block_features[
        "block_500m_active_months"
    ].max() > 12
):

    raise ValueError(
        "Active months cannot exceed 12."
    )

print("Block feature validation passed.")

# ------------------------------------------------------------
# 20. ATTACH BLOCK BEHAVIOR TO EVERY FIRMS OBSERVATION
# ------------------------------------------------------------

print("\nAttaching annual block behavior...")

block_features_reset = (
    block_features
    .reset_index()
)

df = df.merge(
    block_features_reset,
    on="block_500m_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 21. ROW COUNT VALIDATION
# ------------------------------------------------------------

expected_rows = 655204

if len(df) != expected_rows:

    raise ValueError(
        f"ERROR: Row count changed! "
        f"Expected {expected_rows:,}, "
        f"got {len(df):,}"
    )

print(
    "Row count preserved:",
    f"{len(df):,}"
)

# ------------------------------------------------------------
# 22. MISSING BLOCK BEHAVIOR
# ------------------------------------------------------------

missing_behavior = (
    df[
        "block_500m_total_detections"
    ]
    .isna()
    .sum()
)

print(
    "Rows missing block behavior:",
    f"{missing_behavior:,}"
)

if missing_behavior != 0:

    raise ValueError(
        "Some FIRMS observations "
        "did not receive block behavior."
    )

# ------------------------------------------------------------
# 23. REMOVE TEMPORARY COLUMNS
# ------------------------------------------------------------

df.drop(
    columns=[
        "_month",
        "_day"
    ],
    inplace=True
)

# ------------------------------------------------------------
# 24. SAVE CHECKPOINT
# ------------------------------------------------------------

OUTPUT_PATH = (
    "/kaggle/working/"
    "firms_2025_500m_block_behavior.csv"
)

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nSaved:")
print(OUTPUT_PATH)

# ------------------------------------------------------------
# 25. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STEP 2B COMPLETE")
print("=" * 80)

print(
    f"Final observations : {len(df):,}"
)

print(
    f"Final columns      : {len(df.columns):,}"
)

print(
    f"500m blocks        : "
    f"{df['block_500m_id'].nunique():,}"
)

print(
    "\nEach FIRMS observation now contains "
    "the complete 2025 behavior of its 500m block."
)

print(
    "\nNo FIRMS observations were removed."
)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:08:15.563024Z","iopub.execute_input":"2026-09-09T12:08:15.564005Z","iopub.status.idle":"2026-09-09T12:10:12.991553Z","shell.execute_reply.started":"2026-09-09T12:08:15.563957Z","shell.execute_reply":"2026-09-09T12:10:12.990460Z"}}
# ============================================================
# STEP 2C — CREATE 1 KM × 1 KM SPATIAL BLOCKS
# ============================================================

import pandas as pd
import numpy as np

INPUT_FILE = "/kaggle/working/firms_2025_500m_block_behavior.csv"
OUTPUT_FILE = "/kaggle/working/firms_2025_1km_block_behavior.csv"

print("=" * 70)
print("STEP 2C — 1 KM × 1 KM SPATIAL GRID")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD EXISTING 500m DATA
# ------------------------------------------------------------
print("\nLoading existing 500m dataset...")

df = pd.read_csv(INPUT_FILE)

print(f"Rows loaded: {len(df):,}")
print(f"Columns loaded: {len(df.columns):,}")

# ------------------------------------------------------------
# 2. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------
required_cols = [
    "latitude",
    "longitude",
    "acq_date",
    "frp",
    "brightness"
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# 3. CREATE 1 KM GRID
# ------------------------------------------------------------
KM_PER_DEG_LAT = 111.32
GRID_SIZE_KM = 1.0

df["block_1km_lat"] = np.floor(
    df["latitude"] * KM_PER_DEG_LAT / GRID_SIZE_KM
).astype(np.int32)

lon_km = (
    KM_PER_DEG_LAT *
    np.cos(np.radians(df["latitude"]))
)

lon_km = np.maximum(lon_km, 1e-6)

df["block_1km_lon"] = np.floor(
    df["longitude"] * lon_km / GRID_SIZE_KM
).astype(np.int32)

df["block_1km_id"] = (
    df["block_1km_lat"].astype(str)
    + "_"
    + df["block_1km_lon"].astype(str)
)

# ------------------------------------------------------------
# 4. BASIC GRID VALIDATION
# ------------------------------------------------------------
print("\n--- GRID VALIDATION ---")

unique_blocks = df["block_1km_id"].nunique()

print(
    f"Unique occupied 1km blocks: "
    f"{unique_blocks:,}"
)

obs_per_block = (
    df.groupby("block_1km_id")
      .size()
)

print(
    f"Average observations/block: "
    f"{obs_per_block.mean():.4f}"
)

print(
    f"Median observations/block: "
    f"{obs_per_block.median():.0f}"
)

print(
    f"Maximum observations/block: "
    f"{obs_per_block.max():,}"
)

# ------------------------------------------------------------
# 5. OBSERVATION DISTRIBUTION
# ------------------------------------------------------------
print("\n--- OBSERVATIONS PER BLOCK ---")

for threshold in [2, 5, 10, 20, 50, 100, 500, 1000]:
    count = (obs_per_block >= threshold).sum()

    print(
        f"Blocks with >= {threshold:4} observations: "
        f"{count:,}"
    )

single_obs = (obs_per_block == 1).sum()

print(
    f"\nBlocks with exactly 1 observation: "
    f"{single_obs:,} "
    f"({single_obs / unique_blocks * 100:.2f}%)"
)

# ------------------------------------------------------------
# 6. PREPARE DATE / MONTH
# ------------------------------------------------------------
df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

if df["acq_date"].isna().any():
    raise ValueError("Invalid acquisition dates detected.")

df["month"] = df["acq_date"].dt.month.astype(np.int8)

# ------------------------------------------------------------
# 7. ACTIVE MONTHS
# ------------------------------------------------------------
block_months = (
    df.groupby("block_1km_id")["month"]
      .nunique()
)

print("\n--- ACTIVE MONTHS PER 1KM BLOCK ---")

print(
    block_months
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 8. FULL-YEAR BLOCK BEHAVIOR
# ------------------------------------------------------------
print("\n--- CALCULATING 1KM FULL-YEAR BEHAVIOR ---")

behavior = (
    df.groupby("block_1km_id")
      .agg(
          block_1km_total_detections=("block_1km_id", "size"),

          block_1km_active_days=("acq_date", "nunique"),

          block_1km_active_months=("month", "nunique"),

          block_1km_mean_frp=("frp", "mean"),

          block_1km_median_frp=("frp", "median"),

          block_1km_max_frp=("frp", "max"),

          block_1km_std_frp=("frp", "std"),

          block_1km_mean_brightness=("brightness", "mean"),

          block_1km_max_brightness=("brightness", "max"),

          block_1km_first_active_date=("acq_date", "min"),

          block_1km_last_active_date=("acq_date", "max")
      )
)

# ------------------------------------------------------------
# 9. DAILY BEHAVIOR
# ------------------------------------------------------------
print("Calculating daily behavior...")

daily_counts = (
    df.groupby(
        ["block_1km_id", "acq_date"]
    )
    .size()
)

daily_summary = (
    daily_counts
    .groupby(level=0)
    .agg(
        block_1km_detections_per_active_day="mean"
    )
)

behavior = behavior.join(daily_summary)

# ------------------------------------------------------------
# 10. MONTHLY COUNTS
# ------------------------------------------------------------
print("Calculating monthly behavior...")

monthly_counts_raw = (
    df.groupby(
        ["block_1km_id", "month"]
    )
    .size()
    .unstack(fill_value=0)
)

# Ensure all 12 months exist BEFORE renaming
monthly_counts_raw = monthly_counts_raw.reindex(
    columns=range(1, 13),
    fill_value=0
)

# ------------------------------------------------------------
# 11. PEAK MONTH
# ------------------------------------------------------------
# IMPORTANT:
# Calculate peak month while columns are still numeric 1..12.
# This fixes the previous ValueError.

peak_month = monthly_counts_raw.idxmax(axis=1)

peak_month_detections = monthly_counts_raw.max(axis=1)

behavior["block_1km_peak_month"] = (
    peak_month.astype(np.int8)
)

behavior["block_1km_peak_month_detections"] = (
    peak_month_detections.astype(np.int32)
)

# ------------------------------------------------------------
# 12. RENAME MONTHLY COUNT COLUMNS
# ------------------------------------------------------------
monthly_counts = monthly_counts_raw.copy()

monthly_counts.columns = [
    f"block_1km_detections_month_{m:02d}"
    for m in range(1, 13)
]

behavior = behavior.join(monthly_counts)

# ------------------------------------------------------------
# 13. MONTHLY CONCENTRATION
# ------------------------------------------------------------
monthly_total = monthly_counts_raw.sum(axis=1)

monthly_shares = monthly_counts_raw.div(
    monthly_total.replace(0, np.nan),
    axis=0
)

behavior["block_1km_monthly_concentration"] = (
    (monthly_shares ** 2).sum(axis=1)
)

# ------------------------------------------------------------
# 14. MONTH PERSISTENCE
# ------------------------------------------------------------
behavior["block_1km_month_persistence_ratio"] = (
    behavior["block_1km_active_months"] / 12.0
)

# ------------------------------------------------------------
# 15. DAY PERSISTENCE
# ------------------------------------------------------------
behavior["block_1km_day_persistence_ratio"] = (
    behavior["block_1km_active_days"] / 365.0
)

# ------------------------------------------------------------
# 16. ACTIVITY SPAN
# ------------------------------------------------------------
behavior["block_1km_activity_span_days"] = (
    behavior["block_1km_last_active_date"]
    - behavior["block_1km_first_active_date"]
).dt.days

# ------------------------------------------------------------
# 17. ACTIVE-DAY GAPS
# ------------------------------------------------------------
print("Calculating active-day gaps...")

active_dates = (
    df[
        ["block_1km_id", "acq_date"]
    ]
    .drop_duplicates()
    .sort_values(
        ["block_1km_id", "acq_date"]
    )
)

active_dates["day_gap"] = (
    active_dates
    .groupby("block_1km_id")["acq_date"]
    .diff()
    .dt.days
)

gap_summary = (
    active_dates
    .groupby("block_1km_id")["day_gap"]
    .agg(
        block_1km_mean_active_day_gap="mean",
        block_1km_median_active_day_gap="median",
        block_1km_max_active_day_gap="max"
    )
)

behavior = behavior.join(gap_summary)

# ------------------------------------------------------------
# 18. CLEAN NaN VALUES
# ------------------------------------------------------------
behavior["block_1km_std_frp"] = (
    behavior["block_1km_std_frp"]
    .fillna(0)
)

for col in [
    "block_1km_mean_active_day_gap",
    "block_1km_median_active_day_gap",
    "block_1km_max_active_day_gap"
]:
    behavior[col] = behavior[col].fillna(0)

# ------------------------------------------------------------
# 19. RESET INDEX
# ------------------------------------------------------------
behavior = behavior.reset_index()

print(
    f"\n1km behavior table: "
    f"{len(behavior):,} blocks × "
    f"{len(behavior.columns)} features"
)

# ------------------------------------------------------------
# 20. ATTACH BEHAVIOR TO EACH FIRMS DETECTION
# ------------------------------------------------------------
print("\nAttaching 1km behavior to FIRMS observations...")

df = df.merge(
    behavior,
    on="block_1km_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 21. FINAL VALIDATION
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print(
    f"Final rows: {len(df):,}"
)

print(
    f"Final columns: {len(df.columns):,}"
)

# Row count MUST remain unchanged
if len(df) != 655204:
    raise ValueError(
        f"Row count changed! "
        f"Expected 655,204, got {len(df):,}"
    )

missing_behavior = (
    df["block_1km_total_detections"]
    .isna()
    .sum()
)

print(
    f"Missing 1km behavior rows: "
    f"{missing_behavior:,}"
)

if missing_behavior != 0:
    raise ValueError(
        "Some FIRMS observations do not have 1km behavior."
    )

# ------------------------------------------------------------
# 22. PEAK MONTH VALIDATION
# ------------------------------------------------------------
peak_values = (
    df["block_1km_peak_month"]
    .dropna()
    .unique()
)

print(
    "\nPeak month values:",
    sorted(peak_values.tolist())
)

if not set(peak_values).issubset(set(range(1, 13))):
    raise ValueError(
        "Invalid peak month detected."
    )

# ------------------------------------------------------------
# 23. SAVE
# ------------------------------------------------------------
print("\nSaving dataset...")

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved successfully:\n"
    f"{OUTPUT_FILE}"
)

print("\n" + "=" * 70)
print("STEP 2C COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:10:12.992857Z","iopub.execute_input":"2026-09-09T12:10:12.993106Z","iopub.status.idle":"2026-09-09T12:10:33.159691Z","shell.execute_reply.started":"2026-09-09T12:10:12.993085Z","shell.execute_reply":"2026-09-09T12:10:33.157568Z"}}
# ============================================================
# STEP 3A — PREPARE DATA FOR FIRE / ACTIVITY EVENT GROUPING
# ============================================================

import pandas as pd
import numpy as np

INPUT_FILE = "/kaggle/working/firms_2025_1km_block_behavior.csv"

print("=" * 70)
print("STEP 3A — PREPARE FIRMS DATA FOR EVENT GROUPING")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------
print("\nLoading 1km behavior dataset...")

df = pd.read_csv(INPUT_FILE)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

# ------------------------------------------------------------
# 2. REQUIRED COLUMNS
# ------------------------------------------------------------
required_cols = [
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
    "frp",
    "brightness",
    "block_500m_id",
    "block_1km_id"
]

missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# ------------------------------------------------------------
# 3. DATE / TIME
# ------------------------------------------------------------
df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

if df["acq_date"].isna().any():
    raise ValueError(
        "Invalid acquisition dates found."
    )

# FIRMS acq_time is generally HHMM
df["acq_time_str"] = (
    df["acq_time"]
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(4)
)

df["acq_hour"] = (
    df["acq_time_str"]
    .str[:2]
    .astype(int)
)

df["acq_minute"] = (
    df["acq_time_str"]
    .str[2:]
    .astype(int)
)

# ------------------------------------------------------------
# 4. CREATE DATETIME
# ------------------------------------------------------------
df["acq_datetime"] = (
    df["acq_date"]
    + pd.to_timedelta(df["acq_hour"], unit="h")
    + pd.to_timedelta(df["acq_minute"], unit="m")
)

# ------------------------------------------------------------
# 5. BASIC VALIDATION
# ------------------------------------------------------------
print("\n--- DATE/TIME VALIDATION ---")

print(
    "Date range:",
    df["acq_date"].min().date(),
    "to",
    df["acq_date"].max().date()
)

print(
    "Datetime range:",
    df["acq_datetime"].min(),
    "to",
    df["acq_datetime"].max()
)

invalid_time = (
    (df["acq_hour"] < 0) |
    (df["acq_hour"] > 23) |
    (df["acq_minute"] < 0) |
    (df["acq_minute"] > 59)
).sum()

print(
    f"Invalid time values: {invalid_time:,}"
)

if invalid_time > 0:
    raise ValueError(
        "Invalid FIRMS acquisition time detected."
    )

# ------------------------------------------------------------
# 6. CREATE TEMPORAL DAY INDEX
# ------------------------------------------------------------
df["day_of_year"] = (
    df["acq_date"]
    .dt.dayofyear
    .astype(np.int16)
)

# ------------------------------------------------------------
# 7. CREATE COARSE EVENT SEARCH GRID
# ------------------------------------------------------------
# This grid is ONLY used to efficiently find nearby points.
# It is NOT the final spatial block.
#
# Approximate:
#   1 degree latitude  ≈ 111.32 km
#
# We use ~1 km cells.

KM_PER_DEG_LAT = 111.32

df["event_grid_lat"] = np.floor(
    df["latitude"] * KM_PER_DEG_LAT
).astype(np.int32)

lon_km = (
    KM_PER_DEG_LAT *
    np.cos(np.radians(df["latitude"]))
)

lon_km = np.maximum(lon_km, 1e-6)

df["event_grid_lon"] = np.floor(
    df["longitude"] * lon_km
).astype(np.int32)

# ------------------------------------------------------------
# 8. EVENT SEARCH KEY
# ------------------------------------------------------------
df["event_search_key"] = (
    df["event_grid_lat"].astype(str)
    + "_"
    + df["event_grid_lon"].astype(str)
    + "_"
    + df["day_of_year"].astype(str)
)

# ------------------------------------------------------------
# 9. SORT CHRONOLOGICALLY
# ------------------------------------------------------------
df = df.sort_values(
    ["acq_datetime", "latitude", "longitude"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 10. CREATE UNIQUE DETECTION ID
# ------------------------------------------------------------
df["detection_id"] = np.arange(
    len(df),
    dtype=np.int64
)

# ------------------------------------------------------------
# 11. VALIDATION
# ------------------------------------------------------------
print("\n--- FINAL PREPARATION VALIDATION ---")

print(
    f"Rows after preparation: {len(df):,}"
)

print(
    f"Unique detection IDs: "
    f"{df['detection_id'].nunique():,}"
)

print(
    f"Unique 500m blocks: "
    f"{df['block_500m_id'].nunique():,}"
)

print(
    f"Unique 1km blocks: "
    f"{df['block_1km_id'].nunique():,}"
)

print(
    f"Unique event-search cells: "
    f"{df['event_search_key'].nunique():,}"
)

if len(df) != 655204:
    raise ValueError(
        f"Row count changed. Expected 655,204, "
        f"got {len(df):,}"
    )

if df["detection_id"].nunique() != len(df):
    raise ValueError(
        "Detection IDs are not unique."
    )

# ------------------------------------------------------------
# 12. MEMORY INFORMATION
# ------------------------------------------------------------
print("\nDataset memory usage:")

memory_gb = (
    df.memory_usage(deep=True).sum()
    / (1024 ** 3)
)

print(
    f"{memory_gb:.2f} GB"
)

print("\n" + "=" * 70)
print("STEP 3A COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:10:33.161270Z","iopub.execute_input":"2026-09-09T12:10:33.161600Z","iopub.status.idle":"2026-09-09T12:12:15.631732Z","shell.execute_reply.started":"2026-09-09T12:10:33.161573Z","shell.execute_reply":"2026-09-09T12:12:15.630535Z"}}
# ============================================================
# STEP 3B — GROUP FIRMS DETECTIONS INTO FIRE / ACTIVITY EVENTS
# ============================================================

import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

INPUT_FILE = "/kaggle/working/firms_2025_1km_block_behavior.csv"
OUTPUT_FILE = "/kaggle/working/firms_2025_events.csv"

print("=" * 70)
print("STEP 3B — FIRE / ACTIVITY EVENT GROUPING")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------
print("\nLoading dataset...")

df = pd.read_csv(INPUT_FILE)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

# ------------------------------------------------------------
# 2. PREPARE DATE/TIME
# ------------------------------------------------------------
df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

if df["acq_date"].isna().any():
    raise ValueError("Invalid acquisition dates found.")

# FIRMS acquisition time
df["acq_time_str"] = (
    df["acq_time"]
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(4)
)

df["acq_hour"] = (
    df["acq_time_str"].str[:2].astype(int)
)

df["acq_minute"] = (
    df["acq_time_str"].str[2:].astype(int)
)

df["acq_datetime"] = (
    df["acq_date"]
    + pd.to_timedelta(df["acq_hour"], unit="h")
    + pd.to_timedelta(df["acq_minute"], unit="m")
)

# ------------------------------------------------------------
# 3. CREATE DETECTION ID
# ------------------------------------------------------------
df["detection_id"] = np.arange(
    len(df),
    dtype=np.int64
)

# ------------------------------------------------------------
# 4. EVENT PARAMETERS
# ------------------------------------------------------------
MAX_DISTANCE_KM = 1.0
MAX_TIME_DAYS = 2

MAX_TIME_HOURS = MAX_TIME_DAYS * 24

print("\nEvent grouping criteria:")
print(f"  Maximum spatial distance: {MAX_DISTANCE_KM} km")
print(f"  Maximum temporal distance: {MAX_TIME_DAYS} days")

# ------------------------------------------------------------
# 5. CONVERT LAT/LON TO APPROXIMATE KM COORDINATES
# ------------------------------------------------------------
print("\nConverting coordinates to km...")

lat = df["latitude"].to_numpy(dtype=np.float64)
lon = df["longitude"].to_numpy(dtype=np.float64)

lat_rad = np.radians(lat)

KM_PER_DEG_LAT = 111.32

# Approximate local conversion
x_km = lon * KM_PER_DEG_LAT * np.cos(lat_rad)
y_km = lat * KM_PER_DEG_LAT

coords = np.column_stack([
    x_km,
    y_km
])

# ------------------------------------------------------------
# 6. SORT BY TIME
# ------------------------------------------------------------
print("Sorting detections by acquisition time...")

time_values = (
    df["acq_datetime"]
    .astype("int64")
    .to_numpy()
)

order = np.argsort(time_values)

coords_sorted = coords[order]
times_sorted = time_values[order]

n = len(df)

# ------------------------------------------------------------
# 7. SPATIAL TREE
# ------------------------------------------------------------
print("\nBuilding spatial index...")

tree = cKDTree(coords_sorted)

# Candidate pairs within 1 km
#
# This is much more efficient than calculating every possible
# pair among 655k detections.

print(
    "Finding spatially nearby candidate pairs..."
)

pairs = tree.query_pairs(
    r=MAX_DISTANCE_KM,
    output_type="ndarray"
)

print(
    f"Spatial candidate pairs: {len(pairs):,}"
)

# ------------------------------------------------------------
# 8. TEMPORAL FILTER
# ------------------------------------------------------------
print("\nApplying temporal constraint...")

if len(pairs) > 0:

    i = pairs[:, 0]
    j = pairs[:, 1]

    time_diff_hours = (
        np.abs(
            times_sorted[j] -
            times_sorted[i]
        )
        / (1e9 * 3600)
    )

    keep = (
        time_diff_hours <= MAX_TIME_HOURS
    )

    pairs = pairs[keep]

print(
    f"Space + time candidate pairs: "
    f"{len(pairs):,}"
)

# ------------------------------------------------------------
# 9. CREATE CONNECTED COMPONENT GRAPH
# ------------------------------------------------------------
print("\nBuilding event connectivity graph...")

if len(pairs) == 0:

    # Every detection becomes its own event
    event_labels_sorted = np.arange(
        n,
        dtype=np.int64
    )

else:

    rows = np.concatenate([
        pairs[:, 0],
        pairs[:, 1]
    ])

    cols = np.concatenate([
        pairs[:, 1],
        pairs[:, 0]
    ])

    data = np.ones(
        len(rows),
        dtype=np.uint8
    )

    graph = coo_matrix(
        (data, (rows, cols)),
        shape=(n, n)
    ).tocsr()

    # Connected components
    print(
        "Finding connected components..."
    )

    number_of_events, event_labels_sorted = (
        connected_components(
            graph,
            directed=False,
            return_labels=True
        )
    )

    print(
        f"Connected events found: "
        f"{number_of_events:,}"
    )

# ------------------------------------------------------------
# 10. RETURN EVENT LABELS TO ORIGINAL ORDER
# ------------------------------------------------------------
event_labels_original = np.empty(
    n,
    dtype=np.int64
)

event_labels_original[order] = (
    event_labels_sorted
)

df["event_id"] = event_labels_original

# ------------------------------------------------------------
# 11. MAKE EVENT IDs EASY TO READ
# ------------------------------------------------------------
df["event_id"] = (
    df["event_id"]
    .astype(np.int64)
)

# ------------------------------------------------------------
# 12. EVENT SIZE
# ------------------------------------------------------------
print("\nCalculating event sizes...")

event_sizes = (
    df.groupby("event_id")
      .size()
      .rename("event_detection_count")
)

df = df.merge(
    event_sizes,
    on="event_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 13. EVENT DISTRIBUTION
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("EVENT DISTRIBUTION")
print("=" * 70)

number_events = df["event_id"].nunique()

print(
    f"Total events: {number_events:,}"
)

print(
    f"Total detections: {len(df):,}"
)

print(
    f"Average detections/event: "
    f"{len(df) / number_events:.3f}"
)

event_size_values = (
    event_sizes
    .value_counts()
    .sort_index()
)

print("\nEvent size distribution:")

for threshold in [
    1, 2, 3, 5, 10, 20, 50, 100, 500
]:

    count = (
        event_sizes >= threshold
    ).sum()

    print(
        f"Events with >= {threshold:4} "
        f"detections: {count:,}"
    )

# ------------------------------------------------------------
# 14. SINGLE-DETECTION EVENTS
# ------------------------------------------------------------
single_events = (
    event_sizes == 1
).sum()

print(
    f"\nSingle-detection events: "
    f"{single_events:,}"
)

print(
    f"Percentage of events that are single detection: "
    f"{single_events / number_events * 100:.2f}%"
)

# ------------------------------------------------------------
# 15. LARGE EVENTS
# ------------------------------------------------------------
print("\nLargest events:")

largest_events = (
    event_sizes
    .sort_values(ascending=False)
    .head(20)
)

print(
    largest_events.to_string()
)

# ------------------------------------------------------------
# 16. VALIDATION
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print(
    f"Original detections: 655,204"
)

print(
    f"Final detections: {len(df):,}"
)

if len(df) != 655204:
    raise ValueError(
        "Detection count changed!"
    )

if df["detection_id"].nunique() != 655204:
    raise ValueError(
        "Detection IDs are not unique."
    )

if df["event_id"].isna().any():
    raise ValueError(
        "Some detections have no event ID."
    )

if df["event_id"].nunique() != number_events:
    raise ValueError(
        "Event count inconsistency."
    )

# Every event size must match its detections
check_sizes = (
    df.groupby("event_id")
      .size()
      .eq(
          df.groupby("event_id")
            ["event_detection_count"]
            .first()
      )
)

if not check_sizes.all():
    raise ValueError(
        "Event size validation failed."
    )

print("\nAll event validations passed.")

# ------------------------------------------------------------
# 17. SAVE
# ------------------------------------------------------------
print("\nSaving event dataset...")

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved successfully:\n"
    f"{OUTPUT_FILE}"
)

print("\n" + "=" * 70)
print("STEP 3B COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:12:15.633675Z","iopub.execute_input":"2026-09-09T12:12:15.634132Z","iopub.status.idle":"2026-09-09T12:12:39.246686Z","shell.execute_reply.started":"2026-09-09T12:12:15.634095Z","shell.execute_reply":"2026-09-09T12:12:39.244362Z"}}
# ============================================================
# STEP 3C — CREATE EVENT-LEVEL FEATURES
# ============================================================

import pandas as pd
import numpy as np

INPUT_FILE = "/kaggle/working/firms_2025_events.csv"
OUTPUT_FILE = "/kaggle/working/firms_2025_event_features.csv"

print("=" * 70)
print("STEP 3C — EVENT-LEVEL FEATURE EXTRACTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD EVENT DATA
# ------------------------------------------------------------
print("\nLoading event dataset...")

df = pd.read_csv(INPUT_FILE)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

# ------------------------------------------------------------
# 2. PREPARE DATE/TIME
# ------------------------------------------------------------
df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

df["acq_datetime"] = pd.to_datetime(
    df["acq_datetime"],
    errors="coerce"
)

if df["acq_date"].isna().any():
    raise ValueError("Invalid acquisition dates found.")

if df["acq_datetime"].isna().any():
    raise ValueError("Invalid acquisition datetimes found.")

df["month"] = df["acq_date"].dt.month.astype(np.int8)

# ------------------------------------------------------------
# 3. EVENT BASIC FEATURES
# ------------------------------------------------------------
print("\nCalculating basic event features...")

event_features = (
    df.groupby("event_id")
      .agg(
          event_detection_count=("detection_id", "size"),

          event_active_days=("acq_date", "nunique"),

          event_start_datetime=("acq_datetime", "min"),

          event_end_datetime=("acq_datetime", "max"),

          event_start_date=("acq_date", "min"),

          event_end_date=("acq_date", "max"),

          event_mean_latitude=("latitude", "mean"),

          event_mean_longitude=("longitude", "mean"),

          event_min_latitude=("latitude", "min"),

          event_max_latitude=("latitude", "max"),

          event_min_longitude=("longitude", "min"),

          event_max_longitude=("longitude", "max"),

          event_mean_frp=("frp", "mean"),

          event_median_frp=("frp", "median"),

          event_max_frp=("frp", "max"),

          event_std_frp=("frp", "std"),

          event_mean_brightness=("brightness", "mean"),

          event_max_brightness=("brightness", "max"),

          event_500m_block_count=("block_500m_id", "nunique"),

          event_1km_block_count=("block_1km_id", "nunique")
      )
)

# ------------------------------------------------------------
# 4. EVENT DURATION
# ------------------------------------------------------------
event_features["event_duration_days"] = (
    event_features["event_end_datetime"]
    - event_features["event_start_datetime"]
).dt.total_seconds() / 86400.0

# Calendar-day span
event_features["event_calendar_span_days"] = (
    event_features["event_end_date"]
    - event_features["event_start_date"]
).dt.days

# ------------------------------------------------------------
# 5. SPATIAL EXTENT
# ------------------------------------------------------------
print("Calculating spatial extent...")

event_features["event_latitude_span_deg"] = (
    event_features["event_max_latitude"]
    - event_features["event_min_latitude"]
)

event_features["event_longitude_span_deg"] = (
    event_features["event_max_longitude"]
    - event_features["event_min_longitude"]
)

# Approximate km extent
event_features["event_latitude_span_km"] = (
    event_features["event_latitude_span_deg"]
    * 111.32
)

mean_lat_rad = np.radians(
    event_features["event_mean_latitude"]
)

event_features["event_longitude_span_km"] = (
    event_features["event_longitude_span_deg"]
    * 111.32
    * np.cos(mean_lat_rad)
)

event_features["event_spatial_extent_km"] = np.sqrt(
    event_features["event_latitude_span_km"] ** 2
    +
    event_features["event_longitude_span_km"] ** 2
)

# ------------------------------------------------------------
# 6. EVENT DETECTION DENSITY
# ------------------------------------------------------------
event_features["event_detections_per_active_day"] = (
    event_features["event_detection_count"]
    /
    event_features["event_active_days"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 7. EVENT TEMPORAL GAPS
# ------------------------------------------------------------
print("Calculating temporal gaps...")

event_dates = (
    df[
        ["event_id", "acq_date"]
    ]
    .drop_duplicates()
    .sort_values(
        ["event_id", "acq_date"]
    )
)

event_dates["day_gap"] = (
    event_dates
    .groupby("event_id")["acq_date"]
    .diff()
    .dt.days
)

gap_features = (
    event_dates
    .groupby("event_id")["day_gap"]
    .agg(
        event_mean_active_day_gap="mean",
        event_median_active_day_gap="median",
        event_max_active_day_gap="max"
    )
)

event_features = event_features.join(
    gap_features
)

# ------------------------------------------------------------
# 8. MONTHLY EVENT BEHAVIOR
# ------------------------------------------------------------
print("Calculating monthly behavior...")

event_month_counts = (
    df.groupby(
        ["event_id", "month"]
    )
    .size()
    .unstack(fill_value=0)
)

event_month_counts = event_month_counts.reindex(
    columns=range(1, 13),
    fill_value=0
)

# Number of active months
event_features["event_active_months"] = (
    (event_month_counts > 0)
    .sum(axis=1)
    .astype(np.int8)
)

# Peak month
event_features["event_peak_month"] = (
    event_month_counts
    .idxmax(axis=1)
    .astype(np.int8)
)

event_features["event_peak_month_detections"] = (
    event_month_counts.max(axis=1)
)

# Monthly concentration
monthly_total = event_month_counts.sum(axis=1)

monthly_share = event_month_counts.div(
    monthly_total.replace(0, np.nan),
    axis=0
)

event_features["event_monthly_concentration"] = (
    (monthly_share ** 2).sum(axis=1)
)

# ------------------------------------------------------------
# 9. DAILY TEMPORAL CONCENTRATION
# ------------------------------------------------------------
print("Calculating daily concentration...")

event_day_counts = (
    df.groupby(
        ["event_id", "acq_date"]
    )
    .size()
)

event_features["event_max_detections_one_day"] = (
    event_day_counts
    .groupby(level=0)
    .max()
)

event_features["event_mean_detections_active_day"] = (
    event_day_counts
    .groupby(level=0)
    .mean()
)

# ------------------------------------------------------------
# 10. UNIQUE SATELLITE / INSTRUMENT INFORMATION
# ------------------------------------------------------------
if "satellite" in df.columns:

    event_features["event_satellite_count"] = (
        df.groupby("event_id")["satellite"]
          .nunique()
    )

if "instrument" in df.columns:

    event_features["event_instrument_count"] = (
        df.groupby("event_id")["instrument"]
          .nunique()
    )

# ------------------------------------------------------------
# 11. DAY / NIGHT INFORMATION
# ------------------------------------------------------------
# We do not assume day/night labels from acquisition time here.
# Instead, retain acquisition-hour statistics.

event_features["event_mean_acquisition_hour"] = (
    df.groupby("event_id")["acq_hour"]
      .mean()
)

event_features["event_min_acquisition_hour"] = (
    df.groupby("event_id")["acq_hour"]
      .min()
)

event_features["event_max_acquisition_hour"] = (
    df.groupby("event_id")["acq_hour"]
      .max()
)

# ------------------------------------------------------------
# 12. CLEAN NaN VALUES
# ------------------------------------------------------------
for col in [
    "event_std_frp",
    "event_mean_active_day_gap",
    "event_median_active_day_gap",
    "event_max_active_day_gap"
]:
    if col in event_features.columns:
        event_features[col] = (
            event_features[col]
            .fillna(0)
        )

# ------------------------------------------------------------
# 13. RESET INDEX
# ------------------------------------------------------------
event_features = event_features.reset_index()

# ------------------------------------------------------------
# 14. VALIDATION
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("EVENT FEATURE VALIDATION")
print("=" * 70)

print(
    f"Total events: "
    f"{len(event_features):,}"
)

print(
    f"Total feature columns: "
    f"{len(event_features.columns):,}"
)

print(
    f"Original detections: "
    f"{len(df):,}"
)

feature_detection_total = (
    event_features["event_detection_count"]
    .sum()
)

print(
    f"Detections represented by events: "
    f"{feature_detection_total:,}"
)

if feature_detection_total != len(df):
    raise ValueError(
        "Event detection counts do not sum to all FIRMS detections."
    )

if event_features["event_id"].nunique() != len(event_features):
    raise ValueError(
        "Duplicate event IDs found."
    )

if event_features["event_detection_count"].min() < 1:
    raise ValueError(
        "Event with zero detections found."
    )

# ------------------------------------------------------------
# 15. EVENT STATISTICS
# ------------------------------------------------------------
print("\n--- EVENT STATISTICS ---")

for col in [
    "event_detection_count",
    "event_active_days",
    "event_active_months",
    "event_duration_days",
    "event_spatial_extent_km",
    "event_500m_block_count",
    "event_1km_block_count",
    "event_mean_frp",
    "event_max_frp",
    "event_mean_brightness"
]:

    print(
        f"{col:40s}"
        f"mean={event_features[col].mean():.3f}, "
        f"median={event_features[col].median():.3f}, "
        f"max={event_features[col].max():.3f}"
    )

# ------------------------------------------------------------
# 16. EVENT SIZE DISTRIBUTION
# ------------------------------------------------------------
print("\n--- EVENT SIZE DISTRIBUTION ---")

for threshold in [
    1, 2, 3, 5, 10, 20, 50, 100, 500
]:

    count = (
        event_features["event_detection_count"]
        >= threshold
    ).sum()

    print(
        f"Events with >= {threshold:4} detections: "
        f"{count:,}"
    )

# ------------------------------------------------------------
# 17. CHECK FOR IMPOSSIBLE VALUES
# ------------------------------------------------------------
print("\n--- SANITY CHECKS ---")

if (
    event_features["event_duration_days"] < 0
).any():

    raise ValueError(
        "Negative event duration detected."
    )

if (
    event_features["event_spatial_extent_km"] < 0
).any():

    raise ValueError(
        "Negative spatial extent detected."
    )

if (
    event_features["event_active_days"] < 1
).any():

    raise ValueError(
        "Event with zero active days detected."
    )

print("Sanity checks passed.")

# ------------------------------------------------------------
# 18. SAVE
# ------------------------------------------------------------
print("\nSaving event feature table...")

event_features.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved successfully:\n"
    f"{OUTPUT_FILE}"
)

print("\n" + "=" * 70)
print("STEP 3C COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:12:39.250346Z","iopub.execute_input":"2026-09-09T12:12:39.250651Z","iopub.status.idle":"2026-09-09T12:12:40.661240Z","shell.execute_reply.started":"2026-09-09T12:12:39.250630Z","shell.execute_reply":"2026-09-09T12:12:40.659371Z"}}
# ============================================================
# STEP 3D — EVENT QUALITY ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

INPUT_FILE = "/kaggle/working/firms_2025_event_features.csv"

print("=" * 70)
print("STEP 3D — EVENT QUALITY ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD EVENT FEATURES
# ------------------------------------------------------------
print("\nLoading event feature table...")

events = pd.read_csv(INPUT_FILE)

print(f"Events: {len(events):,}")
print(f"Features: {len(events.columns):,}")

# ------------------------------------------------------------
# 2. EVENT SIZE CATEGORIES
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("1. EVENT SIZE")
print("=" * 70)

size_bins = [
    1,
    2,
    3,
    5,
    10,
    20,
    50,
    100,
    500
]

for threshold in size_bins:
    count = (
        events["event_detection_count"] >= threshold
    ).sum()

    pct = (
        count / len(events) * 100
    )

    print(
        f">= {threshold:4} detections: "
        f"{count:8,} ({pct:6.2f}%)"
    )

# Exact size distribution for first 10
print("\nExact event sizes 1–10:")

exact_sizes = (
    events["event_detection_count"]
    .value_counts()
    .sort_index()
)

print(
    exact_sizes[
        exact_sizes.index <= 10
    ].to_string()
)

# ------------------------------------------------------------
# 3. EVENT DURATION
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("2. EVENT DURATION")
print("=" * 70)

duration_bins = [
    ("0 days", events["event_duration_days"] == 0),
    ("< 1 day", events["event_duration_days"] < 1),
    ("1–2 days", (
        (events["event_duration_days"] >= 1) &
        (events["event_duration_days"] <= 2)
    )),
    ("3–7 days", (
        (events["event_duration_days"] > 2) &
        (events["event_duration_days"] <= 7)
    )),
    ("8–30 days", (
        (events["event_duration_days"] > 7) &
        (events["event_duration_days"] <= 30)
    )),
    (">30 days", events["event_duration_days"] > 30)
]

for label, condition in duration_bins:
    count = condition.sum()
    pct = count / len(events) * 100

    print(
        f"{label:12s}: "
        f"{count:8,} ({pct:6.2f}%)"
    )

# ------------------------------------------------------------
# 4. ACTIVE DAYS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("3. ACTIVE DAYS")
print("=" * 70)

for threshold in [1, 2, 3, 5, 10, 20, 50, 100]:

    count = (
        events["event_active_days"] >= threshold
    ).sum()

    pct = count / len(events) * 100

    print(
        f">= {threshold:3} active days: "
        f"{count:8,} ({pct:6.2f}%)"
    )

# ------------------------------------------------------------
# 5. ACTIVE MONTHS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("4. ACTIVE MONTHS")
print("=" * 70)

print(
    events["event_active_months"]
    .value_counts()
    .sort_index()
    .to_string()
)

# ------------------------------------------------------------
# 6. SPATIAL EXTENT
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("5. SPATIAL EXTENT")
print("=" * 70)

spatial_bins = [
    ("0 km", events["event_spatial_extent_km"] == 0),
    ("<0.5 km", events["event_spatial_extent_km"] < 0.5),
    ("0.5–1 km", (
        (events["event_spatial_extent_km"] >= 0.5) &
        (events["event_spatial_extent_km"] <= 1)
    )),
    ("1–2 km", (
        (events["event_spatial_extent_km"] > 1) &
        (events["event_spatial_extent_km"] <= 2)
    )),
    ("2–5 km", (
        (events["event_spatial_extent_km"] > 2) &
        (events["event_spatial_extent_km"] <= 5)
    )),
    ("5–10 km", (
        (events["event_spatial_extent_km"] > 5) &
        (events["event_spatial_extent_km"] <= 10)
    )),
    (">10 km", events["event_spatial_extent_km"] > 10)
]

for label, condition in spatial_bins:
    count = condition.sum()
    pct = count / len(events) * 100

    print(
        f"{label:12s}: "
        f"{count:8,} ({pct:6.2f}%)"
    )

# ------------------------------------------------------------
# 7. BLOCK SPREAD
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("6. SPATIAL BLOCK SPREAD")
print("=" * 70)

for threshold in [1, 2, 3, 5, 10, 20, 50, 100]:

    count_500 = (
        events["event_500m_block_count"] >= threshold
    ).sum()

    count_1km = (
        events["event_1km_block_count"] >= threshold
    ).sum()

    print(
        f">= {threshold:3} blocks | "
        f"500m: {count_500:8,} | "
        f"1km: {count_1km:8,}"
    )

# ------------------------------------------------------------
# 8. POTENTIALLY LARGE / CHAINED EVENTS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("7. POTENTIALLY LARGE / CHAINED EVENTS")
print("=" * 70)

# These are NOT automatically considered bad.
# They are candidates for inspection.

large_events = events[
    (
        (events["event_detection_count"] >= 50)
        |
        (events["event_spatial_extent_km"] > 10)
        |
        (events["event_active_days"] > 30)
    )
].copy()

print(
    f"Large/persistent candidate events: "
    f"{len(large_events):,}"
)

if len(large_events) > 0:

    print("\nLargest candidate events:")

    display_cols = [
        "event_id",
        "event_detection_count",
        "event_active_days",
        "event_active_months",
        "event_duration_days",
        "event_spatial_extent_km",
        "event_500m_block_count",
        "event_1km_block_count",
        "event_mean_frp",
        "event_max_frp"
    ]

    print(
        large_events
        .sort_values(
            "event_detection_count",
            ascending=False
        )
        [display_cols]
        .head(25)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 9. SINGLE-DETECTION EVENTS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("8. SINGLE-DETECTION EVENTS")
print("=" * 70)

single = events[
    events["event_detection_count"] == 1
]

print(
    f"Single-detection events: "
    f"{len(single):,}"
)

print(
    f"Percentage: "
    f"{len(single) / len(events) * 100:.2f}%"
)

print(
    f"Median FRP: "
    f"{single['event_mean_frp'].median():.3f}"
)

print(
    f"Median brightness: "
    f"{single['event_mean_brightness'].median():.3f}"
)

# ------------------------------------------------------------
# 10. MULTI-DETECTION EVENTS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("9. MULTI-DETECTION EVENTS")
print("=" * 70)

multi = events[
    events["event_detection_count"] >= 2
]

print(
    f"Multi-detection events: "
    f"{len(multi):,}"
)

print(
    f"Percentage: "
    f"{len(multi) / len(events) * 100:.2f}%"
)

print(
    f"Median detections/event: "
    f"{multi['event_detection_count'].median():.3f}"
)

print(
    f"Median active days: "
    f"{multi['event_active_days'].median():.3f}"
)

print(
    f"Median spatial extent: "
    f"{multi['event_spatial_extent_km'].median():.3f} km"
)

# ------------------------------------------------------------
# 11. VERY PERSISTENT EVENTS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("10. VERY PERSISTENT EVENTS")
print("=" * 70)

persistent = events[
    (
        (events["event_active_days"] >= 10)
        |
        (events["event_active_months"] >= 3)
    )
]

print(
    f"Events active >=10 days OR >=3 months: "
    f"{len(persistent):,}"
)

print(
    f"Percentage: "
    f"{len(persistent) / len(events) * 100:.3f}%"
)

# ------------------------------------------------------------
# 12. SUMMARY TABLE
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("11. OVERALL EVENT SUMMARY")
print("=" * 70)

summary = pd.DataFrame({
    "metric": [
        "Total events",
        "Total detections",
        "Average detections/event",
        "Median detections/event",
        "Single-detection events",
        "Multi-detection events",
        "Events >=10 detections",
        "Events >=50 detections",
        "Events >=100 detections",
        "Events >=500 detections",
        "Events active >=10 days",
        "Events active >=3 months",
        "Events spatial extent >10 km"
    ],
    "value": [
        len(events),
        events["event_detection_count"].sum(),
        events["event_detection_count"].mean(),
        events["event_detection_count"].median(),
        (events["event_detection_count"] == 1).sum(),
        (events["event_detection_count"] >= 2).sum(),
        (events["event_detection_count"] >= 10).sum(),
        (events["event_detection_count"] >= 50).sum(),
        (events["event_detection_count"] >= 100).sum(),
        (events["event_detection_count"] >= 500).sum(),
        (events["event_active_days"] >= 10).sum(),
        (events["event_active_months"] >= 3).sum(),
        (events["event_spatial_extent_km"] > 10).sum()
    ]
})

print(
    summary.to_string(index=False)
)

# ------------------------------------------------------------
# 13. SAVE QUALITY REPORT
# ------------------------------------------------------------
REPORT_FILE = "/kaggle/working/firms_2025_event_quality_report.csv"

summary.to_csv(
    REPORT_FILE,
    index=False
)

print(
    f"\nQuality summary saved to:\n"
    f"{REPORT_FILE}"
)

print("\n" + "=" * 70)
print("STEP 3D COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:12:40.664484Z","iopub.execute_input":"2026-09-09T12:12:40.664920Z","iopub.status.idle":"2026-09-09T12:13:08.070608Z","shell.execute_reply.started":"2026-09-09T12:12:40.664885Z","shell.execute_reply":"2026-09-09T12:13:08.068393Z"}}
# ============================================================
# STEP 3E — EVENT TEMPORAL + SPATIAL BEHAVIOR FEATURES
# ============================================================

import pandas as pd
import numpy as np

INPUT_FILE = "/kaggle/working/firms_2025_events.csv"
OUTPUT_FILE = "/kaggle/working/firms_2025_event_features_v2.csv"

print("=" * 70)
print("STEP 3E — EVENT TEMPORAL + SPATIAL BEHAVIOR FEATURES")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD DETECTION-LEVEL EVENT DATA
# ------------------------------------------------------------
print("\nLoading event detections...")

df = pd.read_csv(INPUT_FILE)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

required = [
    "event_id",
    "detection_id",
    "latitude",
    "longitude",
    "acq_date",
    "acq_datetime",
    "frp",
    "brightness",
    "block_500m_id",
    "block_1km_id"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# ------------------------------------------------------------
# 2. DATE/TIME
# ------------------------------------------------------------
df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

df["acq_datetime"] = pd.to_datetime(
    df["acq_datetime"],
    errors="coerce"
)

if df["acq_date"].isna().any():
    raise ValueError("Invalid dates found.")

if df["acq_datetime"].isna().any():
    raise ValueError("Invalid datetimes found.")

df["month"] = (
    df["acq_date"]
    .dt.month
    .astype(np.int8)
)

# ------------------------------------------------------------
# 3. SORT BY EVENT + TIME
# ------------------------------------------------------------
print("\nSorting detections within events...")

df = df.sort_values(
    ["event_id", "acq_datetime"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. TEMPORAL GAP BETWEEN CONSECUTIVE DETECTIONS
# ------------------------------------------------------------
print("Calculating temporal behavior...")

df["detection_time_gap_hours"] = (
    df.groupby("event_id")["acq_datetime"]
      .diff()
      .dt.total_seconds()
      / 3600.0
)

# ------------------------------------------------------------
# 5. EVENT TEMPORAL FEATURES
# ------------------------------------------------------------
event = (
    df.groupby("event_id")
      .agg(
          event_detection_count=("detection_id", "size"),

          event_active_days=("acq_date", "nunique"),

          event_start_datetime=("acq_datetime", "min"),

          event_end_datetime=("acq_datetime", "max"),

          event_mean_frp=("frp", "mean"),

          event_median_frp=("frp", "median"),

          event_max_frp=("frp", "max"),

          event_std_frp=("frp", "std"),

          event_mean_brightness=("brightness", "mean"),

          event_max_brightness=("brightness", "max"),

          event_mean_latitude=("latitude", "mean"),

          event_mean_longitude=("longitude", "mean"),

          event_min_latitude=("latitude", "min"),

          event_max_latitude=("latitude", "max"),

          event_min_longitude=("longitude", "min"),

          event_max_longitude=("longitude", "max"),

          event_500m_block_count=("block_500m_id", "nunique"),

          event_1km_block_count=("block_1km_id", "nunique")
      )
)

# ------------------------------------------------------------
# 6. EVENT DURATION
# ------------------------------------------------------------
event["event_duration_hours"] = (
    event["event_end_datetime"]
    - event["event_start_datetime"]
).dt.total_seconds() / 3600.0

event["event_duration_days"] = (
    event["event_duration_hours"] / 24.0
)

# ------------------------------------------------------------
# 7. GAP FEATURES
# ------------------------------------------------------------
gap_features = (
    df.groupby("event_id")["detection_time_gap_hours"]
      .agg(
          event_mean_detection_gap_hours="mean",
          event_median_detection_gap_hours="median",
          event_max_detection_gap_hours="max",
          event_min_detection_gap_hours="min"
      )
)

event = event.join(gap_features)

# First detection has no previous gap.
# For single-detection events, these become NaN.
for col in [
    "event_mean_detection_gap_hours",
    "event_median_detection_gap_hours",
    "event_max_detection_gap_hours",
    "event_min_detection_gap_hours"
]:
    event[col] = event[col].fillna(0)

# ------------------------------------------------------------
# 8. ACTIVE-DAY GAPS
# ------------------------------------------------------------
print("Calculating active-day gaps...")

active_dates = (
    df[
        ["event_id", "acq_date"]
    ]
    .drop_duplicates()
    .sort_values(
        ["event_id", "acq_date"]
    )
)

active_dates["active_day_gap"] = (
    active_dates
    .groupby("event_id")["acq_date"]
    .diff()
    .dt.days
)

day_gap_features = (
    active_dates
    .groupby("event_id")["active_day_gap"]
    .agg(
        event_mean_active_day_gap="mean",
        event_median_active_day_gap="median",
        event_max_active_day_gap="max"
    )
)

event = event.join(day_gap_features)

for col in [
    "event_mean_active_day_gap",
    "event_median_active_day_gap",
    "event_max_active_day_gap"
]:
    event[col] = event[col].fillna(0)

# ------------------------------------------------------------
# 9. DETECTION DENSITY
# ------------------------------------------------------------
event["event_detections_per_active_day"] = (
    event["event_detection_count"]
    /
    event["event_active_days"].replace(0, np.nan)
)

event["event_detections_per_hour"] = (
    event["event_detection_count"]
    /
    event["event_duration_hours"].replace(0, np.nan)
)

event["event_detections_per_hour"] = (
    event["event_detections_per_hour"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# ------------------------------------------------------------
# 10. SPATIAL EXTENT
# ------------------------------------------------------------
print("Calculating spatial behavior...")

event["event_latitude_span_deg"] = (
    event["event_max_latitude"]
    - event["event_min_latitude"]
)

event["event_longitude_span_deg"] = (
    event["event_max_longitude"]
    - event["event_min_longitude"]
)

event["event_latitude_span_km"] = (
    event["event_latitude_span_deg"]
    * 111.32
)

mean_lat_rad = np.radians(
    event["event_mean_latitude"]
)

event["event_longitude_span_km"] = (
    event["event_longitude_span_deg"]
    * 111.32
    * np.cos(mean_lat_rad)
)

event["event_spatial_extent_km"] = np.sqrt(
    event["event_latitude_span_km"] ** 2
    +
    event["event_longitude_span_km"] ** 2
)

# ------------------------------------------------------------
# 11. SPATIAL SPREAD PER DAY
# ------------------------------------------------------------
event["event_spatial_extent_per_active_day_km"] = (
    event["event_spatial_extent_km"]
    /
    event["event_active_days"].replace(0, np.nan)
)

event[
    "event_spatial_extent_per_active_day_km"
] = (
    event[
        "event_spatial_extent_per_active_day_km"
    ]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# ------------------------------------------------------------
# 12. SPATIAL BLOCK DENSITY
# ------------------------------------------------------------
event["event_detections_per_500m_block"] = (
    event["event_detection_count"]
    /
    event["event_500m_block_count"].replace(0, np.nan)
)

event["event_detections_per_1km_block"] = (
    event["event_detection_count"]
    /
    event["event_1km_block_count"].replace(0, np.nan)
)

# ------------------------------------------------------------
# 13. MONTHLY BEHAVIOR
# ------------------------------------------------------------
print("Calculating monthly behavior...")

monthly = (
    df.groupby(
        ["event_id", "month"]
    )
    .size()
    .unstack(fill_value=0)
)

monthly = monthly.reindex(
    columns=range(1, 13),
    fill_value=0
)

event["event_active_months"] = (
    (monthly > 0)
    .sum(axis=1)
    .astype(np.int8)
)

event["event_peak_month"] = (
    monthly
    .idxmax(axis=1)
    .astype(np.int8)
)

event["event_peak_month_detections"] = (
    monthly.max(axis=1)
)

monthly_total = monthly.sum(axis=1)

monthly_share = monthly.div(
    monthly_total.replace(0, np.nan),
    axis=0
)

event["event_monthly_concentration"] = (
    (monthly_share ** 2).sum(axis=1)
)

# ------------------------------------------------------------
# 14. DAILY DETECTION DISTRIBUTION
# ------------------------------------------------------------
print("Calculating daily detection distribution...")

daily_counts = (
    df.groupby(
        ["event_id", "acq_date"]
    )
    .size()
)

daily_features = (
    daily_counts
    .groupby(level=0)
    .agg(
        event_max_detections_one_day="max",
        event_mean_detections_active_day="mean",
        event_median_detections_active_day="median"
    )
)

event = event.join(daily_features)

# ------------------------------------------------------------
# 15. FRP VARIABILITY
# ------------------------------------------------------------
event["event_frp_cv"] = (
    event["event_std_frp"]
    /
    event["event_mean_frp"].replace(0, np.nan)
)

event["event_frp_cv"] = (
    event["event_frp_cv"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

event["event_frp_range"] = (
    event["event_max_frp"]
    -
    event["event_mean_frp"]
)

# ------------------------------------------------------------
# 16. BRIGHTNESS VARIABILITY
# ------------------------------------------------------------
brightness_stats = (
    df.groupby("event_id")["brightness"]
      .agg(
          event_median_brightness="median",
          event_std_brightness="std",
          event_min_brightness="min"
      )
)

event = event.join(brightness_stats)

event["event_std_brightness"] = (
    event["event_std_brightness"]
    .fillna(0)
)

# ------------------------------------------------------------
# 17. ACQUISITION HOUR BEHAVIOR
# ------------------------------------------------------------
if "acq_hour" in df.columns:

    hour_features = (
        df.groupby("event_id")["acq_hour"]
          .agg(
              event_mean_acquisition_hour="mean",
              event_min_acquisition_hour="min",
              event_max_acquisition_hour="max",
              event_acquisition_hour_count="nunique"
          )
    )

    event = event.join(hour_features)

# ------------------------------------------------------------
# 18. SATELLITE / INSTRUMENT DIVERSITY
# ------------------------------------------------------------
if "satellite" in df.columns:

    event["event_satellite_count"] = (
        df.groupby("event_id")["satellite"]
          .nunique()
    )

if "instrument" in df.columns:

    event["event_instrument_count"] = (
        df.groupby("event_id")["instrument"]
          .nunique()
    )

# ------------------------------------------------------------
# 19. CLEAN NUMERIC VALUES
# ------------------------------------------------------------
event = event.replace(
    [np.inf, -np.inf],
    np.nan
)

numeric_cols = event.select_dtypes(
    include=[np.number]
).columns

event[numeric_cols] = (
    event[numeric_cols]
    .fillna(0)
)

# ------------------------------------------------------------
# 20. RESET INDEX
# ------------------------------------------------------------
event = event.reset_index()

# ------------------------------------------------------------
# 21. VALIDATION
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("EVENT FEATURE VALIDATION")
print("=" * 70)

print(
    f"Total events: {len(event):,}"
)

print(
    f"Feature columns: {len(event.columns):,}"
)

print(
    f"Total detections represented: "
    f"{event['event_detection_count'].sum():,}"
)

if len(event) != 302070:
    raise ValueError(
        f"Expected 302,070 events, "
        f"got {len(event):,}"
    )

if event["event_detection_count"].sum() != 655204:
    raise ValueError(
        "Detection counts do not sum to 655,204."
    )

if event["event_id"].nunique() != len(event):
    raise ValueError(
        "Duplicate event IDs detected."
    )

if (
    event["event_duration_days"] < 0
).any():
    raise ValueError(
        "Negative event duration detected."
    )

if (
    event["event_spatial_extent_km"] < 0
).any():
    raise ValueError(
        "Negative spatial extent detected."
    )

print("All validations passed.")

# ------------------------------------------------------------
# 22. IMPORTANT EVENT CATEGORIES
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("EVENT BEHAVIOR CATEGORIES")
print("=" * 70)

single = (
    event["event_detection_count"] == 1
)

repeated = (
    event["event_detection_count"] >= 2
)

persistent = (
    (event["event_active_days"] >= 10)
    |
    (event["event_active_months"] >= 3)
)

spatially_large = (
    event["event_spatial_extent_km"] > 10
)

highly_repeated = (
    event["event_detection_count"] >= 50
)

print(
    f"Single detection:       {single.sum():,}"
)

print(
    f"Repeated detections:    {repeated.sum():,}"
)

print(
    f"Persistent events:      {persistent.sum():,}"
)

print(
    f">10 km spatial extent:  {spatially_large.sum():,}"
)

print(
    f">=50 detections:         {highly_repeated.sum():,}"
)

# ------------------------------------------------------------
# 23. SHOW REPRESENTATIVE EVENTS
# ------------------------------------------------------------
print("\n--- LARGEST EVENTS ---")

display_cols = [
    "event_id",
    "event_detection_count",
    "event_active_days",
    "event_active_months",
    "event_duration_days",
    "event_spatial_extent_km",
    "event_500m_block_count",
    "event_1km_block_count",
    "event_detections_per_active_day",
    "event_mean_frp",
    "event_max_frp",
    "event_frp_cv"
]

print(
    event
    .sort_values(
        "event_detection_count",
        ascending=False
    )[display_cols]
    .head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 24. SAVE
# ------------------------------------------------------------
print("\nSaving enhanced event feature table...")

event.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved successfully:\n"
    f"{OUTPUT_FILE}"
)

print("\n" + "=" * 70)
print("STEP 3E COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:13:08.075089Z","iopub.execute_input":"2026-09-09T12:13:08.075613Z","iopub.status.idle":"2026-09-09T12:13:18.203520Z","shell.execute_reply.started":"2026-09-09T12:13:08.075567Z","shell.execute_reply":"2026-09-09T12:13:18.199470Z"}}
# ============================================================
# STEP 4A — INSPECT WORLD BANK GAS FLARE DATASET
# ============================================================

import pandas as pd
import numpy as np
import os

FLARE_FILE = (
    "/kaggle/input/datasets/"
    "gautamkumar0036/flare-real-correct-data/"
    "Flare-Volume-Estimates-by-individual-Flare-Location-2012-2025.xlsx"
)

print("=" * 70)
print("STEP 4A — WORLD BANK GAS FLARE DATA INSPECTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. CHECK FILE
# ------------------------------------------------------------
print("\nChecking file...")

if not os.path.exists(FLARE_FILE):
    raise FileNotFoundError(
        f"Flare file not found:\n{FLARE_FILE}"
    )

print("File found:")
print(FLARE_FILE)

# ------------------------------------------------------------
# 2. LIST EXCEL SHEETS
# ------------------------------------------------------------
print("\nReading Excel workbook...")

xls = pd.ExcelFile(FLARE_FILE)

print("\nSheets:")
for i, sheet in enumerate(xls.sheet_names):
    print(f"{i}: {sheet}")

# ------------------------------------------------------------
# 3. INSPECT EVERY SHEET
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("SHEET INSPECTION")
print("=" * 70)

for sheet in xls.sheet_names:

    temp = pd.read_excel(
        FLARE_FILE,
        sheet_name=sheet,
        nrows=10
    )

    print("\n" + "-" * 70)
    print(f"SHEET: {sheet}")
    print("-" * 70)

    print(
        f"Columns: {len(temp.columns)}"
    )

    print("\nColumn names:")
    for col in temp.columns:
        print(f"  - {col}")

    print("\nFirst rows:")
    print(
        temp.head(3).to_string(index=False)
    )

# ------------------------------------------------------------
# 4. LOAD FIRST LIKELY DATA SHEET
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FULL DATASET INSPECTION")
print("=" * 70)

# Prefer a sheet containing flare/location information.
# Otherwise use the first sheet.

preferred_sheet = None

for sheet in xls.sheet_names:

    name = sheet.lower()

    if (
        "flare" in name
        or "location" in name
        or "volume" in name
    ):
        preferred_sheet = sheet
        break

if preferred_sheet is None:
    preferred_sheet = xls.sheet_names[0]

print(
    f"\nSelected sheet: {preferred_sheet}"
)

flare = pd.read_excel(
    FLARE_FILE,
    sheet_name=preferred_sheet
)

print(
    f"Rows: {len(flare):,}"
)

print(
    f"Columns: {len(flare.columns):,}"
)

# ------------------------------------------------------------
# 5. DATA TYPES
# ------------------------------------------------------------
print("\n--- DATA TYPES ---")

print(
    flare.dtypes.to_string()
)

# ------------------------------------------------------------
# 6. MISSING VALUES
# ------------------------------------------------------------
print("\n--- MISSING VALUES ---")

missing = (
    flare.isna()
    .sum()
    .sort_values(ascending=False)
)

print(
    missing[
        missing > 0
    ].to_string()
)

# ------------------------------------------------------------
# 7. SAMPLE DATA
# ------------------------------------------------------------
print("\n--- SAMPLE DATA ---")

print(
    flare.head(10).to_string(index=False)
)

# ------------------------------------------------------------
# 8. SEARCH FOR COORDINATE COLUMNS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("COORDINATE COLUMN DETECTION")
print("=" * 70)

for col in flare.columns:

    col_lower = str(col).lower()

    if any(
        term in col_lower
        for term in [
            "lat",
            "lon",
            "longitude",
            "latitude",
            "coord"
        ]
    ):

        print(
            f"Potential coordinate column: {col}"
        )

# ------------------------------------------------------------
# 9. SEARCH FOR YEAR COLUMNS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("YEAR / 2025 COLUMN DETECTION")
print("=" * 70)

for col in flare.columns:

    col_str = str(col)

    if (
        "2025" in col_str
        or "year" in col_str.lower()
        or "date" in col_str.lower()
    ):

        print(
            f"Potential temporal column: {col}"
        )

# ------------------------------------------------------------
# 10. CHECK INDIA-LIKE COORDINATES
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("POTENTIAL INDIA COORDINATE CHECK")
print("=" * 70)

numeric_cols = flare.select_dtypes(
    include=np.number
).columns

print(
    f"Numeric columns: {len(numeric_cols)}"
)

for col in numeric_cols:

    values = pd.to_numeric(
        flare[col],
        errors="coerce"
    )

    valid = values.dropna()

    if len(valid) == 0:
        continue

    minimum = valid.min()
    maximum = valid.max()

    # Latitude-like range
    if minimum >= -90 and maximum <= 90:
        print(
            f"{col}: range "
            f"{minimum:.4f} to {maximum:.4f}"
        )

# ------------------------------------------------------------
# 11. FINAL
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 4A INSPECTION COMPLETE")
print("=" * 70)

print(
    "\nIMPORTANT:"
    "\nWe have NOT yet assigned any FIRMS detection"
    "\nto gas."
    "\nThis step only identifies the structure of the"
    "\nindependent World Bank flare dataset."
)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:13:18.205332Z","iopub.execute_input":"2026-09-09T12:13:18.205640Z","iopub.status.idle":"2026-09-09T12:13:25.566002Z","shell.execute_reply.started":"2026-09-09T12:13:18.205616Z","shell.execute_reply":"2026-09-09T12:13:25.563660Z"}}
# ============================================================
# STEP 4B — EXTRACT INDIA 2025 ACTIVE FLARE LOCATIONS
# ============================================================

import pandas as pd
import numpy as np
import os

FLARE_FILE = (
    "/kaggle/input/datasets/"
    "gautamkumar0036/flare-real-correct-data/"
    "Flare-Volume-Estimates-by-individual-Flare-Location-2012-2025.xlsx"
)

OUTPUT_FILE = (
    "/kaggle/working/india_worldbank_flare_locations_2025.csv"
)

print("=" * 70)
print("STEP 4B — INDIA 2025 WORLD BANK FLARE LOCATIONS")
print("=" * 70)

# ------------------------------------------------------------
# 1. CHECK FILE
# ------------------------------------------------------------
if not os.path.exists(FLARE_FILE):
    raise FileNotFoundError(
        f"Flare file not found:\n{FLARE_FILE}"
    )

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------
flare = pd.read_excel(
    FLARE_FILE,
    sheet_name="2012-2025-Flare-Volume-Estimate"
)

print(
    f"\nGlobal flare records: {len(flare):,}"
)

# ------------------------------------------------------------
# 3. STANDARDIZE COLUMN NAMES
# ------------------------------------------------------------
# Excel may interpret year headers as integers.
# Convert every column name to a string.

flare.columns = (
    flare.columns
    .astype(str)
    .str.strip()
)

print("\nColumns detected:")

print(
    flare.columns.tolist()
)

# ------------------------------------------------------------
# 4. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------
required_columns = [
    "Flare id",
    "Country",
    "Latitude",
    "Longitude",
    "Location",
    "Field Type",
    "Field name",
    "Operator",
    "2025"
]

missing = [
    col
    for col in required_columns
    if col not in flare.columns
]

if missing:
    raise ValueError(
        f"Required columns missing: {missing}"
    )

# ------------------------------------------------------------
# 5. CLEAN IMPORTANT COLUMNS
# ------------------------------------------------------------
flare["Country"] = (
    flare["Country"]
    .astype(str)
    .str.strip()
)

flare["Latitude"] = pd.to_numeric(
    flare["Latitude"],
    errors="coerce"
)

flare["Longitude"] = pd.to_numeric(
    flare["Longitude"],
    errors="coerce"
)

flare["2025"] = pd.to_numeric(
    flare["2025"],
    errors="coerce"
)

# ------------------------------------------------------------
# 6. FILTER INDIA
# ------------------------------------------------------------
india = flare[
    flare["Country"]
    .str.upper()
    .eq("INDIA")
].copy()

print(
    f"\nIndia flare records: {len(india):,}"
)

if len(india) == 0:
    raise ValueError(
        "No India records found."
    )

# ------------------------------------------------------------
# 7. CHECK INDIA COORDINATES
# ------------------------------------------------------------
print("\n--- INDIA COORDINATE CHECK ---")

print(
    f"Latitude range: "
    f"{india['Latitude'].min():.6f} "
    f"to "
    f"{india['Latitude'].max():.6f}"
)

print(
    f"Longitude range: "
    f"{india['Longitude'].min():.6f} "
    f"to "
    f"{india['Longitude'].max():.6f}"
)

# ------------------------------------------------------------
# 8. FILTER APPROXIMATE INDIA BOUNDS
# ------------------------------------------------------------
india_valid = india[
    india["Latitude"].between(6, 38)
    &
    india["Longitude"].between(67, 98)
].copy()

print(
    f"India records inside approximate bounds: "
    f"{len(india_valid):,}"
)

# ------------------------------------------------------------
# 9. CHECK 2025 ACTIVITY
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("2025 FLARE ACTIVITY")
print("=" * 70)

print(
    f"2025 non-null: "
    f"{india_valid['2025'].notna().sum():,}"
)

print(
    f"2025 > 0: "
    f"{(india_valid['2025'] > 0).sum():,}"
)

print(
    f"2025 = 0: "
    f"{(india_valid['2025'] == 0).sum():,}"
)

print(
    f"2025 missing: "
    f"{india_valid['2025'].isna().sum():,}"
)

# ------------------------------------------------------------
# 10. EXTRACT ACTIVE 2025 FLARES
# ------------------------------------------------------------
active = india_valid[
    india_valid["2025"].fillna(0) > 0
].copy()

print(
    f"\nActive India flare locations in 2025: "
    f"{len(active):,}"
)

if len(active) == 0:
    raise ValueError(
        "No active India flare locations found for 2025."
    )

# ------------------------------------------------------------
# 11. 2025 ACTIVITY STATISTICS
# ------------------------------------------------------------
print("\n--- 2025 ACTIVITY STATISTICS ---")

print(
    f"Total 2025 value: "
    f"{active['2025'].sum():.6f}"
)

print(
    f"Mean 2025 value: "
    f"{active['2025'].mean():.6f}"
)

print(
    f"Median 2025 value: "
    f"{active['2025'].median():.6f}"
)

print(
    f"Maximum 2025 value: "
    f"{active['2025'].max():.6f}"
)

# ------------------------------------------------------------
# 12. FIELD TYPE
# ------------------------------------------------------------
print("\n--- FIELD TYPE ---")

print(
    active["Field Type"]
    .fillna("Unknown")
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 13. TOP ACTIVE FLARE LOCATIONS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("TOP INDIA FLARE LOCATIONS BY 2025 ACTIVITY")
print("=" * 70)

display_cols = [
    "Flare id",
    "Country",
    "Latitude",
    "Longitude",
    "Location",
    "Field Type",
    "Field name",
    "Operator",
    "2025"
]

print(
    active
    .sort_values(
        "2025",
        ascending=False
    )[display_cols]
    .head(30)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 14. DUPLICATE COORDINATE CHECK
# ------------------------------------------------------------
print("\n--- DUPLICATE LOCATION CHECK ---")

duplicate_coords = (
    active
    .duplicated(
        subset=["Latitude", "Longitude"],
        keep=False
    )
    .sum()
)

print(
    f"Rows sharing exact coordinates: "
    f"{duplicate_coords:,}"
)

print(
    f"Unique active coordinates: "
    f"{active[['Latitude', 'Longitude']].drop_duplicates().shape[0]:,}"
)

# ------------------------------------------------------------
# 15. SAVE
# ------------------------------------------------------------
active.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved successfully:\n"
    f"{OUTPUT_FILE}"
)

# ------------------------------------------------------------
# 16. FINAL VALIDATION
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

assert len(active) > 0

assert active["Latitude"].notna().all()
assert active["Longitude"].notna().all()

assert active["Latitude"].between(6, 38).all()
assert active["Longitude"].between(67, 98).all()

assert (active["2025"] > 0).all()

assert active["Flare id"].notna().all()

print(
    f"Validated active India flare locations: "
    f"{len(active):,}"
)

print(
    "All coordinate and 2025 activity checks passed."
)

print("\n" + "=" * 70)
print("STEP 4B COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:13:25.569447Z","iopub.execute_input":"2026-09-09T12:13:25.570399Z","iopub.status.idle":"2026-09-09T12:13:47.729743Z","shell.execute_reply.started":"2026-09-09T12:13:25.570356Z","shell.execute_reply":"2026-09-09T12:13:47.728647Z"}}
# ============================================================
# STEP 4C — MATCH FIRMS EVENTS TO WORLD BANK FLARE LOCATIONS
# ============================================================

import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

EVENT_FILE = (
    "/kaggle/working/firms_2025_event_features_v2.csv"
)

FLARE_FILE = (
    "/kaggle/working/india_worldbank_flare_locations_2025.csv"
)

OUTPUT_FILE = (
    "/kaggle/working/firms_2025_event_features_gas_evidence.csv"
)

print("=" * 70)
print("STEP 4C — FIRMS EVENT ↔ WORLD BANK FLARE MATCHING")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD EVENTS
# ------------------------------------------------------------
print("\nLoading event features...")

events = pd.read_csv(EVENT_FILE)

print(
    f"Events loaded: {len(events):,}"
)

# ------------------------------------------------------------
# 2. LOAD FLARE LOCATIONS
# ------------------------------------------------------------
print("\nLoading active India flare locations...")

flares = pd.read_csv(FLARE_FILE)

print(
    f"Active flare locations: {len(flares):,}"
)

# ------------------------------------------------------------
# 3. VALIDATE REQUIRED COLUMNS
# ------------------------------------------------------------
event_required = [
    "event_id",
    "event_mean_latitude",
    "event_mean_longitude"
]

flare_required = [
    "Flare id",
    "Latitude",
    "Longitude",
    "Field Type",
    "2025"
]

missing_event = [
    c for c in event_required
    if c not in events.columns
]

missing_flare = [
    c for c in flare_required
    if c not in flares.columns
]

if missing_event:
    raise ValueError(
        f"Missing event columns: {missing_event}"
    )

if missing_flare:
    raise ValueError(
        f"Missing flare columns: {missing_flare}"
    )

# ------------------------------------------------------------
# 4. CLEAN COORDINATES
# ------------------------------------------------------------
events["event_mean_latitude"] = pd.to_numeric(
    events["event_mean_latitude"],
    errors="coerce"
)

events["event_mean_longitude"] = pd.to_numeric(
    events["event_mean_longitude"],
    errors="coerce"
)

flares["Latitude"] = pd.to_numeric(
    flares["Latitude"],
    errors="coerce"
)

flares["Longitude"] = pd.to_numeric(
    flares["Longitude"],
    errors="coerce"
)

flares["2025"] = pd.to_numeric(
    flares["2025"],
    errors="coerce"
)

# ------------------------------------------------------------
# 5. VALIDATE COORDINATES
# ------------------------------------------------------------
if events[
    ["event_mean_latitude", "event_mean_longitude"]
].isna().any().any():

    raise ValueError(
        "Missing event coordinates detected."
    )

if flares[
    ["Latitude", "Longitude"]
].isna().any().any():

    raise ValueError(
        "Missing flare coordinates detected."
    )

# ------------------------------------------------------------
# 6. CONVERT LAT/LON TO APPROXIMATE KM
# ------------------------------------------------------------
print("\nConverting coordinates to km...")

KM_PER_DEG_LAT = 111.32

event_lat = (
    events["event_mean_latitude"]
    .to_numpy(dtype=np.float64)
)

event_lon = (
    events["event_mean_longitude"]
    .to_numpy(dtype=np.float64)
)

flare_lat = (
    flares["Latitude"]
    .to_numpy(dtype=np.float64)
)

flare_lon = (
    flares["Longitude"]
    .to_numpy(dtype=np.float64)
)

event_x = (
    event_lon
    * KM_PER_DEG_LAT
    * np.cos(np.radians(event_lat))
)

event_y = (
    event_lat
    * KM_PER_DEG_LAT
)

flare_x = (
    flare_lon
    * KM_PER_DEG_LAT
    * np.cos(np.radians(flare_lat))
)

flare_y = (
    flare_lat
    * KM_PER_DEG_LAT
)

event_coords = np.column_stack([
    event_x,
    event_y
])

flare_coords = np.column_stack([
    flare_x,
    flare_y
])

# ------------------------------------------------------------
# 7. BUILD FLARE SPATIAL INDEX
# ------------------------------------------------------------
print("\nBuilding flare spatial index...")

tree = cKDTree(flare_coords)

# ------------------------------------------------------------
# 8. NEAREST FLARE
# ------------------------------------------------------------
print("\nFinding nearest known flare for every event...")

nearest_distance_km, nearest_index = (
    tree.query(
        event_coords,
        k=1
    )
)

events["nearest_flare_distance_km"] = (
    nearest_distance_km
)

# ------------------------------------------------------------
# 9. NEAREST FLARE INFORMATION
# ------------------------------------------------------------
nearest_flares = (
    flares.iloc[nearest_index]
    .reset_index(drop=True)
)

events["nearest_flare_id"] = (
    nearest_flares["Flare id"].values
)

events["nearest_flare_field_type"] = (
    nearest_flares["Field Type"]
    .fillna("Unknown")
    .values
)

events["nearest_flare_2025_activity"] = (
    nearest_flares["2025"]
    .fillna(0)
    .values
)

events["nearest_flare_location"] = (
    nearest_flares["Location"]
    .fillna("Unknown")
    .values
)

events["nearest_flare_field_name"] = (
    nearest_flares["Field name"]
    .fillna("Unknown")
    .values
)

events["nearest_flare_operator"] = (
    nearest_flares["Operator"]
    .fillna("Unknown")
    .values
)

# ------------------------------------------------------------
# 10. DISTANCE-BASED EVIDENCE FLAGS
# ------------------------------------------------------------
print("\nCreating distance evidence features...")

for radius in [1, 2, 5, 10]:

    neighbors = tree.query_ball_point(
        event_coords,
        r=radius
    )

    counts = np.array(
        [len(x) for x in neighbors],
        dtype=np.int16
    )

    events[
        f"active_flare_count_within_{radius}km"
    ] = counts

    events[
        f"has_active_flare_within_{radius}km"
    ] = (
        counts > 0
    ).astype(np.int8)

# ------------------------------------------------------------
# 11. GAS-SPECIFIC EVIDENCE
# ------------------------------------------------------------
print("\nCreating gas-specific evidence...")

gas_mask = (
    flares["Field Type"]
    .fillna("")
    .astype(str)
    .str.upper()
    .eq("GAS")
)

gas_flares = flares[
    gas_mask
].copy()

print(
    f"Active India flares classified as GAS: "
    f"{len(gas_flares):,}"
)

if len(gas_flares) > 0:

    gas_lat = (
        gas_flares["Latitude"]
        .to_numpy(dtype=np.float64)
    )

    gas_lon = (
        gas_flares["Longitude"]
        .to_numpy(dtype=np.float64)
    )

    gas_x = (
        gas_lon
        * KM_PER_DEG_LAT
        * np.cos(np.radians(gas_lat))
    )

    gas_y = (
        gas_lat
        * KM_PER_DEG_LAT
    )

    gas_coords = np.column_stack([
        gas_x,
        gas_y
    ])

    gas_tree = cKDTree(
        gas_coords
    )

    gas_distance, gas_index = (
        gas_tree.query(
            event_coords,
            k=1
        )
    )

    events["nearest_gas_flare_distance_km"] = (
        gas_distance
    )

    nearest_gas = (
        gas_flares
        .iloc[gas_index]
        .reset_index(drop=True)
    )

    events["nearest_gas_flare_id"] = (
        nearest_gas["Flare id"].values
    )

    events["nearest_gas_flare_2025_activity"] = (
        nearest_gas["2025"]
        .fillna(0)
        .values
    )

    for radius in [1, 2, 5, 10]:

        gas_neighbors = (
            gas_tree.query_ball_point(
                event_coords,
                r=radius
            )
        )

        gas_counts = np.array(
            [len(x) for x in gas_neighbors],
            dtype=np.int16
        )

        events[
            f"gas_flare_count_within_{radius}km"
        ] = gas_counts

        events[
            f"has_gas_flare_within_{radius}km"
        ] = (
            gas_counts > 0
        ).astype(np.int8)

else:

    events["nearest_gas_flare_distance_km"] = np.inf

    events["nearest_gas_flare_id"] = "None"

    events["nearest_gas_flare_2025_activity"] = 0

    for radius in [1, 2, 5, 10]:

        events[
            f"gas_flare_count_within_{radius}km"
        ] = 0

        events[
            f"has_gas_flare_within_{radius}km"
        ] = 0

# ------------------------------------------------------------
# 12. OIL / UNKNOWN EVIDENCE
# ------------------------------------------------------------
events["nearest_flare_is_gas"] = (
    events["nearest_flare_field_type"]
    .astype(str)
    .str.upper()
    .eq("GAS")
    .astype(np.int8)
)

events["nearest_flare_is_oil"] = (
    events["nearest_flare_field_type"]
    .astype(str)
    .str.upper()
    .eq("OIL")
    .astype(np.int8)
)

events["nearest_flare_is_unknown"] = (
    events["nearest_flare_field_type"]
    .astype(str)
    .str.upper()
    .eq("UNKNOWN")
    .astype(np.int8)
)

# ------------------------------------------------------------
# 13. VALIDATION
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("STEP 4C VALIDATION")
print("=" * 70)

print(
    f"Events: {len(events):,}"
)

print(
    f"Nearest flare distances calculated: "
    f"{events['nearest_flare_distance_km'].notna().sum():,}"
)

print(
    f"Events with active flare within 1 km: "
    f"{events['has_active_flare_within_1km'].sum():,}"
)

print(
    f"Events with active flare within 2 km: "
    f"{events['has_active_flare_within_2km'].sum():,}"
)

print(
    f"Events with active flare within 5 km: "
    f"{events['has_active_flare_within_5km'].sum():,}"
)

print(
    f"Events with active flare within 10 km: "
    f"{events['has_active_flare_within_10km'].sum():,}"
)

print(
    f"Events with GAS flare within 1 km: "
    f"{events['has_gas_flare_within_1km'].sum():,}"
)

print(
    f"Events with GAS flare within 2 km: "
    f"{events['has_gas_flare_within_2km'].sum():,}"
)

print(
    f"Events with GAS flare within 5 km: "
    f"{events['has_gas_flare_within_5km'].sum():,}"
)

print(
    f"Events with GAS flare within 10 km: "
    f"{events['has_gas_flare_within_10km'].sum():,}"
)

# ------------------------------------------------------------
# 14. SANITY CHECKS
# ------------------------------------------------------------
if len(events) != 302070:
    raise ValueError(
        "Event count changed!"
    )

if events[
    "nearest_flare_distance_km"
].isna().any():

    raise ValueError(
        "Some events have no nearest flare."
    )

if (
    events["nearest_flare_distance_km"] < 0
).any():

    raise ValueError(
        "Negative flare distance detected."
    )

# ------------------------------------------------------------
# 15. DISTANCE SUMMARY
# ------------------------------------------------------------
print("\n--- NEAREST ACTIVE FLARE DISTANCE ---")

print(
    events[
        "nearest_flare_distance_km"
    ].describe().to_string()
)

# ------------------------------------------------------------
# 16. NEAREST FLARE FIELD TYPE
# ------------------------------------------------------------
print("\n--- NEAREST FLARE FIELD TYPE ---")

print(
    events[
        "nearest_flare_field_type"
    ]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 17. CLOSEST EVENTS TO KNOWN GAS FLARES
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("CLOSEST EVENTS TO KNOWN GAS FLARES")
print("=" * 70)

gas_display_cols = [
    "event_id",
    "event_detection_count",
    "event_active_days",
    "event_duration_days",
    "event_spatial_extent_km",
    "event_mean_frp",
    "event_max_frp",
    "nearest_gas_flare_distance_km",
    "nearest_gas_flare_id",
    "nearest_gas_flare_2025_activity"
]

print(
    events
    .sort_values(
        "nearest_gas_flare_distance_km"
    )[gas_display_cols]
    .head(30)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 18. SAVE
# ------------------------------------------------------------
print("\nSaving gas evidence dataset...")

events.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved successfully:\n"
    f"{OUTPUT_FILE}"
)

print("\n" + "=" * 70)
print("STEP 4C COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:13:47.731294Z","iopub.execute_input":"2026-09-09T12:13:47.731970Z","iopub.status.idle":"2026-09-09T12:13:47.751563Z","shell.execute_reply.started":"2026-09-09T12:13:47.731937Z","shell.execute_reply":"2026-09-09T12:13:47.750317Z"}}
# ============================================================
# STEP 4D-1 — FIND WRI GLOBAL POWER PLANT DATABASE
# ============================================================

import os

print("=" * 70)
print("SEARCHING FOR WRI POWER PLANT DATA")
print("=" * 70)

matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(
            (".csv", ".xlsx", ".xls", ".parquet")
        ):
            path = os.path.join(root, file)

            name = file.lower()

            if (
                "power" in name
                or "plant" in name
                or "global" in name
                or "wri" in name
            ):
                matches.append(path)

for path in sorted(matches):
    print(path)

print("\nTotal candidates:", len(matches))

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:13:47.753947Z","iopub.execute_input":"2026-09-09T12:13:47.754397Z","iopub.status.idle":"2026-09-09T12:13:51.404545Z","shell.execute_reply.started":"2026-09-09T12:13:47.754360Z","shell.execute_reply":"2026-09-09T12:13:51.403003Z"}}
# ============================================================
# STEP 4D — INSPECT EXISTING INDUSTRIAL EVIDENCE
# ============================================================

import os
import pandas as pd

print("=" * 70)
print("STEP 4D — EXISTING INDUSTRIAL EVIDENCE INSPECTION")
print("=" * 70)

# Find the corrected industrial evidence file
matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "firms_industrial_evidence_indicators_2025_corrected.csv":
            matches.append(os.path.join(root, file))

print("\nMatching file(s):")

for path in matches:
    print(path)

if len(matches) == 0:
    raise FileNotFoundError(
        "firms_industrial_evidence_indicators_2025_corrected.csv "
        "was not found in /kaggle/input"
    )

INDUSTRIAL_FILE = matches[0]

print("\nLoading:")
print(INDUSTRIAL_FILE)

industrial = pd.read_csv(INDUSTRIAL_FILE)

print("\n" + "=" * 70)
print("BASIC INFORMATION")
print("=" * 70)

print("Rows:", f"{len(industrial):,}")
print("Columns:", len(industrial.columns))

print("\nColumn names:")
for i, col in enumerate(industrial.columns, 1):
    print(f"{i:3}. {col}")

print("\nFirst 5 rows:")
display(industrial.head())

print("\nData types:")
display(
    industrial.dtypes
    .to_frame("dtype")
)

print("\nMissing values:")
missing = (
    industrial.isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing[missing > 0]
    .head(30)
    .to_frame("missing_count")
)

print("\n" + "=" * 70)
print("STEP 4D INSPECTION COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:13:51.406855Z","iopub.execute_input":"2026-09-09T12:13:51.407297Z","iopub.status.idle":"2026-09-09T12:14:31.503887Z","shell.execute_reply.started":"2026-09-09T12:13:51.407271Z","shell.execute_reply":"2026-09-09T12:14:31.503050Z"}}
# ============================================================
# STEP 4D-2
# AGGREGATE INDUSTRIAL DETECTION EVIDENCE TO EVENT LEVEL
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 4D-2 — DETECTION INDUSTRIAL EVIDENCE → EVENT EVIDENCE")
print("=" * 70)

# ------------------------------------------------------------
# 1. FIND INDUSTRIAL EVIDENCE FILE
# ------------------------------------------------------------

industrial_matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "firms_industrial_evidence_indicators_2025_corrected.csv":
            industrial_matches.append(
                os.path.join(root, file)
            )

if len(industrial_matches) == 0:
    raise FileNotFoundError(
        "Industrial evidence file not found."
    )

INDUSTRIAL_FILE = industrial_matches[0]

# ------------------------------------------------------------
# 2. FIND EVENTS FILE
# ------------------------------------------------------------

EVENT_FILE = "/kaggle/working/firms_2025_events.csv"

if not os.path.exists(EVENT_FILE):
    raise FileNotFoundError(
        f"Events file not found: {EVENT_FILE}"
    )

# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

print("\nLoading events...")

events = pd.read_csv(EVENT_FILE)

print(
    f"Events loaded: {len(events):,}"
)

print("\nLoading industrial evidence...")

industrial = pd.read_csv(
    INDUSTRIAL_FILE
)

print(
    f"Industrial detections loaded: "
    f"{len(industrial):,}"
)

# ------------------------------------------------------------
# 4. BASIC VALIDATION
# ------------------------------------------------------------

if "_fire_index" not in industrial.columns:
    raise ValueError(
        "Industrial evidence does not contain _fire_index."
    )

if "_fire_index" not in events.columns:
    raise ValueError(
        "Events file does not contain _fire_index."
    )

if "event_id" not in events.columns:
    raise ValueError(
        "Events file does not contain event_id."
    )

# ------------------------------------------------------------
# 5. CHECK DETECTION COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DETECTION VALIDATION")
print("=" * 70)

print(
    "Industrial rows:",
    f"{len(industrial):,}"
)

print(
    "Event rows:",
    f"{len(events):,}"
)

print(
    "Unique industrial detection IDs:",
    industrial["_fire_index"].nunique()
)

print(
    "Unique event detection IDs:",
    events["_fire_index"].nunique()
)

# ------------------------------------------------------------
# 6. CHECK OVERLAP
# ------------------------------------------------------------

industrial_ids = set(
    industrial["_fire_index"]
)

event_ids = set(
    events["_fire_index"]
)

intersection = (
    industrial_ids & event_ids
)

print(
    "Matching detection IDs:",
    f"{len(intersection):,}"
)

if len(intersection) != len(event_ids):
    print(
        "WARNING: Not every event detection has "
        "industrial evidence."
    )
else:
    print(
        "All event detections have industrial evidence."
    )

# ------------------------------------------------------------
# 7. SELECT INDUSTRIAL FEATURES
# ------------------------------------------------------------

industrial_feature_cols = [
    # Distances
    "steel_distance_m",
    "cement_distance_m",
    "wri_power_distance_m",
    "fertilizer_distance_m",
    "refinery_petro_distance_m",

    # Spatial proximity
    "steel_within_500m",
    "steel_within_1km",
    "steel_within_2km",

    "cement_within_500m",
    "cement_within_1km",
    "cement_within_2km",

    "wri_power_within_500m",
    "wri_power_within_1km",
    "wri_power_within_2km",

    "fertilizer_within_500m",
    "fertilizer_within_1km",
    "fertilizer_within_2km",

    "refinery_petro_within_500m",
    "refinery_petro_within_1km",
    "refinery_petro_within_2km",

    # Temporal recurrence
    "same_cell_fire_count_3d",
    "same_cell_fire_count_7d",
    "same_cell_fire_count_30d",

    "nearby_fire_count_3d_1km",
    "nearby_fire_count_7d_1km",
    "nearby_fire_count_30d_1km",

    # Source-specific temporal evidence
    "steel_strong_7d",
    "steel_strong_30d",
    "steel_moderate_7d",
    "steel_moderate_30d",

    "cement_strong_7d",
    "cement_strong_30d",
    "cement_moderate_7d",
    "cement_moderate_30d",

    "wri_power_strong_7d",
    "wri_power_strong_30d",
    "wri_power_moderate_7d",
    "wri_power_moderate_30d",

    "fertilizer_strong_7d",
    "fertilizer_strong_30d",
    "fertilizer_moderate_7d",
    "fertilizer_moderate_30d",

    "refinery_petro_strong_7d",
    "refinery_petro_strong_30d",
    "refinery_petro_moderate_7d",
    "refinery_petro_moderate_30d",

    # Overall industrial evidence
    "industrial_source_count_500m",
    "industrial_source_count_1km",
    "industrial_multi_source_500m",
    "industrial_multi_source_1km",
    "industrial_evidence_score"
]

missing_features = [
    c for c in industrial_feature_cols
    if c not in industrial.columns
]

if missing_features:
    raise ValueError(
        f"Missing industrial features: {missing_features}"
    )

# ------------------------------------------------------------
# 8. KEEP ONLY REQUIRED COLUMNS
# ------------------------------------------------------------

industrial_small = industrial[
    ["_fire_index"] + industrial_feature_cols
].copy()

# ------------------------------------------------------------
# 9. MERGE EVENT ID INTO INDUSTRIAL EVIDENCE
# ------------------------------------------------------------

print("\nConnecting detections to event IDs...")

event_mapping = events[
    ["_fire_index", "event_id"]
].copy()

industrial_event = industrial_small.merge(
    event_mapping,
    on="_fire_index",
    how="inner",
    validate="one_to_one"
)

print(
    "Industrial detections connected to events:",
    f"{len(industrial_event):,}"
)

if len(industrial_event) != len(industrial):
    print(
        "WARNING: Some industrial detections "
        "could not be matched to an event."
    )

# ------------------------------------------------------------
# 10. AGGREGATION RULES
# ------------------------------------------------------------

print("\nAggregating evidence by event...")

# Distance:
# closest detection in an event = minimum distance
distance_cols = [
    "steel_distance_m",
    "cement_distance_m",
    "wri_power_distance_m",
    "fertilizer_distance_m",
    "refinery_petro_distance_m"
]

# Boolean evidence:
# if ANY detection satisfies condition, event has evidence
boolean_cols = [
    c for c in industrial_feature_cols
    if c not in distance_cols
    and (
        industrial[c].dtype == bool
        or c.endswith("_within_500m")
        or c.endswith("_within_1km")
        or c.endswith("_within_2km")
        or "_strong_" in c
        or "_moderate_" in c
        or c.startswith("industrial_multi_source")
    )
]

# Count / score columns:
# strongest observed value in the event
numeric_max_cols = [
    "same_cell_fire_count_3d",
    "same_cell_fire_count_7d",
    "same_cell_fire_count_30d",
    "nearby_fire_count_3d_1km",
    "nearby_fire_count_7d_1km",
    "nearby_fire_count_30d_1km",
    "industrial_source_count_500m",
    "industrial_source_count_1km",
    "industrial_evidence_score"
]

# Remove anything accidentally duplicated between groups
boolean_cols = [
    c for c in boolean_cols
    if c not in numeric_max_cols
]

# ------------------------------------------------------------
# 11. BUILD AGGREGATION DICTIONARY
# ------------------------------------------------------------

agg_dict = {}

for c in distance_cols:
    agg_dict[c] = "min"

for c in boolean_cols:
    agg_dict[c] = "max"

for c in numeric_max_cols:
    agg_dict[c] = "max"

# ------------------------------------------------------------
# 12. EVENT AGGREGATION
# ------------------------------------------------------------

event_industrial = (
    industrial_event
    .groupby("event_id", sort=False)
    .agg(agg_dict)
    .reset_index()
)

print(
    "Event-level industrial evidence:",
    f"{len(event_industrial):,} events"
)

# ------------------------------------------------------------
# 13. RENAME COLUMNS
# ------------------------------------------------------------

rename_map = {}

for c in distance_cols:
    rename_map[c] = (
        "event_min_" +
        c
    )

for c in boolean_cols:
    rename_map[c] = (
        "event_any_" +
        c
    )

for c in numeric_max_cols:
    rename_map[c] = (
        "event_max_" +
        c
    )

event_industrial = event_industrial.rename(
    columns=rename_map
)

# ------------------------------------------------------------
# 14. MERGE WITH TODAY'S EVENT FEATURES
# ------------------------------------------------------------

GAS_EVENT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_gas_evidence.csv"
)

print("\nLoading today's gas-evidence event dataset...")

event_features = pd.read_csv(
    GAS_EVENT_FILE
)

print(
    "Current event feature rows:",
    f"{len(event_features):,}"
)

if "event_id" not in event_features.columns:
    raise ValueError(
        "event_id missing from current event features."
    )

# ------------------------------------------------------------
# 15. ATTACH INDUSTRIAL EVIDENCE
# ------------------------------------------------------------

print("\nAttaching industrial evidence...")

final_events = event_features.merge(
    event_industrial,
    on="event_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 16. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4D-2 VALIDATION")
print("=" * 70)

print(
    "Original event rows:",
    f"{len(event_features):,}"
)

print(
    "Final event rows:",
    f"{len(final_events):,}"
)

print(
    "Industrial event rows:",
    f"{len(event_industrial):,}"
)

missing_industrial = (
    final_events[
        "event_min_steel_distance_m"
    ].isna().sum()
)

print(
    "Events without industrial evidence:",
    f"{missing_industrial:,}"
)

if len(final_events) != len(event_features):
    raise ValueError(
        "Event count changed during industrial merge!"
    )

# ------------------------------------------------------------
# 17. INDUSTRIAL COVERAGE
# ------------------------------------------------------------

print("\n--- INDUSTRIAL PROXIMITY COVERAGE ---")

coverage_cols = [
    "event_any_steel_within_500m",
    "event_any_steel_within_1km",

    "event_any_cement_within_500m",
    "event_any_cement_within_1km",

    "event_any_wri_power_within_500m",
    "event_any_wri_power_within_1km",

    "event_any_fertilizer_within_500m",
    "event_any_fertilizer_within_1km",

    "event_any_refinery_petro_within_500m",
    "event_any_refinery_petro_within_1km"
]

for c in coverage_cols:

    if c in final_events.columns:

        print(
            f"{c}: "
            f"{final_events[c].sum():,}"
        )

# ------------------------------------------------------------
# 18. INDUSTRIAL SCORE SUMMARY
# ------------------------------------------------------------

print("\n--- INDUSTRIAL EVIDENCE SCORE ---")

print(
    final_events[
        "event_max_industrial_evidence_score"
    ]
    .describe()
    .to_string()
)

# ------------------------------------------------------------
# 19. SAVE
# ------------------------------------------------------------

OUTPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_gas_industrial_evidence.csv"
)

print("\nSaving final event evidence dataset...")

final_events.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved successfully:\n{OUTPUT_FILE}"
)

print("\n" + "=" * 70)
print("STEP 4D-2 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:14:31.505012Z","iopub.execute_input":"2026-09-09T12:14:31.505385Z","iopub.status.idle":"2026-09-09T12:14:31.710420Z","shell.execute_reply.started":"2026-09-09T12:14:31.505359Z","shell.execute_reply":"2026-09-09T12:14:31.708699Z"}}
# ============================================================
# STEP 4E-1 — INSPECT EXISTING MINING EVIDENCE FILES
# ============================================================

import os
import pandas as pd

print("=" * 70)
print("STEP 4E-1 — MINING EVIDENCE INSPECTION")
print("=" * 70)

target_files = [
    "firms_coal_mine_distance_evidence_2025.csv",
    "firms_coal_mine_evidence_2025.csv",
    "firms_mining_evidence_analysis_2025.csv",
    "firms_mining_labeling_functions_2025.csv",
    "india_coal_mines_gcmt_2025.csv",
    "coal_mine_firms_activity_2025.csv",
    "mining_lf_coverage_2025.csv"
]

# ------------------------------------------------------------
# 1. FIND FILES
# ------------------------------------------------------------

found = {}

for root, dirs, files in os.walk("/kaggle/input"):

    for file in files:

        if file in target_files:

            found[file] = os.path.join(
                root,
                file
            )

print("\nFiles found:")

for name, path in sorted(found.items()):

    print(f"\n{name}")
    print(f"  {path}")

# ------------------------------------------------------------
# 2. INSPECT EACH FILE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FILE STRUCTURES")
print("=" * 70)

for name, path in sorted(found.items()):

    print("\n" + "-" * 70)
    print(name)
    print("-" * 70)

    # Only read a small sample first
    sample = pd.read_csv(
        path,
        nrows=5
    )

    # Get row count efficiently from file size is not enough,
    # so don't load the huge files completely here.
    print("Columns:", len(sample.columns))

    print("\nColumn names:")

    for i, col in enumerate(
        sample.columns,
        1
    ):
        print(f"{i:3}. {col}")

    print("\nFirst 5 rows:")

    display(sample)

print("\n" + "=" * 70)
print("STEP 4E-1 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:14:31.712483Z","iopub.execute_input":"2026-09-09T12:14:31.713336Z","iopub.status.idle":"2026-09-09T12:14:31.828121Z","shell.execute_reply.started":"2026-09-09T12:14:31.713281Z","shell.execute_reply":"2026-09-09T12:14:31.825938Z"}}
# ============================================================
# STEP 4E-2 FIX — CHECK EVENT FILE IDENTIFIERS
# ============================================================

import pandas as pd

EVENT_FILE = "/kaggle/working/firms_2025_events.csv"

events = pd.read_csv(EVENT_FILE, nrows=5)

print("=" * 70)
print("EVENT FILE IDENTIFIER CHECK")
print("=" * 70)

print("\nColumns:")
for i, col in enumerate(events.columns, 1):
    print(f"{i:3}. {col}")

print("\nFirst 5 rows:")
display(events)

print("\nPossible identifier columns:")

for col in events.columns:
    name = col.lower()

    if (
        "id" in name
        or "index" in name
        or "fire" in name
        or "detection" in name
    ):
        print(f"  {col}")

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:14:31.830091Z","iopub.execute_input":"2026-09-09T12:14:31.830592Z","iopub.status.idle":"2026-09-09T12:15:05.795157Z","shell.execute_reply.started":"2026-09-09T12:14:31.830556Z","shell.execute_reply":"2026-09-09T12:15:05.789862Z"}}
# ============================================================
# STEP 4E-2
# AGGREGATE MINING DETECTION EVIDENCE → EVENT LEVEL
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 4E-2 — MINING DETECTION EVIDENCE → EVENT EVIDENCE")
print("=" * 70)

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------

MINING_FILE = (
    "/kaggle/input/datasets/gautamkumar0036/"
    "google-colab-all-real-data/"
    "firms_coal_mine_distance_evidence_2025.csv"
)

EVENT_FILE = (
    "/kaggle/working/firms_2025_events.csv"
)

CURRENT_EVENT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_gas_industrial_evidence.csv"
)

OUTPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_gas_industrial_mining_evidence.csv"
)

# ------------------------------------------------------------
# 2. CHECK FILES
# ------------------------------------------------------------

for path in [
    MINING_FILE,
    EVENT_FILE,
    CURRENT_EVENT_FILE
]:

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File not found:\n{path}"
        )

# ------------------------------------------------------------
# 3. LOAD EVENT-DETECTION MAPPING
# ------------------------------------------------------------

print("\nLoading event-detection mapping...")

events_detection = pd.read_csv(
    EVENT_FILE,
    usecols=[
        "_fire_index",
        "event_id"
    ]
)

print(
    f"Detection rows: "
    f"{len(events_detection):,}"
)

print(
    f"Unique detections: "
    f"{events_detection['_fire_index'].nunique():,}"
)

print(
    f"Unique events: "
    f"{events_detection['event_id'].nunique():,}"
)

# ------------------------------------------------------------
# 4. LOAD MINING EVIDENCE
# ------------------------------------------------------------

print("\nLoading mining evidence...")

mining = pd.read_csv(
    MINING_FILE
)

print(
    f"Mining detection rows: "
    f"{len(mining):,}"
)

print(
    f"Unique mining detection IDs: "
    f"{mining['_fire_index'].nunique():,}"
)

# ------------------------------------------------------------
# 5. VALIDATE IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("IDENTIFIER VALIDATION")
print("=" * 70)

if events_detection[
    "_fire_index"
].duplicated().any():

    raise ValueError(
        "Duplicate _fire_index found in event mapping."
    )

if mining[
    "_fire_index"
].duplicated().any():

    raise ValueError(
        "Duplicate _fire_index found in mining evidence."
    )

event_detection_ids = set(
    events_detection["_fire_index"]
)

mining_detection_ids = set(
    mining["_fire_index"]
)

matching = (
    event_detection_ids
    &
    mining_detection_ids
)

print(
    f"Matching detection IDs: "
    f"{len(matching):,}"
)

if len(matching) != len(event_detection_ids):

    missing = (
        event_detection_ids
        - mining_detection_ids
    )

    raise ValueError(
        f"{len(missing):,} event detections "
        "are missing mining evidence."
    )

print(
    "All 655,204 detections have mining evidence."
)

# ------------------------------------------------------------
# 6. SELECT MINING FEATURES
# ------------------------------------------------------------

mining_cols = [
    "nearest_coal_mine_distance_km",
    "nearest_coal_mine_distance_m",
    "nearest_coal_mine_id",
    "nearest_coal_mine_name",
    "nearest_coal_mine_state",

    "coal_mine_within_250m",
    "coal_mine_within_500m",
    "coal_mine_within_750m",
    "coal_mine_within_1000m",
    "coal_mine_within_2000m",
    "coal_mine_within_1km",
    "coal_mine_within_2km",

    "mining_evidence_close_500m",
    "mining_evidence_close_1km",

    "mining_evidence_close_500m_recurrent_7d",
    "mining_evidence_close_500m_recurrent_30d",

    "mining_evidence_close_1km_recurrent_7d",
    "mining_evidence_close_1km_recurrent_30d",

    "mining_evidence_close_500m_nearby_7d",
    "mining_evidence_close_1km_nearby_30d"
]

missing = [
    c for c in mining_cols
    if c not in mining.columns
]

if missing:
    raise ValueError(
        f"Missing mining columns: {missing}"
    )

mining_small = mining[
    ["_fire_index"] + mining_cols
].copy()

# ------------------------------------------------------------
# 7. ATTACH EVENT ID
# ------------------------------------------------------------

print("\nAttaching event IDs to mining detections...")

mining_event = mining_small.merge(
    events_detection,
    on="_fire_index",
    how="inner",
    validate="one_to_one"
)

print(
    f"Mining detections connected to events: "
    f"{len(mining_event):,}"
)

if len(mining_event) != len(mining_small):

    raise ValueError(
        "Mining detection → event mapping lost rows."
    )

# ------------------------------------------------------------
# 8. CLEAN DISTANCES
# ------------------------------------------------------------

for col in [
    "nearest_coal_mine_distance_km",
    "nearest_coal_mine_distance_m"
]:

    mining_event[col] = pd.to_numeric(
        mining_event[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# 9. FIND NEAREST MINE PER EVENT
# ------------------------------------------------------------

print("\nFinding nearest coal mine for each event...")

nearest_idx = (
    mining_event
    .groupby("event_id")[
        "nearest_coal_mine_distance_km"
    ]
    .idxmin()
)

nearest_mine = (
    mining_event.loc[
        nearest_idx,
        [
            "event_id",
            "nearest_coal_mine_id",
            "nearest_coal_mine_name",
            "nearest_coal_mine_state"
        ]
    ]
    .copy()
)

# ------------------------------------------------------------
# 10. AGGREGATE DISTANCES
# ------------------------------------------------------------

distance_agg = {
    "nearest_coal_mine_distance_km": "min",
    "nearest_coal_mine_distance_m": "min"
}

# ------------------------------------------------------------
# 11. AGGREGATE BOOLEAN EVIDENCE
# ------------------------------------------------------------

boolean_cols = [
    "coal_mine_within_250m",
    "coal_mine_within_500m",
    "coal_mine_within_750m",
    "coal_mine_within_1000m",
    "coal_mine_within_2000m",
    "coal_mine_within_1km",
    "coal_mine_within_2km",

    "mining_evidence_close_500m",
    "mining_evidence_close_1km",

    "mining_evidence_close_500m_recurrent_7d",
    "mining_evidence_close_500m_recurrent_30d",

    "mining_evidence_close_1km_recurrent_7d",
    "mining_evidence_close_1km_recurrent_30d",

    "mining_evidence_close_500m_nearby_7d",
    "mining_evidence_close_1km_nearby_30d"
]

# ------------------------------------------------------------
# 12. BUILD AGGREGATION DICTIONARY
# ------------------------------------------------------------

agg_dict = {}

for col, rule in distance_agg.items():
    agg_dict[col] = rule

for col in boolean_cols:
    agg_dict[col] = "max"

# ------------------------------------------------------------
# 13. EVENT-LEVEL AGGREGATION
# ------------------------------------------------------------

print("\nAggregating mining evidence to event level...")

event_mining = (
    mining_event
    .groupby(
        "event_id",
        sort=False
    )
    .agg(agg_dict)
    .reset_index()
)

print(
    f"Event-level mining records: "
    f"{len(event_mining):,}"
)

# ------------------------------------------------------------
# 14. RENAME FEATURES
# ------------------------------------------------------------

rename_map = {}

for col in distance_agg:
    rename_map[col] = (
        "event_min_" + col
    )

for col in boolean_cols:
    rename_map[col] = (
        "event_any_" + col
    )

event_mining = event_mining.rename(
    columns=rename_map
)

# ------------------------------------------------------------
# 15. ADD NEAREST MINE IDENTITY
# ------------------------------------------------------------

event_mining = event_mining.merge(
    nearest_mine,
    on="event_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 16. LOAD CURRENT EVENT FEATURE DATASET
# ------------------------------------------------------------

print("\nLoading current combined event dataset...")

current_events = pd.read_csv(
    CURRENT_EVENT_FILE
)

print(
    f"Current events: "
    f"{len(current_events):,}"
)

if current_events[
    "event_id"
].duplicated().any():

    raise ValueError(
        "Duplicate event_id found in current event dataset."
    )

# ------------------------------------------------------------
# 17. MERGE MINING EVIDENCE
# ------------------------------------------------------------

print("\nMerging mining evidence...")

final_events = current_events.merge(
    event_mining,
    on="event_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 18. VALIDATE EVENT COUNT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4E-2 VALIDATION")
print("=" * 70)

print(
    "Input events:",
    f"{len(current_events):,}"
)

print(
    "Mining event records:",
    f"{len(event_mining):,}"
)

print(
    "Final events:",
    f"{len(final_events):,}"
)

if len(final_events) != len(current_events):

    raise ValueError(
        "Event count changed during mining merge!"
    )

# ------------------------------------------------------------
# 19. MINING COVERAGE
# ------------------------------------------------------------

print("\n--- COAL MINE PROXIMITY COVERAGE ---")

coverage_cols = [
    "event_any_coal_mine_within_250m",
    "event_any_coal_mine_within_500m",
    "event_any_coal_mine_within_750m",
    "event_any_coal_mine_within_1000m",
    "event_any_coal_mine_within_2000m"
]

for col in coverage_cols:

    print(
        f"{col}: "
        f"{int(final_events[col].sum()):,}"
    )

# ------------------------------------------------------------
# 20. MINING EVIDENCE COVERAGE
# ------------------------------------------------------------

print("\n--- MINING EVIDENCE COVERAGE ---")

evidence_cols = [
    "event_any_mining_evidence_close_500m",
    "event_any_mining_evidence_close_1km",
    "event_any_mining_evidence_close_500m_recurrent_7d",
    "event_any_mining_evidence_close_500m_recurrent_30d",
    "event_any_mining_evidence_close_1km_recurrent_7d",
    "event_any_mining_evidence_close_1km_recurrent_30d",
    "event_any_mining_evidence_close_500m_nearby_7d",
    "event_any_mining_evidence_close_1km_nearby_30d"
]

for col in evidence_cols:

    print(
        f"{col}: "
        f"{int(final_events[col].sum()):,}"
    )

# ------------------------------------------------------------
# 21. DISTANCE SUMMARY
# ------------------------------------------------------------

print("\n--- NEAREST COAL MINE DISTANCE ---")

print(
    final_events[
        "event_min_nearest_coal_mine_distance_km"
    ]
    .describe()
    .to_string()
)

# ------------------------------------------------------------
# 22. CLOSEST EVENTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("30 CLOSEST EVENTS TO KNOWN COAL MINES")
print("=" * 70)

display_cols = [
    "event_id",
    "event_detection_count",
    "event_active_days",
    "event_duration_days",
    "event_spatial_extent_km",
    "event_min_nearest_coal_mine_distance_km",
    "nearest_coal_mine_id",
    "nearest_coal_mine_name",
    "nearest_coal_mine_state",
    "event_any_coal_mine_within_250m",
    "event_any_coal_mine_within_500m",
    "event_any_mining_evidence_close_500m_recurrent_7d",
    "event_any_mining_evidence_close_500m_recurrent_30d"
]

display(
    final_events
    .sort_values(
        "event_min_nearest_coal_mine_distance_km"
    )
    [display_cols]
    .head(30)
)

# ------------------------------------------------------------
# 23. SAVE
# ------------------------------------------------------------

print("\nSaving combined event dataset...")

final_events.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved successfully:\n{OUTPUT_FILE}"
)

print("\n" + "=" * 70)
print("STEP 4E-2 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:15:05.799457Z","iopub.execute_input":"2026-09-09T12:15:05.800320Z","iopub.status.idle":"2026-09-09T12:15:06.015590Z","shell.execute_reply.started":"2026-09-09T12:15:05.800222Z","shell.execute_reply":"2026-09-09T12:15:06.014146Z"}}
# ============================================================
# STEP 4F-1 — INSPECT EXISTING AGRICULTURE EVIDENCE
# ============================================================

import os
import pandas as pd

print("=" * 70)
print("STEP 4F-1 — AGRICULTURE EVIDENCE INSPECTION")
print("=" * 70)

target_files = [
    "firms_agriculture_evidence_indicators_2025.csv",
    "firms_agriculture_labeling_functions_2025.csv",
    "agriculture_daily_seasonality_2025.csv",
    "agriculture_latitude_month_seasonality_2025.csv",
    "agriculture_monthly_seasonality_2025.csv"
]

found = {}

for root, dirs, files in os.walk("/kaggle/input"):

    for file in files:

        if file in target_files:

            found[file] = os.path.join(
                root,
                file
            )

print("\nFiles found:")

for name, path in sorted(found.items()):

    print(f"\n{name}")
    print(f"  {path}")

print("\n" + "=" * 70)
print("FILE STRUCTURES")
print("=" * 70)

for name, path in sorted(found.items()):

    print("\n" + "-" * 70)
    print(name)
    print("-" * 70)

    sample = pd.read_csv(
        path,
        nrows=5
    )

    print(
        "Columns:",
        len(sample.columns)
    )

    print("\nColumn names:")

    for i, col in enumerate(
        sample.columns,
        1
    ):
        print(
            f"{i:3}. {col}"
        )

    print("\nFirst 5 rows:")

    display(sample)

print("\n" + "=" * 70)
print("STEP 4F-1 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:15:06.017289Z","iopub.execute_input":"2026-09-09T12:15:06.017613Z","iopub.status.idle":"2026-09-09T12:16:04.170478Z","shell.execute_reply.started":"2026-09-09T12:15:06.017576Z","shell.execute_reply":"2026-09-09T12:16:04.169390Z"}}
# ============================================================
# STEP 4F-2
# AGRICULTURE DETECTION EVIDENCE → EVENT LEVEL
# WITH COLUMN VALIDATION / FIX
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 4F-2 — AGRICULTURE DETECTION EVIDENCE → EVENT EVIDENCE")
print("=" * 70)

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------

AGRI_FILE = (
    "/kaggle/input/datasets/gautamkumar0036/"
    "google-colab-all-real-data/"
    "firms_agriculture_evidence_indicators_2025.csv"
)

EVENT_FILE = (
    "/kaggle/working/firms_2025_events.csv"
)

CURRENT_EVENT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_gas_industrial_mining_evidence.csv"
)

OUTPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_gas_industrial_mining_agriculture_evidence.csv"
)

# ------------------------------------------------------------
# 2. CHECK FILES
# ------------------------------------------------------------

print("\nChecking input files...")

for path in [
    AGRI_FILE,
    EVENT_FILE,
    CURRENT_EVENT_FILE
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File not found:\n{path}"
        )

print("All required files found.")

# ------------------------------------------------------------
# 3. LOAD EVENT → DETECTION MAPPING
# ------------------------------------------------------------

print("\nLoading event-detection mapping...")

event_mapping = pd.read_csv(
    EVENT_FILE,
    usecols=[
        "_fire_index",
        "event_id"
    ]
)

print(
    f"Detection rows: "
    f"{len(event_mapping):,}"
)

print(
    f"Unique detections: "
    f"{event_mapping['_fire_index'].nunique():,}"
)

print(
    f"Unique events: "
    f"{event_mapping['event_id'].nunique():,}"
)

# ------------------------------------------------------------
# 4. LOAD AGRICULTURE EVIDENCE
# ------------------------------------------------------------

print("\nLoading agriculture evidence...")

agri = pd.read_csv(
    AGRI_FILE
)

print(
    f"Agriculture detection rows: "
    f"{len(agri):,}"
)

print(
    f"Unique agriculture detection IDs: "
    f"{agri['_fire_index'].nunique():,}"
)

# ------------------------------------------------------------
# 5. VALIDATE IDENTIFIERS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("IDENTIFIER VALIDATION")
print("=" * 70)

if event_mapping[
    "_fire_index"
].duplicated().any():

    raise ValueError(
        "Duplicate _fire_index found in event mapping."
    )

if agri[
    "_fire_index"
].duplicated().any():

    raise ValueError(
        "Duplicate _fire_index found in agriculture evidence."
    )

event_ids = set(
    event_mapping["_fire_index"]
)

agri_ids = set(
    agri["_fire_index"]
)

matching = (
    event_ids & agri_ids
)

print(
    f"Matching detection IDs: "
    f"{len(matching):,}"
)

if len(matching) != len(event_ids):

    missing = event_ids - agri_ids

    raise ValueError(
        f"{len(missing):,} event detections "
        "are missing agriculture evidence."
    )

print(
    f"All {len(event_ids):,} detections "
    "have agriculture evidence."
)

# ------------------------------------------------------------
# 6. REQUIRED AGRICULTURE FEATURES
# ------------------------------------------------------------

agri_cols = [
    "dw_crop_probability",
    "dw_top_class",
    "dw_top_probability",
    "dw_land_cover",
    "dw_prob_available",

    "agri_osm_distance_m",
    "agri_osm_within_500m",
    "agri_osm_within_1km",
    "agri_osm_within_2km",

    "same_cell_fire_count_3d",
    "same_cell_fire_count_7d",
    "same_cell_fire_count_30d",

    "nearby_fire_count_3d_1km",
    "nearby_fire_count_7d_1km",
    "nearby_fire_count_30d_1km",

    "agri_dw_crop_50",
    "agri_dw_crop_70",
    "agri_dw_crop_80",
    "agri_dw_crop_90",

    "agri_dw_top_class",
    "agri_dw_landcover_crop",

    "agri_recurrent_3d",
    "agri_recurrent_7d",
    "agri_recurrent_30d",

    "agri_nearby_activity_7d",
    "agri_nearby_activity_30d",

    "agriculture_evidence_score"
]

missing = [
    c for c in agri_cols
    if c not in agri.columns
]

if missing:
    raise ValueError(
        f"Missing agriculture columns: {missing}"
    )

print(
    f"\nAll {len(agri_cols)} required agriculture "
    "features found."
)

# ------------------------------------------------------------
# 7. SELECT REQUIRED COLUMNS
# ------------------------------------------------------------

agri_small = agri[
    ["_fire_index"] + agri_cols
].copy()

# ------------------------------------------------------------
# 8. CONNECT DETECTIONS TO EVENTS
# ------------------------------------------------------------

print("\nAttaching event IDs...")

agri_event = agri_small.merge(
    event_mapping,
    on="_fire_index",
    how="inner",
    validate="one_to_one"
)

print(
    f"Agriculture detections connected to events: "
    f"{len(agri_event):,}"
)

if len(agri_event) != len(agri_small):
    raise ValueError(
        "Agriculture detection → event mapping lost rows."
    )

# ------------------------------------------------------------
# 9. NUMERIC CLEANING
# ------------------------------------------------------------

numeric_cols = [
    "dw_crop_probability",
    "dw_top_probability",
    "agri_osm_distance_m",
    "same_cell_fire_count_3d",
    "same_cell_fire_count_7d",
    "same_cell_fire_count_30d",
    "nearby_fire_count_3d_1km",
    "nearby_fire_count_7d_1km",
    "nearby_fire_count_30d_1km",
    "agriculture_evidence_score"
]

for col in numeric_cols:

    agri_event[col] = pd.to_numeric(
        agri_event[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# 10. AGGREGATION DEFINITIONS
# ------------------------------------------------------------

# Closest agriculture OSM feature
distance_cols = [
    "agri_osm_distance_m"
]

# Boolean / evidence indicators
boolean_cols = [
    "agri_osm_within_500m",
    "agri_osm_within_1km",
    "agri_osm_within_2km",

    "agri_dw_crop_50",
    "agri_dw_crop_70",
    "agri_dw_crop_80",
    "agri_dw_crop_90",

    "agri_dw_top_class",
    "agri_dw_landcover_crop",

    "agri_recurrent_3d",
    "agri_recurrent_7d",
    "agri_recurrent_30d",

    "agri_nearby_activity_7d",
    "agri_nearby_activity_30d"
]

# Numeric features where maximum event evidence is retained
numeric_max_cols = [
    "same_cell_fire_count_3d",
    "same_cell_fire_count_7d",
    "same_cell_fire_count_30d",

    "nearby_fire_count_3d_1km",
    "nearby_fire_count_7d_1km",
    "nearby_fire_count_30d_1km",

    "agriculture_evidence_score"
]

# Dynamic World probabilities
dw_probability_cols = [
    "dw_crop_probability",
    "dw_top_probability"
]

# ------------------------------------------------------------
# 11. BUILD AGGREGATION DICTIONARY
# ------------------------------------------------------------

agg_dict = {}

for col in distance_cols:
    agg_dict[col] = "min"

for col in boolean_cols:
    agg_dict[col] = "max"

for col in numeric_max_cols:
    agg_dict[col] = "max"

for col in dw_probability_cols:
    agg_dict[col] = ["max", "mean"]

# ------------------------------------------------------------
# 12. AGGREGATE TO EVENT LEVEL
# ------------------------------------------------------------

print("\nAggregating agriculture evidence by event...")

event_agri = (
    agri_event
    .groupby(
        "event_id",
        sort=False
    )
    .agg(agg_dict)
)

# Flatten multi-level column names
event_agri.columns = [
    "_".join(
        [str(x) for x in col]
    ).strip("_")
    if isinstance(col, tuple)
    else str(col)
    for col in event_agri.columns
]

event_agri = (
    event_agri
    .reset_index()
)

print(
    f"Event-level agriculture records: "
    f"{len(event_agri):,}"
)

# ------------------------------------------------------------
# 13. RENAME EVENT-LEVEL FEATURES
# ------------------------------------------------------------

rename_map = {}

for col in distance_cols:
    rename_map[col] = (
        "event_min_" + col
    )

for col in boolean_cols:
    rename_map[col] = (
        "event_any_" + col
    )

for col in numeric_max_cols:
    rename_map[col] = (
        "event_max_" + col
    )

rename_map[
    "dw_crop_probability_max"
] = "event_max_dw_crop_probability"

rename_map[
    "dw_crop_probability_mean"
] = "event_mean_dw_crop_probability"

rename_map[
    "dw_top_probability_max"
] = "event_max_dw_top_probability"

rename_map[
    "dw_top_probability_mean"
] = "event_mean_dw_top_probability"

event_agri = event_agri.rename(
    columns=rename_map
)

# ------------------------------------------------------------
# 14. DOMINANT DYNAMIC WORLD LAND COVER
# ------------------------------------------------------------

print("\nCalculating dominant Dynamic World land cover...")

landcover_counts = (
    agri_event[
        [
            "event_id",
            "dw_land_cover"
        ]
    ]
    .dropna(
        subset=["dw_land_cover"]
    )
    .groupby(
        ["event_id", "dw_land_cover"]
    )
    .size()
    .reset_index(
        name="count"
    )
)

if len(landcover_counts) > 0:

    dominant_idx = (
        landcover_counts
        .groupby("event_id")[
            "count"
        ]
        .idxmax()
    )

    dominant_landcover = (
        landcover_counts
        .loc[
            dominant_idx,
            [
                "event_id",
                "dw_land_cover"
            ]
        ]
        .rename(
            columns={
                "dw_land_cover":
                "event_dominant_dw_land_cover"
            }
        )
    )

    event_agri = event_agri.merge(
        dominant_landcover,
        on="event_id",
        how="left",
        validate="one_to_one"
    )

# ------------------------------------------------------------
# 15. LOAD CURRENT EVENT DATASET
# ------------------------------------------------------------

print("\nLoading current combined event dataset...")

current_events = pd.read_csv(
    CURRENT_EVENT_FILE
)

print(
    f"Current events: "
    f"{len(current_events):,}"
)

if current_events[
    "event_id"
].duplicated().any():

    raise ValueError(
        "Duplicate event_id found."
    )

# ------------------------------------------------------------
# 16. MERGE AGRICULTURE EVIDENCE
# ------------------------------------------------------------

print("\nMerging agriculture evidence...")

final_events = current_events.merge(
    event_agri,
    on="event_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 17. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4F-2 VALIDATION")
print("=" * 70)

print(
    "Input events:",
    f"{len(current_events):,}"
)

print(
    "Agriculture event records:",
    f"{len(event_agri):,}"
)

print(
    "Final events:",
    f"{len(final_events):,}"
)

if len(final_events) != len(current_events):

    raise ValueError(
        "Event count changed during agriculture merge!"
    )

if final_events[
    "event_id"
].nunique() != len(final_events):

    raise ValueError(
        "Duplicate event_id created during merge!"
    )

# ------------------------------------------------------------
# 18. CHECK ACTUAL AGRICULTURE COLUMN NAMES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AGRICULTURE COLUMN VALIDATION")
print("=" * 70)

agri_event_cols = [
    c for c in final_events.columns
    if (
        "agri_" in c.lower()
        or "agriculture" in c.lower()
        or "crop_" in c.lower()
        or "dw_crop" in c.lower()
    )
]

print(
    f"\nAgriculture-related columns found: "
    f"{len(agri_event_cols)}"
)

for i, col in enumerate(
    agri_event_cols,
    1
):
    print(
        f"{i:3}. {col}"
    )

# ------------------------------------------------------------
# 19. AUTOMATIC FEATURE FINDER
# ------------------------------------------------------------

def find_column(feature):

    matches = [
        c for c in final_events.columns
        if feature.lower() in c.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:

        # Prefer exact event_any name
        preferred = [
            c for c in matches
            if c.lower() == (
                "event_any_" + feature
            ).lower()
        ]

        if preferred:
            return preferred[0]

        return matches[0]

    return None

# ------------------------------------------------------------
# 20. AGRICULTURE OSM COVERAGE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("AGRICULTURE OSM COVERAGE")
print("-" * 70)

osm_features = [
    "agri_osm_within_500m",
    "agri_osm_within_1km",
    "agri_osm_within_2km"
]

for feature in osm_features:

    col = find_column(feature)

    if col is not None:

        value = (
            pd.to_numeric(
                final_events[col],
                errors="coerce"
            )
            .fillna(0)
            .astype(bool)
            .sum()
        )

        print(
            f"{col}: "
            f"{int(value):,}"
        )

    else:

        print(
            f"WARNING: {feature} not found"
        )

# ------------------------------------------------------------
# 21. DYNAMIC WORLD CROP COVERAGE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("DYNAMIC WORLD CROP EVIDENCE")
print("-" * 70)

crop_features = [
    "agri_dw_crop_50",
    "agri_dw_crop_70",
    "agri_dw_crop_80",
    "agri_dw_crop_90"
]

for feature in crop_features:

    col = find_column(feature)

    if col is not None:

        value = (
            pd.to_numeric(
                final_events[col],
                errors="coerce"
            )
            .fillna(0)
            .astype(bool)
            .sum()
        )

        print(
            f"{col}: "
            f"{int(value):,}"
        )

    else:

        print(
            f"WARNING: {feature} not found"
        )

# ------------------------------------------------------------
# 22. AGRICULTURE RECURRENCE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("AGRICULTURE RECURRENCE")
print("-" * 70)

recurrence_features = [
    "agri_recurrent_3d",
    "agri_recurrent_7d",
    "agri_recurrent_30d",
    "agri_nearby_activity_7d",
    "agri_nearby_activity_30d"
]

for feature in recurrence_features:

    col = find_column(feature)

    if col is not None:

        value = (
            pd.to_numeric(
                final_events[col],
                errors="coerce"
            )
            .fillna(0)
            .astype(bool)
            .sum()
        )

        print(
            f"{col}: "
            f"{int(value):,}"
        )

    else:

        print(
            f"WARNING: {feature} not found"
        )

# ------------------------------------------------------------
# 23. AGRICULTURE EVIDENCE SCORE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("AGRICULTURE EVIDENCE SCORE")
print("-" * 70)

score_col = find_column(
    "agriculture_evidence_score"
)

if score_col is not None:

    print(
        final_events[
            score_col
        ]
        .describe()
        .to_string()
    )

else:

    print(
        "WARNING: Agriculture evidence score "
        "column not found."
    )

# ------------------------------------------------------------
# 24. CROP PROBABILITY SUMMARY
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("EVENT CROP PROBABILITY")
print("-" * 70)

probability_cols = [
    c for c in [
        "event_max_dw_crop_probability",
        "event_mean_dw_crop_probability"
    ]
    if c in final_events.columns
]

if probability_cols:

    print(
        final_events[
            probability_cols
        ]
        .describe()
        .to_string()
    )

else:

    print(
        "WARNING: Crop probability columns "
        "not found."
    )

# ------------------------------------------------------------
# 25. DOMINANT LAND COVER
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("DOMINANT EVENT LAND COVER")
print("-" * 70)

landcover_col = (
    "event_dominant_dw_land_cover"
)

if landcover_col in final_events.columns:

    print(
        final_events[
            landcover_col
        ]
        .value_counts(
            dropna=False
        )
        .head(20)
        .to_string()
    )

else:

    print(
        "WARNING: Dominant land cover "
        "column not found."
    )

# ------------------------------------------------------------
# 26. FINAL AGRICULTURE FEATURE COUNT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL DATASET SUMMARY")
print("=" * 70)

print(
    f"Final event rows: "
    f"{len(final_events):,}"
)

print(
    f"Final columns: "
    f"{len(final_events.columns):,}"
)

print(
    f"Agriculture-related columns: "
    f"{len(agri_event_cols):,}"
)

# ------------------------------------------------------------
# 27. SAVE
# ------------------------------------------------------------

print("\nSaving combined event dataset...")

final_events.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved successfully:\n"
    f"{OUTPUT_FILE}"
)

print("\n" + "=" * 70)
print("STEP 4F-2 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:16:04.172517Z","iopub.execute_input":"2026-09-09T12:16:04.172961Z","iopub.status.idle":"2026-09-09T12:16:04.191540Z","shell.execute_reply.started":"2026-09-09T12:16:04.172927Z","shell.execute_reply":"2026-09-09T12:16:04.190547Z"}}
# ============================================================
# STEP 4G-1 — INSPECT EXISTING WILDFIRE EVIDENCE
# ============================================================

import os
import pandas as pd

print("=" * 70)
print("STEP 4G-1 — WILDFIRE EVIDENCE INSPECTION")
print("=" * 70)

target_files = [
    "firms_wildfire_evidence_indicators_2025.csv",
    "firms_wildfire_labeling_functions_2025.csv",
    "wildfire_evidence_analysis_2025.csv",
    "wildfire_seasonality_2025.csv",
    "wildfire_monthly_seasonality_2025.csv"
]

found = {}

for root, dirs, files in os.walk("/kaggle/input"):

    for file in files:

        if file in target_files:

            found[file] = os.path.join(
                root,
                file
            )

print("\nFiles found:")

for name, path in sorted(found.items()):

    print(f"\n{name}")
    print(f"  {path}")

print("\n" + "=" * 70)
print("FILE STRUCTURES")
print("=" * 70)

for name, path in sorted(found.items()):

    print("\n" + "-" * 70)
    print(name)
    print("-" * 70)

    sample = pd.read_csv(
        path,
        nrows=5
    )

    print(
        "Columns:",
        len(sample.columns)
    )

    print("\nColumn names:")

    for i, col in enumerate(
        sample.columns,
        1
    ):
        print(
            f"{i:3}. {col}"
        )

    print("\nFirst 5 rows:")

    display(sample)

print("\n" + "=" * 70)
print("STEP 4G-1 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:16:04.193472Z","iopub.execute_input":"2026-09-09T12:16:04.193837Z","iopub.status.idle":"2026-09-09T12:16:04.222811Z","shell.execute_reply.started":"2026-09-09T12:16:04.193804Z","shell.execute_reply":"2026-09-09T12:16:04.220401Z"}}
# ============================================================
# STEP 4G-2 — FIND EXISTING WILDFIRE-RELATED FILES
# ============================================================

import os

print("=" * 70)
print("STEP 4G-2 — SEARCHING FOR WILDFIRE-RELATED FILES")
print("=" * 70)

keywords = [
    "wildfire",
    "forest",
    "burn",
    "vegetation",
    "landcover",
    "land_cover"
]

matches = []

for root, dirs, files in os.walk("/kaggle/input"):

    for file in files:

        file_lower = file.lower()

        if any(
            keyword in file_lower
            for keyword in keywords
        ):

            matches.append(
                os.path.join(root, file)
            )

print(
    f"\nFiles found: {len(matches)}"
)

for i, path in enumerate(
    sorted(matches),
    1
):

    print(
        f"{i:3}. {path}"
    )

print("\n" + "=" * 70)
print("STEP 4G-2 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:16:04.225529Z","iopub.execute_input":"2026-09-09T12:16:04.225928Z","iopub.status.idle":"2026-09-09T12:16:04.280483Z","shell.execute_reply.started":"2026-09-09T12:16:04.225896Z","shell.execute_reply":"2026-09-09T12:16:04.279083Z"}}
# ============================================================
# STEP 4G-3 — INSPECT WILDFIRE EVIDENCE FILE
# ============================================================

import pandas as pd
import os

print("=" * 70)
print("STEP 4G-3 — WILDFIRE EVIDENCE INSPECTION")
print("=" * 70)

WILDFIRE_FILE = (
    "/kaggle/input/datasets/gautamkumar0036/"
    "google-colab-all-real-data/"
    "firms_wildfire_evidence_2025.csv"
)

if not os.path.exists(WILDFIRE_FILE):
    raise FileNotFoundError(
        f"Wildfire file not found:\n{WILDFIRE_FILE}"
    )

# ------------------------------------------------------------
# LOAD SAMPLE
# ------------------------------------------------------------

wildfire_sample = pd.read_csv(
    WILDFIRE_FILE,
    nrows=5
)

print("\nFile:")
print(WILDFIRE_FILE)

print(
    "\nColumns:",
    len(wildfire_sample.columns)
)

print("\nColumn names:")

for i, col in enumerate(
    wildfire_sample.columns,
    1
):
    print(
        f"{i:3}. {col}"
    )

print("\nFirst 5 rows:")
display(wildfire_sample)

# ------------------------------------------------------------
# BASIC FILE INFORMATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASIC FILE INFORMATION")
print("=" * 70)

file_size_mb = (
    os.path.getsize(WILDFIRE_FILE)
    / (1024 ** 2)
)

print(
    f"File size: "
    f"{file_size_mb:.2f} MB"
)

# ------------------------------------------------------------
# IDENTIFY IMPORTANT COLUMNS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("POTENTIALLY IMPORTANT WILDFIRE COLUMNS")
print("=" * 70)

keywords = [
    "fire",
    "wildfire",
    "forest",
    "tree",
    "vegetation",
    "burn",
    "land",
    "dw_",
    "osm",
    "recurrent",
    "nearby",
    "distance",
    "score",
    "season",
    "crop"
]

for col in wildfire_sample.columns:

    col_lower = col.lower()

    if any(
        keyword in col_lower
        for keyword in keywords
    ):

        print(col)

print("\n" + "=" * 70)
print("STEP 4G-3 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:16:04.282256Z","iopub.execute_input":"2026-09-09T12:16:04.282626Z","iopub.status.idle":"2026-09-09T12:17:12.193852Z","shell.execute_reply.started":"2026-09-09T12:16:04.282593Z","shell.execute_reply":"2026-09-09T12:17:12.193036Z"}}
# ============================================================
# STEP 4G-4
# AGGREGATE WILDFIRE EVIDENCE → EVENT LEVEL
# WITH COLUMN VALIDATION / FIX
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 4G-4 — WILDFIRE DETECTION EVIDENCE → EVENT EVIDENCE")
print("=" * 70)

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------

WILDFIRE_FILE = (
    "/kaggle/input/datasets/gautamkumar0036/"
    "google-colab-all-real-data/"
    "firms_wildfire_evidence_2025.csv"
)

EVENT_FILE = (
    "/kaggle/working/firms_2025_events.csv"
)

CURRENT_EVENT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_gas_industrial_mining_agriculture_evidence.csv"
)

OUTPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

# ------------------------------------------------------------
# 2. CHECK FILES
# ------------------------------------------------------------

print("\nChecking input files...")

for path in [
    WILDFIRE_FILE,
    EVENT_FILE,
    CURRENT_EVENT_FILE
]:

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"File not found:\n{path}"
        )

print("All required files found.")

# ------------------------------------------------------------
# 3. LOAD EVENT → DETECTION MAPPING
# ------------------------------------------------------------

print("\nLoading event-detection mapping...")

event_mapping = pd.read_csv(
    EVENT_FILE,
    usecols=[
        "_fire_index",
        "event_id"
    ]
)

print(
    f"Detection rows: "
    f"{len(event_mapping):,}"
)

print(
    f"Unique detections: "
    f"{event_mapping['_fire_index'].nunique():,}"
)

print(
    f"Unique events: "
    f"{event_mapping['event_id'].nunique():,}"
)

# ------------------------------------------------------------
# 4. LOAD WILDFIRE EVIDENCE
# ------------------------------------------------------------

print("\nLoading wildfire evidence...")

wildfire = pd.read_csv(
    WILDFIRE_FILE
)

print(
    f"Wildfire detection rows: "
    f"{len(wildfire):,}"
)

print(
    f"Unique wildfire detection IDs: "
    f"{wildfire['_fire_index'].nunique():,}"
)

# ------------------------------------------------------------
# 5. IDENTIFIER VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("IDENTIFIER VALIDATION")
print("=" * 70)

if event_mapping[
    "_fire_index"
].duplicated().any():

    raise ValueError(
        "Duplicate _fire_index found in event mapping."
    )

if wildfire[
    "_fire_index"
].duplicated().any():

    raise ValueError(
        "Duplicate _fire_index found in wildfire evidence."
    )

event_ids = set(
    event_mapping["_fire_index"]
)

wildfire_ids = set(
    wildfire["_fire_index"]
)

matching = (
    event_ids & wildfire_ids
)

print(
    f"Matching detection IDs: "
    f"{len(matching):,}"
)

if len(matching) != len(event_ids):

    missing = event_ids - wildfire_ids

    raise ValueError(
        f"{len(missing):,} detections are missing "
        "wildfire evidence."
    )

print(
    f"All {len(event_ids):,} FIRMS detections "
    "have wildfire evidence."
)

# ------------------------------------------------------------
# 6. REQUIRED WILDFIRE FEATURES
# ------------------------------------------------------------

distance_cols = [
    "distance_to_nearest_flare_m",
    "distance_to_nearest_oil_well_m",
    "distance_to_nearest_mineshaft_m",
    "distance_to_nearest_adit_m",
    "distance_to_nearest_gasometer_m",
    "distance_to_nearest_industrial_m",
    "distance_to_nearest_quarry_m",

    "distance_to_nearest_farmland_m",
    "distance_to_nearest_farmyard_m",
    "distance_to_nearest_orchard_m",
    "distance_to_nearest_vineyard_m",
    "distance_to_nearest_plant_nursery_m",
    "distance_to_nearest_greenhouse_m",
    "distance_to_nearest_allotment_m",

    "distance_to_nearest_forest_m",
    "distance_to_nearest_scrub_m",
    "distance_to_nearest_grassland_m",
    "distance_to_nearest_heath_m",
    "distance_to_nearest_agriculture_m",
    "distance_to_nearest_natural_vegetation_m"
]

near_cols = [
    "near_flare_1km",
    "near_oil_well_1km",
    "near_mine_1km",
    "near_industrial_1km",
    "near_quarry_1km",
    "near_agriculture_1km",
    "near_forest_1km",
    "near_natural_vegetation_1km"
]

dw_probability_cols = [
    "dw_trees_prob",
    "dw_grass_prob",
    "dw_shrub_prob",
    "dw_flooded_vegetation_prob",
    "dw_crop_prob",
    "dw_natural_vegetation_prob"
]

indicator_cols = [
    "natural_landcover_context",
    "natural_vegetation_strong",
    "natural_vegetation_moderate",
    "vegetation_recent_nearby_activity",
    "vegetation_recurrent_activity"
]

required_cols = (
    distance_cols
    + near_cols
    + dw_probability_cols
    + indicator_cols
)

missing = [
    c for c in required_cols
    if c not in wildfire.columns
]

if missing:

    raise ValueError(
        f"Missing wildfire columns:\n{missing}"
    )

print(
    f"\nAll {len(required_cols)} required "
    "wildfire features found."
)

# ------------------------------------------------------------
# 7. SELECT REQUIRED DATA
# ------------------------------------------------------------

wildfire_small = wildfire[
    ["_fire_index"] + required_cols
].copy()

# ------------------------------------------------------------
# 8. ATTACH EVENT IDs
# ------------------------------------------------------------

print("\nAttaching event IDs...")

wildfire_event = wildfire_small.merge(
    event_mapping,
    on="_fire_index",
    how="inner",
    validate="one_to_one"
)

print(
    f"Wildfire detections connected to events: "
    f"{len(wildfire_event):,}"
)

if len(wildfire_event) != len(wildfire_small):

    raise ValueError(
        "Wildfire detection → event mapping lost rows."
    )

# ------------------------------------------------------------
# 9. NUMERIC CONVERSION
# ------------------------------------------------------------

numeric_cols = (
    distance_cols
    + dw_probability_cols
)

for col in numeric_cols:

    wildfire_event[col] = pd.to_numeric(
        wildfire_event[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# 10. BUILD AGGREGATION DICTIONARY
# ------------------------------------------------------------

agg_dict = {}

# Distances → minimum
for col in distance_cols:

    agg_dict[col] = "min"

# Nearby flags → maximum / ANY
for col in near_cols:

    agg_dict[col] = "max"

# Vegetation indicators → maximum / ANY
for col in indicator_cols:

    agg_dict[col] = "max"

# Dynamic World probabilities → max + mean
for col in dw_probability_cols:

    agg_dict[col] = [
        "max",
        "mean"
    ]

# ------------------------------------------------------------
# 11. AGGREGATE BY EVENT
# ------------------------------------------------------------

print("\nAggregating wildfire evidence by event...")

event_wildfire = (
    wildfire_event
    .groupby(
        "event_id",
        sort=False
    )
    .agg(agg_dict)
)

# Flatten MultiIndex columns
event_wildfire.columns = [
    "_".join(
        [str(x) for x in col]
    ).strip("_")
    if isinstance(col, tuple)
    else str(col)
    for col in event_wildfire.columns
]

event_wildfire = (
    event_wildfire
    .reset_index()
)

print(
    f"Event-level wildfire records: "
    f"{len(event_wildfire):,}"
)

# ------------------------------------------------------------
# 12. RENAME EVENT-LEVEL COLUMNS
# ------------------------------------------------------------

rename_map = {}

for col in distance_cols:

    rename_map[col] = (
        "event_min_" + col
    )

for col in near_cols:

    rename_map[col] = (
        "event_any_" + col
    )

for col in indicator_cols:

    rename_map[col] = (
        "event_any_" + col
    )

for col in dw_probability_cols:

    rename_map[
        f"{col}_max"
    ] = (
        f"event_max_{col}"
    )

    rename_map[
        f"{col}_mean"
    ] = (
        f"event_mean_{col}"
    )

event_wildfire = event_wildfire.rename(
    columns=rename_map
)

# ------------------------------------------------------------
# 13. LOAD CURRENT EVENT DATASET
# ------------------------------------------------------------

print("\nLoading current combined event dataset...")

current_events = pd.read_csv(
    CURRENT_EVENT_FILE
)

print(
    f"Current events: "
    f"{len(current_events):,}"
)

if current_events[
    "event_id"
].duplicated().any():

    raise ValueError(
        "Duplicate event_id found."
    )

# ------------------------------------------------------------
# 14. MERGE WILDFIRE EVIDENCE
# ------------------------------------------------------------

print("\nMerging wildfire evidence...")

final_events = current_events.merge(
    event_wildfire,
    on="event_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 15. EVENT COUNT VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 4G-4 VALIDATION")
print("=" * 70)

print(
    f"Input events: "
    f"{len(current_events):,}"
)

print(
    f"Wildfire event records: "
    f"{len(event_wildfire):,}"
)

print(
    f"Final events: "
    f"{len(final_events):,}"
)

if len(final_events) != len(current_events):

    raise ValueError(
        "Event count changed during wildfire merge!"
    )

if final_events[
    "event_id"
].nunique() != len(final_events):

    raise ValueError(
        "Duplicate event IDs after merge!"
    )

# ------------------------------------------------------------
# 16. SHOW ACTUAL WILDFIRE COLUMNS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WILDFIRE COLUMN VALIDATION")
print("=" * 70)

wildfire_event_cols = [
    c for c in final_events.columns
    if (
        "wildfire" in c.lower()
        or "vegetation" in c.lower()
        or "forest" in c.lower()
        or "natural_" in c.lower()
        or "near_" in c.lower()
        or "dw_" in c.lower()
    )
]

print(
    f"\nWildfire-related columns found: "
    f"{len(wildfire_event_cols)}"
)

for i, col in enumerate(
    wildfire_event_cols,
    1
):

    print(
        f"{i:3}. {col}"
    )

# ------------------------------------------------------------
# 17. HELPER FUNCTION
#     FIND ACTUAL COLUMN NAME
# ------------------------------------------------------------

def find_column(feature):

    matches = [
        c for c in final_events.columns
        if feature.lower() in c.lower()
    ]

    if len(matches) == 1:

        return matches[0]

    if len(matches) > 1:

        preferred = [
            c for c in matches
            if c.lower() == (
                "event_any_" + feature
            ).lower()
        ]

        if preferred:

            return preferred[0]

        return matches[0]

    return None

# ------------------------------------------------------------
# 18. VEGETATION PROXIMITY COVERAGE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("VEGETATION CONTEXT")
print("-" * 70)

vegetation_features = [
    "near_forest_1km",
    "near_natural_vegetation_1km"
]

for feature in vegetation_features:

    col = find_column(feature)

    if col is not None:

        values = pd.to_numeric(
            final_events[col],
            errors="coerce"
        ).fillna(0)

        print(
            f"{col}: "
            f"{int(values.astype(bool).sum()):,}"
        )

    else:

        print(
            f"WARNING: {feature} not found"
        )

# ------------------------------------------------------------
# 19. NATURAL VEGETATION INDICATORS
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("NATURAL VEGETATION INDICATORS")
print("-" * 70)

indicator_features = [
    "natural_landcover_context",
    "natural_vegetation_strong",
    "natural_vegetation_moderate",
    "vegetation_recent_nearby_activity",
    "vegetation_recurrent_activity"
]

for feature in indicator_features:

    col = find_column(feature)

    if col is not None:

        values = pd.to_numeric(
            final_events[col],
            errors="coerce"
        ).fillna(0)

        print(
            f"{col}: "
            f"{int(values.astype(bool).sum()):,}"
        )

    else:

        print(
            f"WARNING: {feature} not found"
        )

# ------------------------------------------------------------
# 20. DYNAMIC WORLD VEGETATION PROBABILITIES
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("DYNAMIC WORLD VEGETATION PROBABILITIES")
print("-" * 70)

probability_features = [
    "dw_trees_prob",
    "dw_grass_prob",
    "dw_shrub_prob",
    "dw_flooded_vegetation_prob",
    "dw_crop_prob",
    "dw_natural_vegetation_prob"
]

for feature in probability_features:

    col = find_column(
        f"event_max_{feature}"
    )

    if col is None:

        col = find_column(feature)

    if col is not None:

        print(
            f"\n{col}"
        )

        print(
            pd.to_numeric(
                final_events[col],
                errors="coerce"
            )
            .describe()
            .to_string()
        )

    else:

        print(
            f"\nWARNING: {feature} not found"
        )

# ------------------------------------------------------------
# 21. FEATURE GROUP VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WILDFIRE / VEGETATION FEATURE VALIDATION")
print("=" * 70)

feature_groups = {

    "Vegetation proximity": [
        "near_forest_1km",
        "near_natural_vegetation_1km"
    ],

    "Natural vegetation indicators": [
        "natural_landcover_context",
        "natural_vegetation_strong",
        "natural_vegetation_moderate",
        "vegetation_recent_nearby_activity",
        "vegetation_recurrent_activity"
    ],

    "Dynamic World vegetation": [
        "dw_trees_prob",
        "dw_grass_prob",
        "dw_shrub_prob",
        "dw_flooded_vegetation_prob",
        "dw_crop_prob",
        "dw_natural_vegetation_prob"
    ],

    "Other source-context proximity": [
        "near_flare_1km",
        "near_oil_well_1km",
        "near_mine_1km",
        "near_industrial_1km",
        "near_quarry_1km",
        "near_agriculture_1km"
    ]
}

for group, features in feature_groups.items():

    print(f"\n{group}:")

    for feature in features:

        matches = [
            c for c in final_events.columns
            if feature.lower() in c.lower()
        ]

        if matches:

            print(
                f"  ✓ {feature} → {matches}"
            )

        else:

            print(
                f"  ✗ {feature} → NOT FOUND"
            )

# ------------------------------------------------------------
# 22. FINAL EVENT COUNT VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL EVENT COUNT VALIDATION")
print("=" * 70)

print(
    f"Events: "
    f"{len(final_events):,}"
)

print(
    f"Unique event IDs: "
    f"{final_events['event_id'].nunique():,}"
)

if (
    len(final_events) != len(current_events)
    or
    final_events["event_id"].nunique()
    != len(current_events)
):

    raise ValueError(
        "Final event count validation failed."
    )

# ------------------------------------------------------------
# 23. SAVE FINAL DATASET
# ------------------------------------------------------------

print("\nSaving final all-source event dataset...")

final_events.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    f"\nSaved successfully:\n"
    f"{OUTPUT_FILE}"
)

print("\n" + "=" * 70)
print("STEP 4G-4 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:17:12.195648Z","iopub.execute_input":"2026-09-09T12:17:12.196404Z","iopub.status.idle":"2026-09-09T12:17:19.500982Z","shell.execute_reply.started":"2026-09-09T12:17:12.196340Z","shell.execute_reply":"2026-09-09T12:17:19.497520Z"}}
# ============================================================
# STEP 4H-1 — FINAL EVIDENCE DATASET AUDIT
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 4H-1 — FINAL EVENT EVIDENCE DATASET AUDIT")
print("=" * 70)

FINAL_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

# ------------------------------------------------------------
# 1. CHECK FILE
# ------------------------------------------------------------

if not os.path.exists(FINAL_FILE):
    raise FileNotFoundError(
        f"Final evidence file not found:\n{FINAL_FILE}"
    )

# ------------------------------------------------------------
# 2. LOAD
# ------------------------------------------------------------

print("\nLoading final event evidence dataset...")

final_events = pd.read_csv(
    FINAL_FILE
)

print(
    f"Rows: {len(final_events):,}"
)

print(
    f"Columns: {len(final_events.columns):,}"
)

# ------------------------------------------------------------
# 3. EVENT ID VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EVENT ID VALIDATION")
print("=" * 70)

unique_events = (
    final_events["event_id"]
    .nunique()
)

duplicate_events = (
    final_events["event_id"]
    .duplicated()
    .sum()
)

print(
    f"Unique event IDs: "
    f"{unique_events:,}"
)

print(
    f"Duplicate event rows: "
    f"{duplicate_events:,}"
)

if len(final_events) != 302070:
    raise ValueError(
        f"Expected 302,070 events, "
        f"got {len(final_events):,}"
    )

if unique_events != 302070:
    raise ValueError(
        "Event IDs are not unique."
    )

print(
    "\n✓ Event count and uniqueness validated."
)

# ------------------------------------------------------------
# 4. LIST EVIDENCE COLUMNS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE EVIDENCE COLUMNS")
print("=" * 70)

source_groups = {

    "GAS / FLARE": [
        c for c in final_events.columns
        if (
            "flare" in c.lower()
            or "gas_" in c.lower()
        )
    ],

    "INDUSTRIAL": [
        c for c in final_events.columns
        if (
            "industrial" in c.lower()
            or "steel" in c.lower()
            or "cement" in c.lower()
            or "fertilizer" in c.lower()
            or "refinery" in c.lower()
            or "power" in c.lower()
        )
    ],

    "MINING": [
        c for c in final_events.columns
        if (
            "mine" in c.lower()
            or "mining" in c.lower()
        )
    ],

    "AGRICULTURE": [
        c for c in final_events.columns
        if (
            "agri" in c.lower()
            or "agriculture" in c.lower()
            or "crop" in c.lower()
        )
    ],

    "WILDFIRE / VEGETATION": [
        c for c in final_events.columns
        if (
            "forest" in c.lower()
            or "vegetation" in c.lower()
            or "wildfire" in c.lower()
            or "natural_" in c.lower()
        )
    ]
}

for group, columns in source_groups.items():

    # Remove duplicates while preserving order
    columns = list(
        dict.fromkeys(columns)
    )

    print(
        f"\n{group}: {len(columns)} columns"
    )

    for col in columns:

        print(
            f"  {col}"
        )

# ------------------------------------------------------------
# 5. CHECK MISSINGNESS OF SOURCE EVIDENCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SOURCE EVIDENCE MISSINGNESS")
print("=" * 70)

all_evidence_columns = list(
    dict.fromkeys(
        sum(
            source_groups.values(),
            []
        )
    )
)

missing_summary = []

for col in all_evidence_columns:

    missing_count = (
        final_events[col]
        .isna()
        .sum()
    )

    missing_pct = (
        missing_count
        / len(final_events)
        * 100
    )

    missing_summary.append(
        (
            col,
            missing_count,
            missing_pct
        )
    )

missing_summary = sorted(
    missing_summary,
    key=lambda x: x[2],
    reverse=True
)

for col, count, pct in missing_summary:

    print(
        f"{col:55} "
        f"{count:8,} missing "
        f"({pct:6.2f}%)"
    )

# ------------------------------------------------------------
# 6. BASIC NUMERIC SANITY CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NUMERIC SANITY CHECK")
print("=" * 70)

numeric_cols = final_events.select_dtypes(
    include=np.number
).columns

print(
    f"Numeric columns: "
    f"{len(numeric_cols):,}"
)

inf_count = 0

for col in numeric_cols:

    values = final_events[col].to_numpy()

    inf_count += np.isinf(
        values
    ).sum()

print(
    f"Infinite numeric values: "
    f"{inf_count:,}"
)

if inf_count > 0:
    raise ValueError(
        "Infinite numeric values detected."
    )

print(
    "✓ No infinite numeric values."
)

# ------------------------------------------------------------
# 7. SAVE CONFIRMED DATASET
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print(
    f"Rows: {len(final_events):,}"
)

print(
    f"Columns: {len(final_events.columns):,}"
)

print(
    f"Unique events: "
    f"{final_events['event_id'].nunique():,}"
)

print(
    f"\nConfirmed file:\n{FINAL_FILE}"
)

print("\n" + "=" * 70)
print("STEP 4H-1 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:17:19.502609Z","iopub.execute_input":"2026-09-09T12:17:19.502907Z","iopub.status.idle":"2026-09-09T12:17:28.496958Z","shell.execute_reply.started":"2026-09-09T12:17:19.502882Z","shell.execute_reply":"2026-09-09T12:17:28.494310Z"}}
# ============================================================
# STEP 5A — INSPECT EVENT-LEVEL EVIDENCE DISTRIBUTIONS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5A — EVIDENCE DISTRIBUTION ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD FINAL EVENT DATASET
# ------------------------------------------------------------

FINAL_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

df = pd.read_csv(FINAL_FILE)

print(
    f"\nEvents: {len(df):,}"
)

# ------------------------------------------------------------
# 2. HELPER FUNCTION
# ------------------------------------------------------------

def summarize_numeric(
    data,
    columns
):

    for col in columns:

        if col not in data.columns:
            print(f"\n[NOT FOUND] {col}")
            continue

        s = pd.to_numeric(
            data[col],
            errors="coerce"
        ).dropna()

        print("\n" + "-" * 70)
        print(col)
        print("-" * 70)

        if len(s) == 0:

            print("No valid numeric values.")
            continue

        print(
            s.describe(
                percentiles=[
                    0.01,
                    0.05,
                    0.10,
                    0.25,
                    0.50,
                    0.75,
                    0.90,
                    0.95,
                    0.99
                ]
            ).to_string()
        )

# ------------------------------------------------------------
# 3. GAS / FLARE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GAS / FLARE EVIDENCE")
print("=" * 70)

summarize_numeric(
    df,
    [
        "nearest_gas_flare_distance_km",
        "gas_flare_count_within_1km",
        "gas_flare_count_within_2km",
        "gas_flare_count_within_5km",
        "gas_flare_count_within_10km",
        "nearest_gas_flare_2025_activity"
    ]
)

# ------------------------------------------------------------
# 4. INDUSTRIAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INDUSTRIAL EVIDENCE")
print("=" * 70)

summarize_numeric(
    df,
    [
        "event_min_steel_distance_m",
        "event_min_cement_distance_m",
        "event_min_wri_power_distance_m",
        "event_min_fertilizer_distance_m",
        "event_min_refinery_petro_distance_m",
        "event_max_industrial_source_count_500m",
        "event_max_industrial_source_count_1km",
        "event_max_industrial_evidence_score"
    ]
)

# ------------------------------------------------------------
# 5. MINING
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MINING EVIDENCE")
print("=" * 70)

summarize_numeric(
    df,
    [
        "event_min_nearest_coal_mine_distance_km",
        "event_min_nearest_coal_mine_distance_m"
    ]
)

# ------------------------------------------------------------
# 6. AGRICULTURE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AGRICULTURE EVIDENCE")
print("=" * 70)

summarize_numeric(
    df,
    [
        "agri_osm_distance_m_min",
        "event_max_dw_crop_probability",
        "event_mean_dw_crop_probability",
        "event_max_dw_crop_prob",
        "event_mean_dw_crop_prob",
        "agriculture_evidence_score_max"
    ]
)

# ------------------------------------------------------------
# 7. WILDFIRE / VEGETATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WILDFIRE / VEGETATION EVIDENCE")
print("=" * 70)

summarize_numeric(
    df,
    [
        "distance_to_nearest_forest_m_min",
        "distance_to_nearest_natural_vegetation_m_min",
        "event_max_dw_trees_prob",
        "event_mean_dw_trees_prob",
        "event_max_dw_grass_prob",
        "event_mean_dw_grass_prob",
        "event_max_dw_shrub_prob",
        "event_mean_dw_shrub_prob",
        "event_max_dw_natural_vegetation_prob",
        "event_mean_dw_natural_vegetation_prob"
    ]
)

# ------------------------------------------------------------
# 8. FIRMS EVENT BEHAVIOR
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FIRMS EVENT BEHAVIOR")
print("=" * 70)

summarize_numeric(
    df,
    [
        "detection_count",
        "active_days",
        "active_months",
        "duration_days",
        "spatial_extent_km",
        "mean_frp",
        "max_frp",
        "mean_brightness",
        "max_brightness"
    ]
)

# ------------------------------------------------------------
# 9. BOOLEAN EVIDENCE COVERAGE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BOOLEAN EVIDENCE COVERAGE")
print("=" * 70)

boolean_groups = {

    "Gas": [
        "has_gas_flare_within_1km",
        "has_gas_flare_within_2km",
        "has_gas_flare_within_5km",
        "has_gas_flare_within_10km",
        "nearest_flare_is_gas"
    ],

    "Industrial": [
        "event_any_steel_within_500m",
        "event_any_cement_within_500m",
        "event_any_wri_power_within_500m",
        "event_any_fertilizer_within_500m",
        "event_any_refinery_petro_within_500m",
        "event_any_industrial_multi_source_500m"
    ],

    "Mining": [
        "event_any_coal_mine_within_500m",
        "event_any_coal_mine_within_1km",
        "event_any_mining_evidence_close_500m",
        "event_any_mining_evidence_close_1km"
    ],

    "Agriculture": [
        "agri_osm_within_500m_max",
        "agri_osm_within_1km_max",
        "agri_dw_crop_50_max",
        "agri_dw_crop_70_max",
        "agri_dw_crop_80_max",
        "agri_dw_crop_90_max"
    ],

    "Wildfire": [
        "near_forest_1km_max",
        "near_natural_vegetation_1km_max",
        "natural_landcover_context_max",
        "natural_vegetation_strong_max",
        "natural_vegetation_moderate_max"
    ]
}

for group, columns in boolean_groups.items():

    print("\n" + "-" * 70)
    print(group)
    print("-" * 70)

    for col in columns:

        if col not in df.columns:
            print(
                f"{col}: NOT FOUND"
            )
            continue

        values = df[col]

        # Handle bool / 0-1 / NaN safely
        positive = (
            values.fillna(False)
            .astype(bool)
            .sum()
        )

        pct = (
            positive
            / len(df)
            * 100
        )

        print(
            f"{col:50} "
            f"{positive:8,} "
            f"({pct:6.2f}%)"
        )

print("\n" + "=" * 70)
print("STEP 5A COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:17:28.498937Z","iopub.execute_input":"2026-09-09T12:17:28.499258Z","iopub.status.idle":"2026-09-09T12:17:28.521214Z","shell.execute_reply.started":"2026-09-09T12:17:28.499238Z","shell.execute_reply":"2026-09-09T12:17:28.517753Z"}}
# ============================================================
# STEP 5B — IDENTIFY ACTUAL EVENT BEHAVIOR COLUMNS
# ============================================================

print("=" * 70)
print("STEP 5B — EVENT BEHAVIOR COLUMN INSPECTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. SEARCH FOR EVENT BEHAVIOR COLUMNS
# ------------------------------------------------------------

behavior_keywords = [
    "detection",
    "active",
    "duration",
    "span",
    "gap",
    "persistence",
    "concentration",
    "peak_month",
    "frp",
    "brightness",
    "spatial",
    "extent",
    "rate",
    "month"
]

behavior_cols = []

for col in final_events.columns:

    col_lower = col.lower()

    if any(
        keyword in col_lower
        for keyword in behavior_keywords
    ):

        behavior_cols.append(col)

print(
    f"\nPotential behavior columns: "
    f"{len(behavior_cols)}"
)

for i, col in enumerate(
    behavior_cols,
    1
):

    print(
        f"{i:3}. {col}"
    )

# ------------------------------------------------------------
# 2. SHOW DATA TYPES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEHAVIOR COLUMN DATA TYPES")
print("=" * 70)

for col in behavior_cols:

    print(
        f"{col:55} "
        f"{str(final_events[col].dtype)}"
    )

# ------------------------------------------------------------
# 3. SEARCH SPECIFIC IMPORTANT TERMS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("IMPORTANT BEHAVIOR FEATURES")
print("=" * 70)

important_terms = [
    "count",
    "active_days",
    "active_months",
    "duration",
    "span",
    "persistence",
    "gap",
    "concentration",
    "peak",
    "frp",
    "brightness"
]

for term in important_terms:

    matches = [
        col for col in final_events.columns
        if term in col.lower()
    ]

    print(
        f"\n{term}:"
    )

    for col in matches:

        print(
            f"  {col}"
        )

print("\n" + "=" * 70)
print("STEP 5B COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:17:28.530982Z","iopub.execute_input":"2026-09-09T12:17:28.531426Z","iopub.status.idle":"2026-09-09T12:18:22.749337Z","shell.execute_reply.started":"2026-09-09T12:17:28.531399Z","shell.execute_reply":"2026-09-09T12:18:22.746875Z"}}
# ============================================================
# STEP 5C — INITIAL SOURCE-SPECIFIC LABELING FUNCTIONS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5C — INITIAL LABELING FUNCTIONS")
print("=" * 70)

# ------------------------------------------------------------
# LABEL DEFINITIONS
# ------------------------------------------------------------

ABSTAIN = -1

INDUSTRIAL = 0
GAS = 1
AGRICULTURE = 2
MINING = 3
WILDFIRE = 4

LABEL_NAMES = {
    ABSTAIN: "ABSTAIN",
    INDUSTRIAL: "INDUSTRIAL",
    GAS: "GAS",
    AGRICULTURE: "AGRICULTURE",
    MINING: "MINING",
    WILDFIRE: "WILDFIRE"
}

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

FINAL_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

df = pd.read_csv(FINAL_FILE)

print(
    f"\nEvents: {len(df):,}"
)

# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------

def b(col):
    """
    Safe boolean conversion.
    """
    return (
        df[col]
        .fillna(False)
        .astype(bool)
    )


def n(col):
    """
    Numeric conversion.
    """
    return pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ============================================================
# GAS LFs
# ============================================================

print("\n" + "=" * 70)
print("GAS LABELING FUNCTIONS")
print("=" * 70)

# Strongest gas evidence:
# active World Bank gas flare very close to event.

LF_GAS_ACTIVE_FLARE_1KM = np.where(
    b("has_gas_flare_within_1km"),
    GAS,
    ABSTAIN
)

# Slightly weaker but still highly specific.

LF_GAS_ACTIVE_FLARE_2KM = np.where(
    b("has_gas_flare_within_2km"),
    GAS,
    ABSTAIN
)

# Gas flare within 5 km.

LF_GAS_ACTIVE_FLARE_5KM = np.where(
    b("has_gas_flare_within_5km"),
    GAS,
    ABSTAIN
)

# Repeated FIRMS activity associated with a very close flare.
# Requires both spatial gas evidence and repeated detections.

LF_GAS_REPEATED_FLARE = np.where(
    (
        b("has_gas_flare_within_2km")
        &
        (n("event_detection_count") >= 2)
    ),
    GAS,
    ABSTAIN
)

# ============================================================
# INDUSTRIAL LFs
# ============================================================

print("\n" + "=" * 70)
print("INDUSTRIAL LABELING FUNCTIONS")
print("=" * 70)

# Multi-source industrial evidence is more specific than
# generic proximity to one industrial facility.

LF_INDUSTRIAL_MULTI_SOURCE_500M = np.where(
    b("event_any_industrial_multi_source_500m"),
    INDUSTRIAL,
    ABSTAIN
)

LF_INDUSTRIAL_MULTI_SOURCE_1KM = np.where(
    b("event_any_industrial_multi_source_1km"),
    INDUSTRIAL,
    ABSTAIN
)

# Strong source-specific industrial evidence.

LF_INDUSTRIAL_STEEL = np.where(
    (
        b("event_any_steel_within_500m")
        &
        (
            b("event_any_steel_strong_7d")
            |
            b("event_any_steel_strong_30d")
        )
    ),
    INDUSTRIAL,
    ABSTAIN
)

LF_INDUSTRIAL_CEMENT = np.where(
    (
        b("event_any_cement_within_500m")
        &
        (
            b("event_any_cement_strong_7d")
            |
            b("event_any_cement_strong_30d")
        )
    ),
    INDUSTRIAL,
    ABSTAIN
)

LF_INDUSTRIAL_POWER = np.where(
    (
        b("event_any_wri_power_within_500m")
        &
        (
            b("event_any_wri_power_strong_7d")
            |
            b("event_any_wri_power_strong_30d")
        )
    ),
    INDUSTRIAL,
    ABSTAIN
)

LF_INDUSTRIAL_FERTILIZER = np.where(
    (
        b("event_any_fertilizer_within_500m")
        &
        (
            b("event_any_fertilizer_strong_7d")
            |
            b("event_any_fertilizer_strong_30d")
        )
    ),
    INDUSTRIAL,
    ABSTAIN
)

LF_INDUSTRIAL_REFINERY = np.where(
    (
        b("event_any_refinery_petro_within_500m")
        &
        (
            b("event_any_refinery_petro_strong_7d")
            |
            b("event_any_refinery_petro_strong_30d")
        )
    ),
    INDUSTRIAL,
    ABSTAIN
)

# Industrial evidence score.
# Score >= 2 means multiple pieces of industrial evidence.

LF_INDUSTRIAL_SCORE = np.where(
    n("event_max_industrial_evidence_score") >= 2,
    INDUSTRIAL,
    ABSTAIN
)

# ============================================================
# MINING LFs
# ============================================================

print("\n" + "=" * 70)
print("MINING LABELING FUNCTIONS")
print("=" * 70)

# Very close coal mine.

LF_MINING_COAL_500M = np.where(
    b("event_any_coal_mine_within_500m"),
    MINING,
    ABSTAIN
)

# Coal mine within 1 km plus repeated activity.

LF_MINING_COAL_RECURRENT = np.where(
    (
        b("event_any_coal_mine_within_1km")
        &
        (
            b("event_any_mining_evidence_close_1km_recurrent_7d")
            |
            b("event_any_mining_evidence_close_1km_recurrent_30d")
        )
    ),
    MINING,
    ABSTAIN
)

# Strong close mining evidence.

LF_MINING_STRONG = np.where(
    (
        b("event_any_mining_evidence_close_500m")
        &
        (
            b("event_any_mining_evidence_close_500m_recurrent_7d")
            |
            b("event_any_mining_evidence_close_500m_recurrent_30d")
        )
    ),
    MINING,
    ABSTAIN
)

# Coal mine within 1 km.

LF_MINING_COAL_1KM = np.where(
    b("event_any_coal_mine_within_1km"),
    MINING,
    ABSTAIN
)

# ============================================================
# AGRICULTURE LFs
# ============================================================

print("\n" + "=" * 70)
print("AGRICULTURE LABELING FUNCTIONS")
print("=" * 70)

# Strong Dynamic World crop evidence.
# 70% is rare (0.14%), therefore highly selective.

LF_AGRI_CROP_70 = np.where(
    b("agri_dw_crop_70_max"),
    AGRICULTURE,
    ABSTAIN
)

# Crop >= 50% combined with agricultural OSM proximity.

LF_AGRI_CROP_OSM = np.where(
    (
        b("agri_dw_crop_50_max")
        &
        (
            b("agri_osm_within_500m_max")
            |
            b("agri_osm_within_1km_max")
        )
    ),
    AGRICULTURE,
    ABSTAIN
)

# Strong agriculture evidence score.

LF_AGRI_SCORE = np.where(
    n("agriculture_evidence_score_max") >= 2,
    AGRICULTURE,
    ABSTAIN
)

# Agriculture recurrence + agricultural context.

LF_AGRI_RECURRENT = np.where(
    (
        (
            b("agri_recurrent_7d_max")
            |
            b("agri_recurrent_30d_max")
        )
        &
        (
            b("agri_osm_within_1km_max")
            |
            b("agri_dw_landcover_crop_max")
        )
    ),
    AGRICULTURE,
    ABSTAIN
)

# ============================================================
# WILDFIRE LFs
# ============================================================

print("\n" + "=" * 70)
print("WILDFIRE LABELING FUNCTIONS")
print("=" * 70)

# Strong natural vegetation context.

LF_WILDFIRE_STRONG_NATURAL = np.where(
    (
        b("natural_vegetation_strong_max")
        &
        b("natural_landcover_context_max")
    ),
    WILDFIRE,
    ABSTAIN
)

# Natural vegetation + forest proximity.

LF_WILDFIRE_FOREST_CONTEXT = np.where(
    (
        b("near_forest_1km_max")
        &
        (
            b("natural_landcover_context_max")
            |
            b("natural_vegetation_moderate_max")
        )
    ),
    WILDFIRE,
    ABSTAIN
)

# Strong Dynamic World natural vegetation probability.

LF_WILDFIRE_NATURAL_PROB = np.where(
    (
        n("event_mean_dw_natural_vegetation_prob")
        >= 0.70
    ),
    WILDFIRE,
    ABSTAIN
)

# Forest/natural vegetation proximity plus high natural
# vegetation probability.

LF_WILDFIRE_VEGETATION_COMBINED = np.where(
    (
        (
            b("near_forest_1km_max")
            |
            b("near_natural_vegetation_1km_max")
        )
        &
        (
            n("event_mean_dw_natural_vegetation_prob")
            >= 0.60
        )
    ),
    WILDFIRE,
    ABSTAIN
)

# ============================================================
# STORE LFs
# ============================================================

lf_columns = {

    "LF_GAS_ACTIVE_FLARE_1KM":
        LF_GAS_ACTIVE_FLARE_1KM,

    "LF_GAS_ACTIVE_FLARE_2KM":
        LF_GAS_ACTIVE_FLARE_2KM,

    "LF_GAS_ACTIVE_FLARE_5KM":
        LF_GAS_ACTIVE_FLARE_5KM,

    "LF_GAS_REPEATED_FLARE":
        LF_GAS_REPEATED_FLARE,

    "LF_INDUSTRIAL_MULTI_SOURCE_500M":
        LF_INDUSTRIAL_MULTI_SOURCE_500M,

    "LF_INDUSTRIAL_MULTI_SOURCE_1KM":
        LF_INDUSTRIAL_MULTI_SOURCE_1KM,

    "LF_INDUSTRIAL_STEEL":
        LF_INDUSTRIAL_STEEL,

    "LF_INDUSTRIAL_CEMENT":
        LF_INDUSTRIAL_CEMENT,

    "LF_INDUSTRIAL_POWER":
        LF_INDUSTRIAL_POWER,

    "LF_INDUSTRIAL_FERTILIZER":
        LF_INDUSTRIAL_FERTILIZER,

    "LF_INDUSTRIAL_REFINERY":
        LF_INDUSTRIAL_REFINERY,

    "LF_INDUSTRIAL_SCORE":
        LF_INDUSTRIAL_SCORE,

    "LF_MINING_COAL_500M":
        LF_MINING_COAL_500M,

    "LF_MINING_COAL_RECURRENT":
        LF_MINING_COAL_RECURRENT,

    "LF_MINING_STRONG":
        LF_MINING_STRONG,

    "LF_MINING_COAL_1KM":
        LF_MINING_COAL_1KM,

    "LF_AGRI_CROP_70":
        LF_AGRI_CROP_70,

    "LF_AGRI_CROP_OSM":
        LF_AGRI_CROP_OSM,

    "LF_AGRI_SCORE":
        LF_AGRI_SCORE,

    "LF_AGRI_RECURRENT":
        LF_AGRI_RECURRENT,

    "LF_WILDFIRE_STRONG_NATURAL":
        LF_WILDFIRE_STRONG_NATURAL,

    "LF_WILDFIRE_FOREST_CONTEXT":
        LF_WILDFIRE_FOREST_CONTEXT,

    "LF_WILDFIRE_NATURAL_PROB":
        LF_WILDFIRE_NATURAL_PROB,

    "LF_WILDFIRE_VEGETATION_COMBINED":
        LF_WILDFIRE_VEGETATION_COMBINED
}

for name, values in lf_columns.items():

    df[name] = values.astype(np.int8)

# ============================================================
# LF COVERAGE
# ============================================================

print("\n" + "=" * 70)
print("LABELING FUNCTION COVERAGE")
print("=" * 70)

coverage_rows = []

for name in lf_columns:

    values = df[name]

    covered = (
        values != ABSTAIN
    ).sum()

    coverage = (
        covered
        / len(df)
        * 100
    )

    coverage_rows.append(
        [
            name,
            int(covered),
            coverage
        ]
    )

coverage_df = pd.DataFrame(
    coverage_rows,
    columns=[
        "LF",
        "covered_events",
        "coverage_pct"
    ]
)

display(
    coverage_df
    .sort_values(
        "coverage_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

# ============================================================
# LF VOTE DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("LF VOTE DISTRIBUTION")
print("=" * 70)

for name in lf_columns:

    counts = (
        df[name]
        .value_counts()
        .to_dict()
    )

    print(
        f"\n{name}"
    )

    for label, count in sorted(
        counts.items()
    ):

        print(
            f"  {LABEL_NAMES.get(label, str(label)):12} "
            f"{count:,}"
        )

# ============================================================
# SAVE
# ============================================================

LF_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_initial.csv"
)

df.to_csv(
    LF_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5C COMPLETE")
print("=" * 70)

print(
    f"\nSaved:\n{LF_FILE}"
)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:18:22.752345Z","iopub.execute_input":"2026-09-09T12:18:22.752941Z","iopub.status.idle":"2026-09-09T12:19:21.356729Z","shell.execute_reply.started":"2026-09-09T12:18:22.752886Z","shell.execute_reply":"2026-09-09T12:19:21.354706Z"}}
# ============================================================
# STEP 5D — LF CONFLICT AND AGREEMENT ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5D — LABELING FUNCTION CONFLICT ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# LOAD INITIAL LF DATA
# ------------------------------------------------------------

LF_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_initial.csv"
)

df = pd.read_csv(LF_FILE)

print(f"\nEvents: {len(df):,}")

# ------------------------------------------------------------
# LF GROUPS
# ------------------------------------------------------------

lf_groups = {
    "GAS": [
        "LF_GAS_ACTIVE_FLARE_1KM",
        "LF_GAS_ACTIVE_FLARE_2KM",
        "LF_GAS_ACTIVE_FLARE_5KM",
        "LF_GAS_REPEATED_FLARE"
    ],

    "INDUSTRIAL": [
        "LF_INDUSTRIAL_MULTI_SOURCE_500M",
        "LF_INDUSTRIAL_MULTI_SOURCE_1KM",
        "LF_INDUSTRIAL_STEEL",
        "LF_INDUSTRIAL_CEMENT",
        "LF_INDUSTRIAL_POWER",
        "LF_INDUSTRIAL_FERTILIZER",
        "LF_INDUSTRIAL_REFINERY",
        "LF_INDUSTRIAL_SCORE"
    ],

    "MINING": [
        "LF_MINING_COAL_500M",
        "LF_MINING_COAL_RECURRENT",
        "LF_MINING_STRONG",
        "LF_MINING_COAL_1KM"
    ],

    "AGRICULTURE": [
        "LF_AGRI_CROP_70",
        "LF_AGRI_CROP_OSM",
        "LF_AGRI_SCORE",
        "LF_AGRI_RECURRENT"
    ],

    "WILDFIRE": [
        "LF_WILDFIRE_STRONG_NATURAL",
        "LF_WILDFIRE_FOREST_CONTEXT",
        "LF_WILDFIRE_NATURAL_PROB",
        "LF_WILDFIRE_VEGETATION_COMBINED"
    ]
}

# ------------------------------------------------------------
# COUNT VOTES WITHIN EACH CLASS
# ------------------------------------------------------------

for class_name, cols in lf_groups.items():

    vote_cols = [
        col for col in cols
        if col in df.columns
    ]

    df[f"{class_name.lower()}_lf_votes"] = (
        (df[vote_cols] != -1)
        .sum(axis=1)
    )

# ------------------------------------------------------------
# TOTAL NUMBER OF SOURCE CLASSES VOTING
# ------------------------------------------------------------

class_vote_counts = pd.DataFrame(index=df.index)

for class_name in lf_groups:

    class_vote_counts[class_name] = (
        df[f"{class_name.lower()}_lf_votes"] > 0
    ).astype(int)

df["lf_active_class_count"] = (
    class_vote_counts.sum(axis=1)
)

# ------------------------------------------------------------
# DETERMINE UNIQUE CLASS COMBINATIONS
# ------------------------------------------------------------

df["lf_active_classes"] = (
    class_vote_counts
    .astype(int)
    .astype(str)
    .agg(
        lambda row: ",".join(
            class_vote_counts.columns[
                row.values.astype(bool)
            ]
        ),
        axis=1
    )
)

# Empty combinations

df.loc[
    df["lf_active_class_count"] == 0,
    "lf_active_classes"
] = "NONE"

# ------------------------------------------------------------
# OVERALL CLASS AGREEMENT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NUMBER OF ACTIVE SOURCE CLASSES")
print("=" * 70)

class_count_summary = (
    df["lf_active_class_count"]
    .value_counts()
    .sort_index()
)

for count, n_events in class_count_summary.items():

    pct = (
        n_events
        / len(df)
        * 100
    )

    print(
        f"{count} active classes: "
        f"{n_events:,} events "
        f"({pct:.2f}%)"
    )

# ------------------------------------------------------------
# EXACT CLASS COMBINATIONS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP CLASS COMBINATIONS")
print("=" * 70)

combination_summary = (
    df["lf_active_classes"]
    .value_counts()
    .head(30)
)

for combination, count in combination_summary.items():

    pct = (
        count
        / len(df)
        * 100
    )

    print(
        f"{combination:45} "
        f"{count:8,} "
        f"({pct:6.2f}%)"
    )

# ------------------------------------------------------------
# CLASS-TO-CLASS CONFLICT MATRIX
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS CONFLICT MATRIX")
print("=" * 70)

classes = list(lf_groups.keys())

conflict_matrix = pd.DataFrame(
    0,
    index=classes,
    columns=classes,
    dtype=int
)

for c1 in classes:

    for c2 in classes:

        if c1 == c2:
            continue

        conflict_matrix.loc[c1, c2] = (
            (
                class_vote_counts[c1] == 1
            )
            &
            (
                class_vote_counts[c2] == 1
            )
        ).sum()

print(conflict_matrix)

# ------------------------------------------------------------
# PAIRWISE CONFLICT DETAILS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PAIRWISE CONFLICTS")
print("=" * 70)

for i in range(len(classes)):

    for j in range(i + 1, len(classes)):

        c1 = classes[i]
        c2 = classes[j]

        conflict = (
            (class_vote_counts[c1] == 1)
            &
            (class_vote_counts[c2] == 1)
        ).sum()

        if conflict > 0:

            pct = (
                conflict
                / len(df)
                * 100
            )

            print(
                f"{c1:12} <-> {c2:12}: "
                f"{conflict:,} "
                f"({pct:.3f}%)"
            )

# ------------------------------------------------------------
# MULTI-CLASS CONFLICTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MULTI-CLASS CONFLICTS")
print("=" * 70)

multi_class = df[
    df["lf_active_class_count"] >= 2
]

print(
    f"Events with >=2 source classes voting: "
    f"{len(multi_class):,} "
    f"({len(multi_class)/len(df)*100:.2f}%)"
)

print(
    f"Events with >=3 source classes voting: "
    f"{(df['lf_active_class_count'] >= 3).sum():,}"
)

print(
    f"Events with >=4 source classes voting: "
    f"{(df['lf_active_class_count'] >= 4).sum():,}"
)

print(
    f"Events with all 5 source classes voting: "
    f"{(df['lf_active_class_count'] == 5).sum():,}"
)

# ------------------------------------------------------------
# WILDFIRE VS AGRICULTURE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WILDFIRE vs AGRICULTURE")
print("=" * 70)

wf_agri = df[
    (
        class_vote_counts["WILDFIRE"] == 1
    )
    &
    (
        class_vote_counts["AGRICULTURE"] == 1
    )
]

print(
    f"Wildfire + Agriculture conflicts: "
    f"{len(wf_agri):,}"
)

# ------------------------------------------------------------
# WILDFIRE VS INDUSTRIAL
# ------------------------------------------------------------

wf_ind = df[
    (
        class_vote_counts["WILDFIRE"] == 1
    )
    &
    (
        class_vote_counts["INDUSTRIAL"] == 1
    )
]

print(
    f"Wildfire + Industrial conflicts: "
    f"{len(wf_ind):,}"
)

# ------------------------------------------------------------
# WILDFIRE VS MINING
# ------------------------------------------------------------

wf_mining = df[
    (
        class_vote_counts["WILDFIRE"] == 1
    )
    &
    (
        class_vote_counts["MINING"] == 1
    )
]

print(
    f"Wildfire + Mining conflicts: "
    f"{len(wf_mining):,}"
)

# ------------------------------------------------------------
# WILDFIRE VS GAS
# ------------------------------------------------------------

wf_gas = df[
    (
        class_vote_counts["WILDFIRE"] == 1
    )
    &
    (
        class_vote_counts["GAS"] == 1
    )
]

print(
    f"Wildfire + Gas conflicts: "
    f"{len(wf_gas):,}"
)

# ------------------------------------------------------------
# EVENTS WITH NO LFs
# ------------------------------------------------------------

no_lf = df[
    df["lf_active_class_count"] == 0
]

print("\n" + "=" * 70)
print("UNLABELED / ABSTAINED EVENTS")
print("=" * 70)

print(
    f"No LF votes: "
    f"{len(no_lf):,} "
    f"({len(no_lf)/len(df)*100:.2f}%)"
)

# ------------------------------------------------------------
# SAVE CONFLICT ANALYSIS
# ------------------------------------------------------------

CONFLICT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_lf_conflict_analysis.csv"
)

df.to_csv(
    CONFLICT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5D COMPLETE")
print("=" * 70)

print(
    f"\nSaved:\n{CONFLICT_FILE}"
)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:19:21.359744Z","iopub.execute_input":"2026-09-09T12:19:21.360138Z","iopub.status.idle":"2026-09-09T12:20:16.989681Z","shell.execute_reply.started":"2026-09-09T12:19:21.360105Z","shell.execute_reply":"2026-09-09T12:20:16.988226Z"}}
# ============================================================
# STEP 5E — CORRECT LF CONFLICT ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5E — CORRECTED LF CONFLICT ANALYSIS")
print("=" * 70)

LF_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_initial.csv"
)

df = pd.read_csv(LF_FILE)

# ------------------------------------------------------------
# LABELS
# ------------------------------------------------------------

ABSTAIN = -1

CLASSES = {
    "GAS": 1,
    "INDUSTRIAL": 0,
    "MINING": 3,
    "AGRICULTURE": 2,
    "WILDFIRE": 4
}

# ------------------------------------------------------------
# LF GROUPS
# ------------------------------------------------------------

lf_groups = {
    "GAS": [
        "LF_GAS_ACTIVE_FLARE_1KM",
        "LF_GAS_ACTIVE_FLARE_2KM",
        "LF_GAS_ACTIVE_FLARE_5KM",
        "LF_GAS_REPEATED_FLARE"
    ],

    "INDUSTRIAL": [
        "LF_INDUSTRIAL_MULTI_SOURCE_500M",
        "LF_INDUSTRIAL_MULTI_SOURCE_1KM",
        "LF_INDUSTRIAL_STEEL",
        "LF_INDUSTRIAL_CEMENT",
        "LF_INDUSTRIAL_POWER",
        "LF_INDUSTRIAL_FERTILIZER",
        "LF_INDUSTRIAL_REFINERY",
        "LF_INDUSTRIAL_SCORE"
    ],

    "MINING": [
        "LF_MINING_COAL_500M",
        "LF_MINING_COAL_RECURRENT",
        "LF_MINING_STRONG",
        "LF_MINING_COAL_1KM"
    ],

    "AGRICULTURE": [
        "LF_AGRI_CROP_70",
        "LF_AGRI_CROP_OSM",
        "LF_AGRI_SCORE",
        "LF_AGRI_RECURRENT"
    ],

    "WILDFIRE": [
        "LF_WILDFIRE_STRONG_NATURAL",
        "LF_WILDFIRE_FOREST_CONTEXT",
        "LF_WILDFIRE_NATURAL_PROB",
        "LF_WILDFIRE_VEGETATION_COMBINED"
    ]
}

# ------------------------------------------------------------
# CALCULATE VOTES PER CLASS
# ------------------------------------------------------------

class_vote_counts = pd.DataFrame(index=df.index)

for class_name, cols in lf_groups.items():

    available_cols = [
        c for c in cols
        if c in df.columns
    ]

    class_vote_counts[class_name] = (
        (df[available_cols] != ABSTAIN)
        .sum(axis=1)
    )

    df[f"{class_name.lower()}_lf_votes"] = (
        class_vote_counts[class_name]
    )

# ------------------------------------------------------------
# ACTIVE CLASS FLAGS
# ------------------------------------------------------------

active_classes = pd.DataFrame(index=df.index)

for class_name in lf_groups:

    active_classes[class_name] = (
        class_vote_counts[class_name] > 0
    )

df["lf_active_class_count"] = (
    active_classes.sum(axis=1)
)

# ------------------------------------------------------------
# CORRECT CLASS COMBINATION
# ------------------------------------------------------------

def get_active_classes(row):

    active = [
        class_name
        for class_name, is_active
        in row.items()
        if bool(is_active)
    ]

    if len(active) == 0:
        return "NONE"

    return ",".join(active)


df["lf_active_classes"] = (
    active_classes
    .apply(get_active_classes, axis=1)
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NUMBER OF ACTIVE SOURCE CLASSES")
print("=" * 70)

summary = (
    df["lf_active_class_count"]
    .value_counts()
    .sort_index()
)

for n_classes, n_events in summary.items():

    print(
        f"{n_classes} active classes: "
        f"{n_events:,} events "
        f"({n_events / len(df) * 100:.2f}%)"
    )

# ------------------------------------------------------------
# ACTUAL CLASS COMBINATIONS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ACTUAL CLASS COMBINATIONS")
print("=" * 70)

combination_summary = (
    df["lf_active_classes"]
    .value_counts()
)

for combination, count in combination_summary.head(30).items():

    print(
        f"{combination:40} "
        f"{count:8,} "
        f"({count / len(df) * 100:6.2f}%)"
    )

# ------------------------------------------------------------
# PAIRWISE CONFLICT MATRIX
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CORRECTED CLASS CONFLICT MATRIX")
print("=" * 70)

classes = list(lf_groups.keys())

conflict_matrix = pd.DataFrame(
    0,
    index=classes,
    columns=classes,
    dtype=int
)

for i, c1 in enumerate(classes):

    for j, c2 in enumerate(classes):

        if i >= j:
            continue

        conflict = (
            active_classes[c1]
            &
            active_classes[c2]
        ).sum()

        conflict_matrix.loc[c1, c2] = conflict
        conflict_matrix.loc[c2, c1] = conflict

print(conflict_matrix)

# ------------------------------------------------------------
# PAIRWISE DETAILS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PAIRWISE CONFLICT DETAILS")
print("=" * 70)

for i in range(len(classes)):

    for j in range(i + 1, len(classes)):

        c1 = classes[i]
        c2 = classes[j]

        conflict = (
            active_classes[c1]
            &
            active_classes[c2]
        ).sum()

        print(
            f"{c1:12} <-> {c2:12}: "
            f"{conflict:,} "
            f"({conflict / len(df) * 100:.3f}%)"
        )

# ------------------------------------------------------------
# SINGLE-CLASS EVENTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SINGLE-CLASS LF EVENTS")
print("=" * 70)

single_class = df[
    df["lf_active_class_count"] == 1
]

for class_name in classes:

    count = (
        single_class["lf_active_classes"]
        .eq(class_name)
        .sum()
    )

    print(
        f"{class_name:12}: "
        f"{count:,}"
    )

# ------------------------------------------------------------
# NO-LF EVENTS
# ------------------------------------------------------------

no_lf = (
    df["lf_active_class_count"] == 0
).sum()

print("\n" + "=" * 70)
print("NO-LF EVENTS")
print("=" * 70)

print(
    f"{no_lf:,} "
    f"({no_lf / len(df) * 100:.2f}%)"
)

# ------------------------------------------------------------
# SAVE CORRECTED FILE
# ------------------------------------------------------------

OUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_lf_conflict_analysis_corrected.csv"
)

df.to_csv(
    OUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5E COMPLETE")
print("=" * 70)

print(f"\nSaved:\n{OUT_FILE}")

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:20:16.992004Z","iopub.execute_input":"2026-09-09T12:20:16.992636Z","iopub.status.idle":"2026-09-09T12:21:17.281017Z","shell.execute_reply.started":"2026-09-09T12:20:16.992589Z","shell.execute_reply":"2026-09-09T12:21:17.278409Z"}}
# ============================================================
# STEP 5F — REFINED LABELING FUNCTIONS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5F — REFINED LABELING FUNCTIONS")
print("=" * 70)

ABSTAIN = -1

INDUSTRIAL = 0
GAS = 1
AGRICULTURE = 2
MINING = 3
WILDFIRE = 4

INPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

df = pd.read_csv(INPUT_FILE)

print(f"\nEvents: {len(df):,}")


# ============================================================
# HELPERS
# ============================================================

def B(col):
    """
    Safe boolean feature.
    Missing values -> False.
    """
    if col not in df.columns:
        print(f"WARNING: missing column -> {col}")
        return pd.Series(False, index=df.index)

    return df[col].fillna(False).astype(bool)


def N(col):
    """
    Safe numeric feature.
    """
    if col not in df.columns:
        print(f"WARNING: missing column -> {col}")
        return pd.Series(np.nan, index=df.index)

    return pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ============================================================
# COMMON CONTRASTIVE EVIDENCE
# ============================================================

# Strong agriculture evidence
strong_agriculture = (
    B("agri_dw_crop_70_max")
    |
    (
        N("agriculture_evidence_score_max") >= 2
    )
)

# Strong industrial evidence
strong_industrial = (
    (
        N("event_max_industrial_evidence_score") >= 2
    )
    |
    B("event_any_industrial_multi_source_500m")
)

# Strong mining evidence
strong_mining = (
    B("event_any_coal_mine_within_500m")
    |
    B("event_any_mining_evidence_close_500m")
)

# Any strong human/source-specific evidence
strong_non_wildfire = (
    strong_agriculture
    |
    strong_industrial
    |
    strong_mining
)


# ============================================================
# GAS
# ============================================================

print("\nGAS LFs")

LF_GAS_1KM = np.where(
    B("has_gas_flare_within_1km"),
    GAS,
    ABSTAIN
)

LF_GAS_2KM = np.where(
    B("has_gas_flare_within_2km"),
    GAS,
    ABSTAIN
)

LF_GAS_REPEATED = np.where(
    (
        B("has_gas_flare_within_2km")
        &
        (N("event_detection_count") >= 2)
    ),
    GAS,
    ABSTAIN
)


# ============================================================
# INDUSTRIAL
# ============================================================

print("INDUSTRIAL LFs")

LF_IND_MULTI_500 = np.where(
    B("event_any_industrial_multi_source_500m"),
    INDUSTRIAL,
    ABSTAIN
)

LF_IND_SCORE = np.where(
    N("event_max_industrial_evidence_score") >= 2,
    INDUSTRIAL,
    ABSTAIN
)

LF_IND_STEEL = np.where(
    (
        B("event_any_steel_within_500m")
        &
        (
            B("event_any_steel_strong_7d")
            |
            B("event_any_steel_strong_30d")
        )
    ),
    INDUSTRIAL,
    ABSTAIN
)

LF_IND_CEMENT = np.where(
    (
        B("event_any_cement_within_500m")
        &
        (
            B("event_any_cement_strong_7d")
            |
            B("event_any_cement_strong_30d")
        )
    ),
    INDUSTRIAL,
    ABSTAIN
)

LF_IND_POWER = np.where(
    (
        B("event_any_wri_power_within_500m")
        &
        (
            B("event_any_wri_power_strong_7d")
            |
            B("event_any_wri_power_strong_30d")
        )
    ),
    INDUSTRIAL,
    ABSTAIN
)


# ============================================================
# MINING
# ============================================================

print("MINING LFs")

LF_MINE_500 = np.where(
    B("event_any_coal_mine_within_500m"),
    MINING,
    ABSTAIN
)

LF_MINE_RECURRENT = np.where(
    (
        B("event_any_coal_mine_within_1km")
        &
        (
            B("event_any_mining_evidence_close_1km_recurrent_7d")
            |
            B("event_any_mining_evidence_close_1km_recurrent_30d")
        )
    ),
    MINING,
    ABSTAIN
)

LF_MINE_STRONG = np.where(
    (
        B("event_any_mining_evidence_close_500m")
        &
        (
            B("event_any_mining_evidence_close_500m_recurrent_7d")
            |
            B("event_any_mining_evidence_close_500m_recurrent_30d")
        )
    ),
    MINING,
    ABSTAIN
)


# ============================================================
# AGRICULTURE
# ============================================================

print("AGRICULTURE LFs")

# Very strong crop probability.
LF_AGRI_CROP_70 = np.where(
    B("agri_dw_crop_70_max"),
    AGRICULTURE,
    ABSTAIN
)

# Crop + agricultural OSM context.
LF_AGRI_CROP_CONTEXT = np.where(
    (
        B("agri_dw_crop_50_max")
        &
        (
            B("agri_osm_within_500m_max")
            |
            B("agri_osm_within_1km_max")
        )
    ),
    AGRICULTURE,
    ABSTAIN
)

# Strong agriculture evidence score.
LF_AGRI_SCORE = np.where(
    N("agriculture_evidence_score_max") >= 2,
    AGRICULTURE,
    ABSTAIN
)

# Repeated agricultural activity.
LF_AGRI_RECURRENT = np.where(
    (
        (
            B("agri_recurrent_7d_max")
            |
            B("agri_recurrent_30d_max")
        )
        &
        (
            B("agri_osm_within_1km_max")
            |
            B("agri_dw_landcover_crop_max")
        )
    ),
    AGRICULTURE,
    ABSTAIN
)


# ============================================================
# WILDFIRE
# ============================================================

print("WILDFIRE LFs")

# ------------------------------------------------------------
# WF 1:
# Strong natural vegetation evidence PLUS forest context.
# ------------------------------------------------------------

LF_WF_STRONG_FOREST = np.where(
    (
        B("natural_vegetation_strong_max")
        &
        B("near_forest_1km_max")
        &
        (~strong_agriculture)
        &
        (~strong_industrial)
        &
        (~strong_mining)
    ),
    WILDFIRE,
    ABSTAIN
)

# ------------------------------------------------------------
# WF 2:
# High natural vegetation probability + forest/natural context.
# ------------------------------------------------------------

LF_WF_HIGH_NATURAL = np.where(
    (
        (
            N("event_mean_dw_natural_vegetation_prob")
            >= 0.70
        )
        &
        (
            B("near_forest_1km_max")
            |
            B("near_natural_vegetation_1km_max")
        )
        &
        (~strong_agriculture)
        &
        (~strong_industrial)
        &
        (~strong_mining)
    ),
    WILDFIRE,
    ABSTAIN
)

# ------------------------------------------------------------
# WF 3:
# Natural vegetation context + moderate vegetation evidence.
# Requires BOTH, not just one.
# ------------------------------------------------------------

LF_WF_VEGETATION_CONTEXT = np.where(
    (
        B("natural_landcover_context_max")
        &
        B("natural_vegetation_moderate_max")
        &
        (
            B("near_forest_1km_max")
            |
            B("near_natural_vegetation_1km_max")
        )
        &
        (~strong_agriculture)
        &
        (~strong_industrial)
        &
        (~strong_mining)
    ),
    WILDFIRE,
    ABSTAIN
)

# ------------------------------------------------------------
# WF 4:
# Persistent event in natural vegetation.
#
# This uses event behavior as additional evidence.
# ------------------------------------------------------------

LF_WF_PERSISTENT_NATURAL = np.where(
    (
        (
            N("event_active_days") >= 2
        )
        |
        (
            N("event_detection_count") >= 3
        )
    )
    &
    (
        N("event_mean_dw_natural_vegetation_prob")
        >= 0.60
    )
    &
    (
        B("near_forest_1km_max")
        |
        B("near_natural_vegetation_1km_max")
    )
    &
    (~strong_agriculture)
    &
    (~strong_industrial)
    &
    (~strong_mining),
    WILDFIRE,
    ABSTAIN
)


# ============================================================
# STORE REFINED LFs
# ============================================================

lf_columns = {

    "LF_GAS_1KM":
        LF_GAS_1KM,

    "LF_GAS_2KM":
        LF_GAS_2KM,

    "LF_GAS_REPEATED":
        LF_GAS_REPEATED,

    "LF_IND_MULTI_500":
        LF_IND_MULTI_500,

    "LF_IND_SCORE":
        LF_IND_SCORE,

    "LF_IND_STEEL":
        LF_IND_STEEL,

    "LF_IND_CEMENT":
        LF_IND_CEMENT,

    "LF_IND_POWER":
        LF_IND_POWER,

    "LF_MINE_500":
        LF_MINE_500,

    "LF_MINE_RECURRENT":
        LF_MINE_RECURRENT,

    "LF_MINE_STRONG":
        LF_MINE_STRONG,

    "LF_AGRI_CROP_70":
        LF_AGRI_CROP_70,

    "LF_AGRI_CROP_CONTEXT":
        LF_AGRI_CROP_CONTEXT,

    "LF_AGRI_SCORE":
        LF_AGRI_SCORE,

    "LF_AGRI_RECURRENT":
        LF_AGRI_RECURRENT,

    "LF_WF_STRONG_FOREST":
        LF_WF_STRONG_FOREST,

    "LF_WF_HIGH_NATURAL":
        LF_WF_HIGH_NATURAL,

    "LF_WF_VEGETATION_CONTEXT":
        LF_WF_VEGETATION_CONTEXT,

    "LF_WF_PERSISTENT_NATURAL":
        LF_WF_PERSISTENT_NATURAL
}

for name, values in lf_columns.items():

    df[name] = values.astype(np.int8)


# ============================================================
# COVERAGE
# ============================================================

print("\n" + "=" * 70)
print("REFINED LF COVERAGE")
print("=" * 70)

coverage = []

for name in lf_columns:

    covered = (
        df[name] != ABSTAIN
    ).sum()

    coverage.append({
        "LF": name,
        "covered_events": int(covered),
        "coverage_pct": (
            covered / len(df) * 100
        )
    })

coverage_df = pd.DataFrame(coverage)

display(
    coverage_df
    .sort_values(
        "coverage_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# CLASS COVERAGE
# ============================================================

print("\n" + "=" * 70)
print("COVERAGE BY CLASS")
print("=" * 70)

class_lfs = {
    "GAS": [
        "LF_GAS_1KM",
        "LF_GAS_2KM",
        "LF_GAS_REPEATED"
    ],

    "INDUSTRIAL": [
        "LF_IND_MULTI_500",
        "LF_IND_SCORE",
        "LF_IND_STEEL",
        "LF_IND_CEMENT",
        "LF_IND_POWER"
    ],

    "MINING": [
        "LF_MINE_500",
        "LF_MINE_RECURRENT",
        "LF_MINE_STRONG"
    ],

    "AGRICULTURE": [
        "LF_AGRI_CROP_70",
        "LF_AGRI_CROP_CONTEXT",
        "LF_AGRI_SCORE",
        "LF_AGRI_RECURRENT"
    ],

    "WILDFIRE": [
        "LF_WF_STRONG_FOREST",
        "LF_WF_HIGH_NATURAL",
        "LF_WF_VEGETATION_CONTEXT",
        "LF_WF_PERSISTENT_NATURAL"
    ]
}

for class_name, cols in class_lfs.items():

    active = (
        (df[cols] != ABSTAIN)
        .any(axis=1)
    )

    count = active.sum()

    print(
        f"{class_name:12}: "
        f"{count:,} "
        f"({count / len(df) * 100:.2f}%)"
    )


# ============================================================
# SAVE
# ============================================================

OUTPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_refined.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5F COMPLETE")
print("=" * 70)

print(f"\nSaved:\n{OUTPUT_FILE}")

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:21:17.283463Z","iopub.execute_input":"2026-09-09T12:21:17.283863Z","iopub.status.idle":"2026-09-09T12:21:26.220147Z","shell.execute_reply.started":"2026-09-09T12:21:17.283832Z","shell.execute_reply":"2026-09-09T12:21:26.218273Z"}}
# ============================================================
# STEP 5G — REFINED LF AGREEMENT AND CORRELATION ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5G — REFINED LF AGREEMENT & CORRELATION")
print("=" * 70)

INPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_refined.csv"
)

df = pd.read_csv(INPUT_FILE)

ABSTAIN = -1

lf_cols = [
    "LF_GAS_1KM",
    "LF_GAS_2KM",
    "LF_GAS_REPEATED",

    "LF_IND_MULTI_500",
    "LF_IND_SCORE",
    "LF_IND_STEEL",
    "LF_IND_CEMENT",
    "LF_IND_POWER",

    "LF_MINE_500",
    "LF_MINE_RECURRENT",
    "LF_MINE_STRONG",

    "LF_AGRI_CROP_70",
    "LF_AGRI_CROP_CONTEXT",
    "LF_AGRI_SCORE",
    "LF_AGRI_RECURRENT",

    "LF_WF_STRONG_FOREST",
    "LF_WF_HIGH_NATURAL",
    "LF_WF_VEGETATION_CONTEXT",
    "LF_WF_PERSISTENT_NATURAL"
]

print(f"\nEvents: {len(df):,}")
print(f"LFs: {len(lf_cols)}")


# ============================================================
# 1. INDIVIDUAL LF COVERAGE
# ============================================================

print("\n" + "=" * 70)
print("INDIVIDUAL LF COVERAGE")
print("=" * 70)

coverage_rows = []

for lf in lf_cols:

    covered = (
        df[lf] != ABSTAIN
    ).sum()

    coverage_rows.append({
        "LF": lf,
        "covered": int(covered),
        "coverage_pct": covered / len(df) * 100
    })

coverage_df = (
    pd.DataFrame(coverage_rows)
    .sort_values(
        "coverage_pct",
        ascending=False
    )
)

display(coverage_df)


# ============================================================
# 2. NUMBER OF LFs VOTING PER EVENT
# ============================================================

lf_active_count = (
    (df[lf_cols] != ABSTAIN)
    .sum(axis=1)
)

df["lf_active_count"] = lf_active_count

print("\n" + "=" * 70)
print("NUMBER OF ACTIVE LFs PER EVENT")
print("=" * 70)

active_summary = (
    lf_active_count
    .value_counts()
    .sort_index()
)

for n_lfs, count in active_summary.items():

    print(
        f"{n_lfs:2} LFs: "
        f"{count:,} "
        f"({count / len(df) * 100:.2f}%)"
    )


# ============================================================
# 3. CLASS-SPECIFIC LF AGREEMENT
# ============================================================

class_groups = {

    "GAS": [
        "LF_GAS_1KM",
        "LF_GAS_2KM",
        "LF_GAS_REPEATED"
    ],

    "INDUSTRIAL": [
        "LF_IND_MULTI_500",
        "LF_IND_SCORE",
        "LF_IND_STEEL",
        "LF_IND_CEMENT",
        "LF_IND_POWER"
    ],

    "MINING": [
        "LF_MINE_500",
        "LF_MINE_RECURRENT",
        "LF_MINE_STRONG"
    ],

    "AGRICULTURE": [
        "LF_AGRI_CROP_70",
        "LF_AGRI_CROP_CONTEXT",
        "LF_AGRI_SCORE",
        "LF_AGRI_RECURRENT"
    ],

    "WILDFIRE": [
        "LF_WF_STRONG_FOREST",
        "LF_WF_HIGH_NATURAL",
        "LF_WF_VEGETATION_CONTEXT",
        "LF_WF_PERSISTENT_NATURAL"
    ]
}

print("\n" + "=" * 70)
print("WITHIN-CLASS LF AGREEMENT")
print("=" * 70)

for class_name, cols in class_groups.items():

    votes = (
        (df[cols] != ABSTAIN)
        .sum(axis=1)
    )

    active = votes > 0

    print(f"\n{class_name}")

    print(
        f"  Any LF: "
        f"{active.sum():,}"
    )

    for k in range(1, len(cols) + 1):

        count = (
            votes >= k
        ).sum()

        print(
            f"  >= {k} agreeing LFs: "
            f"{count:,}"
        )


# ============================================================
# 4. PAIRWISE LF OVERLAP
# ============================================================

print("\n" + "=" * 70)
print("PAIRWISE LF OVERLAP")
print("=" * 70)

overlap_rows = []

for i in range(len(lf_cols)):

    for j in range(i + 1, len(lf_cols)):

        lf1 = lf_cols[i]
        lf2 = lf_cols[j]

        active1 = df[lf1] != ABSTAIN
        active2 = df[lf2] != ABSTAIN

        both = (
            active1
            &
            active2
        ).sum()

        either = (
            active1
            |
            active2
        ).sum()

        if either > 0:

            jaccard = both / either

        else:

            jaccard = 0

        overlap_rows.append({
            "LF1": lf1,
            "LF2": lf2,
            "both_active": int(both),
            "either_active": int(either),
            "jaccard": jaccard
        })

overlap_df = (
    pd.DataFrame(overlap_rows)
    .sort_values(
        "jaccard",
        ascending=False
    )
)

print("\nTop 30 most overlapping LF pairs:")

display(
    overlap_df.head(30)
)


# ============================================================
# 5. SAME-CLASS CORRELATION
# ============================================================

print("\n" + "=" * 70)
print("WITHIN-CLASS LF CORRELATIONS")
print("=" * 70)

for class_name, cols in class_groups.items():

    # Convert abstain to 0, vote to 1
    binary = (
        df[cols] != ABSTAIN
    ).astype(int)

    if len(cols) >= 2:

        corr = binary.corr()

        print(f"\n{class_name}")
        display(corr)


# ============================================================
# 6. WILDFIRE AGREEMENT DETAILS
# ============================================================

print("\n" + "=" * 70)
print("WILDFIRE AGREEMENT DETAILS")
print("=" * 70)

wf_cols = class_groups["WILDFIRE"]

wf_votes = (
    (df[wf_cols] != ABSTAIN)
    .sum(axis=1)
)

for k in range(1, 5):

    count = (
        wf_votes >= k
    ).sum()

    print(
        f">= {k} wildfire LFs agree: "
        f"{count:,} "
        f"({count / len(df) * 100:.2f}%)"
    )


# ============================================================
# 7. CROSS-CLASS CONFLICTS
# ============================================================

print("\n" + "=" * 70)
print("CROSS-CLASS CONFLICTS")
print("=" * 70)

class_active = {}

for class_name, cols in class_groups.items():

    class_active[class_name] = (
        (df[cols] != ABSTAIN)
        .any(axis=1)
    )

classes = list(class_groups.keys())

for i in range(len(classes)):

    for j in range(i + 1, len(classes)):

        c1 = classes[i]
        c2 = classes[j]

        conflict = (
            class_active[c1]
            &
            class_active[c2]
        ).sum()

        print(
            f"{c1:12} + {c2:12}: "
            f"{conflict:,}"
        )


# ============================================================
# 8. SAVE ANALYSIS
# ============================================================

OVERLAP_FILE = (
    "/kaggle/working/"
    "firms_2025_refined_lf_overlap_analysis.csv"
)

overlap_df.to_csv(
    OVERLAP_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5G COMPLETE")
print("=" * 70)

print(
    f"\nSaved overlap analysis:\n{OVERLAP_FILE}"
)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:21:26.222917Z","iopub.execute_input":"2026-09-09T12:21:26.224911Z","iopub.status.idle":"2026-09-09T12:21:34.531678Z","shell.execute_reply.started":"2026-09-09T12:21:26.224828Z","shell.execute_reply":"2026-09-09T12:21:34.530275Z"}}
# ============================================================
# STEP 5H — SOURCE BEHAVIOR PROFILE
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5H — SOURCE BEHAVIOR PROFILE")
print("=" * 70)

FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_refined.csv"
)

df = pd.read_csv(FILE)

ABSTAIN = -1

# ------------------------------------------------------------
# SOURCE LF GROUPS
# ------------------------------------------------------------

groups = {

    "GAS": [
        "LF_GAS_1KM",
        "LF_GAS_2KM",
        "LF_GAS_REPEATED"
    ],

    "INDUSTRIAL": [
        "LF_IND_MULTI_500",
        "LF_IND_SCORE",
        "LF_IND_STEEL",
        "LF_IND_CEMENT",
        "LF_IND_POWER"
    ],

    "MINING": [
        "LF_MINE_500",
        "LF_MINE_RECURRENT",
        "LF_MINE_STRONG"
    ],

    "AGRICULTURE": [
        "LF_AGRI_CROP_70",
        "LF_AGRI_CROP_CONTEXT",
        "LF_AGRI_SCORE",
        "LF_AGRI_RECURRENT"
    ],

    "WILDFIRE": [
        "LF_WF_STRONG_FOREST",
        "LF_WF_HIGH_NATURAL",
        "LF_WF_VEGETATION_CONTEXT",
        "LF_WF_PERSISTENT_NATURAL"
    ]
}

# ------------------------------------------------------------
# BEHAVIOR FEATURES
# ------------------------------------------------------------

behavior_features = [
    "event_detection_count",
    "event_active_days",
    "event_duration_hours",
    "event_duration_days",
    "event_spatial_extent_km",
    "event_spatial_extent_per_active_day_km",
    "event_detections_per_active_day",
    "event_detections_per_hour",
    "event_active_months",
    "event_monthly_concentration",
    "event_max_detections_one_day",
    "event_mean_detections_active_day",
    "event_median_detections_active_day",
    "event_mean_detection_gap_hours",
    "event_max_detection_gap_hours",
    "event_mean_active_day_gap",
    "event_max_active_day_gap",
    "event_frp_cv",
    "event_mean_frp",
    "event_max_frp"
]

behavior_features = [
    c for c in behavior_features
    if c in df.columns
]

print(
    f"\nBehavior features available: "
    f"{len(behavior_features)}"
)

# ------------------------------------------------------------
# CREATE SOURCE-SUPPORTED MASKS
# ------------------------------------------------------------

source_masks = {}

for source, lfs in groups.items():

    source_masks[source] = (
        (df[lfs] != ABSTAIN)
        .any(axis=1)
    )

# ------------------------------------------------------------
# PROFILE EACH SOURCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MEDIAN BEHAVIOR BY LF-SUPPORTED SOURCE")
print("=" * 70)

profile_rows = []

for source, mask in source_masks.items():

    subset = df.loc[
        mask,
        behavior_features
    ]

    print(
        f"\n{source}: "
        f"{len(subset):,} supported events"
    )

    for feature in behavior_features:

        values = pd.to_numeric(
            subset[feature],
            errors="coerce"
        ).dropna()

        if len(values) == 0:
            continue

        profile_rows.append({
            "source": source,
            "feature": feature,
            "count": len(values),
            "median": values.median(),
            "mean": values.mean(),
            "p25": values.quantile(0.25),
            "p75": values.quantile(0.75),
            "p90": values.quantile(0.90)
        })

profile_df = pd.DataFrame(profile_rows)

# ------------------------------------------------------------
# DISPLAY IMPORTANT FEATURES
# ------------------------------------------------------------

important_features = [
    "event_detection_count",
    "event_active_days",
    "event_duration_days",
    "event_spatial_extent_km",
    "event_spatial_extent_per_active_day_km",
    "event_active_months",
    "event_monthly_concentration",
    "event_mean_detection_gap_hours",
    "event_mean_active_day_gap",
    "event_mean_frp",
    "event_max_frp"
]

important_features = [
    x for x in important_features
    if x in profile_df["feature"].unique()
]

for feature in important_features:

    print("\n" + "-" * 70)
    print(feature)

    display(
        profile_df[
            profile_df["feature"] == feature
        ][
            [
                "source",
                "count",
                "median",
                "mean",
                "p25",
                "p75",
                "p90"
            ]
        ]
        .sort_values(
            "median"
        )
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# SAVE PROFILE
# ------------------------------------------------------------

PROFILE_FILE = (
    "/kaggle/working/"
    "firms_2025_source_behavior_profile.csv"
)

profile_df.to_csv(
    PROFILE_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5H COMPLETE")
print("=" * 70)

print(
    f"\nSaved:\n{PROFILE_FILE}"
)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:21:34.533067Z","iopub.execute_input":"2026-09-09T12:21:34.533344Z","iopub.status.idle":"2026-09-09T12:21:42.396158Z","shell.execute_reply.started":"2026-09-09T12:21:34.533323Z","shell.execute_reply":"2026-09-09T12:21:42.395041Z"}}
# ============================================================
# STEP 5I — GLOBAL EVENT BEHAVIOR DISTRIBUTIONS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5I — GLOBAL EVENT BEHAVIOR DISTRIBUTIONS")
print("=" * 70)

FILE = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

df = pd.read_csv(FILE)

print(f"\nTotal events: {len(df):,}")

# ------------------------------------------------------------
# FEATURES
# ------------------------------------------------------------

features = [
    "event_detection_count",
    "event_active_days",
    "event_duration_hours",
    "event_duration_days",
    "event_spatial_extent_km",
    "event_spatial_extent_per_active_day_km",
    "event_detections_per_active_day",
    "event_detections_per_hour",
    "event_active_months",
    "event_monthly_concentration",
    "event_max_detections_one_day",
    "event_mean_detections_active_day",
    "event_median_detections_active_day",
    "event_mean_detection_gap_hours",
    "event_max_detection_gap_hours",
    "event_mean_active_day_gap",
    "event_max_active_day_gap",
    "event_mean_frp",
    "event_max_frp",
    "event_frp_cv"
]

features = [
    c for c in features
    if c in df.columns
]

# ------------------------------------------------------------
# GLOBAL DISTRIBUTIONS
# ------------------------------------------------------------

rows = []

for feature in features:

    values = pd.to_numeric(
        df[feature],
        errors="coerce"
    ).dropna()

    rows.append({
        "feature": feature,
        "count": len(values),
        "missing": df[feature].isna().sum(),
        "min": values.min(),
        "p01": values.quantile(0.01),
        "p05": values.quantile(0.05),
        "p10": values.quantile(0.10),
        "p25": values.quantile(0.25),
        "median": values.median(),
        "p75": values.quantile(0.75),
        "p90": values.quantile(0.90),
        "p95": values.quantile(0.95),
        "p99": values.quantile(0.99),
        "max": values.max(),
        "mean": values.mean()
    })

distribution_df = pd.DataFrame(rows)

display(
    distribution_df
)

# ------------------------------------------------------------
# IMPORTANT THRESHOLD COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EVENT BEHAVIOR THRESHOLD COUNTS")
print("=" * 70)

threshold_tests = {

    "detections >= 2":
        df["event_detection_count"] >= 2,

    "detections >= 3":
        df["event_detection_count"] >= 3,

    "detections >= 5":
        df["event_detection_count"] >= 5,

    "detections >= 10":
        df["event_detection_count"] >= 10,

    "active_days >= 2":
        df["event_active_days"] >= 2,

    "active_days >= 3":
        df["event_active_days"] >= 3,

    "active_days >= 5":
        df["event_active_days"] >= 5,

    "duration >= 1 day":
        df["event_duration_days"] >= 1,

    "duration >= 3 days":
        df["event_duration_days"] >= 3,

    "duration >= 7 days":
        df["event_duration_days"] >= 7,

    "spatial extent >= 0.5 km":
        df["event_spatial_extent_km"] >= 0.5,

    "spatial extent >= 1 km":
        df["event_spatial_extent_km"] >= 1,

    "spatial extent >= 2 km":
        df["event_spatial_extent_km"] >= 2,

    "spatial extent >= 5 km":
        df["event_spatial_extent_km"] >= 5,

    "active months >= 2":
        df["event_active_months"] >= 2,

    "active months >= 3":
        df["event_active_months"] >= 3
}

threshold_rows = []

for name, mask in threshold_tests.items():

    count = int(mask.sum())

    threshold_rows.append({
        "condition": name,
        "events": count,
        "percentage": count / len(df) * 100
    })

threshold_df = pd.DataFrame(
    threshold_rows
)

display(
    threshold_df
)

# ------------------------------------------------------------
# PEAK MONTH DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EVENT PEAK MONTH DISTRIBUTION")
print("=" * 70)

peak_month = (
    df["event_peak_month"]
    .value_counts()
    .sort_index()
)

for month, count in peak_month.items():

    print(
        f"Month {int(month):2d}: "
        f"{count:,} "
        f"({count / len(df) * 100:.2f}%)"
    )

# ------------------------------------------------------------
# DETECTION COUNT DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DETECTION COUNT DISTRIBUTION")
print("=" * 70)

detection_counts = (
    df["event_detection_count"]
    .value_counts()
    .sort_index()
)

for count, n_events in detection_counts.head(20).items():

    print(
        f"{int(count):3d} detections: "
        f"{n_events:,}"
    )

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUT_FILE = (
    "/kaggle/working/"
    "firms_2025_global_event_behavior_distribution.csv"
)

distribution_df.to_csv(
    OUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5I COMPLETE")
print("=" * 70)

print(f"\nSaved:\n{OUT_FILE}")

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:21:42.398249Z","iopub.execute_input":"2026-09-09T12:21:42.398616Z","iopub.status.idle":"2026-09-09T12:22:34.772200Z","shell.execute_reply.started":"2026-09-09T12:21:42.398595Z","shell.execute_reply":"2026-09-09T12:22:34.770607Z"}}
# ============================================================
# STEP 5J — BEHAVIOR + SOURCE CONTEXT LFs
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5J — BEHAVIOR + SOURCE CONTEXT LFs")
print("=" * 70)

INPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_refined.csv"
)

df = pd.read_csv(INPUT_FILE)

ABSTAIN = -1

INDUSTRIAL = 0
GAS = 1
AGRICULTURE = 2
MINING = 3
WILDFIRE = 4

print(f"\nEvents: {len(df):,}")


# ============================================================
# HELPERS
# ============================================================

def B(col):
    if col not in df.columns:
        print(f"WARNING: missing column: {col}")
        return pd.Series(False, index=df.index)

    return df[col].fillna(False).astype(bool)


def N(col):
    if col not in df.columns:
        print(f"WARNING: missing column: {col}")
        return pd.Series(np.nan, index=df.index)

    return pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ============================================================
# EXISTING SOURCE-SPECIFIC EVIDENCE
# ============================================================

strong_agriculture = (
    B("agri_dw_crop_70_max")
    |
    (N("agriculture_evidence_score_max") >= 2)
)

strong_industrial = (
    (N("event_max_industrial_evidence_score") >= 2)
    |
    B("event_any_industrial_multi_source_500m")
)

strong_mining = (
    B("event_any_coal_mine_within_500m")
    |
    B("event_any_mining_evidence_close_500m")
)


# ============================================================
# 1. WILDFIRE — REPEATED NATURAL ACTIVITY
# ============================================================

# At least 2 active days + natural vegetation context.

LF_WF_REPEATED_NATURAL = np.where(
    (
        (N("event_active_days") >= 2)
        &
        (
            B("natural_landcover_context_max")
            |
            B("natural_vegetation_strong_max")
        )
        &
        (
            B("near_forest_1km_max")
            |
            B("near_natural_vegetation_1km_max")
        )
        &
        (~strong_agriculture)
        &
        (~strong_industrial)
        &
        (~strong_mining)
    ),
    WILDFIRE,
    ABSTAIN
)


# ============================================================
# 2. WILDFIRE — MULTIPLE DETECTIONS + NATURAL CONTEXT
# ============================================================

LF_WF_MULTIPLE_NATURAL = np.where(
    (
        (N("event_detection_count") >= 3)
        &
        (
            N("event_mean_dw_natural_vegetation_prob")
            >= 0.60
        )
        &
        (
            B("near_forest_1km_max")
            |
            B("near_natural_vegetation_1km_max")
        )
        &
        (~strong_agriculture)
        &
        (~strong_industrial)
        &
        (~strong_mining)
    ),
    WILDFIRE,
    ABSTAIN
)


# ============================================================
# 3. WILDFIRE — SPATIAL SPREAD + NATURAL CONTEXT
# ============================================================

LF_WF_SPATIAL_NATURAL = np.where(
    (
        (N("event_spatial_extent_km") >= 1.0)
        &
        (
            B("natural_landcover_context_max")
            |
            B("natural_vegetation_strong_max")
        )
        &
        (
            B("near_forest_1km_max")
            |
            B("near_natural_vegetation_1km_max")
        )
        &
        (~strong_agriculture)
        &
        (~strong_industrial)
        &
        (~strong_mining)
    ),
    WILDFIRE,
    ABSTAIN
)


# ============================================================
# 4. WILDFIRE — HIGH FRP + NATURAL CONTEXT
# ============================================================

LF_WF_HIGH_FRP_NATURAL = np.where(
    (
        (N("event_mean_frp") >= 8.0)
        &
        (
            N("event_mean_dw_natural_vegetation_prob")
            >= 0.60
        )
        &
        (
            B("near_forest_1km_max")
            |
            B("near_natural_vegetation_1km_max")
        )
        &
        (~strong_agriculture)
        &
        (~strong_industrial)
        &
        (~strong_mining)
    ),
    WILDFIRE,
    ABSTAIN
)


# ============================================================
# 5. AGRICULTURE — REPEATED ACTIVITY + CROP
# ============================================================

LF_AGRI_REPEATED_CROP = np.where(
    (
        (N("event_detection_count") >= 3)
        &
        (
            B("agri_dw_crop_50_max")
            |
            B("agri_dw_landcover_crop_max")
        )
        &
        (~strong_industrial)
        &
        (~strong_mining)
    ),
    AGRICULTURE,
    ABSTAIN
)


# ============================================================
# 6. AGRICULTURE — HIGH FRP + CROP CONTEXT
# ============================================================

LF_AGRI_FRP_CROP = np.where(
    (
        (N("event_mean_frp") >= 4.0)
        &
        (
            B("agri_dw_crop_50_max")
            |
            B("agri_dw_landcover_crop_max")
        )
        &
        (~strong_industrial)
        &
        (~strong_mining)
    ),
    AGRICULTURE,
    ABSTAIN
)


# ============================================================
# 7. INDUSTRIAL — REPEATED ACTIVITY + INDUSTRIAL CONTEXT
# ============================================================

LF_IND_REPEATED = np.where(
    (
        (N("event_detection_count") >= 3)
        &
        (
            B("event_any_industrial_multi_source_500m")
            |
            (N("event_max_industrial_evidence_score") >= 2)
        )
    ),
    INDUSTRIAL,
    ABSTAIN
)


# ============================================================
# 8. MINING — REPEATED ACTIVITY + MINE CONTEXT
# ============================================================

LF_MINE_REPEATED_BEHAVIOR = np.where(
    (
        (N("event_active_days") >= 2)
        &
        (
            B("event_any_coal_mine_within_1km")
            |
            B("event_any_mining_evidence_close_1km")
        )
    ),
    MINING,
    ABSTAIN
)


# ============================================================
# 9. GAS — REPEATED FLARE ACTIVITY
# ============================================================

LF_GAS_REPEATED_BEHAVIOR = np.where(
    (
        (N("event_detection_count") >= 2)
        &
        B("has_gas_flare_within_2km")
    ),
    GAS,
    ABSTAIN
)


# ============================================================
# STORE
# ============================================================

behavior_lfs = {

    "LF_WF_REPEATED_NATURAL":
        LF_WF_REPEATED_NATURAL,

    "LF_WF_MULTIPLE_NATURAL":
        LF_WF_MULTIPLE_NATURAL,

    "LF_WF_SPATIAL_NATURAL":
        LF_WF_SPATIAL_NATURAL,

    "LF_WF_HIGH_FRP_NATURAL":
        LF_WF_HIGH_FRP_NATURAL,

    "LF_AGRI_REPEATED_CROP":
        LF_AGRI_REPEATED_CROP,

    "LF_AGRI_FRP_CROP":
        LF_AGRI_FRP_CROP,

    "LF_IND_REPEATED":
        LF_IND_REPEATED,

    "LF_MINE_REPEATED_BEHAVIOR":
        LF_MINE_REPEATED_BEHAVIOR,

    "LF_GAS_REPEATED_BEHAVIOR":
        LF_GAS_REPEATED_BEHAVIOR
}

for name, values in behavior_lfs.items():

    df[name] = values.astype(np.int8)


# ============================================================
# COVERAGE
# ============================================================

print("\n" + "=" * 70)
print("BEHAVIOR LF COVERAGE")
print("=" * 70)

coverage_rows = []

for name in behavior_lfs:

    covered = (
        df[name] != ABSTAIN
    ).sum()

    coverage_rows.append({
        "LF": name,
        "covered_events": int(covered),
        "coverage_pct":
            covered / len(df) * 100
    })

coverage_df = (
    pd.DataFrame(coverage_rows)
    .sort_values(
        "coverage_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    coverage_df
)


# ============================================================
# VOTE COUNTS
# ============================================================

print("\n" + "=" * 70)
print("BEHAVIOR LF VOTE COUNTS")
print("=" * 70)

for name in behavior_lfs:

    votes = (
        df[name] != ABSTAIN
    ).sum()

    print(
        f"{name:35} "
        f"{votes:,}"
    )


# ============================================================
# SAVE
# ============================================================

OUTPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_behavior_refined.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5J COMPLETE")
print("=" * 70)

print(f"\nSaved:\n{OUTPUT_FILE}")

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:22:34.773896Z","iopub.execute_input":"2026-09-09T12:22:34.774233Z","iopub.status.idle":"2026-09-09T12:22:44.833628Z","shell.execute_reply.started":"2026-09-09T12:22:34.774200Z","shell.execute_reply":"2026-09-09T12:22:44.831837Z"}}
# ============================================================
# STEP 5K — FINAL CANDIDATE LF ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5K — FINAL CANDIDATE LF ANALYSIS")
print("=" * 70)

INPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_behavior_refined.csv"
)

df = pd.read_csv(INPUT_FILE)

ABSTAIN = -1

print(f"\nEvents: {len(df):,}")


# ============================================================
# FINAL CANDIDATE LF GROUPS
# ============================================================

groups = {

    "GAS": [
        "LF_GAS_1KM",
        "LF_GAS_2KM",
        "LF_GAS_REPEATED",
        "LF_GAS_REPEATED_BEHAVIOR"
    ],

    "INDUSTRIAL": [
        "LF_IND_MULTI_500",
        "LF_IND_SCORE",
        "LF_IND_STEEL",
        "LF_IND_CEMENT",
        "LF_IND_POWER",
        "LF_IND_REPEATED"
    ],

    "MINING": [
        "LF_MINE_500",
        "LF_MINE_RECURRENT",
        "LF_MINE_STRONG",
        "LF_MINE_REPEATED_BEHAVIOR"
    ],

    "AGRICULTURE": [
        "LF_AGRI_CROP_70",
        "LF_AGRI_CROP_CONTEXT",
        "LF_AGRI_SCORE",
        "LF_AGRI_RECURRENT",
        "LF_AGRI_REPEATED_CROP",
        "LF_AGRI_FRP_CROP"
    ],

    "WILDFIRE": [
        "LF_WF_STRONG_FOREST",
        "LF_WF_HIGH_NATURAL",
        "LF_WF_VEGETATION_CONTEXT",
        "LF_WF_PERSISTENT_NATURAL",
        "LF_WF_REPEATED_NATURAL",
        "LF_WF_MULTIPLE_NATURAL",
        "LF_WF_SPATIAL_NATURAL",
        "LF_WF_HIGH_FRP_NATURAL"
    ]
}


# ============================================================
# CHECK COLUMNS
# ============================================================

all_lfs = []

for source, lfs in groups.items():

    for lf in lfs:

        if lf in df.columns:
            all_lfs.append(lf)

        else:
            print(
                f"WARNING: missing LF -> {lf}"
            )

print(
    f"\nCandidate LFs available: "
    f"{len(all_lfs)}"
)


# ============================================================
# INDIVIDUAL COVERAGE
# ============================================================

print("\n" + "=" * 70)
print("INDIVIDUAL LF COVERAGE")
print("=" * 70)

coverage_rows = []

for source, lfs in groups.items():

    for lf in lfs:

        if lf not in df.columns:
            continue

        covered = (
            df[lf] != ABSTAIN
        ).sum()

        coverage_rows.append({
            "source": source,
            "LF": lf,
            "covered_events": int(covered),
            "coverage_pct":
                covered / len(df) * 100
        })

coverage_df = (
    pd.DataFrame(coverage_rows)
    .sort_values(
        "coverage_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    coverage_df
)


# ============================================================
# ACTIVE LF COUNT
# ============================================================

active_lfs = (
    (df[all_lfs] != ABSTAIN)
    .sum(axis=1)
)

print("\n" + "=" * 70)
print("NUMBER OF ACTIVE LFs PER EVENT")
print("=" * 70)

active_summary = (
    active_lfs
    .value_counts()
    .sort_index()
)

for count, n_events in active_summary.items():

    print(
        f"{count:2} active LFs: "
        f"{n_events:,} "
        f"({n_events / len(df) * 100:.2f}%)"
    )


# ============================================================
# SOURCE-LEVEL ACTIVE FLAGS
# ============================================================

source_active = pd.DataFrame(
    index=df.index
)

for source, lfs in groups.items():

    available = [
        lf for lf in lfs
        if lf in df.columns
    ]

    source_active[source] = (
        (df[available] != ABSTAIN)
        .any(axis=1)
    )

df["candidate_active_source_count"] = (
    source_active.sum(axis=1)
)


# ============================================================
# SOURCE COMBINATIONS
# ============================================================

def source_combination(row):

    active = [
        source
        for source in source_active.columns
        if row[source]
    ]

    if not active:
        return "NONE"

    return ",".join(active)


df["candidate_source_combination"] = (
    source_active.apply(
        source_combination,
        axis=1
    )
)

print("\n" + "=" * 70)
print("SOURCE COMBINATIONS")
print("=" * 70)

combos = (
    df["candidate_source_combination"]
    .value_counts()
)

for combo, count in combos.head(30).items():

    print(
        f"{combo:45} "
        f"{count:8,} "
        f"({count / len(df) * 100:6.2f}%)"
    )


# ============================================================
# WITHIN-CLASS AGREEMENT
# ============================================================

print("\n" + "=" * 70)
print("WITHIN-CLASS AGREEMENT")
print("=" * 70)

for source, lfs in groups.items():

    available = [
        lf for lf in lfs
        if lf in df.columns
    ]

    votes = (
        (df[available] != ABSTAIN)
        .sum(axis=1)
    )

    print(f"\n{source}")

    for k in range(
        1,
        len(available) + 1
    ):

        count = (
            votes >= k
        ).sum()

        print(
            f"  >= {k} agreeing LFs: "
            f"{count:,}"
        )


# ============================================================
# CROSS-SOURCE CONFLICTS
# ============================================================

print("\n" + "=" * 70)
print("CROSS-SOURCE CONFLICTS")
print("=" * 70)

sources = list(groups.keys())

conflict_rows = []

for i in range(len(sources)):

    for j in range(i + 1, len(sources)):

        s1 = sources[i]
        s2 = sources[j]

        conflict = (
            source_active[s1]
            &
            source_active[s2]
        ).sum()

        conflict_rows.append({
            "source_1": s1,
            "source_2": s2,
            "conflict_events": int(conflict),
            "conflict_pct":
                conflict / len(df) * 100
        })

        print(
            f"{s1:12} + {s2:12}: "
            f"{conflict:,}"
        )

conflict_df = pd.DataFrame(
    conflict_rows
)


# ============================================================
# PAIRWISE LF OVERLAP
# ============================================================

print("\n" + "=" * 70)
print("TOP LF OVERLAPS")
print("=" * 70)

overlap_rows = []

for i in range(len(all_lfs)):

    for j in range(i + 1, len(all_lfs)):

        lf1 = all_lfs[i]
        lf2 = all_lfs[j]

        a = (
            df[lf1] != ABSTAIN
        )

        b = (
            df[lf2] != ABSTAIN
        )

        both = (
            a & b
        ).sum()

        either = (
            a | b
        ).sum()

        jaccard = (
            both / either
            if either > 0
            else 0
        )

        overlap_rows.append({
            "LF1": lf1,
            "LF2": lf2,
            "both_active": int(both),
            "either_active": int(either),
            "jaccard": jaccard
        })

overlap_df = (
    pd.DataFrame(overlap_rows)
    .sort_values(
        "jaccard",
        ascending=False
    )
)

display(
    overlap_df.head(30)
)


# ============================================================
# SAVE ANALYSIS
# ============================================================

COVERAGE_FILE = (
    "/kaggle/working/"
    "firms_2025_final_candidate_lf_coverage.csv"
)

CONFLICT_FILE = (
    "/kaggle/working/"
    "firms_2025_final_candidate_lf_conflicts.csv"
)

OVERLAP_FILE = (
    "/kaggle/working/"
    "firms_2025_final_candidate_lf_overlap.csv"
)

coverage_df.to_csv(
    COVERAGE_FILE,
    index=False
)

conflict_df.to_csv(
    CONFLICT_FILE,
    index=False
)

overlap_df.to_csv(
    OVERLAP_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5K COMPLETE")
print("=" * 70)

print("\nSaved:")
print(COVERAGE_FILE)
print(CONFLICT_FILE)
print(OVERLAP_FILE)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:22:44.835013Z","iopub.execute_input":"2026-09-09T12:22:44.835337Z","iopub.status.idle":"2026-09-09T12:23:40.880513Z","shell.execute_reply.started":"2026-09-09T12:22:44.835311Z","shell.execute_reply":"2026-09-09T12:23:40.879384Z"}}
# ============================================================
# STEP 5L — FINAL LF CANDIDATE SET
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5L — FINAL LF CANDIDATE SET")
print("=" * 70)

INPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_behavior_refined.csv"
)

df = pd.read_csv(INPUT_FILE)

ABSTAIN = -1

# ------------------------------------------------------------
# FINAL LF GROUPS
# ------------------------------------------------------------

final_lf_groups = {

    "GAS": [
        "LF_GAS_1KM",
        "LF_GAS_2KM",
        "LF_GAS_REPEATED"
    ],

    "INDUSTRIAL": [
        "LF_IND_MULTI_500",
        "LF_IND_SCORE",
        "LF_IND_STEEL",
        "LF_IND_CEMENT",
        "LF_IND_POWER",
        "LF_IND_REPEATED"
    ],

    "MINING": [
        "LF_MINE_500",
        "LF_MINE_RECURRENT",
        "LF_MINE_STRONG"
    ],

    "AGRICULTURE": [
        "LF_AGRI_CROP_70",
        "LF_AGRI_CROP_CONTEXT",
        "LF_AGRI_SCORE",
        "LF_AGRI_RECURRENT"
    ],

    "WILDFIRE": [
        "LF_WF_STRONG_FOREST",
        "LF_WF_REPEATED_NATURAL"
    ]
}

# ------------------------------------------------------------
# CHECK LFs
# ------------------------------------------------------------

all_lfs = []

for source, lfs in final_lf_groups.items():

    print(f"\n{source}")

    for lf in lfs:

        if lf not in df.columns:
            raise ValueError(
                f"Missing LF: {lf}"
            )

        all_lfs.append(lf)

        print(f"  {lf}")

print(
    f"\nTotal final candidate LFs: "
    f"{len(all_lfs)}"
)

# ------------------------------------------------------------
# COVERAGE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL LF COVERAGE")
print("=" * 70)

coverage_rows = []

for source, lfs in final_lf_groups.items():

    for lf in lfs:

        covered = (
            df[lf] != ABSTAIN
        ).sum()

        coverage_rows.append({
            "source": source,
            "LF": lf,
            "covered_events": int(covered),
            "coverage_pct":
                covered / len(df) * 100
        })

coverage_df = pd.DataFrame(
    coverage_rows
)

display(
    coverage_df
    .sort_values(
        "coverage_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# ACTIVE LFs PER EVENT
# ------------------------------------------------------------

df["final_lf_count"] = (
    (df[all_lfs] != ABSTAIN)
    .sum(axis=1)
)

print("\n" + "=" * 70)
print("ACTIVE FINAL LFs PER EVENT")
print("=" * 70)

summary = (
    df["final_lf_count"]
    .value_counts()
    .sort_index()
)

for count, n_events in summary.items():

    print(
        f"{count:2} LFs: "
        f"{n_events:,} "
        f"({n_events / len(df) * 100:.2f}%)"
    )

# ------------------------------------------------------------
# SOURCE ACTIVE FLAGS
# ------------------------------------------------------------

source_active = pd.DataFrame(
    index=df.index
)

for source, lfs in final_lf_groups.items():

    source_active[source] = (
        (df[lfs] != ABSTAIN)
        .any(axis=1)
    )

# ------------------------------------------------------------
# SOURCE COMBINATIONS
# ------------------------------------------------------------

def make_combination(row):

    active = [
        source
        for source in source_active.columns
        if row[source]
    ]

    if not active:
        return "NONE"

    return ",".join(active)


df["final_source_combination"] = (
    source_active.apply(
        make_combination,
        axis=1
    )
)

print("\n" + "=" * 70)
print("FINAL SOURCE COMBINATIONS")
print("=" * 70)

combos = (
    df["final_source_combination"]
    .value_counts()
)

for combo, count in combos.head(25).items():

    print(
        f"{combo:40} "
        f"{count:8,} "
        f"({count / len(df) * 100:6.2f}%)"
    )

# ------------------------------------------------------------
# WITHIN-CLASS AGREEMENT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WITHIN-CLASS AGREEMENT")
print("=" * 70)

for source, lfs in final_lf_groups.items():

    votes = (
        (df[lfs] != ABSTAIN)
        .sum(axis=1)
    )

    print(f"\n{source}")

    for k in range(
        1,
        len(lfs) + 1
    ):

        count = (
            votes >= k
        ).sum()

        print(
            f"  >= {k} agreeing LFs: "
            f"{count:,}"
        )

# ------------------------------------------------------------
# CROSS-SOURCE CONFLICTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CROSS-SOURCE CONFLICTS")
print("=" * 70)

sources = list(final_lf_groups.keys())

conflict_rows = []

for i in range(len(sources)):

    for j in range(i + 1, len(sources)):

        s1 = sources[i]
        s2 = sources[j]

        conflict = (
            source_active[s1]
            &
            source_active[s2]
        ).sum()

        conflict_rows.append({
            "source_1": s1,
            "source_2": s2,
            "conflict_events": int(conflict),
            "conflict_pct":
                conflict / len(df) * 100
        })

        print(
            f"{s1:12} + {s2:12}: "
            f"{conflict:,}"
        )

conflict_df = pd.DataFrame(
    conflict_rows
)

# ------------------------------------------------------------
# PAIRWISE OVERLAP
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP FINAL LF OVERLAPS")
print("=" * 70)

overlap_rows = []

for i in range(len(all_lfs)):

    for j in range(i + 1, len(all_lfs)):

        lf1 = all_lfs[i]
        lf2 = all_lfs[j]

        a = (
            df[lf1] != ABSTAIN
        )

        b = (
            df[lf2] != ABSTAIN
        )

        both = (
            a & b
        ).sum()

        either = (
            a | b
        ).sum()

        jaccard = (
            both / either
            if either > 0
            else 0
        )

        overlap_rows.append({
            "LF1": lf1,
            "LF2": lf2,
            "both_active": int(both),
            "either_active": int(either),
            "jaccard": jaccard
        })

overlap_df = (
    pd.DataFrame(overlap_rows)
    .sort_values(
        "jaccard",
        ascending=False
    )
)

display(
    overlap_df.head(20)
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUTPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_final_candidates.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5L COMPLETE")
print("=" * 70)

print(f"\nSaved:\n{OUTPUT_FILE}")

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:23:40.881884Z","iopub.execute_input":"2026-09-09T12:23:40.882224Z","iopub.status.idle":"2026-09-09T12:23:48.532774Z","shell.execute_reply.started":"2026-09-09T12:23:40.882195Z","shell.execute_reply":"2026-09-09T12:23:48.530279Z"}}
# ============================================================
# STEP 5M — CONFLICT CASE INSPECTION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5M — CONFLICT CASE INSPECTION")
print("=" * 70)

INPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_final_candidates.csv"
)

df = pd.read_csv(INPUT_FILE)

ABSTAIN = -1

# ------------------------------------------------------------
# FINAL LF GROUPS
# ------------------------------------------------------------

final_lf_groups = {

    "GAS": [
        "LF_GAS_1KM",
        "LF_GAS_2KM",
        "LF_GAS_REPEATED"
    ],

    "INDUSTRIAL": [
        "LF_IND_MULTI_500",
        "LF_IND_SCORE",
        "LF_IND_STEEL",
        "LF_IND_CEMENT",
        "LF_IND_POWER",
        "LF_IND_REPEATED"
    ],

    "MINING": [
        "LF_MINE_500",
        "LF_MINE_RECURRENT",
        "LF_MINE_STRONG"
    ],

    "AGRICULTURE": [
        "LF_AGRI_CROP_70",
        "LF_AGRI_CROP_CONTEXT",
        "LF_AGRI_SCORE",
        "LF_AGRI_RECURRENT"
    ],

    "WILDFIRE": [
        "LF_WF_STRONG_FOREST",
        "LF_WF_REPEATED_NATURAL"
    ]
}

# ------------------------------------------------------------
# SOURCE VOTE COUNTS
# ------------------------------------------------------------

source_votes = pd.DataFrame(index=df.index)

for source, lfs in final_lf_groups.items():
    source_votes[source] = (
        (df[lfs] != ABSTAIN).sum(axis=1)
    )

# ------------------------------------------------------------
# FUNCTION TO INSPECT A CONFLICT
# ------------------------------------------------------------

def inspect_conflict(source1, source2):

    mask = (
        (source_votes[source1] > 0)
        &
        (source_votes[source2] > 0)
    )

    subset = df.loc[mask].copy()

    print("\n" + "=" * 70)
    print(f"{source1} + {source2}")
    print("=" * 70)

    print(
        f"Conflict events: {len(subset):,}"
    )

    if len(subset) == 0:
        return

    # Vote counts
    print("\nVote-count distribution:")

    counts = (
        source_votes.loc[
            mask,
            [source1, source2]
        ]
        .value_counts()
        .sort_index()
    )

    print(counts)

    # LF combinations
    print("\nLF combinations:")

    rows = []

    for idx in subset.index:

        active = []

        for source, lfs in final_lf_groups.items():

            active_lfs = [
                lf
                for lf in lfs
                if df.loc[idx, lf] != ABSTAIN
            ]

            if active_lfs:
                active.append(
                    f"{source}:"
                    + "|".join(active_lfs)
                )

        rows.append(
            " || ".join(active)
        )

    combo_counts = (
        pd.Series(rows)
        .value_counts()
        .head(15)
    )

    for combo, count in combo_counts.items():

        print(
            f"{count:6,} : {combo}"
        )

    # --------------------------------------------------------
    # EVIDENCE SUMMARY
    # --------------------------------------------------------

    evidence_cols = [

        # Gas
        "nearest_gas_flare_distance_km",
        "gas_flare_count_within_1km",
        "gas_flare_count_within_2km",

        # Industrial
        "industrial_evidence_score_max",
        "industrial_source_count_500m_max",
        "industrial_source_count_1km_max",

        # Mining
        "nearest_coal_mine_distance_km",
        "coal_mine_within_500m_max",
        "coal_mine_within_1km_max",

        # Agriculture
        "agri_osm_distance_m_min",
        "agri_osm_within_500m_max",
        "agri_osm_within_1km_max",
        "event_max_dw_crop_probability",
        "event_mean_dw_crop_probability",
        "agriculture_evidence_score_max",

        # Wildfire
        "distance_to_nearest_forest_m_min",
        "distance_to_nearest_natural_vegetation_m_min",
        "natural_vegetation_strong_max",
        "event_max_dw_natural_vegetation_prob",
        "event_mean_dw_natural_vegetation_prob",

        # Behavior
        "event_detection_count",
        "event_active_days",
        "event_duration_days",
        "event_spatial_extent_km",
        "event_mean_frp",
        "event_max_frp"
    ]

    available = [
        c for c in evidence_cols
        if c in subset.columns
    ]

    print("\nEvidence summary:")

    display(
        subset[available]
        .describe()
        .T
    )

    # --------------------------------------------------------
    # SAMPLE CONFLICT EVENTS
    # --------------------------------------------------------

    print("\nSample conflict events:")

    sample_cols = [
        "event_id",
        "event_detection_count",
        "event_active_days",
        "event_duration_days",
        "event_spatial_extent_km"
    ]

    sample_cols += [
        c for c in available
        if c not in sample_cols
    ]

    display(
        subset[
            sample_cols
        ]
        .head(20)
    )


# ============================================================
# RUN IMPORTANT CONFLICTS
# ============================================================

inspect_conflict(
    "INDUSTRIAL",
    "AGRICULTURE"
)

inspect_conflict(
    "MINING",
    "AGRICULTURE"
)

inspect_conflict(
    "AGRICULTURE",
    "WILDFIRE"
)

inspect_conflict(
    "GAS",
    "AGRICULTURE"
)

print("\n" + "=" * 70)
print("STEP 5M COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:23:48.534782Z","iopub.execute_input":"2026-09-09T12:23:48.535240Z","iopub.status.idle":"2026-09-09T12:24:44.568359Z","shell.execute_reply.started":"2026-09-09T12:23:48.535209Z","shell.execute_reply":"2026-09-09T12:24:44.565071Z"}}
# ============================================================
# STEP 5N — REVISE AGRICULTURE LFs
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5N — REVISED AGRICULTURE LFs")
print("=" * 70)

INPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_final_candidates.csv"
)

df = pd.read_csv(INPUT_FILE)

ABSTAIN = -1
AGRICULTURE = 2

# ------------------------------------------------------------
# ORIGINAL AGRICULTURE LFs
# ------------------------------------------------------------

original_agri = [
    "LF_AGRI_CROP_70",
    "LF_AGRI_CROP_CONTEXT",
    "LF_AGRI_SCORE",
    "LF_AGRI_RECURRENT"
]

print("\nOriginal agriculture LF coverage:")

for lf in original_agri:

    n = (
        df[lf] != ABSTAIN
    ).sum()

    print(
        f"{lf:30} "
        f"{n:8,} "
        f"({n / len(df) * 100:.3f}%)"
    )

# ------------------------------------------------------------
# REMOVE BROAD AGRICULTURE LFs
# ------------------------------------------------------------

df["LF_AGRI_SCORE_REVISED"] = ABSTAIN
df["LF_AGRI_RECURRENT_REVISED"] = ABSTAIN

# ------------------------------------------------------------
# CONSERVATIVE AGRICULTURE LF
#
# Requires actual crop evidence AND recurrence.
#
# We do NOT use agriculture_evidence_score alone.
# ------------------------------------------------------------

crop_available = (
    df["event_max_dw_crop_probability"]
    .notna()
)

crop_strong = (
    df["event_max_dw_crop_probability"] >= 0.50
)

crop_context = (
    df["LF_AGRI_CROP_CONTEXT"] != ABSTAIN
)

recurrent = (
    (
        df["event_active_days"] >= 2
    )
    |
    (
        df["event_detection_count"] >= 3
    )
)

# Conservative combined rule
agri_repeated_crop = (
    crop_available
    &
    (
        crop_strong
        |
        crop_context
    )
    &
    recurrent
)

df["LF_AGRI_REPEATED_CROP_REVISED"] = np.where(
    agri_repeated_crop,
    AGRICULTURE,
    ABSTAIN
)

# ------------------------------------------------------------
# STRICT CROP LF
# ------------------------------------------------------------

df["LF_AGRI_CROP_70_REVISED"] = np.where(
    (
        crop_available
        &
        (
            df["event_max_dw_crop_probability"] >= 0.70
        )
    ),
    AGRICULTURE,
    ABSTAIN
)

# ------------------------------------------------------------
# CROP CONTEXT LF
# ------------------------------------------------------------

df["LF_AGRI_CROP_CONTEXT_REVISED"] = np.where(
    (
        crop_context
        &
        crop_available
        &
        (
            df["event_max_dw_crop_probability"] >= 0.50
        )
    ),
    AGRICULTURE,
    ABSTAIN
)

# ------------------------------------------------------------
# COVERAGE
# ------------------------------------------------------------

revised_agri = [
    "LF_AGRI_CROP_70_REVISED",
    "LF_AGRI_CROP_CONTEXT_REVISED",
    "LF_AGRI_REPEATED_CROP_REVISED"
]

print("\n" + "=" * 70)
print("REVISED AGRICULTURE LF COVERAGE")
print("=" * 70)

for lf in revised_agri:

    n = (
        df[lf] != ABSTAIN
    ).sum()

    print(
        f"{lf:40} "
        f"{n:8,} "
        f"({n / len(df) * 100:.3f}%)"
    )

# ------------------------------------------------------------
# AGRICULTURE AGREEMENT
# ------------------------------------------------------------

agri_votes = (
    df[revised_agri] != ABSTAIN
).sum(axis=1)

print("\n" + "=" * 70)
print("REVISED AGRICULTURE AGREEMENT")
print("=" * 70)

for k in range(1, 4):

    n = (
        agri_votes >= k
    ).sum()

    print(
        f">= {k} agriculture LFs: "
        f"{n:,} "
        f"({n / len(df) * 100:.3f}%)"
    )

# ------------------------------------------------------------
# REVISED AGRICULTURE VS INDUSTRIAL
# ------------------------------------------------------------

industrial_lfs = [
    "LF_IND_MULTI_500",
    "LF_IND_SCORE",
    "LF_IND_STEEL",
    "LF_IND_CEMENT",
    "LF_IND_POWER",
    "LF_IND_REPEATED"
]

industrial_active = (
    (df[industrial_lfs] != ABSTAIN)
    .any(axis=1)
)

revised_agri_active = (
    agri_votes > 0
)

print("\n" + "=" * 70)
print("REVISED AGRICULTURE + INDUSTRIAL CONFLICT")
print("=" * 70)

conflict = (
    revised_agri_active
    &
    industrial_active
)

print(
    f"Conflict events: "
    f"{conflict.sum():,} "
    f"({conflict.mean() * 100:.3f}%)"
)

# ------------------------------------------------------------
# REVISED AGRICULTURE VS MINING
# ------------------------------------------------------------

mining_lfs = [
    "LF_MINE_500",
    "LF_MINE_RECURRENT",
    "LF_MINE_STRONG"
]

mining_active = (
    (df[mining_lfs] != ABSTAIN)
    .any(axis=1)
)

print("\n" + "=" * 70)
print("REVISED AGRICULTURE + MINING CONFLICT")
print("=" * 70)

conflict = (
    revised_agri_active
    &
    mining_active
)

print(
    f"Conflict events: "
    f"{conflict.sum():,} "
    f"({conflict.mean() * 100:.3f}%)"
)

# ------------------------------------------------------------
# REVISED AGRICULTURE VS WILDFIRE
# ------------------------------------------------------------

wildfire_lfs = [
    "LF_WF_STRONG_FOREST",
    "LF_WF_REPEATED_NATURAL"
]

wildfire_active = (
    (df[wildfire_lfs] != ABSTAIN)
    .any(axis=1)
)

print("\n" + "=" * 70)
print("REVISED AGRICULTURE + WILDFIRE CONFLICT")
print("=" * 70)

conflict = (
    revised_agri_active
    &
    wildfire_active
)

print(
    f"Conflict events: "
    f"{conflict.sum():,} "
    f"({conflict.mean() * 100:.3f}%)"
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUTPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_agriculture_revised.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5N COMPLETE")
print("=" * 70)

print(f"\nSaved:\n{OUTPUT_FILE}")

# %% [code] {"jupyter":{"outputs_hidden":false},"execution":{"iopub.status.busy":"2026-09-09T12:24:44.570639Z","iopub.execute_input":"2026-09-09T12:24:44.571033Z"}}
# ============================================================
# STEP 5O — BUILD FINAL SNORKEL LF MATRIX
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5O — FINAL SNORKEL LF MATRIX")
print("=" * 70)

INPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_agriculture_revised.csv"
)

df = pd.read_csv(INPUT_FILE)

ABSTAIN = -1

# Class IDs
INDUSTRIAL = 0
GAS = 1
AGRICULTURE = 2
MINING = 3
WILDFIRE = 4

# ------------------------------------------------------------
# FINAL APPROVED LFs
# ------------------------------------------------------------

final_lfs = {

    # GAS
    "LF_GAS_1KM": GAS,
    "LF_GAS_2KM": GAS,
    "LF_GAS_REPEATED": GAS,

    # INDUSTRIAL
    "LF_IND_MULTI_500": INDUSTRIAL,
    "LF_IND_SCORE": INDUSTRIAL,
    "LF_IND_STEEL": INDUSTRIAL,
    "LF_IND_CEMENT": INDUSTRIAL,
    "LF_IND_POWER": INDUSTRIAL,
    "LF_IND_REPEATED": INDUSTRIAL,

    # MINING
    "LF_MINE_500": MINING,
    "LF_MINE_RECURRENT": MINING,
    "LF_MINE_STRONG": MINING,

    # AGRICULTURE — REVISED
    "LF_AGRI_CROP_70_REVISED": AGRICULTURE,
    "LF_AGRI_CROP_CONTEXT_REVISED": AGRICULTURE,
    "LF_AGRI_REPEATED_CROP_REVISED": AGRICULTURE,

    # WILDFIRE
    "LF_WF_STRONG_FOREST": WILDFIRE,
    "LF_WF_REPEATED_NATURAL": WILDFIRE,
}

# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print(
    f"\nEvents: {len(df):,}"
)

print(
    f"Candidate LFs: {len(final_lfs)}"
)

missing_lfs = [
    lf for lf in final_lfs
    if lf not in df.columns
]

if missing_lfs:

    raise ValueError(
        "Missing LFs:\n"
        + "\n".join(missing_lfs)
    )

print("\nAll final LFs found.")

# ------------------------------------------------------------
# CONSTRUCT LABEL MATRIX
# ------------------------------------------------------------

L = np.full(
    (len(df), len(final_lfs)),
    ABSTAIN,
    dtype=np.int8
)

lf_names = list(final_lfs.keys())

for j, lf in enumerate(lf_names):

    source_label = final_lfs[lf]

    active = (
        df[lf].values != ABSTAIN
    )

    L[active, j] = source_label

L_df = pd.DataFrame(
    L,
    columns=lf_names
)

# ------------------------------------------------------------
# VALIDATE MATRIX
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MATRIX VALIDATION")
print("=" * 70)

print(
    f"Shape: {L_df.shape}"
)

print(
    f"Expected rows: {len(df):,}"
)

print(
    f"Expected LFs: {len(final_lfs)}"
)

# Only valid values?
valid_values = {-1, 0, 1, 2, 3, 4}

actual_values = set(
    np.unique(L)
)

print(
    f"Values present: "
    f"{sorted(actual_values)}"
)

if not actual_values.issubset(valid_values):

    raise ValueError(
        "Invalid values detected in LF matrix."
    )

# ------------------------------------------------------------
# OVERALL COVERAGE
# ------------------------------------------------------------

active_count = (
    (L != ABSTAIN)
    .sum(axis=1)
)

print("\nActive LF distribution:")

for k in range(
    int(active_count.max()) + 1
):

    n = (
        active_count == k
    ).sum()

    print(
        f"{k:2} active LFs: "
        f"{n:8,} "
        f"({n / len(df) * 100:6.2f}%)"
    )

# ------------------------------------------------------------
# LF COVERAGE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LF COVERAGE")
print("=" * 70)

coverage_rows = []

for j, lf in enumerate(lf_names):

    n = (
        L[:, j] != ABSTAIN
    ).sum()

    coverage_rows.append({
        "LF": lf,
        "class": final_lfs[lf],
        "covered_events": int(n),
        "coverage_pct":
            n / len(df) * 100
    })

coverage_df = (
    pd.DataFrame(coverage_rows)
    .sort_values(
        "coverage_pct",
        ascending=False
    )
    .reset_index(drop=True)
)

display(coverage_df)

# ------------------------------------------------------------
# CLASS COVERAGE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS-LEVEL LF COVERAGE")
print("=" * 70)

class_names = {
    0: "INDUSTRIAL",
    1: "GAS",
    2: "AGRICULTURE",
    3: "MINING",
    4: "WILDFIRE"
}

for class_id, class_name in class_names.items():

    class_active = (
        L == class_id
    ).any(axis=1)

    n = class_active.sum()

    print(
        f"{class_name:15} "
        f"{n:8,} "
        f"({n / len(df) * 100:6.2f}%)"
    )

# ------------------------------------------------------------
# SAVE MATRIX
# ------------------------------------------------------------

MATRIX_FILE = (
    "/kaggle/working/"
    "firms_2025_snorkel_lf_matrix.csv"
)

L_df.to_csv(
    MATRIX_FILE,
    index=False
)

# Save event IDs separately so matrix rows remain traceable
EVENT_FILE = (
    "/kaggle/working/"
    "firms_2025_snorkel_event_ids.csv"
)

df[["event_id"]].to_csv(
    EVENT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5O COMPLETE")
print("=" * 70)

print("\nSaved:")
print(MATRIX_FILE)
print(EVENT_FILE)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5P — CHECK SNORKEL INSTALLATION
# ============================================================

print("=" * 70)
print("STEP 5P — SNORKEL ENVIRONMENT CHECK")
print("=" * 70)

try:
    import snorkel

    print("\nSnorkel is installed.")
    print("Version:", snorkel.__version__)

except ImportError:
    print("\nSnorkel is NOT installed.")

# Check the important LabelModel import
try:
    from snorkel.labeling import LabelModel

    print("LabelModel import: SUCCESS")

except ImportError as e:
    print("LabelModel import: FAILED")
    print("Error:", e)

print("\nPython environment:")
import sys
print(sys.version)

print("\n" + "=" * 70)
print("STEP 5P COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5Q — VERIFY SNORKEL 0.10.0
# ============================================================

print("=" * 70)
print("STEP 5Q — VERIFY SNORKEL 0.10.0")
print("=" * 70)

import snorkel

print("\nSnorkel version:")
print(snorkel.__version__)

# Correct import for this installation
from snorkel.labeling.model import LabelModel

print("\nLabelModel import: SUCCESS")

# Also verify LFAnalysis
try:
    from snorkel.labeling import LFAnalysis

    print("LFAnalysis import: SUCCESS")

except ImportError as e:

    print("LFAnalysis import: FAILED")
    print("Error:", e)

print("\n" + "=" * 70)
print("STEP 5Q COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5R — SNORKEL LF DIAGNOSTICS
# ============================================================

import pandas as pd
import numpy as np

from snorkel.labeling import LFAnalysis

print("=" * 70)
print("STEP 5R — SNORKEL LF DIAGNOSTICS")
print("=" * 70)

# ------------------------------------------------------------
# LOAD LF MATRIX
# ------------------------------------------------------------

MATRIX_FILE = (
    "/kaggle/working/"
    "firms_2025_snorkel_lf_matrix.csv"
)

L_df = pd.read_csv(MATRIX_FILE)

L = L_df.values.astype(np.int8)

lf_names = list(L_df.columns)

print("\nLF matrix shape:")
print(L.shape)

print("\nNumber of LFs:")
print(len(lf_names))

print("\nNumber of events:")
print(len(L))

# ------------------------------------------------------------
# LABEL DEFINITIONS
# ------------------------------------------------------------

ABSTAIN = -1

class_names = {
    0: "INDUSTRIAL",
    1: "GAS",
    2: "AGRICULTURE",
    3: "MINING",
    4: "WILDFIRE"
}

print("\nClasses:")
for k, v in class_names.items():
    print(f"  {k}: {v}")

# ------------------------------------------------------------
# BASIC MATRIX CHECK
# ------------------------------------------------------------

valid_values = {-1, 0, 1, 2, 3, 4}

actual_values = set(
    np.unique(L)
)

print("\nMatrix values:")
print(sorted(actual_values))

if not actual_values.issubset(valid_values):

    raise ValueError(
        "Invalid label found in LF matrix."
    )

# ------------------------------------------------------------
# SNORKEL LF ANALYSIS
# ------------------------------------------------------------

analysis = LFAnalysis(
    L=L,
    lfs=[
        type(
            "LF",
            (),
            {"name": name}
        )()
        for name in lf_names
    ]
)

# Get polarity / coverage / overlap / conflict
summary = analysis.lf_summary()

print("\n" + "=" * 70)
print("SNORKEL LF SUMMARY")
print("=" * 70)

display(summary)

# ------------------------------------------------------------
# EVENTS WITH AT LEAST ONE LABEL
# ------------------------------------------------------------

active_count = (
    (L != ABSTAIN)
    .sum(axis=1)
)

covered_events = (
    active_count > 0
).sum()

print("\n" + "=" * 70)
print("OVERALL COVERAGE")
print("=" * 70)

print(
    f"Events with >=1 LF: "
    f"{covered_events:,}"
)

print(
    f"Coverage: "
    f"{covered_events / len(L) * 100:.3f}%"
)

print(
    f"Events with no LF: "
    f"{(active_count == 0).sum():,}"
)

# ------------------------------------------------------------
# LF-BY-LF CLASS VOTES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LF CLASS VOTES")
print("=" * 70)

vote_rows = []

for j, lf in enumerate(lf_names):

    row = {
        "LF": lf
    }

    for class_id, class_name in class_names.items():

        row[class_name] = int(
            (L[:, j] == class_id).sum()
        )

    row["ABSTAIN"] = int(
        (L[:, j] == ABSTAIN).sum()
    )

    vote_rows.append(row)

vote_df = pd.DataFrame(
    vote_rows
)

display(vote_df)

# ------------------------------------------------------------
# SAVE DIAGNOSTICS
# ------------------------------------------------------------

SUMMARY_FILE = (
    "/kaggle/working/"
    "firms_2025_snorkel_lf_summary.csv"
)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)

VOTES_FILE = (
    "/kaggle/working/"
    "firms_2025_snorkel_lf_class_votes.csv"
)

vote_df.to_csv(
    VOTES_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5R COMPLETE")
print("=" * 70)

print("\nSaved:")
print(SUMMARY_FILE)
print(VOTES_FILE)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5S — TIGHTEN AGRICULTURE REPEATED-CROP LF
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5S — TIGHTEN AGRICULTURE LF")
print("=" * 70)

INPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_agriculture_revised.csv"
)

df = pd.read_csv(INPUT_FILE)

ABSTAIN = -1
AGRICULTURE = 2

# ------------------------------------------------------------
# CROP PROBABILITY
# ------------------------------------------------------------

crop_prob = df[
    "event_max_dw_crop_probability"
]

crop_available = crop_prob.notna()

# Require actual crop probability >= 50%
crop_50 = (
    crop_available
    &
    (crop_prob >= 0.50)
)

# ------------------------------------------------------------
# REPEATED ACTIVITY
# ------------------------------------------------------------

repeated_activity = (
    (df["event_active_days"] >= 2)
    |
    (df["event_detection_count"] >= 3)
)

# ------------------------------------------------------------
# NEW CONSERVATIVE LF
# ------------------------------------------------------------

df["LF_AGRI_REPEATED_CROP_FINAL"] = np.where(
    crop_50
    &
    repeated_activity,
    AGRICULTURE,
    ABSTAIN
)

# ------------------------------------------------------------
# COVERAGE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AGRICULTURE LF COVERAGE")
print("=" * 70)

lfs = [
    "LF_AGRI_CROP_70_REVISED",
    "LF_AGRI_CROP_CONTEXT_REVISED",
    "LF_AGRI_REPEATED_CROP_REVISED",
    "LF_AGRI_REPEATED_CROP_FINAL"
]

for lf in lfs:

    n = (
        df[lf] != ABSTAIN
    ).sum()

    print(
        f"{lf:40} "
        f"{n:8,} "
        f"({n / len(df) * 100:.3f}%)"
    )

# ------------------------------------------------------------
# AGREEMENT
# ------------------------------------------------------------

final_agri_lfs = [
    "LF_AGRI_CROP_70_REVISED",
    "LF_AGRI_CROP_CONTEXT_REVISED",
    "LF_AGRI_REPEATED_CROP_FINAL"
]

agri_votes = (
    (df[final_agri_lfs] != ABSTAIN)
    .sum(axis=1)
)

print("\n" + "=" * 70)
print("AGRICULTURE AGREEMENT")
print("=" * 70)

for k in range(1, 4):

    n = (
        agri_votes >= k
    ).sum()

    print(
        f">= {k} agriculture LFs: "
        f"{n:,} "
        f"({n / len(df) * 100:.3f}%)"
    )

# ------------------------------------------------------------
# REMOVE OLD VERSION FROM FUTURE USE
# ------------------------------------------------------------

# We keep the old column for comparison/audit,
# but it will NOT be used in the final Snorkel matrix.

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

OUTPUT_FILE = (
    "/kaggle/working/"
    "firms_2025_event_labeling_functions_final_revised.csv"
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 70)
print("STEP 5S COMPLETE")
print("=" * 70)

print(f"\nSaved:\n{OUTPUT_FILE}")

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5T — BUILD FINAL SNORKEL LABELING MATRIX
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5T — BUILD FINAL SNORKEL LABELING MATRIX")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD FINAL LF DATA
# ------------------------------------------------------------

lf_file = "/kaggle/working/firms_2025_event_labeling_functions_final_revised.csv"

lf_df = pd.read_csv(lf_file)

print("\nLoaded LF dataset:")
print("Rows:", len(lf_df))
print("Columns:", len(lf_df.columns))


# ------------------------------------------------------------
# 2. FINAL 17 LABELING FUNCTIONS
# ------------------------------------------------------------

final_lfs = [
    # ---------------- GAS ----------------
    "LF_GAS_1KM",
    "LF_GAS_2KM",
    "LF_GAS_REPEATED",

    # ---------------- INDUSTRIAL ----------------
    "LF_IND_MULTI_500",
    "LF_IND_SCORE",
    "LF_IND_STEEL",
    "LF_IND_CEMENT",
    "LF_IND_POWER",
    "LF_IND_REPEATED",

    # ---------------- MINING ----------------
    "LF_MINE_500",
    "LF_MINE_RECURRENT",
    "LF_MINE_STRONG",

    # ---------------- AGRICULTURE ----------------
    "LF_AGRI_CROP_70_REVISED",
    "LF_AGRI_CROP_CONTEXT_REVISED",
    "LF_AGRI_REPEATED_CROP_FINAL",

    # ---------------- WILDFIRE ----------------
    "LF_WF_STRONG_FOREST",
    "LF_WF_REPEATED_NATURAL",
]

print("\nNumber of final LFs:", len(final_lfs))

missing_lfs = [lf for lf in final_lfs if lf not in lf_df.columns]

if missing_lfs:
    raise ValueError(
        "Missing LF columns:\n" + "\n".join(missing_lfs)
    )

print("All 17 LF columns found.")


# ------------------------------------------------------------
# 3. CHECK LF VALUES
# ------------------------------------------------------------

print("\nChecking LF values...")

allowed_values = {-1, 0, 1, 2, 3, 4}

for lf in final_lfs:
    values = set(lf_df[lf].dropna().unique())

    unexpected = values - allowed_values

    if unexpected:
        raise ValueError(
            f"{lf} contains unexpected values: {unexpected}"
        )

print("All LF values are valid.")
print("Expected encoding:")
print("  -1 = ABSTAIN")
print("   0 = Industrial")
print("   1 = Gas")
print("   2 = Agriculture")
print("   3 = Mining")
print("   4 = Wildfire")


# ------------------------------------------------------------
# 4. BUILD MATRIX
# ------------------------------------------------------------

L = lf_df[final_lfs].astype(np.int8).to_numpy()

print("\nFinal Snorkel matrix shape:")
print(L.shape)

print(f"Events : {L.shape[0]:,}")
print(f"LFs    : {L.shape[1]}")


# ------------------------------------------------------------
# 5. VERIFY NO MISSING VALUES
# ------------------------------------------------------------

missing_count = np.isnan(L.astype(float)).sum()

print("\nMissing LF values:", missing_count)

if missing_count != 0:
    raise ValueError("LF matrix contains missing values.")


# ------------------------------------------------------------
# 6. ACTIVE LF COUNT PER EVENT
# ------------------------------------------------------------

active_counts = (L != -1).sum(axis=1)

print("\n" + "=" * 70)
print("ACTIVE LF COUNT PER EVENT")
print("=" * 70)

for n in sorted(np.unique(active_counts)):
    count = np.sum(active_counts == n)
    pct = count / len(L) * 100

    print(f"{n:2d} LFs active: {count:8,} ({pct:6.2f}%)")


# ------------------------------------------------------------
# 7. EVENTS WITH AT LEAST ONE LF
# ------------------------------------------------------------

covered_events = np.sum(active_counts > 0)
uncovered_events = np.sum(active_counts == 0)

print("\n" + "=" * 70)
print("OVERALL COVERAGE")
print("=" * 70)

print(
    f"Events covered by >=1 LF : "
    f"{covered_events:,} ({covered_events / len(L) * 100:.2f}%)"
)

print(
    f"Events with no LF         : "
    f"{uncovered_events:,} ({uncovered_events / len(L) * 100:.2f}%)"
)


# ------------------------------------------------------------
# 8. CLASS-WISE LF COVERAGE
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

print("\n" + "=" * 70)
print("CLASS-WISE LF COVERAGE")
print("=" * 70)

for class_id, class_name in class_names.items():

    class_mask = (L == class_id)

    # At least one LF votes for this class
    events_with_class_vote = np.any(class_mask, axis=1)

    count = events_with_class_vote.sum()
    pct = count / len(L) * 100

    print(
        f"{class_name:15s}: "
        f"{count:8,} ({pct:6.3f}%)"
    )


# ------------------------------------------------------------
# 9. SAVE FINAL MATRIX
# ------------------------------------------------------------

matrix_df = pd.DataFrame(
    L,
    columns=final_lfs
)

matrix_df.insert(
    0,
    "event_id",
    lf_df["event_id"].values
)

matrix_file = "/kaggle/working/firms_2025_snorkel_final_L_matrix.csv"

matrix_df.to_csv(
    matrix_file,
    index=False
)

print("\nSaved final LF matrix:")
print(matrix_file)


# ------------------------------------------------------------
# 10. SAVE EVENT IDS
# ------------------------------------------------------------

event_ids_file = "/kaggle/working/firms_2025_snorkel_final_event_ids.csv"

pd.DataFrame({
    "event_id": lf_df["event_id"].values
}).to_csv(
    event_ids_file,
    index=False
)

print("\nSaved event IDs:")
print(event_ids_file)


print("\n" + "=" * 70)
print("STEP 5T COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5U — FINAL LF QUALITY ANALYSIS
# CORRECTED FOR CURRENT SNORKEL VERSION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5U — FINAL LF QUALITY ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD FINAL MATRIX
# ------------------------------------------------------------

matrix_file = "/kaggle/working/firms_2025_snorkel_final_L_matrix.csv"

matrix_df = pd.read_csv(matrix_file)

final_lfs = [
    "LF_GAS_1KM",
    "LF_GAS_2KM",
    "LF_GAS_REPEATED",

    "LF_IND_MULTI_500",
    "LF_IND_SCORE",
    "LF_IND_STEEL",
    "LF_IND_CEMENT",
    "LF_IND_POWER",
    "LF_IND_REPEATED",

    "LF_MINE_500",
    "LF_MINE_RECURRENT",
    "LF_MINE_STRONG",

    "LF_AGRI_CROP_70_REVISED",
    "LF_AGRI_CROP_CONTEXT_REVISED",
    "LF_AGRI_REPEATED_CROP_FINAL",

    "LF_WF_STRONG_FOREST",
    "LF_WF_REPEATED_NATURAL",
]

L = matrix_df[final_lfs].to_numpy(dtype=np.int8)

n_events, n_lfs = L.shape

print("\nMatrix shape:", L.shape)


# ------------------------------------------------------------
# 2. LF COVERAGE
# ------------------------------------------------------------

coverage_count = (L != -1).sum(axis=0)
coverage_percent = coverage_count / n_events * 100

coverage_df = pd.DataFrame({
    "LF": final_lfs,
    "Coverage_Count": coverage_count,
    "Coverage_Percent": coverage_percent
})

coverage_df = coverage_df.sort_values(
    "Coverage_Count",
    ascending=False
).reset_index(drop=True)


print("\n" + "=" * 70)
print("LF COVERAGE")
print("=" * 70)

display(coverage_df)


# ------------------------------------------------------------
# 3. OVERLAP BETWEEN LFs
#
# Overlap = fraction of ALL events where BOTH LFs
# give a non-abstain vote.
# ------------------------------------------------------------

overlap_pairs = []

for i in range(n_lfs):

    for j in range(i + 1, n_lfs):

        both_active = (
            (L[:, i] != -1) &
            (L[:, j] != -1)
        )

        count = both_active.sum()

        if count > 0:

            overlap_percent = count / n_events * 100

            overlap_pairs.append({
                "LF_1": final_lfs[i],
                "LF_2": final_lfs[j],
                "Both_Active_Count": int(count),
                "Overlap_Percent": overlap_percent
            })


overlap_df = pd.DataFrame(overlap_pairs)

overlap_df = overlap_df.sort_values(
    "Both_Active_Count",
    ascending=False
).reset_index(drop=True)


print("\n" + "=" * 70)
print("TOP LF OVERLAPS")
print("=" * 70)

display(overlap_df.head(20))


# ------------------------------------------------------------
# 4. LF CONFLICTS
#
# Conflict = both LFs vote, but for DIFFERENT classes.
# ------------------------------------------------------------

conflict_pairs = []

for i in range(n_lfs):

    for j in range(i + 1, n_lfs):

        both_active = (
            (L[:, i] != -1) &
            (L[:, j] != -1)
        )

        different_class = (
            L[:, i] != L[:, j]
        )

        conflict_mask = both_active & different_class

        count = conflict_mask.sum()

        if count > 0:

            conflict_percent = count / n_events * 100

            conflict_pairs.append({
                "LF_1": final_lfs[i],
                "LF_2": final_lfs[j],
                "Conflict_Count": int(count),
                "Conflict_Percent": conflict_percent
            })


conflict_df = pd.DataFrame(conflict_pairs)

if len(conflict_df) > 0:

    conflict_df = conflict_df.sort_values(
        "Conflict_Count",
        ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 70)
    print("TOP LF CONFLICTS")
    print("=" * 70)

    display(conflict_df.head(20))

else:

    print("\nNo LF conflicts found.")


# ------------------------------------------------------------
# 5. WITHIN-CLASS AGREEMENT
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

print("\n" + "=" * 70)
print("WITHIN-CLASS LF AGREEMENT")
print("=" * 70)

agreement_rows = []

for class_id, class_name in class_names.items():

    class_votes = (L == class_id)

    vote_count = class_votes.sum(axis=1)

    row = {
        "Class": class_name,
        "Class_ID": class_id,
        ">=1_LF": int((vote_count >= 1).sum()),
        ">=2_LFs": int((vote_count >= 2).sum()),
        ">=3_LFs": int((vote_count >= 3).sum()),
        ">=4_LFs": int((vote_count >= 4).sum()),
    }

    agreement_rows.append(row)

agreement_df = pd.DataFrame(agreement_rows)

display(agreement_df)


# ------------------------------------------------------------
# 6. CROSS-CLASS CONFLICT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CROSS-CLASS CONFLICT SUMMARY")
print("=" * 70)

class_conflicts = []

for class_a in range(5):

    for class_b in range(class_a + 1, 5):

        mask_a = (L == class_a).any(axis=1)
        mask_b = (L == class_b).any(axis=1)

        count = (mask_a & mask_b).sum()

        if count > 0:

            class_conflicts.append({
                "Class_A": class_names[class_a],
                "Class_B": class_names[class_b],
                "Conflict_Events": int(count),
                "Percent_of_all_events": count / n_events * 100
            })


class_conflict_df = pd.DataFrame(class_conflicts)

if len(class_conflict_df) > 0:

    class_conflict_df = class_conflict_df.sort_values(
        "Conflict_Events",
        ascending=False
    ).reset_index(drop=True)

    display(class_conflict_df)

else:

    print("No cross-class conflicts.")


# ------------------------------------------------------------
# 7. SAVE REPORTS
# ------------------------------------------------------------

coverage_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_final_lf_coverage.csv"
)

overlap_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_final_lf_overlap.csv"
)

conflict_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_final_lf_conflicts.csv"
)

agreement_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_final_lf_agreement.csv"
)

class_conflict_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_final_class_conflicts.csv"
)

coverage_df.to_csv(coverage_file, index=False)
overlap_df.to_csv(overlap_file, index=False)
conflict_df.to_csv(conflict_file, index=False)
agreement_df.to_csv(agreement_file, index=False)
class_conflict_df.to_csv(class_conflict_file, index=False)


print("\n" + "=" * 70)
print("REPORTS SAVED")
print("=" * 70)

print(coverage_file)
print(overlap_file)
print(conflict_file)
print(agreement_file)
print(class_conflict_file)


print("\n" + "=" * 70)
print("STEP 5U COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5V — TRAIN SNORKEL LABEL MODEL
# ============================================================

import pandas as pd
import numpy as np

from snorkel.labeling.model import LabelModel

print("=" * 70)
print("STEP 5V — TRAIN SNORKEL LABEL MODEL")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD FINAL LF MATRIX
# ------------------------------------------------------------

matrix_file = "/kaggle/working/firms_2025_snorkel_final_L_matrix.csv"

matrix_df = pd.read_csv(matrix_file)

final_lfs = [
    "LF_GAS_1KM",
    "LF_GAS_2KM",
    "LF_GAS_REPEATED",

    "LF_IND_MULTI_500",
    "LF_IND_SCORE",
    "LF_IND_STEEL",
    "LF_IND_CEMENT",
    "LF_IND_POWER",
    "LF_IND_REPEATED",

    "LF_MINE_500",
    "LF_MINE_RECURRENT",
    "LF_MINE_STRONG",

    "LF_AGRI_CROP_70_REVISED",
    "LF_AGRI_CROP_CONTEXT_REVISED",
    "LF_AGRI_REPEATED_CROP_FINAL",

    "LF_WF_STRONG_FOREST",
    "LF_WF_REPEATED_NATURAL",
]

L = matrix_df[final_lfs].to_numpy(dtype=np.int8)

print("\nLF matrix shape:", L.shape)


# ------------------------------------------------------------
# 2. VERIFY LABEL ENCODING
# ------------------------------------------------------------

valid_values = {-1, 0, 1, 2, 3, 4}

actual_values = set(np.unique(L))

print("\nUnique LF values:", sorted(actual_values))

if not actual_values.issubset(valid_values):
    raise ValueError(
        f"Unexpected LF values found: "
        f"{actual_values - valid_values}"
    )

print("LF encoding verified.")


# ------------------------------------------------------------
# 3. CREATE LABEL MODEL
# ------------------------------------------------------------

print("\nCreating Snorkel Label Model...")

label_model = LabelModel(
    cardinality=5,
    verbose=True
)

print("Cardinality:", 5)
print("Classes:")
print("  0 = Industrial")
print("  1 = Gas")
print("  2 = Agriculture")
print("  3 = Mining")
print("  4 = Wildfire")


# ------------------------------------------------------------
# 4. TRAIN LABEL MODEL
# ------------------------------------------------------------

print("\nTraining Label Model...")

label_model.fit(
    L_train=L,
    n_epochs=500,
    log_freq=100,
    seed=42
)

print("\nLabel Model training complete.")


# ------------------------------------------------------------
# 5. PREDICT PROBABILITIES
# ------------------------------------------------------------

print("\nGenerating class probabilities...")

probs = label_model.predict_proba(L)

print("Probability matrix shape:", probs.shape)


# ------------------------------------------------------------
# 6. PREDICT LABELS
# ------------------------------------------------------------

pred_labels = label_model.predict(L)

print("Prediction vector shape:", pred_labels.shape)


# ------------------------------------------------------------
# 7. CALCULATE CONFIDENCE
# ------------------------------------------------------------

confidence = probs.max(axis=1)

second_best = np.partition(
    probs,
    -2,
    axis=1
)[:, -2]

margin = confidence - second_best


# ------------------------------------------------------------
# 8. CREATE LABEL-MODEL OUTPUT
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

result_df = pd.DataFrame({
    "event_id": matrix_df["event_id"],
    "label_model_label": pred_labels,
    "label_model_class": [
        class_names.get(x, "Unknown")
        for x in pred_labels
    ],
    "label_model_confidence": confidence,
    "label_model_margin": margin,

    "prob_industrial": probs[:, 0],
    "prob_gas": probs[:, 1],
    "prob_agriculture": probs[:, 2],
    "prob_mining": probs[:, 3],
    "prob_wildfire": probs[:, 4],
})


# ------------------------------------------------------------
# 9. LABEL DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LABEL MODEL PREDICTION DISTRIBUTION")
print("=" * 70)

label_counts = (
    result_df["label_model_class"]
    .value_counts()
)

for class_name in class_names.values():

    count = label_counts.get(class_name, 0)

    print(
        f"{class_name:15s}: "
        f"{count:8,} "
        f"({count / len(result_df) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 10. CONFIDENCE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFIDENCE DISTRIBUTION")
print("=" * 70)

confidence_bins = [
    (0.00, 0.50),
    (0.50, 0.60),
    (0.60, 0.70),
    (0.70, 0.80),
    (0.80, 0.90),
    (0.90, 0.95),
    (0.95, 1.00),
]

for low, high in confidence_bins:

    if high == 1.00:
        mask = (
            (confidence >= low) &
            (confidence <= high)
        )
    else:
        mask = (
            (confidence >= low) &
            (confidence < high)
        )

    count = mask.sum()

    print(
        f"{low:.2f} - {high:.2f}: "
        f"{count:8,} "
        f"({count / len(confidence) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 11. HIGH-CONFIDENCE COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE LABELS")
print("=" * 70)

for threshold in [0.70, 0.80, 0.90, 0.95]:

    count = (confidence >= threshold).sum()

    print(
        f"Confidence >= {threshold:.2f}: "
        f"{count:8,} "
        f"({count / len(confidence) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 12. SAVE LABEL MODEL OUTPUT
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_label_model_predictions.csv"
)

result_df.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)


print("\n" + "=" * 70)
print("STEP 5V COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5W — LABEL MODEL CONFIDENCE BY CLASS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5W — LABEL MODEL CONFIDENCE BY CLASS")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD LABEL MODEL PREDICTIONS
# ------------------------------------------------------------

prediction_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_label_model_predictions.csv"
)

pred_df = pd.read_csv(prediction_file)

print("\nRows:", len(pred_df))

# ------------------------------------------------------------
# 2. CLASS DEFINITIONS
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

# ------------------------------------------------------------
# 3. CONFIDENCE BY PREDICTED CLASS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PREDICTIONS BY CLASS")
print("=" * 70)

class_summary = []

for class_id, class_name in class_names.items():

    subset = pred_df[
        pred_df["label_model_label"] == class_id
    ]

    count = len(subset)

    if count > 0:

        class_summary.append({
            "Class_ID": class_id,
            "Class": class_name,
            "Total_Predictions": count,
            "Percent_All_Events": count / len(pred_df) * 100,
            "Mean_Confidence": subset["label_model_confidence"].mean(),
            "Median_Confidence": subset["label_model_confidence"].median(),
            "Min_Confidence": subset["label_model_confidence"].min(),
            "Max_Confidence": subset["label_model_confidence"].max(),
            ">=0.70": (subset["label_model_confidence"] >= 0.70).sum(),
            ">=0.80": (subset["label_model_confidence"] >= 0.80).sum(),
            ">=0.90": (subset["label_model_confidence"] >= 0.90).sum(),
            ">=0.95": (subset["label_model_confidence"] >= 0.95).sum(),
        })

class_summary_df = pd.DataFrame(class_summary)

display(class_summary_df)


# ------------------------------------------------------------
# 4. HIGH-CONFIDENCE DATASET BY CLASS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE COUNTS BY CLASS")
print("=" * 70)

thresholds = [0.70, 0.80, 0.90, 0.95]

high_conf_rows = []

for threshold in thresholds:

    high_conf = pred_df[
        pred_df["label_model_confidence"] >= threshold
    ]

    print(
        f"\nConfidence >= {threshold:.2f}: "
        f"{len(high_conf):,} events"
    )

    for class_id, class_name in class_names.items():

        count = (
            high_conf["label_model_label"] == class_id
        ).sum()

        pct = (
            count / len(high_conf) * 100
            if len(high_conf) > 0
            else 0
        )

        print(
            f"  {class_name:15s}: "
            f"{count:8,} ({pct:6.2f}%)"
        )

        high_conf_rows.append({
            "Threshold": threshold,
            "Class_ID": class_id,
            "Class": class_name,
            "Count": int(count),
            "Percent_of_High_Confidence": pct
        })

high_conf_class_df = pd.DataFrame(high_conf_rows)


# ------------------------------------------------------------
# 5. CONFIDENCE QUANTILES BY CLASS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFIDENCE QUANTILES BY CLASS")
print("=" * 70)

quantile_rows = []

for class_id, class_name in class_names.items():

    values = pred_df.loc[
        pred_df["label_model_label"] == class_id,
        "label_model_confidence"
    ]

    if len(values) == 0:
        continue

    quantile_rows.append({
        "Class": class_name,
        "Count": len(values),
        "Q25": values.quantile(0.25),
        "Q50": values.quantile(0.50),
        "Q75": values.quantile(0.75),
        "Q90": values.quantile(0.90),
        "Q95": values.quantile(0.95),
        "Q99": values.quantile(0.99)
    })

quantile_df = pd.DataFrame(quantile_rows)

display(quantile_df)


# ------------------------------------------------------------
# 6. SAVE REPORTS
# ------------------------------------------------------------

summary_file = (
    "/kaggle/working/"
    "firms_2025_label_model_class_confidence.csv"
)

threshold_file = (
    "/kaggle/working/"
    "firms_2025_label_model_high_confidence_by_class.csv"
)

quantile_file = (
    "/kaggle/working/"
    "firms_2025_label_model_confidence_quantiles.csv"
)

class_summary_df.to_csv(
    summary_file,
    index=False
)

high_conf_class_df.to_csv(
    threshold_file,
    index=False
)

quantile_df.to_csv(
    quantile_file,
    index=False
)

print("\nSaved:")
print(summary_file)
print(threshold_file)
print(quantile_file)


print("\n" + "=" * 70)
print("STEP 5W COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5Y — CORRECT LF AGREEMENT ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5Y — CORRECT LF AGREEMENT ANALYSIS")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD FINAL LF MATRIX
# ------------------------------------------------------------

matrix_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_final_L_matrix.csv"
)

L_df = pd.read_csv(matrix_file)

final_lfs = [
    "LF_GAS_1KM",
    "LF_GAS_2KM",
    "LF_GAS_REPEATED",

    "LF_IND_MULTI_500",
    "LF_IND_SCORE",
    "LF_IND_STEEL",
    "LF_IND_CEMENT",
    "LF_IND_POWER",
    "LF_IND_REPEATED",

    "LF_MINE_500",
    "LF_MINE_RECURRENT",
    "LF_MINE_STRONG",

    "LF_AGRI_CROP_70_REVISED",
    "LF_AGRI_CROP_CONTEXT_REVISED",
    "LF_AGRI_REPEATED_CROP_FINAL",

    "LF_WF_STRONG_FOREST",
    "LF_WF_REPEATED_NATURAL",
]

L = L_df[final_lfs].to_numpy(dtype=np.int8)


# ------------------------------------------------------------
# 2. LOAD LABEL MODEL PREDICTIONS
# ------------------------------------------------------------

prediction_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_label_model_predictions.csv"
)

pred_df = pd.read_csv(prediction_file)

predicted_labels = (
    pred_df["label_model_label"]
    .to_numpy(dtype=np.int8)
)


# ------------------------------------------------------------
# 3. ACTIVE LF COUNT
# ------------------------------------------------------------

active_mask = L != -1

active_lf_count = active_mask.sum(axis=1)


# ------------------------------------------------------------
# 4. CORRECT AGREEMENT COUNT
#
# Only count an LF as agreeing when:
#
#   LF is NOT abstaining
#   AND
#   LF vote == Label Model prediction
# ------------------------------------------------------------

agreeing_mask = (
    active_mask &
    (L == predicted_labels[:, None])
)

agreeing_lf_count = agreeing_mask.sum(axis=1)


# ------------------------------------------------------------
# 5. DISAGREEMENT COUNT
# ------------------------------------------------------------

disagreeing_lf_count = (
    active_lf_count -
    agreeing_lf_count
)


pred_df["active_lf_count"] = active_lf_count
pred_df["agreeing_lf_count"] = agreeing_lf_count
pred_df["disagreeing_lf_count"] = disagreeing_lf_count


# ------------------------------------------------------------
# 6. VERIFY LOGIC
# ------------------------------------------------------------

print("\nChecking agreement logic...")

if not np.all(
    agreeing_lf_count +
    disagreeing_lf_count ==
    active_lf_count
):
    raise ValueError(
        "Agreement + disagreement != active LF count."
    )

print("Agreement logic verified.")


# ------------------------------------------------------------
# 7. ACTIVE LF DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ACTIVE LF COUNT")
print("=" * 70)

for n in sorted(np.unique(active_lf_count)):

    count = np.sum(active_lf_count == n)

    print(
        f"{n:2d} active LFs: "
        f"{count:8,} "
        f"({count / len(L) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 8. CORRECT AGREEMENT DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CORRECT AGREEMENT COUNT")
print("=" * 70)

for n in sorted(np.unique(agreeing_lf_count)):

    count = np.sum(agreeing_lf_count == n)

    print(
        f"{n:2d} agreeing LFs: "
        f"{count:8,} "
        f"({count / len(L) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 9. AGREEMENT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AGREEMENT SUMMARY")
print("=" * 70)

agreement_summary = (
    pred_df
    .groupby("agreeing_lf_count")
    .agg(
        Events=("event_id", "count"),
        Mean_Confidence=(
            "label_model_confidence",
            "mean"
        ),
        Median_Confidence=(
            "label_model_confidence",
            "median"
        ),
        Mean_Active_LFs=(
            "active_lf_count",
            "mean"
        ),
        Mean_Disagreeing_LFs=(
            "disagreeing_lf_count",
            "mean"
        )
    )
    .reset_index()
)

display(agreement_summary)


# ------------------------------------------------------------
# 10. CONFIDENCE + AGREEMENT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFIDENCE × CORRECT LF AGREEMENT")
print("=" * 70)

pred_df["confidence_band"] = pd.cut(
    pred_df["label_model_confidence"],
    bins=[
        0,
        0.50,
        0.70,
        0.80,
        0.90,
        0.95,
        1.00
    ],
    labels=[
        "<0.50",
        "0.50-0.70",
        "0.70-0.80",
        "0.80-0.90",
        "0.90-0.95",
        "0.95-1.00"
    ],
    include_lowest=True
)

cross_tab = pd.crosstab(
    pred_df["confidence_band"],
    pred_df["agreeing_lf_count"]
)

display(cross_tab)


# ------------------------------------------------------------
# 11. CLASS-WISE CORRECT AGREEMENT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CORRECT AGREEMENT BY PREDICTED CLASS")
print("=" * 70)

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

class_rows = []

for class_id, class_name in class_names.items():

    subset = pred_df[
        pred_df["label_model_label"] == class_id
    ]

    class_rows.append({
        "Class_ID": class_id,
        "Class": class_name,
        "Events": len(subset),

        "Mean_Active_LFs":
            subset["active_lf_count"].mean(),

        "Median_Active_LFs":
            subset["active_lf_count"].median(),

        "Mean_Agreeing_LFs":
            subset["agreeing_lf_count"].mean(),

        "Median_Agreeing_LFs":
            subset["agreeing_lf_count"].median(),

        ">=2_Agreeing_LFs":
            (
                subset["agreeing_lf_count"] >= 2
            ).sum(),

        ">=3_Agreeing_LFs":
            (
                subset["agreeing_lf_count"] >= 3
            ).sum(),

        "No_Disagreement":
            (
                subset["disagreeing_lf_count"] == 0
            ).sum(),

        "Has_Disagreement":
            (
                subset["disagreeing_lf_count"] > 0
            ).sum(),

        ">=0.70_Confidence":
            (
                subset["label_model_confidence"] >= 0.70
            ).sum(),

        ">=0.80_Confidence":
            (
                subset["label_model_confidence"] >= 0.80
            ).sum()
    })

class_agreement_df = pd.DataFrame(class_rows)

display(class_agreement_df)


# ------------------------------------------------------------
# 12. SAVE CORRECTED OUTPUT
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_predictions_corrected_agreement.csv"
)

pred_df.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)


print("\n" + "=" * 70)
print("STEP 5Y COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5Z — CANDIDATE WEAK-LABEL TIERS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5Z — CANDIDATE WEAK-LABEL TIERS")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD CORRECTED PREDICTIONS
# ------------------------------------------------------------

prediction_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_predictions_corrected_agreement.csv"
)

df = pd.read_csv(prediction_file)

print("\nEvents:", len(df))


# ------------------------------------------------------------
# 2. CREATE CANDIDATE TIERS
# ------------------------------------------------------------

# Tier A:
# At least 2 independent LFs agree with the Label Model
tier_a = (
    df["agreeing_lf_count"] >= 2
)

# Tier B:
# Exactly one LF agrees, but Label Model confidence >= 0.80
tier_b = (
    (df["agreeing_lf_count"] == 1) &
    (df["label_model_confidence"] >= 0.80)
)

# Tier C:
# Exactly one agreeing LF and moderate confidence
tier_c = (
    (df["agreeing_lf_count"] == 1) &
    (df["label_model_confidence"] >= 0.70) &
    (df["label_model_confidence"] < 0.80)
)

# Unknown:
# No LF supports the prediction
unknown = (
    df["active_lf_count"] == 0
)

df["candidate_tier"] = "UNRESOLVED"

df.loc[tier_a, "candidate_tier"] = "TIER_A"
df.loc[tier_b, "candidate_tier"] = "TIER_B"
df.loc[tier_c, "candidate_tier"] = "TIER_C"

# Any remaining single-LF low-confidence predictions
# remain unresolved/weak.
df.loc[
    (df["agreeing_lf_count"] == 1) &
    (df["label_model_confidence"] < 0.70),
    "candidate_tier"
] = "WEAK_SINGLE_LF"


# ------------------------------------------------------------
# 3. OVERALL TIER COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL CANDIDATE TIERS")
print("=" * 70)

tier_counts = (
    df["candidate_tier"]
    .value_counts()
)

for tier, count in tier_counts.items():

    print(
        f"{tier:20s}: "
        f"{count:8,} "
        f"({count / len(df) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 4. CLASS × TIER
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS × CANDIDATE TIER")
print("=" * 70)

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

class_tier = pd.crosstab(
    df["label_model_class"],
    df["candidate_tier"]
)

# Ensure consistent class ordering
class_tier = class_tier.reindex(
    list(class_names.values()),
    fill_value=0
)

display(class_tier)


# ------------------------------------------------------------
# 5. TIER × CONFIDENCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TIER × CONFIDENCE")
print("=" * 70)

tier_summary = (
    df
    .groupby("candidate_tier")
    .agg(
        Events=("event_id", "count"),
        Mean_Confidence=(
            "label_model_confidence",
            "mean"
        ),
        Median_Confidence=(
            "label_model_confidence",
            "median"
        ),
        Min_Confidence=(
            "label_model_confidence",
            "min"
        ),
        Max_Confidence=(
            "label_model_confidence",
            "max"
        ),
        Mean_Agreeing_LFs=(
            "agreeing_lf_count",
            "mean"
        ),
        Mean_Active_LFs=(
            "active_lf_count",
            "mean"
        )
    )
    .reset_index()
)

display(tier_summary)


# ------------------------------------------------------------
# 6. TIER A CLASS COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TIER A — >=2 AGREEMENT")
print("=" * 70)

tier_a_df = df[tier_a]

for class_id, class_name in class_names.items():

    count = (
        tier_a_df["label_model_label"] == class_id
    ).sum()

    print(
        f"{class_name:15s}: "
        f"{count:8,}"
    )


# ------------------------------------------------------------
# 7. TIER B CLASS COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TIER B — 1 LF + CONFIDENCE >= 0.80")
print("=" * 70)

tier_b_df = df[tier_b]

for class_id, class_name in class_names.items():

    count = (
        tier_b_df["label_model_label"] == class_id
    ).sum()

    print(
        f"{class_name:15s}: "
        f"{count:8,}"
    )


# ------------------------------------------------------------
# 8. TIER C CLASS COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TIER C — 1 LF + 0.70–0.80")
print("=" * 70)

tier_c_df = df[tier_c]

for class_id, class_name in class_names.items():

    count = (
        tier_c_df["label_model_label"] == class_id
    ).sum()

    print(
        f"{class_name:15s}: "
        f"{count:8,}"
    )


# ------------------------------------------------------------
# 9. UNKNOWN / UNRESOLVED
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNRESOLVED EVENTS")
print("=" * 70)

unknown_count = unknown.sum()

print(
    f"No active LF: "
    f"{unknown_count:,} "
    f"({unknown_count / len(df) * 100:.2f}%)"
)


# ------------------------------------------------------------
# 10. SAVE CANDIDATE TIERS
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_candidate_weak_label_tiers.csv"
)

df.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)


print("\n" + "=" * 70)
print("STEP 5Z COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5AA — INSPECT TIER A LF COMBINATIONS
# ============================================================

import pandas as pd
import numpy as np
from collections import Counter

print("=" * 70)
print("STEP 5AA — INSPECT TIER A LF COMBINATIONS")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD FINAL LF MATRIX
# ------------------------------------------------------------

matrix_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_final_L_matrix.csv"
)

L_df = pd.read_csv(matrix_file)

final_lfs = [
    "LF_GAS_1KM",
    "LF_GAS_2KM",
    "LF_GAS_REPEATED",

    "LF_IND_MULTI_500",
    "LF_IND_SCORE",
    "LF_IND_STEEL",
    "LF_IND_CEMENT",
    "LF_IND_POWER",
    "LF_IND_REPEATED",

    "LF_MINE_500",
    "LF_MINE_RECURRENT",
    "LF_MINE_STRONG",

    "LF_AGRI_CROP_70_REVISED",
    "LF_AGRI_CROP_CONTEXT_REVISED",
    "LF_AGRI_REPEATED_CROP_FINAL",

    "LF_WF_STRONG_FOREST",
    "LF_WF_REPEATED_NATURAL",
]

L = L_df[final_lfs].to_numpy(dtype=np.int8)


# ------------------------------------------------------------
# 2. LOAD CORRECTED LABEL MODEL OUTPUT
# ------------------------------------------------------------

prediction_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_predictions_corrected_agreement.csv"
)

pred_df = pd.read_csv(prediction_file)

predicted_labels = (
    pred_df["label_model_label"]
    .to_numpy(dtype=np.int8)
)


# ------------------------------------------------------------
# 3. FIND TIER A
# ------------------------------------------------------------

tier_a_mask = (
    pred_df["agreeing_lf_count"] >= 2
)

tier_a_indices = np.where(tier_a_mask)[0]

print("\nTier A events:", len(tier_a_indices))


# ------------------------------------------------------------
# 4. CREATE LF COMBINATION
# ------------------------------------------------------------

combination_counter = Counter()

for idx in tier_a_indices:

    label = predicted_labels[idx]

    supporting_lfs = [
        final_lfs[j]
        for j in range(len(final_lfs))
        if L[idx, j] == label
    ]

    key = (
        int(label),
        tuple(supporting_lfs)
    )

    combination_counter[key] += 1


# ------------------------------------------------------------
# 5. DISPLAY TOP COMBINATIONS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP TIER A LF COMBINATIONS")
print("=" * 70)

rows = []

for (label, lfs), count in combination_counter.most_common(30):

    rows.append({
        "Class": class_names.get(
            label,
            str(label)
        ) if "class_names" in globals()
        else {
            0: "Industrial",
            1: "Gas",
            2: "Agriculture",
            3: "Mining",
            4: "Wildfire"
        }.get(label, str(label)),

        "LF_Count": len(lfs),

        "Supporting_LFs": " + ".join(lfs),

        "Events": count,

        "Percent_Tier_A": (
            count / len(tier_a_indices) * 100
        )
    })

combination_df = pd.DataFrame(rows)

display(combination_df)


# ------------------------------------------------------------
# 6. CLASS-WISE LF COMBINATIONS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TIER A COMBINATIONS BY CLASS")
print("=" * 70)

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

for class_id, class_name in class_names.items():

    class_indices = [
        idx
        for idx in tier_a_indices
        if predicted_labels[idx] == class_id
    ]

    counter = Counter()

    for idx in class_indices:

        supporting_lfs = tuple(
            final_lfs[j]
            for j in range(len(final_lfs))
            if L[idx, j] == class_id
        )

        counter[supporting_lfs] += 1

    print("\n" + "-" * 70)
    print(
        f"{class_name} — "
        f"{len(class_indices):,} Tier A events"
    )
    print("-" * 70)

    for lfs, count in counter.most_common(10):

        print(
            f"{count:6,} : "
            f"{' + '.join(lfs)}"
        )


# ------------------------------------------------------------
# 7. COUNT DISTINCT LF TYPES SUPPORTING EACH EVENT
# ------------------------------------------------------------

# We define evidence families:
# Gas, Industrial, Mining, Agriculture, Wildfire

family_map = {}

for lf in final_lfs:

    if lf.startswith("LF_GAS"):
        family_map[lf] = "Gas"

    elif lf.startswith("LF_IND"):
        family_map[lf] = "Industrial"

    elif lf.startswith("LF_MINE"):
        family_map[lf] = "Mining"

    elif lf.startswith("LF_AGRI"):
        family_map[lf] = "Agriculture"

    elif lf.startswith("LF_WF"):
        family_map[lf] = "Wildfire"


family_count_rows = []

for idx in tier_a_indices:

    label = predicted_labels[idx]

    supporting_lfs = [
        final_lfs[j]
        for j in range(len(final_lfs))
        if L[idx, j] == label
    ]

    families = set(
        family_map[lf]
        for lf in supporting_lfs
    )

    family_count_rows.append({
        "event_id": pred_df.iloc[idx]["event_id"],
        "label": label,
        "class": class_names[label],
        "supporting_lf_count": len(supporting_lfs),
        "supporting_family_count": len(families),
        "supporting_families": ", ".join(
            sorted(families)
        )
    })


family_df = pd.DataFrame(
    family_count_rows
)


# ------------------------------------------------------------
# 8. FAMILY SUPPORT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TIER A — NUMBER OF SUPPORTING EVIDENCE FAMILIES")
print("=" * 70)

family_summary = (
    family_df
    .groupby(
        ["class", "supporting_family_count"]
    )
    .size()
    .reset_index(
        name="Events"
    )
)

display(family_summary)


# ------------------------------------------------------------
# 9. SAVE COMBINATION REPORT
# ------------------------------------------------------------

combination_file = (
    "/kaggle/working/"
    "firms_2025_tier_a_lf_combinations.csv"
)

family_file = (
    "/kaggle/working/"
    "firms_2025_tier_a_evidence_family_support.csv"
)

combination_df.to_csv(
    combination_file,
    index=False
)

family_df.to_csv(
    family_file,
    index=False
)

print("\nSaved:")
print(combination_file)
print(family_file)


print("\n" + "=" * 70)
print("STEP 5AA COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 5AB — WEAK LABEL EVIDENCE QUALITY
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 5AB — WEAK LABEL EVIDENCE QUALITY")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD FINAL LF MATRIX
# ------------------------------------------------------------

matrix_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_final_L_matrix.csv"
)

L_df = pd.read_csv(matrix_file)

final_lfs = [
    "LF_GAS_1KM",
    "LF_GAS_2KM",
    "LF_GAS_REPEATED",

    "LF_IND_MULTI_500",
    "LF_IND_SCORE",
    "LF_IND_STEEL",
    "LF_IND_CEMENT",
    "LF_IND_POWER",
    "LF_IND_REPEATED",

    "LF_MINE_500",
    "LF_MINE_RECURRENT",
    "LF_MINE_STRONG",

    "LF_AGRI_CROP_70_REVISED",
    "LF_AGRI_CROP_CONTEXT_REVISED",
    "LF_AGRI_REPEATED_CROP_FINAL",

    "LF_WF_STRONG_FOREST",
    "LF_WF_REPEATED_NATURAL",
]

L = L_df[final_lfs].to_numpy(dtype=np.int8)


# ------------------------------------------------------------
# 2. LOAD LABEL MODEL OUTPUT
# ------------------------------------------------------------

prediction_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_predictions_corrected_agreement.csv"
)

pred_df = pd.read_csv(prediction_file)

predicted_labels = (
    pred_df["label_model_label"]
    .to_numpy(dtype=np.int8)
)


# ------------------------------------------------------------
# 3. CLASS DEFINITIONS
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


# ------------------------------------------------------------
# 4. DEFINE EVIDENCE TYPE
# ------------------------------------------------------------

evidence_type = {

    # Gas
    "LF_GAS_1KM":
        "Source-specific spatial",

    "LF_GAS_2KM":
        "Source-specific spatial",

    "LF_GAS_REPEATED":
        "Source-specific + temporal",

    # Industrial
    "LF_IND_MULTI_500":
        "Multi-facility spatial",

    "LF_IND_SCORE":
        "Composite industrial",

    "LF_IND_STEEL":
        "Facility-specific spatial",

    "LF_IND_CEMENT":
        "Facility-specific spatial",

    "LF_IND_POWER":
        "Facility-specific spatial",

    "LF_IND_REPEATED":
        "Industrial + temporal",

    # Mining
    "LF_MINE_500":
        "Mine-specific spatial",

    "LF_MINE_RECURRENT":
        "Mine + temporal",

    "LF_MINE_STRONG":
        "Strong mine evidence",

    # Agriculture
    "LF_AGRI_CROP_70_REVISED":
        "Strong crop context",

    "LF_AGRI_CROP_CONTEXT_REVISED":
        "Crop context",

    "LF_AGRI_REPEATED_CROP_FINAL":
        "Crop + temporal",

    # Wildfire
    "LF_WF_STRONG_FOREST":
        "Strong natural/forest context",

    "LF_WF_REPEATED_NATURAL":
        "Natural vegetation + temporal",
}


# ------------------------------------------------------------
# 5. ANALYZE EVERY LF
# ------------------------------------------------------------

rows = []

for lf_index, lf in enumerate(final_lfs):

    lf_mask = L[:, lf_index] != -1

    coverage = lf_mask.sum()

    # How often does this LF agree with final prediction?
    agrees = (
        lf_mask &
        (L[:, lf_index] == predicted_labels)
    ).sum()

    # How often does it disagree?
    disagrees = (
        lf_mask &
        (L[:, lf_index] != predicted_labels)
    ).sum()

    # LF's own class
    active_values = L[lf_mask, lf_index]

    if len(active_values) > 0:

        unique, counts = np.unique(
            active_values,
            return_counts=True
        )

        dominant_class_id = int(
            unique[np.argmax(counts)]
        )

        dominant_class = class_names.get(
            dominant_class_id,
            str(dominant_class_id)
        )

    else:

        dominant_class = "None"

    rows.append({

        "LF": lf,

        "Evidence_Type":
            evidence_type[lf],

        "Coverage":
            int(coverage),

        "Coverage_Percent":
            coverage / len(L) * 100,

        "Agrees_With_Label_Model":
            int(agrees),

        "Disagrees_With_Label_Model":
            int(disagrees),

        "Agreement_Rate":
            agrees / coverage
            if coverage > 0
            else np.nan,

        "Dominant_Class":
            dominant_class
    })


lf_quality_df = pd.DataFrame(rows)


# ------------------------------------------------------------
# 6. DISPLAY LF QUALITY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LF EVIDENCE QUALITY")
print("=" * 70)

display(
    lf_quality_df.sort_values(
        "Coverage",
        ascending=False
    )
)


# ------------------------------------------------------------
# 7. CLASS-SPECIFIC EVIDENCE DIVERSITY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS-SPECIFIC EVIDENCE DIVERSITY")
print("=" * 70)

for class_id, class_name in class_names.items():

    class_mask = (
        predicted_labels == class_id
    )

    print(
        f"\n{class_name}: "
        f"{class_mask.sum():,} predicted events"
    )

    for lf_index, lf in enumerate(final_lfs):

        count = (
            class_mask &
            (L[:, lf_index] == class_id)
        ).sum()

        if count > 0:

            print(
                f"  {lf:40s}: "
                f"{count:6,}"
            )


# ------------------------------------------------------------
# 8. SAVE
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_lf_evidence_quality.csv"
)

lf_quality_df.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)


print("\n" + "=" * 70)
print("STEP 5AB COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6A — BUILD INITIAL WEAK-LABEL TRAINING DATASET
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 6A — BUILD INITIAL WEAK-LABEL TRAINING DATASET")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD CORRECTED SNORKEL OUTPUT
# ------------------------------------------------------------

prediction_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_predictions_corrected_agreement.csv"
)

df = pd.read_csv(prediction_file)

print("\nTotal events:", len(df))


# ------------------------------------------------------------
# 2. DEFINE CLASS NAMES
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


# ------------------------------------------------------------
# 3. STRONG WEAK-LABEL CRITERIA
# ------------------------------------------------------------
#
# We require:
#
#   >= 2 LFs agree with the Label Model
#   AND
#   no LF disagrees with the Label Model
#
# This prevents events with direct cross-class conflicts
# from entering the initial training set.
#
# These are still WEAK LABELS, not ground truth.
# ------------------------------------------------------------

strong_mask = (
    (df["agreeing_lf_count"] >= 2) &
    (df["disagreeing_lf_count"] == 0)
)


# ------------------------------------------------------------
# 4. CREATE TRAINING DATASET
# ------------------------------------------------------------

train_df = df.loc[
    strong_mask
].copy()

train_df["weak_label"] = (
    train_df["label_model_label"]
    .astype(np.int8)
)

train_df["weak_label_class"] = (
    train_df["label_model_class"]
)


# ------------------------------------------------------------
# 5. LABEL QUALITY CATEGORY
# ------------------------------------------------------------

train_df["label_quality"] = "STRONG_WEAK_LABEL"


# ------------------------------------------------------------
# 6. BASIC SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INITIAL TRAINING DATASET")
print("=" * 70)

print(
    f"Selected events : "
    f"{len(train_df):,}"
)

print(
    f"Percentage of all events : "
    f"{len(train_df) / len(df) * 100:.2f}%"
)


# ------------------------------------------------------------
# 7. CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WEAK-LABEL CLASS DISTRIBUTION")
print("=" * 70)

class_counts = (
    train_df["weak_label"]
    .value_counts()
    .sort_index()
)

for class_id, class_name in class_names.items():

    count = class_counts.get(class_id, 0)

    print(
        f"{class_id} - {class_name:15s}: "
        f"{count:8,} "
        f"({count / len(train_df) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 8. CONFIDENCE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WEAK-LABEL CONFIDENCE")
print("=" * 70)

print(
    "Mean   :",
    train_df["label_model_confidence"].mean()
)

print(
    "Median :",
    train_df["label_model_confidence"].median()
)

print(
    "Minimum:",
    train_df["label_model_confidence"].min()
)

print(
    "Maximum:",
    train_df["label_model_confidence"].max()
)


# ------------------------------------------------------------
# 9. CLASS-WISE CONFIDENCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS-WISE CONFIDENCE")
print("=" * 70)

confidence_summary = (
    train_df
    .groupby(
        ["weak_label", "weak_label_class"]
    )
    .agg(
        Events=("event_id", "count"),
        Mean_Confidence=(
            "label_model_confidence",
            "mean"
        ),
        Median_Confidence=(
            "label_model_confidence",
            "median"
        ),
        Min_Confidence=(
            "label_model_confidence",
            "min"
        ),
        Max_Confidence=(
            "label_model_confidence",
            "max"
        ),
        Mean_Agreeing_LFs=(
            "agreeing_lf_count",
            "mean"
        ),
    )
    .reset_index()
)

display(confidence_summary)


# ------------------------------------------------------------
# 10. AGREEMENT DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AGREEING LF DISTRIBUTION")
print("=" * 70)

agreement_counts = (
    train_df["agreeing_lf_count"]
    .value_counts()
    .sort_index()
)

for n, count in agreement_counts.items():

    print(
        f"{n} agreeing LFs: "
        f"{count:8,} "
        f"({count / len(train_df) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 11. CHECK CLASS IMBALANCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS IMBALANCE")
print("=" * 70)

largest_class = class_counts.max()
smallest_class = class_counts.min()

print(
    "Largest class:",
    class_names[class_counts.idxmax()],
    largest_class
)

print(
    "Smallest class:",
    class_names[class_counts.idxmin()],
    smallest_class
)

print(
    "Largest / smallest ratio:",
    round(largest_class / smallest_class, 2)
)


# ------------------------------------------------------------
# 12. CHECK FOR CONFLICTING EVENTS
# ------------------------------------------------------------

conflicting_selected = (
    train_df["disagreeing_lf_count"] > 0
).sum()

print("\nSelected events with conflicts:", conflicting_selected)

if conflicting_selected != 0:
    raise ValueError(
        "Training dataset contains conflicting LF votes."
    )

print("Conflict check passed.")


# ------------------------------------------------------------
# 13. SAVE INITIAL TRAINING DATASET
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_initial_weak_label_training_dataset.csv"
)

train_df.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)


print("\n" + "=" * 70)
print("STEP 6A COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6B — INSPECT TRAINING FEATURE DATASET
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 6B — INSPECT TRAINING FEATURE DATASET")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD EVENT FEATURES
# ------------------------------------------------------------

event_file = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

events = pd.read_csv(event_file)

print("\nEvent feature dataset:")
print("Rows   :", len(events))
print("Columns:", len(events.columns))


# ------------------------------------------------------------
# 2. LOAD TRAINING LABELS
# ------------------------------------------------------------

label_file = (
    "/kaggle/working/"
    "firms_2025_initial_weak_label_training_dataset.csv"
)

labels = pd.read_csv(label_file)

print("\nTraining labels:")
print("Rows:", len(labels))


# ------------------------------------------------------------
# 3. CHECK EVENT ID
# ------------------------------------------------------------

if "event_id" not in events.columns:
    raise ValueError(
        "event_id not found in event feature dataset."
    )

if "event_id" not in labels.columns:
    raise ValueError(
        "event_id not found in training label dataset."
    )

print("\nevent_id exists in both datasets.")


# ------------------------------------------------------------
# 4. MATCH TRAINING EVENTS TO FEATURES
# ------------------------------------------------------------

train = labels[
    [
        "event_id",
        "weak_label",
        "weak_label_class",
        "label_model_confidence",
        "agreeing_lf_count",
        "active_lf_count",
        "disagreeing_lf_count"
    ]
].merge(
    events,
    on="event_id",
    how="left",
    validate="one_to_one"
)

print("\nMerged training dataset:")
print("Rows   :", len(train))
print("Columns:", len(train.columns))


# ------------------------------------------------------------
# 5. VERIFY ALL EVENTS MATCHED
# ------------------------------------------------------------

missing_feature_rows = train[
    train["event_id"].isna()
].shape[0]

# Better check: find rows where all event features failed
# by checking the number of columns from events.
feature_only_cols = [
    c for c in events.columns
    if c != "event_id"
]

missing_feature_count = train[
    feature_only_cols
].isna().all(axis=1).sum()

print(
    "\nEvents with no event-feature match:",
    missing_feature_count
)

if missing_feature_count > 0:
    raise ValueError(
        "Some training events could not be matched "
        "to event features."
    )


# ------------------------------------------------------------
# 6. IDENTIFY LABEL-RELATED COLUMNS
# ------------------------------------------------------------

label_related_keywords = [
    "label_model",
    "weak_label",
    "candidate_tier",
    "agreeing_lf",
    "disagreeing_lf",
    "active_lf",
    "LF_"
]

label_leakage_columns = []

for col in train.columns:

    col_lower = col.lower()

    if any(
        keyword.lower() in col_lower
        for keyword in label_related_keywords
    ):
        label_leakage_columns.append(col)


print("\n" + "=" * 70)
print("POTENTIAL LABEL-LEAKAGE COLUMNS")
print("=" * 70)

for col in label_leakage_columns:
    print(col)

print(
    "\nTotal potential leakage columns:",
    len(label_leakage_columns)
)


# ------------------------------------------------------------
# 7. EVENT FEATURE COLUMNS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EVENT FEATURE COLUMNS")
print("=" * 70)

event_feature_columns = [
    c for c in events.columns
    if c != "event_id"
]

print(
    "Total event feature columns:",
    len(event_feature_columns)
)

for i, col in enumerate(event_feature_columns, 1):
    print(f"{i:3d}. {col}")


# ------------------------------------------------------------
# 8. NUMERIC FEATURE SUMMARY
# ------------------------------------------------------------

numeric_features = events[
    event_feature_columns
].select_dtypes(
    include=[np.number]
).columns.tolist()

print("\n" + "=" * 70)
print("NUMERIC FEATURES")
print("=" * 70)

print(
    "Numeric feature count:",
    len(numeric_features)
)

for col in numeric_features:
    print(col)


# ------------------------------------------------------------
# 9. MISSINGNESS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE MISSINGNESS")
print("=" * 70)

missing_summary = pd.DataFrame({
    "Feature": event_feature_columns,
    "Missing_Count": [
        train[c].isna().sum()
        for c in event_feature_columns
    ]
})

missing_summary["Missing_Percent"] = (
    missing_summary["Missing_Count"]
    / len(train)
    * 100
)

missing_summary = missing_summary.sort_values(
    "Missing_Percent",
    ascending=False
)

display(
    missing_summary.head(30)
)


# ------------------------------------------------------------
# 10. SAVE MERGED TRAINING DATASET
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_training_labels_with_event_features.csv"
)

train.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)


print("\n" + "=" * 70)
print("STEP 6B COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6C — FEATURE AUDIT AND MODEL-FEATURE GROUPING
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 6C — FEATURE AUDIT AND MODEL-FEATURE GROUPING")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD MERGED TRAINING DATA
# ------------------------------------------------------------

train_file = (
    "/kaggle/working/"
    "firms_2025_training_labels_with_event_features.csv"
)

df = pd.read_csv(train_file)

print("\nRows   :", len(df))
print("Columns:", len(df.columns))


# ------------------------------------------------------------
# 2. FEATURE GROUP RULES
# ------------------------------------------------------------

def assign_group(col):

    c = col.lower()

    # IDs / metadata
    if (
        c == "event_id"
        or c.endswith("_id")
        or c.endswith("_name")
        or c.endswith("_state")
        or c.endswith("_operator")
        or c.endswith("_location")
        or c in [
            "nearest_flare_field_type",
            "nearest_flare_field_name"
        ]
    ):
        return "IDENTIFIER_OR_CATEGORICAL"

    # Dates
    if "datetime" in c:
        return "DATETIME"

    # Label leakage
    leakage_terms = [
        "weak_label",
        "label_model",
        "agreeing_lf",
        "disagreeing_lf",
        "active_lf",
        "candidate_tier",
        "lf_"
    ]

    if any(term in c for term in leakage_terms):
        return "LABEL_LEAKAGE"

    # FIRMS behavior
    if c.startswith("event_"):
        if any(
            x in c for x in [
                "frp",
                "brightness",
                "acquisition_hour",
                "satellite",
                "instrument"
            ]
        ):
            return "FIRMS_FRP_BRIGHTNESS"

        if any(
            x in c for x in [
                "latitude",
                "longitude",
                "spatial",
                "block_count",
                "per_500m",
                "per_1km"
            ]
        ):
            return "SPATIAL_BEHAVIOR"

        if any(
            x in c for x in [
                "duration",
                "gap",
                "active_day",
                "active_month",
                "peak_month",
                "monthly",
                "detections_per"
            ]
        ):
            return "TEMPORAL_BEHAVIOR"

        if any(
            x in c for x in [
                "same_cell",
                "nearby_fire"
            ]
        ):
            return "FIRE_ACTIVITY_CONTEXT"

    # Gas
    if (
        "flare" in c
        or "gas_flare" in c
        or "oil_well" in c
        or "gasometer" in c
    ):
        return "GAS_EVIDENCE"

    # Industrial
    if any(
        x in c for x in [
            "steel",
            "cement",
            "wri_power",
            "fertilizer",
            "refinery",
            "petro",
            "industrial"
        ]
    ):
        return "INDUSTRIAL_EVIDENCE"

    # Mining
    if any(
        x in c for x in [
            "coal_mine",
            "mineshaft",
            "adit",
            "mine",
            "quarry",
            "mining"
        ]
    ):
        return "MINING_EVIDENCE"

    # Agriculture
    if any(
        x in c for x in [
            "agri",
            "agriculture",
            "farmland",
            "farmyard",
            "orchard",
            "vineyard",
            "nursery",
            "greenhouse",
            "allotment",
            "crop"
        ]
    ):
        return "AGRICULTURE_EVIDENCE"

    # Wildfire / natural vegetation
    if any(
        x in c for x in [
            "forest",
            "scrub",
            "grassland",
            "heath",
            "natural_vegetation",
            "natural_landcover",
            "vegetation",
            "dw_trees",
            "dw_grass",
            "dw_shrub",
            "dw_flooded"
        ]
    ):
        return "WILDFIRE_VEGETATION_EVIDENCE"

    return "OTHER"


# ------------------------------------------------------------
# 3. AUDIT ALL COLUMNS
# ------------------------------------------------------------

audit_rows = []

for col in df.columns:

    audit_rows.append({
        "Feature": col,
        "Dtype": str(df[col].dtype),
        "Group": assign_group(col),
        "Missing_Count": int(df[col].isna().sum()),
        "Missing_Percent":
            df[col].isna().mean() * 100,
        "Unique_Values":
            df[col].nunique(dropna=True)
    })

audit_df = pd.DataFrame(audit_rows)


# ------------------------------------------------------------
# 4. GROUP SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FEATURE GROUP SUMMARY")
print("=" * 70)

group_summary = (
    audit_df
    .groupby("Group")
    .agg(
        Features=("Feature", "count"),
        Missing_Features=(
            "Missing_Count",
            lambda x: (x > 0).sum()
        )
    )
    .sort_values(
        "Features",
        ascending=False
    )
)

display(group_summary)


# ------------------------------------------------------------
# 5. FEATURES BY GROUP
# ------------------------------------------------------------

groups = audit_df["Group"].unique()

for group in sorted(groups):

    subset = audit_df[
        audit_df["Group"] == group
    ]

    print("\n" + "-" * 70)
    print(
        f"{group} "
        f"({len(subset)} features)"
    )
    print("-" * 70)

    for feature in subset["Feature"]:
        print(feature)


# ------------------------------------------------------------
# 6. CONSTANT FEATURES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONSTANT FEATURES")
print("=" * 70)

constant_features = [
    col
    for col in df.columns
    if df[col].nunique(dropna=True) <= 1
]

print(
    "Number of constant features:",
    len(constant_features)
)

for col in constant_features:
    print(col)


# ------------------------------------------------------------
# 7. 100% MISSING FEATURES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("100% MISSING FEATURES")
print("=" * 70)

fully_missing = [
    col
    for col in df.columns
    if df[col].isna().all()
]

for col in fully_missing:
    print(col)

print(
    "\nCount:",
    len(fully_missing)
)


# ------------------------------------------------------------
# 8. NUMERIC FEATURE COUNT BY GROUP
# ------------------------------------------------------------

numeric_mask = (
    audit_df["Dtype"]
    .str.contains(
        "int|float",
        regex=True
    )
)

numeric_group_summary = (
    audit_df[numeric_mask]
    .groupby("Group")
    .size()
    .sort_values(
        ascending=False
    )
)

print("\n" + "=" * 70)
print("NUMERIC FEATURES BY GROUP")
print("=" * 70)

display(
    numeric_group_summary
)


# ------------------------------------------------------------
# 9. SAVE FEATURE AUDIT
# ------------------------------------------------------------

audit_file = (
    "/kaggle/working/"
    "firms_2025_model_feature_audit.csv"
)

audit_df.to_csv(
    audit_file,
    index=False
)

print("\nSaved:")
print(audit_file)


print("\n" + "=" * 70)
print("STEP 6C COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6D — BUILD CLEAN ML FEATURE MATRIX
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 6D — BUILD CLEAN ML FEATURE MATRIX")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD TRAINING DATA
# ------------------------------------------------------------

train_file = (
    "/kaggle/working/"
    "firms_2025_training_labels_with_event_features.csv"
)

df = pd.read_csv(train_file)

print("\nTraining events:", len(df))


# ------------------------------------------------------------
# 2. TARGET
# ------------------------------------------------------------

target_column = "weak_label"

y = df[target_column].astype(np.int8)

print("\nTarget:", target_column)

print("\nTarget distribution:")

target_counts = y.value_counts().sort_index()

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

for class_id, class_name in class_names.items():

    count = target_counts.get(class_id, 0)

    print(
        f"{class_id} - {class_name:15s}: "
        f"{count:8,}"
    )


# ------------------------------------------------------------
# 3. COLUMNS THAT MUST NEVER ENTER X
# ------------------------------------------------------------

# These directly contain the target or information generated
# by the Snorkel labeling process.

label_columns = [
    "weak_label",
    "weak_label_class",
    "label_model_confidence",
    "agreeing_lf_count",
    "active_lf_count",
    "disagreeing_lf_count",
]

# Identifiers / categorical metadata
identifier_columns = [
    "event_id",
    "nearest_flare_id",
    "nearest_flare_field_type",
    "nearest_flare_location",
    "nearest_flare_field_name",
    "nearest_flare_operator",
    "nearest_gas_flare_id",
    "nearest_coal_mine_id",
    "nearest_coal_mine_name",
    "nearest_coal_mine_state",
]

# Datetimes
datetime_columns = [
    "event_start_datetime",
    "event_end_datetime",
]


# ------------------------------------------------------------
# 4. START WITH NUMERIC FEATURES ONLY
# ------------------------------------------------------------

excluded_columns = (
    label_columns +
    identifier_columns +
    datetime_columns
)

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns.tolist()

feature_columns = [
    col
    for col in numeric_columns
    if col not in excluded_columns
]


print("\nInitial numeric model features:", len(feature_columns))


# ------------------------------------------------------------
# 5. REMOVE CONSTANT FEATURES
# ------------------------------------------------------------

constant_features = []

for col in feature_columns:

    if df[col].nunique(dropna=True) <= 1:
        constant_features.append(col)

feature_columns = [
    col
    for col in feature_columns
    if col not in constant_features
]

print(
    "Removed constant features:",
    len(constant_features)
)

for col in constant_features:
    print("  ", col)


# ------------------------------------------------------------
# 6. REMOVE 100% MISSING FEATURES
# ------------------------------------------------------------

fully_missing = [
    col
    for col in feature_columns
    if df[col].isna().all()
]

feature_columns = [
    col
    for col in feature_columns
    if col not in fully_missing
]

print(
    "\nRemoved 100% missing features:",
    len(fully_missing)
)

for col in fully_missing:
    print("  ", col)


# ------------------------------------------------------------
# 7. CREATE X
# ------------------------------------------------------------

X = df[
    feature_columns
].copy()

print("\n" + "=" * 70)
print("FINAL FEATURE MATRIX")
print("=" * 70)

print(
    "X shape:",
    X.shape
)

print(
    "Number of features:",
    X.shape[1]
)


# ------------------------------------------------------------
# 8. CHECK DATA TYPES
# ------------------------------------------------------------

non_numeric = X.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print(
    "\nNon-numeric features:",
    len(non_numeric)
)

if non_numeric:
    print(non_numeric)
    raise ValueError(
        "Non-numeric features remain in X."
    )

print("All X features are numeric.")


# ------------------------------------------------------------
# 9. CHECK MISSINGNESS
# ------------------------------------------------------------

missing_summary = pd.DataFrame({
    "Feature": X.columns,
    "Missing_Count": X.isna().sum(),
    "Missing_Percent":
        X.isna().mean() * 100
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    "Missing_Percent",
    ascending=False
)

print("\n" + "=" * 70)
print("REMAINING MISSING VALUES")
print("=" * 70)

if len(missing_summary) == 0:

    print("No missing values.")

else:

    display(missing_summary)


# ------------------------------------------------------------
# 10. CHECK INFINITE VALUES
# ------------------------------------------------------------

infinite_counts = np.isinf(
    X.to_numpy(
        dtype=np.float64
    )
).sum()

print(
    "\nInfinite values:",
    infinite_counts
)

if infinite_counts > 0:

    raise ValueError(
        "Infinite values found in X."
    )


# ------------------------------------------------------------
# 11. CHECK TARGET VALIDITY
# ------------------------------------------------------------

valid_classes = set(class_names.keys())

actual_classes = set(
    y.unique()
)

print(
    "\nTarget classes:",
    sorted(actual_classes)
)

if not actual_classes.issubset(valid_classes):

    raise ValueError(
        "Unexpected target class found."
    )


# ------------------------------------------------------------
# 12. SAVE FEATURE COLUMN LIST
# ------------------------------------------------------------

feature_list_file = (
    "/kaggle/working/"
    "firms_2025_model_feature_columns.csv"
)

pd.DataFrame({
    "feature": feature_columns
}).to_csv(
    feature_list_file,
    index=False
)


# ------------------------------------------------------------
# 13. SAVE CLEAN DATASET
# ------------------------------------------------------------

clean_df = X.copy()

clean_df.insert(
    0,
    "event_id",
    df["event_id"].values
)

clean_df["target"] = y.values

clean_file = (
    "/kaggle/working/"
    "firms_2025_clean_ml_training_dataset.csv"
)

clean_df.to_csv(
    clean_file,
    index=False
)


# ------------------------------------------------------------
# 14. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(
    "Events:",
    len(clean_df)
)

print(
    "Features:",
    len(feature_columns)
)

print(
    "Target classes:",
    len(actual_classes)
)

print(
    "Feature list saved:",
    feature_list_file
)

print(
    "Clean dataset saved:",
    clean_file
)


print("\n" + "=" * 70)
print("STEP 6D COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6E — TRAIN / VALIDATION / TEST SPLIT
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

print("=" * 70)
print("STEP 6E — TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD CLEAN DATASET
# ------------------------------------------------------------

clean_file = (
    "/kaggle/working/"
    "firms_2025_clean_ml_training_dataset.csv"
)

df = pd.read_csv(clean_file)

print("\nDataset shape:", df.shape)


# ------------------------------------------------------------
# 2. SEPARATE ID / FEATURES / TARGET
# ------------------------------------------------------------

event_ids = df["event_id"].copy()

feature_columns = [
    c for c in df.columns
    if c not in ["event_id", "target"]
]

X = df[feature_columns].copy()
y = df["target"].astype(np.int8)


print("Features:", X.shape[1])
print("Events  :", X.shape[0])


# ------------------------------------------------------------
# 3. FIRST SPLIT
#
# 70% TRAIN
# 30% TEMPORARY
# ------------------------------------------------------------

X_train, X_temp, y_train, y_temp, id_train, id_temp = (
    train_test_split(
        X,
        y,
        event_ids,
        test_size=0.30,
        stratify=y,
        random_state=42
    )
)


# ------------------------------------------------------------
# 4. SECOND SPLIT
#
# TEMPORARY → 15% VALIDATION + 15% TEST
# ------------------------------------------------------------

X_val, X_test, y_val, y_test, id_val, id_test = (
    train_test_split(
        X_temp,
        y_temp,
        id_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=42
    )
)


# ------------------------------------------------------------
# 5. PRINT SPLIT SIZES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPLIT SIZES")
print("=" * 70)

print(
    f"Train      : {len(X_train):,} "
    f"({len(X_train) / len(df) * 100:.2f}%)"
)

print(
    f"Validation : {len(X_val):,} "
    f"({len(X_val) / len(df) * 100:.2f}%)"
)

print(
    f"Test       : {len(X_test):,} "
    f"({len(X_test) / len(df) * 100:.2f}%)"
)


# ------------------------------------------------------------
# 6. CLASS DISTRIBUTION
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

split_data = {
    "Train": y_train,
    "Validation": y_val,
    "Test": y_test
}

distribution_rows = []

for split_name, labels in split_data.items():

    counts = labels.value_counts().sort_index()

    print(f"\n{split_name}")

    for class_id, class_name in class_names.items():

        count = counts.get(class_id, 0)

        print(
            f"  {class_id} - {class_name:15s}: "
            f"{count:6,} "
            f"({count / len(labels) * 100:6.2f}%)"
        )

        distribution_rows.append({
            "Split": split_name,
            "Class_ID": class_id,
            "Class": class_name,
            "Count": int(count),
            "Percent": count / len(labels) * 100
        })


distribution_df = pd.DataFrame(
    distribution_rows
)


# ------------------------------------------------------------
# 7. CHECK STRATIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STRATIFICATION CHECK")
print("=" * 70)

overall_pct = (
    y.value_counts(normalize=True)
    .sort_index()
)

for class_id, class_name in class_names.items():

    train_pct = (
        y_train.value_counts(normalize=True)
        .get(class_id, 0)
    )

    val_pct = (
        y_val.value_counts(normalize=True)
        .get(class_id, 0)
    )

    test_pct = (
        y_test.value_counts(normalize=True)
        .get(class_id, 0)
    )

    original_pct = overall_pct.get(
        class_id,
        0
    )

    print(
        f"{class_name:15s} | "
        f"Overall {original_pct * 100:6.2f}% | "
        f"Train {train_pct * 100:6.2f}% | "
        f"Val {val_pct * 100:6.2f}% | "
        f"Test {test_pct * 100:6.2f}%"
    )


# ------------------------------------------------------------
# 8. FIT IMPUTER ONLY ON TRAINING DATA
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING-VALUE IMPUTATION")
print("=" * 70)

missing_train = X_train.isna().sum().sum()
missing_val = X_val.isna().sum().sum()
missing_test = X_test.isna().sum().sum()

print(
    "Missing values before imputation:"
)

print("  Train:", missing_train)
print("  Val  :", missing_val)
print("  Test :", missing_test)


# Median imputation is robust to the highly skewed
# distance / FRP / count features in this dataset.

imputer = SimpleImputer(
    strategy="median"
)

X_train_imp = imputer.fit_transform(X_train)

X_val_imp = imputer.transform(X_val)

X_test_imp = imputer.transform(X_test)


# ------------------------------------------------------------
# 9. VERIFY IMPUTATION
# ------------------------------------------------------------

print("\nMissing values after imputation:")

print(
    "  Train:",
    np.isnan(X_train_imp).sum()
)

print(
    "  Val  :",
    np.isnan(X_val_imp).sum()
)

print(
    "  Test :",
    np.isnan(X_test_imp).sum()
)

if (
    np.isnan(X_train_imp).sum() > 0 or
    np.isnan(X_val_imp).sum() > 0 or
    np.isnan(X_test_imp).sum() > 0
):
    raise ValueError(
        "Missing values remain after imputation."
    )

print("Imputation check passed.")


# ------------------------------------------------------------
# 10. SAVE SPLIT IDS
# ------------------------------------------------------------

split_ids = pd.concat([
    pd.DataFrame({
        "event_id": id_train.values,
        "split": "train"
    }),

    pd.DataFrame({
        "event_id": id_val.values,
        "split": "validation"
    }),

    pd.DataFrame({
        "event_id": id_test.values,
        "split": "test"
    })
], ignore_index=True)


split_file = (
    "/kaggle/working/"
    "firms_2025_ml_train_val_test_event_ids.csv"
)

split_ids.to_csv(
    split_file,
    index=False
)


# ------------------------------------------------------------
# 11. SAVE IMPUTER STATISTICS
# ------------------------------------------------------------

imputer_stats = pd.DataFrame({
    "feature": feature_columns,
    "median_value": imputer.statistics_
})

imputer_file = (
    "/kaggle/working/"
    "firms_2025_ml_imputer_statistics.csv"
)

imputer_stats.to_csv(
    imputer_file,
    index=False
)


# ------------------------------------------------------------
# 12. SAVE DISTRIBUTION
# ------------------------------------------------------------

distribution_file = (
    "/kaggle/working/"
    "firms_2025_ml_split_class_distribution.csv"
)

distribution_df.to_csv(
    distribution_file,
    index=False
)


# ------------------------------------------------------------
# 13. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(split_file)
print(imputer_file)
print(distribution_file)


print("\n" + "=" * 70)
print("STEP 6E COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6F — RANDOM FOREST BASELINE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

print("=" * 70)
print("STEP 6F — RANDOM FOREST BASELINE")
print("=" * 70)


# ------------------------------------------------------------
# 1. RECREATE TRAIN / VALIDATION / TEST DATA
# ------------------------------------------------------------

clean_file = (
    "/kaggle/working/"
    "firms_2025_clean_ml_training_dataset.csv"
)

split_file = (
    "/kaggle/working/"
    "firms_2025_ml_train_val_test_event_ids.csv"
)

imputer_file = (
    "/kaggle/working/"
    "firms_2025_ml_imputer_statistics.csv"
)

df = pd.read_csv(clean_file)

split_ids = pd.read_csv(split_file)

imputer_stats = pd.read_csv(imputer_file)


# ------------------------------------------------------------
# 2. FEATURE COLUMNS
# ------------------------------------------------------------

feature_columns = (
    imputer_stats["feature"]
    .tolist()
)

X = df[feature_columns].copy()
y = df["target"].astype(np.int8)

event_ids = df["event_id"]


# ------------------------------------------------------------
# 3. RECONSTRUCT SPLITS USING SAVED EVENT IDS
# ------------------------------------------------------------

train_ids = set(
    split_ids.loc[
        split_ids["split"] == "train",
        "event_id"
    ]
)

val_ids = set(
    split_ids.loc[
        split_ids["split"] == "validation",
        "event_id"
    ]
)

test_ids = set(
    split_ids.loc[
        split_ids["split"] == "test",
        "event_id"
    ]
)

train_mask = event_ids.isin(train_ids)
val_mask = event_ids.isin(val_ids)
test_mask = event_ids.isin(test_ids)


X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_val = X.loc[val_mask].copy()
y_val = y.loc[val_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()


# ------------------------------------------------------------
# 4. MEDIAN IMPUTATION
#
# Use the medians calculated from the training split only.
# ------------------------------------------------------------

for col, median_value in zip(
    imputer_stats["feature"],
    imputer_stats["median_value"]
):

    X_train[col] = X_train[col].fillna(
        median_value
    )

    X_val[col] = X_val[col].fillna(
        median_value
    )

    X_test[col] = X_test[col].fillna(
        median_value
    )


# ------------------------------------------------------------
# 5. VERIFY
# ------------------------------------------------------------

print("\nSplit shapes:")

print(
    "Train:",
    X_train.shape,
    y_train.shape
)

print(
    "Validation:",
    X_val.shape,
    y_val.shape
)

print(
    "Test:",
    X_test.shape,
    y_test.shape
)

print("\nRemaining missing values:")

print(
    "Train:",
    X_train.isna().sum().sum()
)

print(
    "Validation:",
    X_val.isna().sum().sum()
)

print(
    "Test:",
    X_test.isna().sum().sum()
)


# ------------------------------------------------------------
# 6. RANDOM FOREST
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING RANDOM FOREST")
print("=" * 70)

model = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=2
)

model.fit(
    X_train,
    y_train
)

print("Random Forest training complete.")


# ------------------------------------------------------------
# 7. VALIDATION PREDICTIONS
# ------------------------------------------------------------

val_pred = model.predict(X_val)

val_proba = model.predict_proba(X_val)


# ------------------------------------------------------------
# 8. VALIDATION METRICS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

print(
    "Accuracy:",
    round(
        accuracy_score(y_val, val_pred),
        4
    )
)

print(
    "Balanced Accuracy:",
    round(
        balanced_accuracy_score(
            y_val,
            val_pred
        ),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_val,
            val_pred,
            average="macro",
            zero_division=0
        ),
        4
    )
)

print(
    "Macro Precision:",
    round(
        precision_score(
            y_val,
            val_pred,
            average="macro",
            zero_division=0
        ),
        4
    )
)

print(
    "Macro Recall:",
    round(
        recall_score(
            y_val,
            val_pred,
            average="macro",
            zero_division=0
        ),
        4
    )
)


# ------------------------------------------------------------
# 9. CLASSIFICATION REPORT
# ------------------------------------------------------------

class_names = [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]

print("\n" + "=" * 70)
print("VALIDATION CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_val,
        val_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 10. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_val,
    val_pred,
    labels=[0, 1, 2, 3, 4]
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print("\n" + "=" * 70)
print("VALIDATION CONFUSION MATRIX")
print("=" * 70)

display(cm_df)


# ------------------------------------------------------------
# 11. PER-CLASS RECALL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PER-CLASS RECALL")
print("=" * 70)

recalls = recall_score(
    y_val,
    val_pred,
    labels=[0, 1, 2, 3, 4],
    average=None,
    zero_division=0
)

for class_id, class_name, value in zip(
    range(5),
    class_names,
    recalls
):

    print(
        f"{class_id} - {class_name:15s}: "
        f"{value:.4f}"
    )


# ------------------------------------------------------------
# 12. FEATURE IMPORTANCE
# ------------------------------------------------------------

importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("TOP 30 FEATURE IMPORTANCES")
print("=" * 70)

display(
    importance_df.head(30)
)


# ------------------------------------------------------------
# 13. SAVE BASELINE MODEL RESULTS
# ------------------------------------------------------------

importance_file = (
    "/kaggle/working/"
    "firms_2025_random_forest_feature_importance.csv"
)

importance_df.to_csv(
    importance_file,
    index=False
)


val_predictions = pd.DataFrame({
    "event_id": event_ids.loc[val_mask].values,
    "true_label": y_val.values,
    "predicted_label": val_pred,
    "prediction_confidence":
        val_proba.max(axis=1)
})

prediction_file = (
    "/kaggle/working/"
    "firms_2025_random_forest_validation_predictions.csv"
)

val_predictions.to_csv(
    prediction_file,
    index=False
)


print("\nSaved:")
print(importance_file)
print(prediction_file)


print("\n" + "=" * 70)
print("STEP 6F COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6G — EVIDENCE-HOLDOUT BASELINE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

print("=" * 70)
print("STEP 6G — EVIDENCE-HOLDOUT BASELINE")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD CLEAN DATA
# ------------------------------------------------------------

clean_file = (
    "/kaggle/working/"
    "firms_2025_clean_ml_training_dataset.csv"
)

split_file = (
    "/kaggle/working/"
    "firms_2025_ml_train_val_test_event_ids.csv"
)

imputer_file = (
    "/kaggle/working/"
    "firms_2025_ml_imputer_statistics.csv"
)

audit_file = (
    "/kaggle/working/"
    "firms_2025_model_feature_audit.csv"
)

df = pd.read_csv(clean_file)
split_ids = pd.read_csv(split_file)
imputer_stats = pd.read_csv(imputer_file)
audit = pd.read_csv(audit_file)


# ------------------------------------------------------------
# 2. INITIAL FEATURES
# ------------------------------------------------------------

feature_columns = (
    imputer_stats["feature"]
    .tolist()
)

print("\nInitial features:", len(feature_columns))


# ------------------------------------------------------------
# 3. REMOVE DIRECT SOURCE-EVIDENCE GROUPS
# ------------------------------------------------------------

holdout_groups = [
    "GAS_EVIDENCE",
    "INDUSTRIAL_EVIDENCE",
    "MINING_EVIDENCE",
    "AGRICULTURE_EVIDENCE",
    "WILDFIRE_VEGETATION_EVIDENCE",
]


evidence_features = audit.loc[
    audit["Group"].isin(holdout_groups),
    "Feature"
].tolist()


# Only retain features that actually exist in the model matrix.
evidence_features = [
    c for c in evidence_features
    if c in feature_columns
]


holdout_features = [
    c
    for c in feature_columns
    if c not in evidence_features
]


print("\nDirect source-evidence features removed:")
print(len(evidence_features))

print("\nRemaining features:")
print(len(holdout_features))


print("\nRemoved feature groups:")

for group in holdout_groups:

    count = sum(
        audit["Group"].eq(group) &
        audit["Feature"].isin(feature_columns)
    )

    print(
        f"  {group:35s}: {count:3d}"
    )


# ------------------------------------------------------------
# 4. CREATE X AND y
# ------------------------------------------------------------

X = df[holdout_features].copy()

y = df["target"].astype(np.int8)

event_ids = df["event_id"]


# ------------------------------------------------------------
# 5. RECREATE SPLITS
# ------------------------------------------------------------

train_ids = set(
    split_ids.loc[
        split_ids["split"] == "train",
        "event_id"
    ]
)

val_ids = set(
    split_ids.loc[
        split_ids["split"] == "validation",
        "event_id"
    ]
)

test_ids = set(
    split_ids.loc[
        split_ids["split"] == "test",
        "event_id"
    ]
)


train_mask = event_ids.isin(train_ids)
val_mask = event_ids.isin(val_ids)
test_mask = event_ids.isin(test_ids)


X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_val = X.loc[val_mask].copy()
y_val = y.loc[val_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()


# ------------------------------------------------------------
# 6. IMPUTE USING TRAINING MEDIANS ONLY
# ------------------------------------------------------------

median_lookup = dict(
    zip(
        imputer_stats["feature"],
        imputer_stats["median_value"]
    )
)


for col in holdout_features:

    median_value = median_lookup[col]

    X_train[col] = X_train[col].fillna(
        median_value
    )

    X_val[col] = X_val[col].fillna(
        median_value
    )

    X_test[col] = X_test[col].fillna(
        median_value
    )


# ------------------------------------------------------------
# 7. VERIFY
# ------------------------------------------------------------

print("\nSplit shapes:")

print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)


print("\nMissing values:")

print(
    "Train:",
    X_train.isna().sum().sum()
)

print(
    "Val:",
    X_val.isna().sum().sum()
)

print(
    "Test:",
    X_test.isna().sum().sum()
)


# ------------------------------------------------------------
# 8. TRAIN RANDOM FOREST
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING EVIDENCE-HOLDOUT RANDOM FOREST")
print("=" * 70)


model_holdout = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=2
)


model_holdout.fit(
    X_train,
    y_train
)


print("Training complete.")


# ------------------------------------------------------------
# 9. VALIDATION PREDICTION
# ------------------------------------------------------------

val_pred = model_holdout.predict(X_val)

val_proba = model_holdout.predict_proba(X_val)


# ------------------------------------------------------------
# 10. METRICS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EVIDENCE-HOLDOUT VALIDATION RESULTS")
print("=" * 70)


accuracy = accuracy_score(
    y_val,
    val_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_val,
    val_pred
)

macro_f1 = f1_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)

macro_precision = precision_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)


print(
    f"Accuracy          : {accuracy:.4f}"
)

print(
    f"Balanced Accuracy : {balanced_accuracy:.4f}"
)

print(
    f"Macro F1          : {macro_f1:.4f}"
)

print(
    f"Macro Precision   : {macro_precision:.4f}"
)

print(
    f"Macro Recall      : {macro_recall:.4f}"
)


# ------------------------------------------------------------
# 11. CLASSIFICATION REPORT
# ------------------------------------------------------------

class_names = [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]


print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)


print(
    classification_report(
        y_val,
        val_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 12. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_val,
    val_pred,
    labels=[0, 1, 2, 3, 4]
)


cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)


print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

display(cm_df)


# ------------------------------------------------------------
# 13. PER-CLASS RECALL
# ------------------------------------------------------------

recalls = recall_score(
    y_val,
    val_pred,
    labels=[0, 1, 2, 3, 4],
    average=None,
    zero_division=0
)


print("\n" + "=" * 70)
print("PER-CLASS RECALL")
print("=" * 70)


for class_id, class_name, value in zip(
    range(5),
    class_names,
    recalls
):

    print(
        f"{class_id} - {class_name:15s}: "
        f"{value:.4f}"
    )


# ------------------------------------------------------------
# 14. FEATURE IMPORTANCE
# ------------------------------------------------------------

importance_df = pd.DataFrame({
    "Feature": holdout_features,
    "Importance":
        model_holdout.feature_importances_
})


importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)


print("\n" + "=" * 70)
print("TOP 30 NON-EVIDENCE FEATURE IMPORTANCES")
print("=" * 70)

display(
    importance_df.head(30)
)


# ------------------------------------------------------------
# 15. SAVE RESULTS
# ------------------------------------------------------------

importance_file = (
    "/kaggle/working/"
    "firms_2025_evidence_holdout_feature_importance.csv"
)

importance_df.to_csv(
    importance_file,
    index=False
)


prediction_file = (
    "/kaggle/working/"
    "firms_2025_evidence_holdout_validation_predictions.csv"
)


pd.DataFrame({
    "event_id": event_ids.loc[val_mask].values,
    "true_label": y_val.values,
    "predicted_label": val_pred,
    "prediction_confidence":
        val_proba.max(axis=1)
}).to_csv(
    prediction_file,
    index=False
)


features_file = (
    "/kaggle/working/"
    "firms_2025_evidence_holdout_features.csv"
)


pd.DataFrame({
    "feature": holdout_features
}).to_csv(
    features_file,
    index=False
)


print("\nSaved:")
print(importance_file)
print(prediction_file)
print(features_file)


print("\n" + "=" * 70)
print("STEP 6G COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6H — SPATIAL LEAKAGE / GEOGRAPHIC HOLDOUT TEST
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

print("=" * 70)
print("STEP 6H — SPATIAL LEAKAGE / GEOGRAPHIC HOLDOUT TEST")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD CLEAN DATASET
# ------------------------------------------------------------

clean_file = (
    "/kaggle/working/"
    "firms_2025_clean_ml_training_dataset.csv"
)

imputer_file = (
    "/kaggle/working/"
    "firms_2025_ml_imputer_statistics.csv"
)

df = pd.read_csv(clean_file)
imputer_stats = pd.read_csv(imputer_file)

print("\nDataset shape:", df.shape)


# ------------------------------------------------------------
# 2. VERIFY REQUIRED COORDINATE FEATURES EXIST
# ------------------------------------------------------------

required_coordinates = [
    "event_mean_latitude",
    "event_mean_longitude"
]

missing_coordinates = [
    c
    for c in required_coordinates
    if c not in df.columns
]

if missing_coordinates:

    raise ValueError(
        "Required coordinate features are missing: "
        + str(missing_coordinates)
    )

print(
    "\nCoordinate features found:",
    required_coordinates
)


# ------------------------------------------------------------
# 3. VERIFY COORDINATE VALUES
# ------------------------------------------------------------

if (
    df["event_mean_latitude"].isna().any()
    or
    df["event_mean_longitude"].isna().any()
):

    raise ValueError(
        "Missing latitude/longitude values found."
    )

print(
    "All events have valid coordinates."
)


# ------------------------------------------------------------
# 4. CREATE COARSE GEOGRAPHIC GROUP
#
# 0.25 degree × 0.25 degree cells.
#
# Events in the same geographic cell are kept together.
# ------------------------------------------------------------

GRID_SIZE = 0.25

df["geo_lat_group"] = np.floor(
    df["event_mean_latitude"] /
    GRID_SIZE
).astype(np.int32)

df["geo_lon_group"] = np.floor(
    df["event_mean_longitude"] /
    GRID_SIZE
).astype(np.int32)

df["geo_group"] = (
    df["geo_lat_group"].astype(str)
    + "_"
    + df["geo_lon_group"].astype(str)
)


print("\n" + "=" * 70)
print("GEOGRAPHIC GROUPING")
print("=" * 70)

print(
    "Grid size:",
    GRID_SIZE,
    "degrees"
)

print(
    "Geographic groups:",
    df["geo_group"].nunique()
)


# ------------------------------------------------------------
# 5. GROUP-LEVEL CLASS COUNTS
# ------------------------------------------------------------

group_counts = (
    df.groupby(
        "geo_group"
    )["target"]
    .value_counts()
    .unstack(
        fill_value=0
    )
)

for class_id in range(5):

    if class_id not in group_counts.columns:

        group_counts[class_id] = 0


group_counts = group_counts[
    [0, 1, 2, 3, 4]
]

group_counts["total"] = (
    group_counts.sum(axis=1)
)


# ------------------------------------------------------------
# 6. RANDOMLY SHUFFLE GEOGRAPHIC GROUPS
# ------------------------------------------------------------

rng = np.random.RandomState(42)

groups = group_counts.index.to_numpy()

rng.shuffle(groups)


# ------------------------------------------------------------
# 7. BUILD ~20% VALIDATION SET
#
# Groups are added as complete units.
# Therefore a geographic group can NEVER occur in both
# training and validation.
# ------------------------------------------------------------

target_validation_size = (
    len(df) * 0.20
)

validation_groups = []

validation_size = 0


for group in groups:

    if validation_size >= target_validation_size:

        break

    group_size = int(
        group_counts.loc[
            group,
            "total"
        ]
    )

    validation_groups.append(
        group
    )

    validation_size += group_size


validation_groups = set(
    validation_groups
)


# ------------------------------------------------------------
# 8. CREATE TRAIN / VALIDATION MASKS
# ------------------------------------------------------------

val_mask = df[
    "geo_group"
].isin(
    validation_groups
)

train_mask = ~val_mask


# ------------------------------------------------------------
# 9. CHECK SPLIT SIZE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPATIAL SPLIT SIZE")
print("=" * 70)

print(
    "Total events:",
    len(df)
)

print(
    "Training events:",
    train_mask.sum()
)

print(
    "Validation events:",
    val_mask.sum()
)

print(
    "Validation percentage:",
    round(
        val_mask.mean() * 100,
        2
    )
)

print(
    "Training geographic groups:",
    df.loc[
        train_mask,
        "geo_group"
    ].nunique()
)

print(
    "Validation geographic groups:",
    df.loc[
        val_mask,
        "geo_group"
    ].nunique()
)


# ------------------------------------------------------------
# 10. VERIFY ZERO GEOGRAPHIC OVERLAP
# ------------------------------------------------------------

train_groups = set(
    df.loc[
        train_mask,
        "geo_group"
    ]
)

val_groups = set(
    df.loc[
        val_mask,
        "geo_group"
    ]
)

overlap = (
    train_groups &
    val_groups
)

print(
    "\nGeographic group overlap:",
    len(overlap)
)

if len(overlap) != 0:

    raise ValueError(
        "SPATIAL LEAKAGE DETECTED."
    )

print(
    "No geographic group overlap."
)


# ------------------------------------------------------------
# 11. CLASS DISTRIBUTION
# ------------------------------------------------------------

class_names = [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]

print("\n" + "=" * 70)
print("SPATIAL CLASS DISTRIBUTION")
print("=" * 70)

for class_id, class_name in enumerate(
    class_names
):

    train_count = int(
        (
            df.loc[
                train_mask,
                "target"
            ] == class_id
        ).sum()
    )

    val_count = int(
        (
            df.loc[
                val_mask,
                "target"
            ] == class_id
        ).sum()
    )

    print(
        f"{class_id} - {class_name:15s}: "
        f"Train={train_count:5d} | "
        f"Val={val_count:5d}"
    )


# ------------------------------------------------------------
# 12. FEATURES
#
# IMPORTANT:
# This is the 137-feature dataset from Step 6D.
# We are NOT using LF/label-model columns.
# ------------------------------------------------------------

feature_columns = (
    imputer_stats[
        "feature"
    ].tolist()
)

X = df[
    feature_columns
].copy()

y = df[
    "target"
].astype(
    np.int8
)


# ------------------------------------------------------------
# 13. TRAIN / VALIDATION DATA
# ------------------------------------------------------------

X_train = X.loc[
    train_mask
].copy()

y_train = y.loc[
    train_mask
].copy()

X_val = X.loc[
    val_mask
].copy()

y_val = y.loc[
    val_mask
].copy()


# ------------------------------------------------------------
# 14. IMPUTE USING TRAINING MEDIANS
# ------------------------------------------------------------

median_lookup = dict(
    zip(
        imputer_stats[
            "feature"
        ],
        imputer_stats[
            "median_value"
        ]
    )
)


for col in feature_columns:

    median_value = median_lookup[
        col
    ]

    X_train[col] = (
        X_train[col]
        .fillna(
            median_value
        )
    )

    X_val[col] = (
        X_val[col]
        .fillna(
            median_value
        )
    )


# ------------------------------------------------------------
# 15. TRAIN RANDOM FOREST
#
# This is the same evidence-holdout model as Step 6G.
# We are testing geographic generalization.
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING SPATIAL-HOLDOUT RANDOM FOREST")
print("=" * 70)

model_spatial = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=2
)

model_spatial.fit(
    X_train,
    y_train
)

print(
    "Training complete."
)


# ------------------------------------------------------------
# 16. VALIDATION PREDICTION
# ------------------------------------------------------------

val_pred = model_spatial.predict(
    X_val
)


# ------------------------------------------------------------
# 17. CALCULATE METRICS
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_val,
    val_pred
)

balanced_accuracy = (
    balanced_accuracy_score(
        y_val,
        val_pred
    )
)

macro_f1 = f1_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)

macro_precision = precision_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)


# ------------------------------------------------------------
# 18. RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPATIAL-HOLDOUT RESULTS")
print("=" * 70)

print(
    f"Accuracy          : {accuracy:.4f}"
)

print(
    f"Balanced Accuracy : {balanced_accuracy:.4f}"
)

print(
    f"Macro F1          : {macro_f1:.4f}"
)

print(
    f"Macro Precision   : {macro_precision:.4f}"
)

print(
    f"Macro Recall      : {macro_recall:.4f}"
)


# ------------------------------------------------------------
# 19. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPATIAL-HOLDOUT CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_val,
        val_pred,
        labels=[
            0,
            1,
            2,
            3,
            4
        ],
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 20. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_val,
    val_pred,
    labels=[
        0,
        1,
        2,
        3,
        4
    ]
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print("\n" + "=" * 70)
print("SPATIAL-HOLDOUT CONFUSION MATRIX")
print("=" * 70)

display(
    cm_df
)


# ------------------------------------------------------------
# 21. SAVE SPATIAL SPLIT
# ------------------------------------------------------------

spatial_split_file = (
    "/kaggle/working/"
    "firms_2025_spatial_holdout_event_ids.csv"
)

pd.DataFrame({
    "event_id":
        df["event_id"].values,

    "geo_group":
        df["geo_group"].values,

    "split":
        np.where(
            val_mask,
            "spatial_validation",
            "spatial_train"
        )
}).to_csv(
    spatial_split_file,
    index=False
)


print("\nSaved:")
print(
    spatial_split_file
)


print("\n" + "=" * 70)
print("STEP 6H COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6I — SPATIAL + EVIDENCE HOLDOUT TEST
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

print("=" * 70)
print("STEP 6I — SPATIAL + EVIDENCE HOLDOUT TEST")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

clean_file = (
    "/kaggle/working/"
    "firms_2025_clean_ml_training_dataset.csv"
)

imputer_file = (
    "/kaggle/working/"
    "firms_2025_ml_imputer_statistics.csv"
)

audit_file = (
    "/kaggle/working/"
    "firms_2025_model_feature_audit.csv"
)

df = pd.read_csv(clean_file)

imputer_stats = pd.read_csv(
    imputer_file
)

audit = pd.read_csv(
    audit_file
)


# ------------------------------------------------------------
# 2. SOURCE-EVIDENCE GROUPS
# ------------------------------------------------------------

holdout_groups = [
    "GAS_EVIDENCE",
    "INDUSTRIAL_EVIDENCE",
    "MINING_EVIDENCE",
    "AGRICULTURE_EVIDENCE",
    "WILDFIRE_VEGETATION_EVIDENCE"
]


all_features = (
    imputer_stats[
        "feature"
    ].tolist()
)


evidence_features = audit.loc[
    audit["Group"].isin(
        holdout_groups
    ),
    "Feature"
].tolist()


evidence_features = [
    c
    for c in evidence_features
    if c in all_features
]


model_features = [
    c
    for c in all_features
    if c not in evidence_features
]


print("\nTotal features:", len(all_features))

print(
    "Evidence features removed:",
    len(evidence_features)
)

print(
    "Remaining model features:",
    len(model_features)
)


# ------------------------------------------------------------
# 3. VERIFY EXPECTED FEATURE COUNT
# ------------------------------------------------------------

if len(model_features) != 65:

    print(
        "\nWARNING:"
    )

    print(
        "Expected approximately 65 "
        "non-evidence features."
    )

    print(
        "Actual:",
        len(model_features)
    )


# ------------------------------------------------------------
# 4. VERIFY COORDINATES
# ------------------------------------------------------------

coordinate_columns = [
    "event_mean_latitude",
    "event_mean_longitude"
]

for col in coordinate_columns:

    if col not in df.columns:

        raise ValueError(
            f"Missing coordinate feature: {col}"
        )

    if df[col].isna().any():

        raise ValueError(
            f"Missing coordinate values: {col}"
        )


# ------------------------------------------------------------
# 5. CREATE GEOGRAPHIC GROUPS
# ------------------------------------------------------------

GRID_SIZE = 0.25

df["geo_lat_group"] = np.floor(
    df["event_mean_latitude"] /
    GRID_SIZE
).astype(np.int32)

df["geo_lon_group"] = np.floor(
    df["event_mean_longitude"] /
    GRID_SIZE
).astype(np.int32)

df["geo_group"] = (
    df["geo_lat_group"].astype(str)
    + "_"
    + df["geo_lon_group"].astype(str)
)


print("\n" + "=" * 70)
print("GEOGRAPHIC GROUPING")
print("=" * 70)

print(
    "Grid size:",
    GRID_SIZE,
    "degrees"
)

print(
    "Total geographic groups:",
    df["geo_group"].nunique()
)


# ------------------------------------------------------------
# 6. GROUP SIZE TABLE
# ------------------------------------------------------------

group_counts = (
    df.groupby(
        "geo_group"
    )["target"]
    .value_counts()
    .unstack(
        fill_value=0
    )
)


for class_id in range(5):

    if class_id not in group_counts.columns:

        group_counts[class_id] = 0


group_counts = group_counts[
    [0, 1, 2, 3, 4]
]

group_counts["total"] = (
    group_counts.sum(axis=1)
)


# ------------------------------------------------------------
# 7. SELECT VALIDATION GROUPS
#
# Target approximately 20%.
# Geographic groups are indivisible.
# ------------------------------------------------------------

rng = np.random.RandomState(42)

groups = group_counts.index.to_numpy()

rng.shuffle(groups)


target_validation_size = (
    len(df) * 0.20
)

validation_groups = []

validation_size = 0


for group in groups:

    if (
        validation_size
        >= target_validation_size
    ):
        break

    group_size = int(
        group_counts.loc[
            group,
            "total"
        ]
    )

    validation_groups.append(
        group
    )

    validation_size += group_size


validation_groups = set(
    validation_groups
)


# ------------------------------------------------------------
# 8. CREATE SPLIT
# ------------------------------------------------------------

val_mask = df[
    "geo_group"
].isin(
    validation_groups
)

train_mask = ~val_mask


print("\n" + "=" * 70)
print("SPATIAL SPLIT")
print("=" * 70)

print(
    "Training events:",
    int(train_mask.sum())
)

print(
    "Validation events:",
    int(val_mask.sum())
)

print(
    "Validation percentage:",
    round(
        val_mask.mean() * 100,
        2
    )
)


# ------------------------------------------------------------
# 9. CHECK GEOGRAPHIC OVERLAP
# ------------------------------------------------------------

train_groups = set(
    df.loc[
        train_mask,
        "geo_group"
    ]
)

val_groups = set(
    df.loc[
        val_mask,
        "geo_group"
    ]
)

overlap = (
    train_groups &
    val_groups
)

print(
    "Geographic group overlap:",
    len(overlap)
)

if len(overlap) != 0:

    raise ValueError(
        "Geographic leakage detected."
    )

print(
    "No geographic group overlap."
)


# ------------------------------------------------------------
# 10. CLASS DISTRIBUTION
# ------------------------------------------------------------

class_names = [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

for class_id, class_name in enumerate(
    class_names
):

    train_count = int(
        (
            df.loc[
                train_mask,
                "target"
            ] == class_id
        ).sum()
    )

    val_count = int(
        (
            df.loc[
                val_mask,
                "target"
            ] == class_id
        ).sum()
    )

    print(
        f"{class_id} - {class_name:15s}: "
        f"Train={train_count:5d} | "
        f"Val={val_count:5d}"
    )


# ------------------------------------------------------------
# 11. BUILD X AND y
# ------------------------------------------------------------

X = df[
    model_features
].copy()

y = df[
    "target"
].astype(
    np.int8
)


X_train = X.loc[
    train_mask
].copy()

y_train = y.loc[
    train_mask
].copy()

X_val = X.loc[
    val_mask
].copy()

y_val = y.loc[
    val_mask
].copy()


# ------------------------------------------------------------
# 12. IMPUTE USING TRAINING MEDIANS ONLY
# ------------------------------------------------------------

median_lookup = dict(
    zip(
        imputer_stats[
            "feature"
        ],
        imputer_stats[
            "median_value"
        ]
    )
)


for col in model_features:

    median_value = median_lookup[
        col
    ]

    X_train[col] = (
        X_train[col]
        .fillna(
            median_value
        )
    )

    X_val[col] = (
        X_val[col]
        .fillna(
            median_value
        )
    )


# ------------------------------------------------------------
# 13. TRAIN RANDOM FOREST
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING SPATIAL + EVIDENCE-HOLDOUT MODEL")
print("=" * 70)

model_spatial_holdout = (
    RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2
    )
)

model_spatial_holdout.fit(
    X_train,
    y_train
)

print(
    "Training complete."
)


# ------------------------------------------------------------
# 14. PREDICTION
# ------------------------------------------------------------

val_pred = (
    model_spatial_holdout.predict(
        X_val
    )
)


# ------------------------------------------------------------
# 15. METRICS
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_val,
    val_pred
)

balanced_accuracy = (
    balanced_accuracy_score(
        y_val,
        val_pred
    )
)

macro_f1 = f1_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)

macro_precision = precision_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)


print("\n" + "=" * 70)
print("SPATIAL + EVIDENCE-HOLDOUT RESULTS")
print("=" * 70)

print(
    f"Accuracy          : {accuracy:.4f}"
)

print(
    f"Balanced Accuracy : {balanced_accuracy:.4f}"
)

print(
    f"Macro F1          : {macro_f1:.4f}"
)

print(
    f"Macro Precision   : {macro_precision:.4f}"
)

print(
    f"Macro Recall      : {macro_recall:.4f}"
)


# ------------------------------------------------------------
# 16. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_val,
        val_pred,
        labels=[
            0,
            1,
            2,
            3,
            4
        ],
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 17. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_val,
    val_pred,
    labels=[
        0,
        1,
        2,
        3,
        4
    ]
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)


print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

display(
    cm_df
)


# ------------------------------------------------------------
# 18. FEATURE IMPORTANCE
# ------------------------------------------------------------

importance_df = pd.DataFrame({
    "Feature": model_features,
    "Importance":
        model_spatial_holdout.feature_importances_
})

importance_df = (
    importance_df
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 70)
print("TOP 30 NON-EVIDENCE FEATURES")
print("=" * 70)

display(
    importance_df.head(30)
)


# ------------------------------------------------------------
# 19. SAVE RESULTS
# ------------------------------------------------------------

split_file = (
    "/kaggle/working/"
    "firms_2025_spatial_evidence_holdout_event_ids.csv"
)

pd.DataFrame({
    "event_id":
        df["event_id"].values,

    "geo_group":
        df["geo_group"].values,

    "split":
        np.where(
            val_mask,
            "spatial_validation",
            "spatial_train"
        )
}).to_csv(
    split_file,
    index=False
)


importance_file = (
    "/kaggle/working/"
    "firms_2025_spatial_evidence_holdout_feature_importance.csv"
)

importance_df.to_csv(
    importance_file,
    index=False
)


print("\nSaved:")
print(split_file)
print(importance_file)


print("\n" + "=" * 70)
print("STEP 6I COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6J — GEOGRAPHY-FREE SPATIAL + EVIDENCE HOLDOUT
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

print("=" * 70)
print("STEP 6J — GEOGRAPHY-FREE SPATIAL + EVIDENCE HOLDOUT")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------

clean_file = (
    "/kaggle/working/"
    "firms_2025_clean_ml_training_dataset.csv"
)

imputer_file = (
    "/kaggle/working/"
    "firms_2025_ml_imputer_statistics.csv"
)

audit_file = (
    "/kaggle/working/"
    "firms_2025_model_feature_audit.csv"
)

df = pd.read_csv(clean_file)

imputer_stats = pd.read_csv(
    imputer_file
)

audit = pd.read_csv(
    audit_file
)


# ------------------------------------------------------------
# 2. ALL 137 FEATURES
# ------------------------------------------------------------

all_features = (
    imputer_stats[
        "feature"
    ].tolist()
)


# ------------------------------------------------------------
# 3. REMOVE DIRECT SOURCE-EVIDENCE FEATURES
# ------------------------------------------------------------

evidence_groups = [
    "GAS_EVIDENCE",
    "INDUSTRIAL_EVIDENCE",
    "MINING_EVIDENCE",
    "AGRICULTURE_EVIDENCE",
    "WILDFIRE_VEGETATION_EVIDENCE"
]


evidence_features = audit.loc[
    audit["Group"].isin(
        evidence_groups
    ),
    "Feature"
].tolist()


evidence_features = [
    c
    for c in evidence_features
    if c in all_features
]


non_evidence_features = [
    c
    for c in all_features
    if c not in evidence_features
]


print("\nAll features:", len(all_features))

print(
    "Direct evidence features:",
    len(evidence_features)
)

print(
    "Non-evidence features:",
    len(non_evidence_features)
)


# ------------------------------------------------------------
# 4. REMOVE GEOGRAPHIC LOCATION FEATURES
#
# These allow the model to learn regional distributions.
# ------------------------------------------------------------

geographic_features = [
    "event_mean_latitude",
    "event_mean_longitude",
    "event_min_latitude",
    "event_max_latitude",
    "event_min_longitude",
    "event_max_longitude",
    "event_latitude_span_deg",
    "event_longitude_span_deg",
    "event_latitude_span_km",
    "event_longitude_span_km"
]


geographic_features = [
    c
    for c in geographic_features
    if c in non_evidence_features
]


model_features = [
    c
    for c in non_evidence_features
    if c not in geographic_features
]


print("\n" + "=" * 70)
print("GEOGRAPHIC FEATURES REMOVED")
print("=" * 70)

print(
    "Number removed:",
    len(geographic_features)
)

for feature in geographic_features:
    print(
        "  ",
        feature
    )


print(
    "\nFinal model features:",
    len(model_features)
)


# ------------------------------------------------------------
# 5. VERIFY EXPECTED FEATURE COUNT
# ------------------------------------------------------------

if len(model_features) != 55:

    print(
        "\nWARNING:"
    )

    print(
        "Expected 55 features "
        "(65 - 10 geographic features)."
    )

    print(
        "Actual:",
        len(model_features)
    )


# ------------------------------------------------------------
# 6. CREATE GEOGRAPHIC GROUPS
#
# Coordinates are used ONLY to create the holdout groups.
# They are NOT given to the ML model.
# ------------------------------------------------------------

if (
    "event_mean_latitude" not in df.columns
    or
    "event_mean_longitude" not in df.columns
):

    raise ValueError(
        "Coordinate columns required "
        "for spatial grouping are missing."
    )


if (
    df["event_mean_latitude"].isna().any()
    or
    df["event_mean_longitude"].isna().any()
):

    raise ValueError(
        "Missing coordinates detected."
    )


GRID_SIZE = 0.25


df["geo_lat_group"] = np.floor(
    df["event_mean_latitude"] /
    GRID_SIZE
).astype(np.int32)


df["geo_lon_group"] = np.floor(
    df["event_mean_longitude"] /
    GRID_SIZE
).astype(np.int32)


df["geo_group"] = (
    df["geo_lat_group"].astype(str)
    + "_"
    + df["geo_lon_group"].astype(str)
)


print("\n" + "=" * 70)
print("SPATIAL GROUPING")
print("=" * 70)

print(
    "Grid size:",
    GRID_SIZE,
    "degrees"
)

print(
    "Geographic groups:",
    df["geo_group"].nunique()
)


# ------------------------------------------------------------
# 7. GROUP COUNTS
# ------------------------------------------------------------

group_counts = (
    df.groupby(
        "geo_group"
    )["target"]
    .value_counts()
    .unstack(
        fill_value=0
    )
)


for class_id in range(5):

    if class_id not in group_counts.columns:

        group_counts[class_id] = 0


group_counts = group_counts[
    [0, 1, 2, 3, 4]
]


group_counts["total"] = (
    group_counts.sum(axis=1)
)


# ------------------------------------------------------------
# 8. SELECT VALIDATION GROUPS
# ------------------------------------------------------------

rng = np.random.RandomState(42)

groups = (
    group_counts
    .index
    .to_numpy()
)

rng.shuffle(groups)


target_validation_size = (
    len(df) * 0.20
)

validation_groups = []

validation_size = 0


for group in groups:

    if (
        validation_size
        >= target_validation_size
    ):
        break

    group_size = int(
        group_counts.loc[
            group,
            "total"
        ]
    )

    validation_groups.append(
        group
    )

    validation_size += group_size


validation_groups = set(
    validation_groups
)


# ------------------------------------------------------------
# 9. TRAIN / VALIDATION MASK
# ------------------------------------------------------------

val_mask = df[
    "geo_group"
].isin(
    validation_groups
)

train_mask = ~val_mask


print("\n" + "=" * 70)
print("SPATIAL SPLIT")
print("=" * 70)

print(
    "Training events:",
    int(train_mask.sum())
)

print(
    "Validation events:",
    int(val_mask.sum())
)

print(
    "Validation percentage:",
    round(
        val_mask.mean() * 100,
        2
    )
)


# ------------------------------------------------------------
# 10. VERIFY NO GEOGRAPHIC OVERLAP
# ------------------------------------------------------------

train_groups = set(
    df.loc[
        train_mask,
        "geo_group"
    ]
)

val_groups = set(
    df.loc[
        val_mask,
        "geo_group"
    ]
)

overlap = (
    train_groups &
    val_groups
)


print(
    "Geographic group overlap:",
    len(overlap)
)


if len(overlap) != 0:

    raise ValueError(
        "Geographic leakage detected."
    )


print(
    "No geographic group overlap."
)


# ------------------------------------------------------------
# 11. CLASS DISTRIBUTION
# ------------------------------------------------------------

class_names = [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]


print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)


for class_id, class_name in enumerate(
    class_names
):

    train_count = int(
        (
            df.loc[
                train_mask,
                "target"
            ] == class_id
        ).sum()
    )

    val_count = int(
        (
            df.loc[
                val_mask,
                "target"
            ] == class_id
        ).sum()
    )

    print(
        f"{class_id} - {class_name:15s}: "
        f"Train={train_count:5d} | "
        f"Val={val_count:5d}"
    )


# ------------------------------------------------------------
# 12. BUILD X / y
#
# IMPORTANT:
# Geographic features are NOT included.
# Coordinates are used only for splitting.
# ------------------------------------------------------------

X = df[
    model_features
].copy()


y = df[
    "target"
].astype(
    np.int8
)


X_train = X.loc[
    train_mask
].copy()


y_train = y.loc[
    train_mask
].copy()


X_val = X.loc[
    val_mask
].copy()


y_val = y.loc[
    val_mask
].copy()


# ------------------------------------------------------------
# 13. IMPUTATION
# ------------------------------------------------------------

median_lookup = dict(
    zip(
        imputer_stats[
            "feature"
        ],
        imputer_stats[
            "median_value"
        ]
    )
)


for col in model_features:

    median_value = median_lookup[
        col
    ]

    X_train[col] = (
        X_train[col]
        .fillna(
            median_value
        )
    )

    X_val[col] = (
        X_val[col]
        .fillna(
            median_value
        )
    )


# ------------------------------------------------------------
# 14. VERIFY NO GEOGRAPHIC FEATURES ENTERED X
# ------------------------------------------------------------

remaining_geographic = [
    c
    for c in model_features
    if c in geographic_features
]


if remaining_geographic:

    raise ValueError(
        "Geographic features still remain: "
        + str(remaining_geographic)
    )


print("\nNo geographic features in X.")


# ------------------------------------------------------------
# 15. TRAIN RANDOM FOREST
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING GEOGRAPHY-FREE RANDOM FOREST")
print("=" * 70)


model_geo_free = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=2
)


model_geo_free.fit(
    X_train,
    y_train
)


print(
    "Training complete."
)


# ------------------------------------------------------------
# 16. VALIDATION PREDICTION
# ------------------------------------------------------------

val_pred = (
    model_geo_free.predict(
        X_val
    )
)


# ------------------------------------------------------------
# 17. METRICS
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_val,
    val_pred
)


balanced_accuracy = (
    balanced_accuracy_score(
        y_val,
        val_pred
    )
)


macro_f1 = f1_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)


macro_precision = precision_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)


macro_recall = recall_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)


print("\n" + "=" * 70)
print("GEOGRAPHY-FREE RESULTS")
print("=" * 70)


print(
    f"Accuracy          : {accuracy:.4f}"
)


print(
    f"Balanced Accuracy : {balanced_accuracy:.4f}"
)


print(
    f"Macro F1          : {macro_f1:.4f}"
)


print(
    f"Macro Precision   : {macro_precision:.4f}"
)


print(
    f"Macro Recall      : {macro_recall:.4f}"
)


# ------------------------------------------------------------
# 18. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)


print(
    classification_report(
        y_val,
        val_pred,
        labels=[
            0,
            1,
            2,
            3,
            4
        ],
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 19. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_val,
    val_pred,
    labels=[
        0,
        1,
        2,
        3,
        4
    ]
)


cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)


print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)


display(
    cm_df
)


# ------------------------------------------------------------
# 20. FEATURE IMPORTANCE
# ------------------------------------------------------------

importance_df = pd.DataFrame({
    "Feature": model_features,
    "Importance":
        model_geo_free.feature_importances_
})


importance_df = (
    importance_df
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 70)
print("TOP 30 GEOGRAPHY-FREE FEATURES")
print("=" * 70)


display(
    importance_df.head(30)
)


# ------------------------------------------------------------
# 21. SAVE RESULTS
# ------------------------------------------------------------

split_file = (
    "/kaggle/working/"
    "firms_2025_geography_free_spatial_holdout_event_ids.csv"
)


pd.DataFrame({
    "event_id":
        df["event_id"].values,

    "geo_group":
        df["geo_group"].values,

    "split":
        np.where(
            val_mask,
            "spatial_validation",
            "spatial_train"
        )
}).to_csv(
    split_file,
    index=False
)


importance_file = (
    "/kaggle/working/"
    "firms_2025_geography_free_feature_importance.csv"
)


importance_df.to_csv(
    importance_file,
    index=False
)


features_file = (
    "/kaggle/working/"
    "firms_2025_geography_free_features.csv"
)


pd.DataFrame({
    "feature":
        model_features
}).to_csv(
    features_file,
    index=False
)


print("\nSaved:")
print(split_file)
print(importance_file)
print(features_file)


print("\n" + "=" * 70)
print("STEP 6J COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6K — WEAK-LABEL GEOGRAPHIC DISTRIBUTION AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 6K — WEAK-LABEL GEOGRAPHIC DISTRIBUTION AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD TRAINING DATA
# ------------------------------------------------------------

clean_file = (
    "/kaggle/working/"
    "firms_2025_clean_ml_training_dataset.csv"
)

df = pd.read_csv(
    clean_file
)


# ------------------------------------------------------------
# 2. CREATE 0.25 DEGREE GEOGRAPHIC GROUP
# ------------------------------------------------------------

GRID_SIZE = 0.25

df["geo_lat_group"] = np.floor(
    df["event_mean_latitude"] /
    GRID_SIZE
).astype(np.int32)

df["geo_lon_group"] = np.floor(
    df["event_mean_longitude"] /
    GRID_SIZE
).astype(np.int32)

df["geo_group"] = (
    df["geo_lat_group"].astype(str)
    + "_"
    + df["geo_lon_group"].astype(str)
)


# ------------------------------------------------------------
# 3. CLASS NAMES
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


# ------------------------------------------------------------
# 4. OVERALL GEOGRAPHIC SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL GEOGRAPHIC DISTRIBUTION")
print("=" * 70)

print(
    "Total events:",
    len(df)
)

print(
    "Geographic groups:",
    df["geo_group"].nunique()
)


# ------------------------------------------------------------
# 5. NUMBER OF GEOGRAPHIC GROUPS PER CLASS
# ------------------------------------------------------------

class_group_summary = []

for class_id, class_name in class_names.items():

    subset = df[
        df["target"] == class_id
    ]

    class_group_summary.append({
        "Class_ID": class_id,
        "Class": class_name,
        "Events": len(subset),
        "Geographic_Groups":
            subset["geo_group"].nunique()
    })


class_group_df = pd.DataFrame(
    class_group_summary
)

class_group_df[
    "Events_per_Geographic_Group"
] = (
    class_group_df["Events"] /
    class_group_df["Geographic_Groups"]
)


display(
    class_group_df
)


# ------------------------------------------------------------
# 6. CLASS CONCENTRATION
#
# For each class:
# What percentage of its events are located in its
# top 10 geographic groups?
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GEOGRAPHIC CONCENTRATION BY CLASS")
print("=" * 70)

concentration_rows = []


for class_id, class_name in class_names.items():

    subset = df[
        df["target"] == class_id
    ]

    group_counts = (
        subset["geo_group"]
        .value_counts()
    )

    total = len(subset)

    top5 = (
        group_counts
        .head(5)
        .sum()
    )

    top10 = (
        group_counts
        .head(10)
        .sum()
    )

    top25 = (
        group_counts
        .head(25)
        .sum()
    )

    top50 = (
        group_counts
        .head(50)
        .sum()
    )

    concentration_rows.append({
        "Class_ID": class_id,
        "Class": class_name,
        "Top_5_Percent":
            top5 / total * 100
            if total > 0 else 0,

        "Top_10_Percent":
            top10 / total * 100
            if total > 0 else 0,

        "Top_25_Percent":
            top25 / total * 100
            if total > 0 else 0,

        "Top_50_Percent":
            top50 / total * 100
            if total > 0 else 0
    })


concentration_df = pd.DataFrame(
    concentration_rows
)

display(
    concentration_df
)


# ------------------------------------------------------------
# 7. GEOGRAPHIC GROUP CLASS MIX
# ------------------------------------------------------------

group_class_counts = (
    df.groupby(
        ["geo_group", "target"]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


for class_id in range(5):

    if class_id not in group_class_counts.columns:

        group_class_counts[class_id] = 0


group_class_counts = (
    group_class_counts[
        [0, 1, 2, 3, 4]
    ]
)


group_class_counts.columns = [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]


group_class_counts[
    "Total"
] = group_class_counts.sum(
    axis=1
)


# ------------------------------------------------------------
# 8. DOMINANT CLASS PER GEOGRAPHIC GROUP
# ------------------------------------------------------------

dominant_class = (
    group_class_counts[
        [
            "Industrial",
            "Gas",
            "Agriculture",
            "Mining",
            "Wildfire"
        ]
    ]
    .idxmax(axis=1)
)


group_class_counts[
    "Dominant_Class"
] = dominant_class


dominant_distribution = (
    group_class_counts[
        "Dominant_Class"
    ]
    .value_counts()
)


print("\n" + "=" * 70)
print("DOMINANT CLASS BY GEOGRAPHIC GROUP")
print("=" * 70)

display(
    dominant_distribution
    .rename_axis("Class")
    .reset_index(
        name="Geographic_Groups"
    )
)


# ------------------------------------------------------------
# 9. PURE GEOGRAPHIC GROUPS
#
# A geographic group is "pure" if all weak-labelled events
# inside it belong to the same class.
# ------------------------------------------------------------

pure_groups = (
    group_class_counts[
        [
            "Industrial",
            "Gas",
            "Agriculture",
            "Mining",
            "Wildfire"
        ]
    ]
    .gt(0)
    .sum(axis=1)
    == 1
)


mixed_groups = ~pure_groups


print("\n" + "=" * 70)
print("GEOGRAPHIC GROUP PURITY")
print("=" * 70)

print(
    "Pure groups:",
    int(pure_groups.sum())
)

print(
    "Mixed groups:",
    int(mixed_groups.sum())
)

print(
    "Pure percentage:",
    round(
        pure_groups.mean() * 100,
        2
    )
)


# ------------------------------------------------------------
# 10. CLASS-PAIR CO-OCCURRENCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS CO-OCCURRENCE IN SAME GEOGRAPHIC GROUP")
print("=" * 70)


classes = [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]


pair_rows = []


for i in range(len(classes)):

    for j in range(i + 1, len(classes)):

        a = classes[i]
        b = classes[j]

        both = (
            (group_class_counts[a] > 0)
            &
            (group_class_counts[b] > 0)
        )

        pair_rows.append({
            "Class_A": a,
            "Class_B": b,
            "Groups_With_Both":
                int(both.sum())
        })


pair_df = pd.DataFrame(
    pair_rows
)

display(
    pair_df
)


# ------------------------------------------------------------
# 11. SAVE RESULTS
# ------------------------------------------------------------

class_group_file = (
    "/kaggle/working/"
    "firms_2025_weak_label_geographic_class_summary.csv"
)

concentration_file = (
    "/kaggle/working/"
    "firms_2025_weak_label_geographic_concentration.csv"
)

group_mix_file = (
    "/kaggle/working/"
    "firms_2025_weak_label_geographic_group_class_mix.csv"
)

pair_file = (
    "/kaggle/working/"
    "firms_2025_weak_label_geographic_class_pairs.csv"
)


class_group_df.to_csv(
    class_group_file,
    index=False
)

concentration_df.to_csv(
    concentration_file,
    index=False
)

group_class_counts.reset_index().to_csv(
    group_mix_file,
    index=False
)

pair_df.to_csv(
    pair_file,
    index=False
)


print("\nSaved:")
print(class_group_file)
print(concentration_file)
print(group_mix_file)
print(pair_file)


print("\n" + "=" * 70)
print("STEP 6K COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 6L — HIGH-CONFIDENCE LABEL GEOGRAPHIC DIVERSITY AUDIT
# CORRECTED FOR ACTUAL LABEL MODEL COLUMN NAMES
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 6L — HIGH-CONFIDENCE LABEL GEOGRAPHIC DIVERSITY AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD CLEAN TRAINING DATA
# ------------------------------------------------------------

clean_file = (
    "/kaggle/working/"
    "firms_2025_clean_ml_training_dataset.csv"
)

df = pd.read_csv(clean_file)

print("\nClean dataset:", df.shape)


# ------------------------------------------------------------
# 2. LOAD LABEL MODEL PREDICTIONS
# ------------------------------------------------------------

prediction_file = (
    "/kaggle/working/"
    "firms_2025_snorkel_label_model_predictions.csv"
)

pred = pd.read_csv(prediction_file)

print("Prediction dataset:", pred.shape)


# ------------------------------------------------------------
# 3. ACTUAL COLUMNS IN YOUR FILE
# ------------------------------------------------------------

print("\nPrediction columns:")
print(pred.columns.tolist())


# ------------------------------------------------------------
# 4. USE THE CORRECT COLUMNS
# ------------------------------------------------------------

event_id_col = "event_id"

label_col = "label_model_label"

confidence_col = "label_model_confidence"


required_prediction_columns = [
    event_id_col,
    label_col,
    confidence_col
]

missing_columns = [
    c for c in required_prediction_columns
    if c not in pred.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ------------------------------------------------------------
# 5. MERGE PREDICTIONS WITH GEOGRAPHIC COORDINATES
# ------------------------------------------------------------

coords = df[
    [
        "event_id",
        "event_mean_latitude",
        "event_mean_longitude"
    ]
].copy()


audit_df = pred.merge(
    coords,
    on="event_id",
    how="inner",
    validate="one_to_one"
)


print(
    "\nMerged rows:",
    len(audit_df)
)


# ------------------------------------------------------------
# 6. KEEP VALID SOURCE CLASSES
# ------------------------------------------------------------

audit_df = audit_df[
    audit_df[label_col].isin(
        [0, 1, 2, 3, 4]
    )
].copy()


print(
    "Valid classified events:",
    len(audit_df)
)


# ------------------------------------------------------------
# 7. CREATE 0.25 DEGREE GEOGRAPHIC GROUP
# ------------------------------------------------------------

GRID_SIZE = 0.25


audit_df["geo_lat_group"] = np.floor(
    audit_df["event_mean_latitude"] /
    GRID_SIZE
).astype(np.int32)


audit_df["geo_lon_group"] = np.floor(
    audit_df["event_mean_longitude"] /
    GRID_SIZE
).astype(np.int32)


audit_df["geo_group"] = (
    audit_df["geo_lat_group"].astype(str)
    + "_"
    + audit_df["geo_lon_group"].astype(str)
)


# ------------------------------------------------------------
# 8. CLASS NAMES
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


# ------------------------------------------------------------
# 9. CONFIDENCE THRESHOLDS
# ------------------------------------------------------------

confidence_thresholds = [
    0.70,
    0.80,
    0.90,
    0.95
]


# ------------------------------------------------------------
# 10. SUMMARY BY CLASS AND CONFIDENCE
# ------------------------------------------------------------

rows = []


for class_id, class_name in class_names.items():

    class_df = audit_df[
        audit_df[label_col] == class_id
    ].copy()


    for threshold in confidence_thresholds:

        subset = class_df[
            class_df[confidence_col] >= threshold
        ]


        rows.append({
            "Class_ID": class_id,

            "Class": class_name,

            "Confidence_Threshold": threshold,

            "Events": len(subset),

            "Geographic_Groups":
                subset["geo_group"].nunique(),

            "Events_per_Group":
                (
                    len(subset) /
                    subset["geo_group"].nunique()
                )
                if len(subset) > 0
                else 0,

            "Mean_Confidence":
                subset[confidence_col].mean()
                if len(subset) > 0
                else np.nan,

            "Median_Confidence":
                subset[confidence_col].median()
                if len(subset) > 0
                else np.nan
        })


diversity_df = pd.DataFrame(rows)


# ------------------------------------------------------------
# 11. DISPLAY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE GEOGRAPHIC DIVERSITY")
print("=" * 70)

display(diversity_df)


# ------------------------------------------------------------
# 12. GEOGRAPHIC GROUP COUNTS PER CLASS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GEOGRAPHIC GROUPS PER CLASS")
print("=" * 70)


for threshold in confidence_thresholds:

    print(
        f"\nConfidence >= {threshold:.2f}"
    )

    subset = audit_df[
        audit_df[confidence_col] >= threshold
    ]


    for class_id, class_name in class_names.items():

        class_subset = subset[
            subset[label_col] == class_id
        ]

        print(
            f"  {class_name:15s}: "
            f"{len(class_subset):5d} events | "
            f"{class_subset['geo_group'].nunique():4d} groups"
        )


# ------------------------------------------------------------
# 13. TOP GEOGRAPHIC GROUPS FOR EACH CLASS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP GEOGRAPHIC GROUPS BY CLASS")
print("=" * 70)


for class_id, class_name in class_names.items():

    print(
        f"\n--- {class_name} ---"
    )

    class_df = audit_df[
        audit_df[label_col] == class_id
    ]


    group_summary = (
        class_df
        .groupby("geo_group")
        .agg(
            events=(
                event_id_col,
                "count"
            ),

            mean_confidence=(
                confidence_col,
                "mean"
            ),

            max_confidence=(
                confidence_col,
                "max"
            )
        )
        .sort_values(
            "events",
            ascending=False
        )
        .head(10)
        .reset_index()
    )


    display(group_summary)


# ------------------------------------------------------------
# 14. SAVE
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_high_confidence_label_geographic_diversity.csv"
)


diversity_df.to_csv(
    output_file,
    index=False
)


print("\nSaved:")
print(output_file)


print("\n" + "=" * 70)
print("STEP 6L COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 7A — BUILD FINAL WEIGHTED TRAINING DATASET
# FINAL CORRECTED VERSION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 7A — BUILD FINAL WEIGHTED TRAINING DATASET")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD WEAK-LABEL TRAINING DATA
# ------------------------------------------------------------

training_file = (
    "/kaggle/working/"
    "firms_2025_initial_weak_label_training_dataset.csv"
)

df = pd.read_csv(training_file)

print("\nInitial weak-label dataset:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 2. ACTUAL COLUMNS
# ------------------------------------------------------------

class_col = "weak_label_class"
confidence_col = "label_model_confidence"
agree_col = "agreeing_lf_count"


# ------------------------------------------------------------
# 3. CLASS NAME → CLASS ID
# ------------------------------------------------------------

class_to_id = {
    "Industrial": 0,
    "Gas": 1,
    "Agriculture": 2,
    "Mining": 3,
    "Wildfire": 4
}

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


# ------------------------------------------------------------
# 4. SHOW ORIGINAL CLASS VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ORIGINAL WEAK-LABEL CLASSES")
print("=" * 70)

print(
    df[class_col]
    .value_counts()
)


# ------------------------------------------------------------
# 5. CONVERT CLASS NAMES TO NUMERIC TARGET
# ------------------------------------------------------------

df["target"] = (
    df[class_col]
    .map(class_to_id)
)


# ------------------------------------------------------------
# 6. VALIDATE MAPPING
# ------------------------------------------------------------

unmapped = df[
    df["target"].isna()
]

if len(unmapped) > 0:

    print(
        "\nUnmapped class values:"
    )

    print(
        unmapped[class_col]
        .value_counts()
    )

    raise ValueError(
        "Some weak-label classes could not be mapped."
    )


df["target"] = (
    df["target"]
    .astype(int)
)


print(
    "\nValid training events:",
    len(df)
)


# ------------------------------------------------------------
# 7. CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WEAK-LABEL CLASS DISTRIBUTION")
print("=" * 70)

class_counts = (
    df["target"]
    .value_counts()
    .sort_index()
)

for class_id in range(5):

    print(
        f"{class_id} - "
        f"{class_names[class_id]:12s}: "
        f"{class_counts.get(class_id, 0):6d}"
    )


# ------------------------------------------------------------
# 8. CONVERT CONFIDENCE / AGREEMENT TO NUMERIC
# ------------------------------------------------------------

df[confidence_col] = pd.to_numeric(
    df[confidence_col],
    errors="coerce"
)

df[agree_col] = pd.to_numeric(
    df[agree_col],
    errors="coerce"
)


# ------------------------------------------------------------
# 9. VALIDATE CONFIDENCE
# ------------------------------------------------------------

if df[confidence_col].isna().any():

    raise ValueError(
        "Missing Label Model confidence values."
    )


if (
    (df[confidence_col] < 0)
    |
    (df[confidence_col] > 1)
).any():

    raise ValueError(
        "Confidence values outside [0, 1]."
    )


# ------------------------------------------------------------
# 10. CREATE CONFIDENCE WEIGHT
#
# Square-root scaling prevents Wildfire, which has many
# extremely high-confidence labels, from completely
# dominating the training process.
# ------------------------------------------------------------

df["confidence_weight"] = np.sqrt(
    df[confidence_col]
)


# ------------------------------------------------------------
# 11. AGREEMENT BONUS
#
# Modest bonus for stronger LF agreement.
# ------------------------------------------------------------

agreement_bonus = np.where(
    df[agree_col] >= 4,
    1.15,
    np.where(
        df[agree_col] >= 3,
        1.10,
        1.00
    )
)


df["agreement_weight"] = (
    agreement_bonus
)


# ------------------------------------------------------------
# 12. FINAL SAMPLE WEIGHT
# ------------------------------------------------------------

df["sample_weight"] = (
    df["confidence_weight"]
    *
    df["agreement_weight"]
)


# ------------------------------------------------------------
# 13. NORMALIZE
# ------------------------------------------------------------

mean_weight = (
    df["sample_weight"].mean()
)

if (
    not np.isfinite(mean_weight)
    or mean_weight <= 0
):

    raise ValueError(
        "Invalid mean sample weight."
    )


df["sample_weight"] = (
    df["sample_weight"]
    /
    mean_weight
)


# ------------------------------------------------------------
# 14. WEIGHT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE WEIGHT SUMMARY")
print("=" * 70)

print(
    df["sample_weight"]
    .describe()
)


# ------------------------------------------------------------
# 15. WEIGHT SUMMARY BY CLASS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE WEIGHTS BY CLASS")
print("=" * 70)

weight_summary = (
    df.groupby("target")
    .agg(
        count=("sample_weight", "count"),
        mean_weight=("sample_weight", "mean"),
        median_weight=("sample_weight", "median"),
        min_weight=("sample_weight", "min"),
        max_weight=("sample_weight", "max")
    )
    .reset_index()
)

weight_summary["Class"] = (
    weight_summary["target"]
    .map(class_names)
)

weight_summary = weight_summary[
    [
        "target",
        "Class",
        "count",
        "mean_weight",
        "median_weight",
        "min_weight",
        "max_weight"
    ]
]

display(
    weight_summary
)


# ------------------------------------------------------------
# 16. FINAL VALIDATION
# ------------------------------------------------------------

if not np.isfinite(
    df["sample_weight"]
).all():

    raise ValueError(
        "Invalid sample weights detected."
    )


if (
    df["sample_weight"] <= 0
).any():

    raise ValueError(
        "Non-positive sample weights detected."
    )


# ------------------------------------------------------------
# 17. SAVE
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_final_weighted_training_dataset.csv"
)

df.to_csv(
    output_file,
    index=False
)


print("\nSaved:")
print(output_file)


print("\n" + "=" * 70)
print("STEP 7A COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 7B — BUILD FINAL ML FEATURE MATRIX
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 7B — BUILD FINAL ML FEATURE MATRIX")
print("=" * 70)


# ------------------------------------------------------------
# 1. FILES
# ------------------------------------------------------------

training_file = (
    "/kaggle/working/"
    "firms_2025_final_weighted_training_dataset.csv"
)

feature_file = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)


# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

train_labels = pd.read_csv(
    training_file
)

event_features = pd.read_csv(
    feature_file
)

print("\nWeighted labels:", train_labels.shape)
print("Event features:", event_features.shape)


# ------------------------------------------------------------
# 3. MERGE
# ------------------------------------------------------------

df = train_labels.merge(
    event_features,
    on="event_id",
    how="left",
    validate="one_to_one"
)

print(
    "\nMerged training dataset:",
    df.shape
)


# ------------------------------------------------------------
# 4. VERIFY MERGE
# ------------------------------------------------------------

if len(df) != len(train_labels):

    raise ValueError(
        "Some training events were lost during merge."
    )


print(
    "Unmatched feature rows:",
    df.isna().all(axis=1).sum()
)


# ------------------------------------------------------------
# 5. LABEL / LF COLUMNS THAT MUST NEVER ENTER X
# ------------------------------------------------------------

label_leakage_columns = [
    "event_id",

    # Snorkel outputs
    "label_model_label",
    "label_model_class",
    "label_model_confidence",
    "label_model_margin",

    # Class probabilities generated by Snorkel
    "prob_industrial",
    "prob_gas",
    "prob_agriculture",
    "prob_mining",
    "prob_wildfire",

    # LF metadata
    "active_lf_count",
    "agreeing_lf_count",
    "disagreeing_lf_count",
    "confidence_band",

    # Weak labels
    "weak_label",
    "weak_label_class",
    "label_quality",

    # Weighting metadata
    "confidence_weight",
    "agreement_weight",
    "sample_weight"
]


# ------------------------------------------------------------
# 6. REMOVE LEAKAGE / LABEL COLUMNS
# ------------------------------------------------------------

existing_leakage = [
    c for c in label_leakage_columns
    if c in df.columns
]


print("\n" + "=" * 70)
print("LABEL / LEEKAGE COLUMNS REMOVED")
print("=" * 70)

for c in existing_leakage:
    print(c)


feature_df = df.drop(
    columns=existing_leakage
)


# ------------------------------------------------------------
# 7. TARGET AND SAMPLE WEIGHT
# ------------------------------------------------------------

y = df["target"].astype(int)

sample_weight = (
    df["sample_weight"]
    .astype(float)
)


# ------------------------------------------------------------
# 8. KEEP NUMERIC FEATURES ONLY
# ------------------------------------------------------------

numeric_features = (
    feature_df
    .select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


# Do not accidentally include target
numeric_features = [
    c for c in numeric_features
    if c != "target"
]


print("\nNumeric candidate features:")
print(len(numeric_features))


# ------------------------------------------------------------
# 9. BUILD X
# ------------------------------------------------------------

X = feature_df[
    numeric_features
].copy()


# ------------------------------------------------------------
# 10. REMOVE 100% MISSING FEATURES
# ------------------------------------------------------------

missing_fraction = (
    X.isna()
    .mean()
)


all_missing_features = (
    missing_fraction[
        missing_fraction >= 1.0
    ]
    .index
    .tolist()
)


if all_missing_features:

    print("\n" + "=" * 70)
    print("100% MISSING FEATURES REMOVED")
    print("=" * 70)

    for c in all_missing_features:
        print(c)

    X = X.drop(
        columns=all_missing_features
    )


# ------------------------------------------------------------
# 11. REMOVE CONSTANT FEATURES
# ------------------------------------------------------------

constant_features = [
    c for c in X.columns
    if X[c].nunique(
        dropna=False
    ) <= 1
]


if constant_features:

    print("\n" + "=" * 70)
    print("CONSTANT FEATURES REMOVED")
    print("=" * 70)

    for c in constant_features:
        print(c)

    X = X.drop(
        columns=constant_features
    )


# ------------------------------------------------------------
# 12. CHECK INF VALUES
# ------------------------------------------------------------

inf_columns = []

for c in X.columns:

    if np.isinf(
        X[c].to_numpy(
            dtype=float,
            na_value=np.nan
        )
    ).any():

        inf_columns.append(c)


if inf_columns:

    print("\nInfinite-value columns:")

    for c in inf_columns:
        print(c)

    X[inf_columns] = X[
        inf_columns
    ].replace(
        [np.inf, -np.inf],
        np.nan
    )


# ------------------------------------------------------------
# 13. MISSING VALUE SUMMARY
# ------------------------------------------------------------

missing_counts = (
    X.isna()
    .sum()
    .sort_values(
        ascending=False
    )
)


missing_counts = missing_counts[
    missing_counts > 0
]


print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

print(
    "Features with missing values:",
    len(missing_counts)
)

print(
    "Total missing cells:",
    int(X.isna().sum().sum())
)

if len(missing_counts) > 0:

    print(
        "\nTop missing features:"
    )

    display(
        missing_counts
        .head(20)
        .to_frame("Missing_Count")
    )


# ------------------------------------------------------------
# 14. FINAL FEATURE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL FEATURE MATRIX")
print("=" * 70)

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "sample_weight shape:",
    sample_weight.shape
)


# ------------------------------------------------------------
# 15. CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


for class_id in range(5):

    count = int(
        (y == class_id).sum()
    )

    print(
        f"{class_id} - "
        f"{class_names[class_id]:12s}: "
        f"{count:6d}"
    )


# ------------------------------------------------------------
# 16. VALIDATION
# ------------------------------------------------------------

if len(X) != len(y):

    raise ValueError(
        "X and y have different row counts."
    )


if len(X) != len(sample_weight):

    raise ValueError(
        "X and sample_weight have different row counts."
    )


if not np.isfinite(
    sample_weight
).all():

    raise ValueError(
        "Invalid sample weights."
    )


# ------------------------------------------------------------
# 17. SAVE FEATURE COLUMN LIST
# ------------------------------------------------------------

feature_columns_file = (
    "/kaggle/working/"
    "firms_2025_final_model_feature_columns.csv"
)


pd.DataFrame({
    "feature": X.columns
}).to_csv(
    feature_columns_file,
    index=False
)


# ------------------------------------------------------------
# 18. SAVE MISSINGNESS SUMMARY
# ------------------------------------------------------------

missingness_file = (
    "/kaggle/working/"
    "firms_2025_final_model_feature_missingness.csv"
)


(
    X.isna()
    .mean()
    .sort_values(
        ascending=False
    )
    .rename(
        "missing_fraction"
    )
    .reset_index()
    .rename(
        columns={
            "index": "feature"
        }
    )
    .to_csv(
        missingness_file,
        index=False
    )
)


# ------------------------------------------------------------
# 19. SAVE FINAL MATRIX
# ------------------------------------------------------------

matrix_file = (
    "/kaggle/working/"
    "firms_2025_final_ml_training_matrix.csv"
)


final_matrix = X.copy()

final_matrix["target"] = y.values

final_matrix["sample_weight"] = (
    sample_weight.values
)

final_matrix.insert(
    0,
    "event_id",
    df["event_id"].values
)


final_matrix.to_csv(
    matrix_file,
    index=False
)


print("\nSaved:")
print(feature_columns_file)
print(missingness_file)
print(matrix_file)


print("\n" + "=" * 70)
print("STEP 7B COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 7C — FINAL TRAIN / VALIDATION / TEST SPLIT
#             + LEAKAGE-SAFE IMPUTATION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

print("=" * 70)
print("STEP 7C — FINAL TRAIN / VALIDATION / TEST SPLIT")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD FINAL MATRIX
# ------------------------------------------------------------

matrix_file = (
    "/kaggle/working/"
    "firms_2025_final_ml_training_matrix.csv"
)

feature_file = (
    "/kaggle/working/"
    "firms_2025_final_model_feature_columns.csv"
)

df = pd.read_csv(matrix_file)

feature_columns = pd.read_csv(
    feature_file
)["feature"].tolist()


print("\nFinal matrix:", df.shape)
print(
    "Feature columns:",
    len(feature_columns)
)


# ------------------------------------------------------------
# 2. BUILD X / y / SAMPLE WEIGHT
# ------------------------------------------------------------

X = df[
    feature_columns
].copy()

y = df[
    "target"
].astype(int)

sample_weight = df[
    "sample_weight"
].astype(float)


# ------------------------------------------------------------
# 3. VERIFY
# ------------------------------------------------------------

if len(X) != len(y):
    raise ValueError(
        "X and y row counts do not match."
    )

if len(X) != len(sample_weight):
    raise ValueError(
        "X and sample_weight row counts do not match."
    )


print(
    "\nX shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "sample_weight shape:",
    sample_weight.shape
)


# ------------------------------------------------------------
# 4. FIRST SPLIT
#
# 70% train
# 30% temporary
# ------------------------------------------------------------

X_train, X_temp, y_train, y_temp, \
w_train, w_temp, id_train, id_temp = train_test_split(
    X,
    y,
    sample_weight,
    df["event_id"],
    test_size=0.30,
    random_state=42,
    stratify=y
)


# ------------------------------------------------------------
# 5. SECOND SPLIT
#
# Temporary 30% → 15% validation + 15% test
# ------------------------------------------------------------

X_val, X_test, y_val, y_test, \
w_val, w_test, id_val, id_test = train_test_split(
    X_temp,
    y_temp,
    w_temp,
    id_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)


print("\n" + "=" * 70)
print("SPLIT SIZES")
print("=" * 70)

print(
    "Training events:",
    len(X_train)
)

print(
    "Validation events:",
    len(X_val)
)

print(
    "Test events:",
    len(X_test)
)


# ------------------------------------------------------------
# 6. CLASS NAMES
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


# ------------------------------------------------------------
# 7. CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)


for class_id in range(5):

    train_count = int(
        (y_train == class_id).sum()
    )

    val_count = int(
        (y_val == class_id).sum()
    )

    test_count = int(
        (y_test == class_id).sum()
    )

    print(
        f"{class_id} - "
        f"{class_names[class_id]:12s}: "
        f"Train={train_count:5d} | "
        f"Val={val_count:5d} | "
        f"Test={test_count:5d}"
    )


# ------------------------------------------------------------
# 8. CHECK MISSING VALUES BEFORE IMPUTATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUES BEFORE IMPUTATION")
print("=" * 70)

print(
    "Train:",
    int(X_train.isna().sum().sum())
)

print(
    "Validation:",
    int(X_val.isna().sum().sum())
)

print(
    "Test:",
    int(X_test.isna().sum().sum())
)


# ------------------------------------------------------------
# 9. LEAKAGE-SAFE MEDIAN IMPUTATION
#
# FIT ONLY ON TRAINING DATA.
# ------------------------------------------------------------

imputer = SimpleImputer(
    strategy="median"
)


X_train_imp = imputer.fit_transform(
    X_train
)

X_val_imp = imputer.transform(
    X_val
)

X_test_imp = imputer.transform(
    X_test
)


# ------------------------------------------------------------
# 10. CONVERT BACK TO DATAFRAMES
# ------------------------------------------------------------

X_train_imp = pd.DataFrame(
    X_train_imp,
    columns=feature_columns,
    index=X_train.index
)

X_val_imp = pd.DataFrame(
    X_val_imp,
    columns=feature_columns,
    index=X_val.index
)

X_test_imp = pd.DataFrame(
    X_test_imp,
    columns=feature_columns,
    index=X_test.index
)


# ------------------------------------------------------------
# 11. VERIFY NO MISSING VALUES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUES AFTER IMPUTATION")
print("=" * 70)

print(
    "Train:",
    int(X_train_imp.isna().sum().sum())
)

print(
    "Validation:",
    int(X_val_imp.isna().sum().sum())
)

print(
    "Test:",
    int(X_test_imp.isna().sum().sum())
)


# ------------------------------------------------------------
# 12. VERIFY FINITE VALUES
# ------------------------------------------------------------

for name, matrix in [
    ("Train", X_train_imp),
    ("Validation", X_val_imp),
    ("Test", X_test_imp)
]:

    if not np.isfinite(
        matrix.to_numpy()
    ).all():

        raise ValueError(
            f"Non-finite values found in {name}."
        )


# ------------------------------------------------------------
# 13. SAVE SPLIT EVENT IDS
# ------------------------------------------------------------

split_ids = pd.DataFrame({
    "event_id": pd.concat(
        [
            id_train,
            id_val,
            id_test
        ],
        ignore_index=True
    ),

    "split": (
        ["train"] * len(id_train)
        +
        ["validation"] * len(id_val)
        +
        ["test"] * len(id_test)
    )
})


split_file = (
    "/kaggle/working/"
    "firms_2025_final_train_val_test_event_ids.csv"
)

split_ids.to_csv(
    split_file,
    index=False
)


# ------------------------------------------------------------
# 14. SAVE IMPUTER STATISTICS
# ------------------------------------------------------------

imputer_file = (
    "/kaggle/working/"
    "firms_2025_final_imputer_statistics.csv"
)

pd.DataFrame({
    "feature": feature_columns,
    "median_value": imputer.statistics_
}).to_csv(
    imputer_file,
    index=False
)


# ------------------------------------------------------------
# 15. SAVE SPLIT DATASET
# ------------------------------------------------------------

train_output = X_train_imp.copy()
train_output["event_id"] = id_train.values
train_output["target"] = y_train.values
train_output["sample_weight"] = w_train.values

train_output = train_output[
    ["event_id"] +
    feature_columns +
    ["target", "sample_weight"]
]


val_output = X_val_imp.copy()
val_output["event_id"] = id_val.values
val_output["target"] = y_val.values
val_output["sample_weight"] = w_val.values

val_output = val_output[
    ["event_id"] +
    feature_columns +
    ["target", "sample_weight"]
]


test_output = X_test_imp.copy()
test_output["event_id"] = id_test.values
test_output["target"] = y_test.values
test_output["sample_weight"] = w_test.values

test_output = test_output[
    ["event_id"] +
    feature_columns +
    ["target", "sample_weight"]
]


train_file = (
    "/kaggle/working/"
    "firms_2025_final_train.csv"
)

val_file = (
    "/kaggle/working/"
    "firms_2025_final_validation.csv"
)

test_file = (
    "/kaggle/working/"
    "firms_2025_final_test.csv"
)


train_output.to_csv(
    train_file,
    index=False
)

val_output.to_csv(
    val_file,
    index=False
)

test_output.to_csv(
    test_file,
    index=False
)


# ------------------------------------------------------------
# 16. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(split_file)
print(imputer_file)
print(train_file)
print(val_file)
print(test_file)


print("\n" + "=" * 70)
print("STEP 7C COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 7D — TRAIN AND COMPARE FINAL ML MODELS
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report
)

print("=" * 70)
print("STEP 7D — TRAIN AND COMPARE FINAL ML MODELS")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD FINAL SPLITS
# ------------------------------------------------------------

train_file = (
    "/kaggle/working/"
    "firms_2025_final_train.csv"
)

val_file = (
    "/kaggle/working/"
    "firms_2025_final_validation.csv"
)

feature_file = (
    "/kaggle/working/"
    "firms_2025_final_model_feature_columns.csv"
)

train_df = pd.read_csv(train_file)
val_df = pd.read_csv(val_file)

feature_columns = pd.read_csv(
    feature_file
)["feature"].tolist()


# ------------------------------------------------------------
# 2. BUILD MATRICES
# ------------------------------------------------------------

X_train = train_df[
    feature_columns
]

y_train = train_df[
    "target"
].astype(int)

w_train = train_df[
    "sample_weight"
].astype(float)


X_val = val_df[
    feature_columns
]

y_val = val_df[
    "target"
].astype(int)


print("\nTraining:", X_train.shape)
print("Validation:", X_val.shape)


# ------------------------------------------------------------
# 3. CLASS NAMES
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


# ------------------------------------------------------------
# 4. DEFINE MODELS
# ------------------------------------------------------------

models = {

    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2
    ),

    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        max_features="sqrt",
        min_samples_leaf=2
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )
}


# ------------------------------------------------------------
# 5. TRAIN AND EVALUATE
# ------------------------------------------------------------

results = []

predictions = {}

probabilities = {}

for model_name, model in models.items():

    print("\n" + "=" * 70)
    print(f"TRAINING: {model_name}")
    print("=" * 70)

    # Train with Snorkel confidence weights
    model.fit(
        X_train,
        y_train,
        sample_weight=w_train
    )

    print("Training complete.")

    # Predictions
    y_pred = model.predict(
        X_val
    )

    y_prob = model.predict_proba(
        X_val
    )

    predictions[
        model_name
    ] = y_pred

    probabilities[
        model_name
    ] = y_prob

    # Metrics
    accuracy = accuracy_score(
        y_val,
        y_pred
    )

    balanced_acc = balanced_accuracy_score(
        y_val,
        y_pred
    )

    macro_f1 = f1_score(
        y_val,
        y_pred,
        average="macro"
    )

    macro_precision = precision_score(
        y_val,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_val,
        y_pred,
        average="macro",
        zero_division=0
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Balanced_Accuracy": balanced_acc,
        "Macro_F1": macro_f1,
        "Macro_Precision": macro_precision,
        "Macro_Recall": macro_recall
    })

    print("\nMetrics:")
    print(
        f"Accuracy          : {accuracy:.4f}"
    )

    print(
        f"Balanced Accuracy : {balanced_acc:.4f}"
    )

    print(
        f"Macro F1          : {macro_f1:.4f}"
    )

    print(
        f"Macro Precision   : {macro_precision:.4f}"
    )

    print(
        f"Macro Recall      : {macro_recall:.4f}"
    )

    # Classification report
    print("\nClassification report:")

    print(
        classification_report(
            y_val,
            y_pred,
            labels=[0, 1, 2, 3, 4],
            target_names=[
                class_names[0],
                class_names[1],
                class_names[2],
                class_names[3],
                class_names[4]
            ],
            digits=4,
            zero_division=0
        )
    )


# ------------------------------------------------------------
# 6. MODEL COMPARISON
# ------------------------------------------------------------

results_df = pd.DataFrame(
    results
).sort_values(
    "Macro_F1",
    ascending=False
).reset_index(
    drop=True
)


print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

display(
    results_df
)


# ------------------------------------------------------------
# 7. SELECT BEST MODEL
# ------------------------------------------------------------

best_model_name = (
    results_df
    .iloc[0]["Model"]
)


print(
    "\nBEST MODEL:",
    best_model_name
)


# ------------------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------------------

results_file = (
    "/kaggle/working/"
    "firms_2025_final_model_comparison.csv"
)

results_df.to_csv(
    results_file,
    index=False
)


# ------------------------------------------------------------
# 9. SAVE VALIDATION PREDICTIONS
# ------------------------------------------------------------

best_predictions = predictions[
    best_model_name
]

best_probabilities = probabilities[
    best_model_name
]


validation_predictions = pd.DataFrame({
    "event_id": val_df["event_id"].values,
    "true_label": y_val.values,
    "predicted_label": best_predictions,
    "prob_industrial": best_probabilities[:, 0],
    "prob_gas": best_probabilities[:, 1],
    "prob_agriculture": best_probabilities[:, 2],
    "prob_mining": best_probabilities[:, 3],
    "prob_wildfire": best_probabilities[:, 4]
})


validation_predictions[
    "prediction_confidence"
] = best_probabilities.max(
    axis=1
)


validation_file = (
    "/kaggle/working/"
    "firms_2025_final_validation_predictions.csv"
)

validation_predictions.to_csv(
    validation_file,
    index=False
)


# ------------------------------------------------------------
# 10. SAVE TRAINED MODELS
# ------------------------------------------------------------

import joblib

for model_name, model in models.items():

    model_file = (
        "/kaggle/working/"
        f"firms_2025_{model_name}_final_validation_model.joblib"
    )

    joblib.dump(
        model,
        model_file
    )

    print(
        "Saved:",
        model_file
    )


print("\nSaved:")
print(results_file)
print(validation_file)


print("\n" + "=" * 70)
print("STEP 7D COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 7E — FINAL RANDOM FOREST TEST EVALUATION
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

print("=" * 70)
print("STEP 7E — FINAL RANDOM FOREST TEST EVALUATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. FILES
# ------------------------------------------------------------

test_file = (
    "/kaggle/working/"
    "firms_2025_final_test.csv"
)

feature_file = (
    "/kaggle/working/"
    "firms_2025_final_model_feature_columns.csv"
)

model_file = (
    "/kaggle/working/"
    "firms_2025_RandomForest_final_validation_model.joblib"
)


# ------------------------------------------------------------
# 2. LOAD TEST DATA
# ------------------------------------------------------------

test_df = pd.read_csv(
    test_file
)

feature_columns = pd.read_csv(
    feature_file
)["feature"].tolist()


print("\nTest dataset:")
print(test_df.shape)

print(
    "Features:",
    len(feature_columns)
)


# ------------------------------------------------------------
# 3. BUILD X / y
# ------------------------------------------------------------

X_test = test_df[
    feature_columns
]

y_test = test_df[
    "target"
].astype(int)


# ------------------------------------------------------------
# 4. LOAD RANDOM FOREST
# ------------------------------------------------------------

model = joblib.load(
    model_file
)


print(
    "\nLoaded model:",
    model_file
)


# ------------------------------------------------------------
# 5. PREDICTION
# ------------------------------------------------------------

y_pred = model.predict(
    X_test
)

y_prob = model.predict_proba(
    X_test
)


# ------------------------------------------------------------
# 6. CLASS NAMES
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


# ------------------------------------------------------------
# 7. OVERALL METRICS
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    y_pred
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

macro_precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(
    f"Accuracy          : {accuracy:.4f}"
)

print(
    f"Balanced Accuracy : {balanced_accuracy:.4f}"
)

print(
    f"Macro F1          : {macro_f1:.4f}"
)

print(
    f"Macro Precision   : {macro_precision:.4f}"
)

print(
    f"Macro Recall      : {macro_recall:.4f}"
)


# ------------------------------------------------------------
# 8. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL TEST CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2, 3, 4],
        target_names=[
            class_names[0],
            class_names[1],
            class_names[2],
            class_names[3],
            class_names[4]
        ],
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 9. CONFUSION MATRIX
# ------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1, 2, 3, 4]
)


print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Industrial",
        "Gas",
        "Agriculture",
        "Mining",
        "Wildfire"
    ],
    columns=[
        "Industrial",
        "Gas",
        "Agriculture",
        "Mining",
        "Wildfire"
    ]
)

display(cm_df)


# ------------------------------------------------------------
# 10. CONFIDENCE
# ------------------------------------------------------------

prediction_confidence = (
    y_prob.max(axis=1)
)


print("\n" + "=" * 70)
print("PREDICTION CONFIDENCE")
print("=" * 70)

print(
    pd.Series(
        prediction_confidence
    ).describe()
)


# ------------------------------------------------------------
# 11. CONFIDENCE BANDS
# ------------------------------------------------------------

confidence_bands = pd.cut(
    prediction_confidence,
    bins=[
        -np.inf,
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
        0.95,
        1.00
    ],
    labels=[
        "<0.50",
        "0.50-0.60",
        "0.60-0.70",
        "0.70-0.80",
        "0.80-0.90",
        "0.90-0.95",
        "0.95-1.00"
    ],
    include_lowest=True
)


confidence_summary = (
    pd.Series(
        confidence_bands
    )
    .value_counts(
        sort=False
    )
    .rename_axis(
        "Confidence_Band"
    )
    .reset_index(
        name="Events"
    )
)


confidence_summary[
    "Percentage"
] = (
    confidence_summary["Events"]
    /
    len(test_df)
    *
    100
)


display(
    confidence_summary
)


# ------------------------------------------------------------
# 12. TEST PREDICTIONS FILE
# ------------------------------------------------------------

test_predictions = pd.DataFrame({

    "event_id":
        test_df["event_id"].values,

    "true_label":
        y_test.values,

    "predicted_label":
        y_pred,

    "predicted_class":
        [
            class_names[x]
            for x in y_pred
        ],

    "prediction_confidence":
        prediction_confidence,

    "prob_industrial":
        y_prob[:, 0],

    "prob_gas":
        y_prob[:, 1],

    "prob_agriculture":
        y_prob[:, 2],

    "prob_mining":
        y_prob[:, 3],

    "prob_wildfire":
        y_prob[:, 4]
})


# ------------------------------------------------------------
# 13. SAVE TEST PREDICTIONS
# ------------------------------------------------------------

test_predictions_file = (
    "/kaggle/working/"
    "firms_2025_final_test_predictions.csv"
)

test_predictions.to_csv(
    test_predictions_file,
    index=False
)


# ------------------------------------------------------------
# 14. SAVE METRICS
# ------------------------------------------------------------

metrics_file = (
    "/kaggle/working/"
    "firms_2025_final_test_metrics.csv"
)

metrics_df = pd.DataFrame([{
    "Model": "RandomForest",
    "Accuracy": accuracy,
    "Balanced_Accuracy": balanced_accuracy,
    "Macro_F1": macro_f1,
    "Macro_Precision": macro_precision,
    "Macro_Recall": macro_recall
}])

metrics_df.to_csv(
    metrics_file,
    index=False
)


# ------------------------------------------------------------
# 15. SAVE CONFUSION MATRIX
# ------------------------------------------------------------

confusion_file = (
    "/kaggle/working/"
    "firms_2025_final_test_confusion_matrix.csv"
)

cm_df.to_csv(
    confusion_file
)


# ------------------------------------------------------------
# 16. SAVE CONFIDENCE SUMMARY
# ------------------------------------------------------------

confidence_file = (
    "/kaggle/working/"
    "firms_2025_final_test_confidence_distribution.csv"
)

confidence_summary.to_csv(
    confidence_file,
    index=False
)


print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(test_predictions_file)
print(metrics_file)
print(confusion_file)
print(confidence_file)


print("\n" + "=" * 70)
print("STEP 7E COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 7F — FINAL PRODUCTION MODEL + ALL EVENT CLASSIFICATION
# ============================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

print("=" * 70)
print("STEP 7F — FINAL PRODUCTION MODEL")
print("=" * 70)


# ------------------------------------------------------------
# 1. FILES
# ------------------------------------------------------------

training_file = (
    "/kaggle/working/"
    "firms_2025_final_weighted_training_dataset.csv"
)

feature_file = (
    "/kaggle/working/"
    "firms_2025_final_model_feature_columns.csv"
)

event_feature_file = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)


# ------------------------------------------------------------
# 2. LOAD TRAINING LABELS
# ------------------------------------------------------------

train_labels = pd.read_csv(
    training_file
)

feature_columns = pd.read_csv(
    feature_file
)["feature"].tolist()


print("\nWeak-label training events:")
print(len(train_labels))

print(
    "Model features:",
    len(feature_columns)
)


# ------------------------------------------------------------
# 3. LOAD ALL EVENT FEATURES
# ------------------------------------------------------------

all_events = pd.read_csv(
    event_feature_file
)

print(
    "All events:",
    all_events.shape
)


# ------------------------------------------------------------
# 4. BUILD TRAINING FEATURE MATRIX
# ------------------------------------------------------------

train_features = train_labels.merge(
    all_events[
        ["event_id"] + feature_columns
    ],
    on="event_id",
    how="inner",
    validate="one_to_one"
)


print(
    "\nMerged training data:",
    train_features.shape
)


if len(train_features) != len(train_labels):

    raise ValueError(
        "Training labels and features did not merge completely."
    )


# ------------------------------------------------------------
# 5. TRAINING X / y / WEIGHTS
# ------------------------------------------------------------

X_train = train_features[
    feature_columns
].copy()

y_train = train_labels[
    "target"
].astype(int)

sample_weight = train_labels[
    "sample_weight"
].astype(float)


# ------------------------------------------------------------
# 6. FIT IMPUTER ON ALL WEAK-LABEL TRAINING DATA
# ------------------------------------------------------------

imputer = SimpleImputer(
    strategy="median"
)

X_train_imp = imputer.fit_transform(
    X_train
)


# ------------------------------------------------------------
# 7. BUILD ALL-EVENT FEATURE MATRIX
# ------------------------------------------------------------

X_all = all_events[
    feature_columns
].copy()


X_all_imp = imputer.transform(
    X_all
)


print(
    "\nTraining matrix:",
    X_train_imp.shape
)

print(
    "All-event matrix:",
    X_all_imp.shape
)


# ------------------------------------------------------------
# 8. TRAIN FINAL RANDOM FOREST
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING FINAL RANDOM FOREST")
print("=" * 70)


final_model = RandomForestClassifier(
    n_estimators=700,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=2
)


final_model.fit(
    X_train_imp,
    y_train,
    sample_weight=sample_weight
)


print("Final model training complete.")


# ------------------------------------------------------------
# 9. PREDICT ALL EVENTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFYING ALL 302,070 EVENTS")
print("=" * 70)


all_predictions = final_model.predict(
    X_all_imp
)

all_probabilities = final_model.predict_proba(
    X_all_imp
)


prediction_confidence = (
    all_probabilities.max(axis=1)
)


# ------------------------------------------------------------
# 10. CLASS NAMES
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


predicted_class_names = [
    class_names[x]
    for x in all_predictions
]


# ------------------------------------------------------------
# 11. CREATE OUTPUT
# ------------------------------------------------------------

final_predictions = pd.DataFrame({

    "event_id":
        all_events["event_id"].values,

    "predicted_label":
        all_predictions,

    "predicted_class":
        predicted_class_names,

    "prediction_confidence":
        prediction_confidence,

    "prob_industrial":
        all_probabilities[:, 0],

    "prob_gas":
        all_probabilities[:, 1],

    "prob_agriculture":
        all_probabilities[:, 2],

    "prob_mining":
        all_probabilities[:, 3],

    "prob_wildfire":
        all_probabilities[:, 4]
})


# ------------------------------------------------------------
# 12. UNKNOWN / UNRESOLVED STATUS
#
# Conservative threshold.
#
# Events below 0.70 remain Unknown.
# ------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.70


final_predictions["status"] = np.where(
    final_predictions[
        "prediction_confidence"
    ] >= CONFIDENCE_THRESHOLD,
    "Classified",
    "Unknown"
)


# ------------------------------------------------------------
# 13. OVERALL DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL EVENT CLASS DISTRIBUTION")
print("=" * 70)


distribution = (
    final_predictions[
        "predicted_class"
    ]
    .value_counts()
)


for class_name in [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]:

    count = int(
        distribution.get(
            class_name,
            0
        )
    )

    percentage = (
        count /
        len(final_predictions) *
        100
    )

    print(
        f"{class_name:15s}: "
        f"{count:7d} "
        f"({percentage:6.2f}%)"
    )


# ------------------------------------------------------------
# 14. CLASSIFIED / UNKNOWN
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION STATUS")
print("=" * 70)


status_counts = (
    final_predictions[
        "status"
    ]
    .value_counts()
)


for status, count in status_counts.items():

    print(
        f"{status:15s}: "
        f"{count:7d} "
        f"({count / len(final_predictions) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 15. CONFIDENCE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PREDICTION CONFIDENCE")
print("=" * 70)

print(
    final_predictions[
        "prediction_confidence"
    ].describe()
)


# ------------------------------------------------------------
# 16. SAVE ALL EVENT PREDICTIONS
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_final_all_event_source_classification.csv"
)


final_predictions.to_csv(
    output_file,
    index=False
)


# ------------------------------------------------------------
# 17. SAVE FINAL MODEL
# ------------------------------------------------------------

model_output = (
    "/kaggle/working/"
    "firms_2025_final_production_random_forest.joblib"
)


joblib.dump(
    final_model,
    model_output
)


# ------------------------------------------------------------
# 18. SAVE IMPUTER
# ------------------------------------------------------------

imputer_output = (
    "/kaggle/working/"
    "firms_2025_final_production_imputer.joblib"
)


joblib.dump(
    imputer,
    imputer_output
)


# ------------------------------------------------------------
# 19. SAVE CLASS DISTRIBUTION
# ------------------------------------------------------------

distribution_output = (
    "/kaggle/working/"
    "firms_2025_final_event_class_distribution.csv"
)


distribution_df = (
    final_predictions[
        [
            "predicted_class",
            "status"
        ]
    ]
    .value_counts()
    .reset_index(
        name="events"
    )
)


distribution_df.to_csv(
    distribution_output,
    index=False
)


# ------------------------------------------------------------
# 20. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(output_file)
print(model_output)
print(imputer_output)
print(distribution_output)


print("\n" + "=" * 70)
print("STEP 7F COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 8 — MAP EVENT CLASSIFICATION TO ALL FIRMS DETECTIONS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 8 — MAP EVENT CLASSIFICATION TO ALL FIRMS DETECTIONS")
print("=" * 70)


# ------------------------------------------------------------
# 1. FILES
# ------------------------------------------------------------

event_classification_file = (
    "/kaggle/working/"
    "firms_2025_final_all_event_source_classification.csv"
)

event_file = (
    "/kaggle/working/"
    "firms_2025_events.csv"
)


# ------------------------------------------------------------
# 2. LOAD EVENT CLASSIFICATION
# ------------------------------------------------------------

event_predictions = pd.read_csv(
    event_classification_file
)

print(
    "\nEvent classification:",
    event_predictions.shape
)


# ------------------------------------------------------------
# 3. LOAD FIRMS EVENT DATA
# ------------------------------------------------------------

events = pd.read_csv(
    event_file
)

print(
    "FIRMS event data:",
    events.shape
)


# ------------------------------------------------------------
# 4. CHECK EVENT ID
# ------------------------------------------------------------

if "event_id" not in event_predictions.columns:
    raise ValueError(
        "event_id missing from event classification file."
    )

if "event_id" not in events.columns:
    raise ValueError(
        "event_id missing from FIRMS event file."
    )


# ------------------------------------------------------------
# 5. KEEP ONLY CLASSIFICATION COLUMNS
# ------------------------------------------------------------

classification_columns = [
    "event_id",
    "predicted_label",
    "predicted_class",
    "prediction_confidence",
    "prob_industrial",
    "prob_gas",
    "prob_agriculture",
    "prob_mining",
    "prob_wildfire",
    "status"
]


missing = [
    c
    for c in classification_columns
    if c not in event_predictions.columns
]


if missing:
    raise ValueError(
        f"Missing classification columns: {missing}"
    )


event_predictions = event_predictions[
    classification_columns
].copy()


# ------------------------------------------------------------
# 6. VERIFY EVENT IDS
# ------------------------------------------------------------

if event_predictions["event_id"].duplicated().any():
    raise ValueError(
        "Duplicate event IDs in classification file."
    )


if events["event_id"].isna().any():
    raise ValueError(
        "Missing event IDs in FIRMS event file."
    )


# ------------------------------------------------------------
# 7. MERGE CLASSIFICATION ONTO EVERY FIRMS DETECTION
# ------------------------------------------------------------

final_detections = events.merge(
    event_predictions,
    on="event_id",
    how="left",
    validate="many_to_one"
)


print(
    "\nFinal detection dataset:",
    final_detections.shape
)


# ------------------------------------------------------------
# 8. VERIFY ALL FIRMS DETECTIONS RETAINED
# ------------------------------------------------------------

if len(final_detections) != len(events):

    raise ValueError(
        "Number of FIRMS detections changed after merge."
    )


print(
    "Original FIRMS detections:",
    len(events)
)

print(
    "Final FIRMS detections:",
    len(final_detections)
)


# ------------------------------------------------------------
# 9. CHECK MISSING CLASSIFICATIONS
# ------------------------------------------------------------

missing_classification = (
    final_detections[
        "predicted_class"
    ].isna()
).sum()


print(
    "Detections without event classification:",
    missing_classification
)


if missing_classification > 0:

    raise ValueError(
        "Some FIRMS detections did not receive an event classification."
    )


# ------------------------------------------------------------
# 10. DETECTION-LEVEL CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DETECTION-LEVEL SOURCE DISTRIBUTION")
print("=" * 70)


source_counts = (
    final_detections[
        "predicted_class"
    ]
    .value_counts()
)


for source in [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]:

    count = int(
        source_counts.get(
            source,
            0
        )
    )

    percentage = (
        count /
        len(final_detections)
        *
        100
    )

    print(
        f"{source:15s}: "
        f"{count:8d} "
        f"({percentage:6.2f}%)"
    )


# ------------------------------------------------------------
# 11. DETECTION-LEVEL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DETECTION-LEVEL CLASSIFICATION STATUS")
print("=" * 70)


status_counts = (
    final_detections[
        "status"
    ]
    .value_counts()
)


for status, count in status_counts.items():

    print(
        f"{status:15s}: "
        f"{count:8d} "
        f"({count / len(final_detections) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 12. VERIFY EVENT COVERAGE
# ------------------------------------------------------------

unique_events_original = (
    events["event_id"]
    .nunique()
)

unique_events_final = (
    final_detections["event_id"]
    .nunique()
)


print("\n" + "=" * 70)
print("EVENT COVERAGE CHECK")
print("=" * 70)

print(
    "Original events:",
    unique_events_original
)

print(
    "Classified events:",
    unique_events_final
)


if (
    unique_events_original
    != unique_events_final
):

    raise ValueError(
        "Some events were lost during detection-level merge."
    )


# ------------------------------------------------------------
# 13. SAVE FINAL DETECTION-LEVEL DATASET
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_final_detection_source_classification.csv"
)


final_detections.to_csv(
    output_file,
    index=False
)


print("\nSaved:")
print(output_file)


print("\n" + "=" * 70)
print("STEP 8 COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 9A — FINAL DATASET AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 9A — FINAL DATASET AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD FINAL DETECTION DATASET
# ------------------------------------------------------------

final_file = (
    "/kaggle/working/"
    "firms_2025_final_detection_source_classification.csv"
)

df = pd.read_csv(final_file)

print("\nDataset shape:")
print(df.shape)


# ------------------------------------------------------------
# 2. BASIC VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASIC VALIDATION")
print("=" * 70)

print("Total detections:", len(df))

if len(df) != 655204:
    raise ValueError(
        f"Expected 655,204 detections but found {len(df)}"
    )

print("Expected detection count: 655204")
print("Detection count check: PASS")


# ------------------------------------------------------------
# 3. REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "event_id",
    "predicted_label",
    "predicted_class",
    "prediction_confidence",
    "prob_industrial",
    "prob_gas",
    "prob_agriculture",
    "prob_mining",
    "prob_wildfire",
    "status"
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Required classification columns: PASS")


# ------------------------------------------------------------
# 4. EVENT COUNT
# ------------------------------------------------------------

unique_events = df["event_id"].nunique()

print("\nUnique events:", unique_events)

if unique_events != 302070:
    raise ValueError(
        f"Expected 302,070 events but found {unique_events}"
    )

print("Event count check: PASS")


# ------------------------------------------------------------
# 5. DUPLICATE EVENT-DETECTION CHECK
# ------------------------------------------------------------

if "detection_id" in df.columns:

    duplicate_detection_ids = (
        df["detection_id"].duplicated().sum()
    )

    print(
        "\nDuplicate detection IDs:",
        duplicate_detection_ids
    )

else:

    duplicate_detection_ids = None

    print(
        "\n'detection_id' column not found."
    )

    print(
        "Checking duplicate rows instead."
    )

    duplicate_rows = df.duplicated().sum()

    print(
        "Duplicate complete rows:",
        duplicate_rows
    )


# ------------------------------------------------------------
# 6. MISSING CLASSIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION COMPLETENESS")
print("=" * 70)

missing_class = df["predicted_class"].isna().sum()
missing_status = df["status"].isna().sum()
missing_confidence = df["prediction_confidence"].isna().sum()

print(
    "Missing predicted class:",
    missing_class
)

print(
    "Missing status:",
    missing_status
)

print(
    "Missing prediction confidence:",
    missing_confidence
)


# ------------------------------------------------------------
# 7. SOURCE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL DETECTION-LEVEL SOURCE DISTRIBUTION")
print("=" * 70)

source_order = [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]

source_summary = []

for source in source_order:

    subset = df[
        df["predicted_class"] == source
    ]

    count = len(subset)

    percentage = (
        count / len(df) * 100
    )

    mean_confidence = (
        subset["prediction_confidence"].mean()
        if count > 0
        else np.nan
    )

    median_confidence = (
        subset["prediction_confidence"].median()
        if count > 0
        else np.nan
    )

    source_summary.append({
        "source": source,
        "detections": count,
        "percentage": percentage,
        "mean_confidence": mean_confidence,
        "median_confidence": median_confidence
    })

source_summary_df = pd.DataFrame(
    source_summary
)

print(
    source_summary_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 8. STATUS DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CLASSIFICATION STATUS")
print("=" * 70)

status_summary = (
    df["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="detections")
)

status_summary["percentage"] = (
    status_summary["detections"]
    / len(df)
    * 100
)

print(
    status_summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 9. CONFIDENCE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PREDICTION CONFIDENCE")
print("=" * 70)

print(
    df["prediction_confidence"]
    .describe()
)


# ------------------------------------------------------------
# 10. CONFIDENCE BANDS
# ------------------------------------------------------------

bins = [
    0.0,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    0.95,
    1.000001
]

labels = [
    "<0.50",
    "0.50-0.60",
    "0.60-0.70",
    "0.70-0.80",
    "0.80-0.90",
    "0.90-0.95",
    "0.95-1.00"
]

df["confidence_band"] = pd.cut(
    df["prediction_confidence"],
    bins=bins,
    labels=labels,
    right=False
)

confidence_summary = (
    df["confidence_band"]
    .value_counts(sort=False)
    .rename_axis("confidence_band")
    .reset_index(name="detections")
)

confidence_summary["percentage"] = (
    confidence_summary["detections"]
    / len(df)
    * 100
)

print(
    confidence_summary.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 11. HIGH-CONFIDENCE COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE COVERAGE")
print("=" * 70)

for threshold in [
    0.70,
    0.80,
    0.90,
    0.95
]:

    count = (
        df["prediction_confidence"]
        >= threshold
    ).sum()

    percentage = (
        count / len(df) * 100
    )

    print(
        f"Confidence >= {threshold:.2f}: "
        f"{count:8d} "
        f"({percentage:6.2f}%)"
    )


# ------------------------------------------------------------
# 12. CHECK PROBABILITY SUM
# ------------------------------------------------------------

probability_columns = [
    "prob_industrial",
    "prob_gas",
    "prob_agriculture",
    "prob_mining",
    "prob_wildfire"
]

prob_sum = df[
    probability_columns
].sum(axis=1)

probability_error = (
    prob_sum - 1.0
).abs()

print("\n" + "=" * 70)
print("PROBABILITY VALIDATION")
print("=" * 70)

print(
    "Maximum probability-sum error:",
    probability_error.max()
)

print(
    "Mean probability-sum error:",
    probability_error.mean()
)

if probability_error.max() > 1e-5:

    print(
        "WARNING: Some probability rows do not sum to 1."
    )

else:

    print(
        "Probability sum check: PASS"
    )


# ------------------------------------------------------------
# 13. SAVE AUDIT SUMMARIES
# ------------------------------------------------------------

source_output = (
    "/kaggle/working/"
    "firms_2025_final_detection_source_summary.csv"
)

status_output = (
    "/kaggle/working/"
    "firms_2025_final_detection_status_summary.csv"
)

confidence_output = (
    "/kaggle/working/"
    "firms_2025_final_detection_confidence_summary.csv"
)

source_summary_df.to_csv(
    source_output,
    index=False
)

status_summary.to_csv(
    status_output,
    index=False
)

confidence_summary.to_csv(
    confidence_output,
    index=False
)


print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(source_output)
print(status_output)
print(confidence_output)

print("\n" + "=" * 70)
print("STEP 9A COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 9B — CLASSIFIED vs UNKNOWN ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 9B — CLASSIFIED vs UNKNOWN ANALYSIS")
print("=" * 70)

FILE = (
    "/kaggle/working/"
    "firms_2025_final_detection_source_classification.csv"
)

df = pd.read_csv(FILE)

# ------------------------------------------------------------
# 1. EVENT-LEVEL STATUS
# ------------------------------------------------------------

event_status = (
    df.groupby("event_id")
      .agg(
          event_status=("status", "first"),
          event_confidence=("prediction_confidence", "first"),
          event_source=("predicted_class", "first"),
          detection_count=("event_id", "size")
      )
      .reset_index()
)

print("\nEvent-level status:")
print(
    event_status["event_status"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 2. EVENT-LEVEL CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EVENT-LEVEL SOURCE DISTRIBUTION")
print("=" * 70)

event_class_counts = (
    event_status["event_source"]
    .value_counts()
)

for source in [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]:

    count = int(
        event_class_counts.get(source, 0)
    )

    pct = (
        count /
        len(event_status)
        * 100
    )

    print(
        f"{source:15s}: "
        f"{count:8d} "
        f"({pct:6.2f}%)"
    )

# ------------------------------------------------------------
# 3. EVENT STATUS BY SOURCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STATUS BY SOURCE — EVENT LEVEL")
print("=" * 70)

event_source_status = pd.crosstab(
    event_status["event_source"],
    event_status["event_status"]
)

print(event_source_status)

# ------------------------------------------------------------
# 4. DETECTION COUNT BY EVENT STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DETECTIONS CONTRIBUTED BY EVENT STATUS")
print("=" * 70)

detection_status = (
    event_status
    .groupby("event_status")["detection_count"]
    .agg(
        events="count",
        detections="sum",
        mean_detections_per_event="mean",
        median_detections_per_event="median",
        max_detections_per_event="max"
    )
    .reset_index()
)

detection_status["detection_percentage"] = (
    detection_status["detections"]
    / len(df)
    * 100
)

print(
    detection_status.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 5. EVENT SIZE VS CONFIDENCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EVENT SIZE VS CONFIDENCE")
print("=" * 70)

event_size_confidence = (
    event_status
    .groupby("event_status")
    .agg(
        events=("event_id", "count"),
        mean_detection_count=("detection_count", "mean"),
        median_detection_count=("detection_count", "median"),
        mean_confidence=("event_confidence", "mean"),
        median_confidence=("event_confidence", "median"),
        min_confidence=("event_confidence", "min"),
        max_confidence=("event_confidence", "max")
    )
    .reset_index()
)

print(
    event_size_confidence.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 6. CONFIDENCE BY SOURCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFIDENCE BY SOURCE — EVENT LEVEL")
print("=" * 70)

source_confidence = (
    event_status
    .groupby("event_source")
    .agg(
        events=("event_id", "count"),
        mean_confidence=("event_confidence", "mean"),
        median_confidence=("event_confidence", "median"),
        p10_confidence=("event_confidence",
                        lambda x: x.quantile(0.10)),
        p25_confidence=("event_confidence",
                        lambda x: x.quantile(0.25)),
        p75_confidence=("event_confidence",
                        lambda x: x.quantile(0.75)),
        p90_confidence=("event_confidence",
                        lambda x: x.quantile(0.90))
    )
    .reset_index()
)

print(
    source_confidence.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 7. CONFIDENCE THRESHOLDS AT EVENT LEVEL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EVENT-LEVEL CONFIDENCE COVERAGE")
print("=" * 70)

for threshold in [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    0.95
]:

    count = (
        event_status["event_confidence"]
        >= threshold
    ).sum()

    pct = (
        count /
        len(event_status)
        * 100
    )

    print(
        f">= {threshold:.2f}: "
        f"{count:8d} "
        f"({pct:6.2f}%)"
    )

# ------------------------------------------------------------
# 8. SAVE EVENT-LEVEL AUDIT
# ------------------------------------------------------------

event_status_output = (
    "/kaggle/working/"
    "firms_2025_final_event_status_audit.csv"
)

event_source_output = (
    "/kaggle/working/"
    "firms_2025_final_event_source_status.csv"
)

event_confidence_output = (
    "/kaggle/working/"
    "firms_2025_final_event_confidence_summary.csv"
)

event_status.to_csv(
    event_status_output,
    index=False
)

event_source_status.to_csv(
    event_source_output
)

source_confidence.to_csv(
    event_confidence_output,
    index=False
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print(event_status_output)
print(event_source_output)
print(event_confidence_output)

print("\n" + "=" * 70)
print("STEP 9B COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 9C — FINAL SOURCE EVIDENCE AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 9C — FINAL SOURCE EVIDENCE AUDIT")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD FINAL EVENT CLASSIFICATION
# ------------------------------------------------------------

classification_file = (
    "/kaggle/working/"
    "firms_2025_final_all_event_source_classification.csv"
)

event_features_file = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

pred = pd.read_csv(classification_file)
features = pd.read_csv(event_features_file)

print("\nPredictions:", pred.shape)
print("Event features:", features.shape)


# ------------------------------------------------------------
# 2. MERGE
# ------------------------------------------------------------

df = pred.merge(
    features,
    on="event_id",
    how="left",
    validate="one_to_one"
)

print("Merged:", df.shape)


# ------------------------------------------------------------
# 3. VERIFY
# ------------------------------------------------------------

if len(df) != len(pred):
    raise ValueError(
        "Event count changed during merge."
    )

missing_features = (
    df["event_detection_count"]
    .isna()
    .sum()
)

print(
    "Events missing feature data:",
    missing_features
)


# ------------------------------------------------------------
# 4. SOURCE SUMMARY
# ------------------------------------------------------------

sources = [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]

summary = []

for source in sources:

    s = df[
        df["predicted_class"] == source
    ]

    if len(s) == 0:
        continue

    classified = (
        s["status"] == "Classified"
    ).sum()

    unknown = (
        s["status"] == "Unknown"
    ).sum()

    summary.append({
        "source": source,
        "events": len(s),
        "classified_events": classified,
        "unknown_events": unknown,
        "classified_percentage": (
            classified / len(s) * 100
        ),
        "detections": s["event_detection_count"].sum(),
        "mean_detection_count": (
            s["event_detection_count"].mean()
        ),
        "median_detection_count": (
            s["event_detection_count"].median()
        ),
        "mean_active_days": (
            s["event_active_days"].mean()
        ),
        "median_active_days": (
            s["event_active_days"].median()
        ),
        "mean_spatial_extent_km": (
            s["event_spatial_extent_km"].mean()
        ),
        "mean_frp": (
            s["event_mean_frp"].mean()
        ),
        "mean_confidence": (
            s["prediction_confidence"].mean()
        ),
        "median_confidence": (
            s["prediction_confidence"].median()
        )
    })

summary_df = pd.DataFrame(summary)

print("\n" + "=" * 70)
print("SOURCE SUMMARY")
print("=" * 70)

print(
    summary_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 5. INDUSTRIAL SUPPORT
# ------------------------------------------------------------

industrial_conditions = {
    "industrial_500m_multi_source":
        "event_any_industrial_multi_source_500m",

    "steel_500m":
        "event_any_steel_within_500m",

    "cement_500m":
        "event_any_cement_within_500m",

    "power_500m":
        "event_any_wri_power_within_500m",

    "fertilizer_500m":
        "event_any_fertilizer_within_500m",

    "refinery_500m":
        "event_any_refinery_petro_within_500m"
}

print("\n" + "=" * 70)
print("INDUSTRIAL PREDICTION SUPPORT")
print("=" * 70)

industrial_events = df[
    df["predicted_class"] == "Industrial"
]

for name, column in industrial_conditions.items():

    if column not in df.columns:
        continue

    supported = (
        industrial_events[column]
        .fillna(0)
        .astype(bool)
        .sum()
    )

    print(
        f"{name:35s}: "
        f"{supported:6d} / "
        f"{len(industrial_events):6d} "
        f"({supported / len(industrial_events) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 6. GAS SUPPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GAS PREDICTION SUPPORT")
print("=" * 70)

gas_events = df[
    df["predicted_class"] == "Gas"
]

gas_conditions = {
    "gas_flare_1km":
        "has_gas_flare_within_1km",

    "gas_flare_2km":
        "has_gas_flare_within_2km",

    "gas_flare_5km":
        "has_gas_flare_within_5km",

    "gas_flare_10km":
        "has_gas_flare_within_10km"
}

for name, column in gas_conditions.items():

    if column not in df.columns:
        continue

    supported = (
        gas_events[column]
        .fillna(0)
        .astype(bool)
        .sum()
    )

    print(
        f"{name:35s}: "
        f"{supported:6d} / "
        f"{len(gas_events):6d} "
        f"({supported / len(gas_events) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 7. AGRICULTURE SUPPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AGRICULTURE PREDICTION SUPPORT")
print("=" * 70)

agri_events = df[
    df["predicted_class"] == "Agriculture"
]

if len(agri_events) > 0:

    for threshold in [0.50, 0.70]:

        supported = (
            agri_events[
                "event_max_dw_crop_probability"
            ]
            >= threshold
        ).sum()

        print(
            f"DW crop probability >= {threshold:.2f}: "
            f"{supported:6d} / "
            f"{len(agri_events):6d} "
            f"({supported / len(agri_events) * 100:6.2f}%)"
        )

    if "agri_osm_within_500m_max" in df.columns:

        supported = (
            agri_events[
                "agri_osm_within_500m_max"
            ]
            .fillna(0)
            .astype(bool)
            .sum()
        )

        print(
            f"OSM agriculture within 500m: "
            f"{supported:6d} / "
            f"{len(agri_events):6d} "
            f"({supported / len(agri_events) * 100:6.2f}%)"
        )


# ------------------------------------------------------------
# 8. MINING SUPPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MINING PREDICTION SUPPORT")
print("=" * 70)

mining_events = df[
    df["predicted_class"] == "Mining"
]

mining_conditions = {
    "coal_mine_500m":
        "event_any_coal_mine_within_500m",

    "coal_mine_1km":
        "event_any_coal_mine_within_1000m",

    "mining_close_500m":
        "event_any_mining_evidence_close_500m",

    "mining_recurrent_500m_7d":
        "event_any_mining_evidence_close_500m_recurrent_7d"
}

for name, column in mining_conditions.items():

    if column not in df.columns:
        continue

    supported = (
        mining_events[column]
        .fillna(0)
        .astype(bool)
        .sum()
    )

    print(
        f"{name:35s}: "
        f"{supported:6d} / "
        f"{len(mining_events):6d} "
        f"({supported / len(mining_events) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 9. WILDFIRE SUPPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WILDFIRE PREDICTION SUPPORT")
print("=" * 70)

wildfire_events = df[
    df["predicted_class"] == "Wildfire"
]

wildfire_conditions = {
    "forest_1km":
        "near_forest_1km_max",

    "natural_vegetation_1km":
        "near_natural_vegetation_1km_max",

    "natural_landcover_context":
        "natural_landcover_context_max",

    "natural_vegetation_strong":
        "natural_vegetation_strong_max",

    "dw_natural_vegetation":
        "event_max_dw_natural_vegetation_prob"
}

for name, column in wildfire_conditions.items():

    if column not in df.columns:
        continue

    values = wildfire_events[column]

    if values.dtype == bool:

        supported = values.fillna(False).sum()

    else:

        supported = (
            values.fillna(0) > 0
        ).sum()

    print(
        f"{name:35s}: "
        f"{supported:6d} / "
        f"{len(wildfire_events):6d} "
        f"({supported / len(wildfire_events) * 100:6.2f}%)"
    )


# ------------------------------------------------------------
# 10. SAVE SUMMARY
# ------------------------------------------------------------

output_file = (
    "/kaggle/working/"
    "firms_2025_final_source_evidence_audit.csv"
)

summary_df.to_csv(
    output_file,
    index=False
)

print("\nSaved:")
print(output_file)

print("\n" + "=" * 70)
print("STEP 9C COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 9D — HIGH-CONFIDENCE SOURCE VALIDATION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 9D — HIGH-CONFIDENCE SOURCE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

prediction_file = (
    "/kaggle/working/"
    "firms_2025_final_all_event_source_classification.csv"
)

feature_file = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

pred = pd.read_csv(prediction_file)
features = pd.read_csv(feature_file)

df = pred.merge(
    features,
    on="event_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 2. HIGH-CONFIDENCE ONLY
# ------------------------------------------------------------

hc = df[
    df["prediction_confidence"] >= 0.70
].copy()

print("\nAll events:", len(df))
print("High-confidence events:", len(hc))
print(
    "High-confidence percentage:",
    len(hc) / len(df) * 100
)

# ------------------------------------------------------------
# 3. SOURCE COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE SOURCE DISTRIBUTION")
print("=" * 70)

source_counts = (
    hc["predicted_class"]
    .value_counts()
)

for source in [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]:

    count = int(
        source_counts.get(source, 0)
    )

    pct = (
        count / len(hc) * 100
        if len(hc) > 0
        else 0
    )

    print(
        f"{source:15s}: "
        f"{count:8d} "
        f"({pct:6.2f}%)"
    )

# ------------------------------------------------------------
# 4. INDUSTRIAL
# ------------------------------------------------------------

industrial = hc[
    hc["predicted_class"] == "Industrial"
]

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE INDUSTRIAL")
print("=" * 70)

industrial_tests = {
    "multi_industrial_500m":
        "event_any_industrial_multi_source_500m",

    "steel_500m":
        "event_any_steel_within_500m",

    "cement_500m":
        "event_any_cement_within_500m",

    "power_500m":
        "event_any_wri_power_within_500m",

    "fertilizer_500m":
        "event_any_fertilizer_within_500m",

    "refinery_500m":
        "event_any_refinery_petro_within_500m"
}

for name, col in industrial_tests.items():

    if col not in industrial.columns:
        continue

    n = (
        industrial[col]
        .fillna(0)
        .astype(bool)
        .sum()
    )

    print(
        f"{name:30s}: "
        f"{n:6d}/{len(industrial):6d} "
        f"({n / len(industrial) * 100:6.2f}%)"
    )

# ------------------------------------------------------------
# 5. GAS
# ------------------------------------------------------------

gas = hc[
    hc["predicted_class"] == "Gas"
]

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE GAS")
print("=" * 70)

gas_tests = {
    "gas_flare_1km":
        "has_gas_flare_within_1km",

    "gas_flare_2km":
        "has_gas_flare_within_2km",

    "gas_flare_5km":
        "has_gas_flare_within_5km"
}

for name, col in gas_tests.items():

    if col not in gas.columns:
        continue

    n = (
        gas[col]
        .fillna(0)
        .astype(bool)
        .sum()
    )

    print(
        f"{name:30s}: "
        f"{n:6d}/{len(gas):6d} "
        f"({n / len(gas) * 100:6.2f}%)"
    )

# ------------------------------------------------------------
# 6. AGRICULTURE
# ------------------------------------------------------------

agri = hc[
    hc["predicted_class"] == "Agriculture"
]

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE AGRICULTURE")
print("=" * 70)

for threshold in [0.50, 0.60, 0.70]:

    n = (
        agri["event_max_dw_crop_probability"]
        >= threshold
    ).sum()

    print(
        f"DW crop probability >= {threshold:.2f}: "
        f"{n:6d}/{len(agri):6d} "
        f"({n / len(agri) * 100:6.2f}%)"
    )

if "agri_osm_within_500m_max" in agri.columns:

    n = (
        agri["agri_osm_within_500m_max"]
        .fillna(0)
        .astype(bool)
        .sum()
    )

    print(
        f"OSM agriculture within 500m: "
        f"{n:6d}/{len(agri):6d} "
        f"({n / len(agri) * 100:6.2f}%)"
    )

# ------------------------------------------------------------
# 7. MINING
# ------------------------------------------------------------

mining = hc[
    hc["predicted_class"] == "Mining"
]

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE MINING")
print("=" * 70)

mining_tests = {
    "coal_mine_500m":
        "event_any_coal_mine_within_500m",

    "coal_mine_1km":
        "event_any_coal_mine_within_1000m",

    "mining_close_500m":
        "event_any_mining_evidence_close_500m",

    "mining_recurrent_500m_7d":
        "event_any_mining_evidence_close_500m_recurrent_7d"
}

for name, col in mining_tests.items():

    if col not in mining.columns:
        continue

    n = (
        mining[col]
        .fillna(0)
        .astype(bool)
        .sum()
    )

    print(
        f"{name:30s}: "
        f"{n:6d}/{len(mining):6d} "
        f"({n / len(mining) * 100:6.2f}%)"
    )

# ------------------------------------------------------------
# 8. WILDFIRE
# ------------------------------------------------------------

wildfire = hc[
    hc["predicted_class"] == "Wildfire"
]

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE WILDFIRE")
print("=" * 70)

wildfire_tests = {
    "forest_1km":
        "near_forest_1km_max",

    "natural_vegetation_1km":
        "near_natural_vegetation_1km_max",

    "natural_landcover":
        "natural_landcover_context_max",

    "natural_vegetation_strong":
        "natural_vegetation_strong_max"
}

for name, col in wildfire_tests.items():

    if col not in wildfire.columns:
        continue

    n = (
        wildfire[col]
        .fillna(0)
        .astype(bool)
        .sum()
    )

    print(
        f"{name:30s}: "
        f"{n:6d}/{len(wildfire):6d} "
        f"({n / len(wildfire) * 100:6.2f}%)"
    )

if "event_max_dw_natural_vegetation_prob" in wildfire.columns:

    for threshold in [0.40, 0.50, 0.60]:

        n = (
            wildfire[
                "event_max_dw_natural_vegetation_prob"
            ]
            >= threshold
        ).sum()

        print(
            f"DW natural vegetation >= {threshold:.2f}: "
            f"{n:6d}/{len(wildfire):6d} "
            f"({n / len(wildfire) * 100:6.2f}%)"
        )

# ------------------------------------------------------------
# 9. SAVE HIGH-CONFIDENCE DATASET
# ------------------------------------------------------------

output = (
    "/kaggle/working/"
    "firms_2025_high_confidence_event_classification.csv"
)

hc[
    [
        "event_id",
        "predicted_class",
        "prediction_confidence",
        "status"
    ]
].to_csv(
    output,
    index=False
)

print("\nSaved:")
print(output)

print("\n" + "=" * 70)
print("STEP 9D COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 10A — FIND TEST FIRMS POINT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 10A — FIND TEST FIRMS POINT")
print("=" * 70)

# ------------------------------------------------------------
# TEST POINT
# ------------------------------------------------------------

TEST_LAT = 24.235000
TEST_LON = 72.182778
TEST_DATE = "2025-04-01"

print("\nTest point:")
print("Latitude :", TEST_LAT)
print("Longitude:", TEST_LON)
print("Date     :", TEST_DATE)


# ------------------------------------------------------------
# LOAD ORIGINAL FIRMS EVENT DATA
# ------------------------------------------------------------

file = (
    "/kaggle/working/"
    "firms_2025_events.csv"
)

df = pd.read_csv(file)

print("\nFIRMS dataset:", df.shape)


# ------------------------------------------------------------
# CONVERT DATE
# ------------------------------------------------------------

date_columns = [
    c for c in df.columns
    if "datetime" in c.lower()
    or "date" in c.lower()
]

print("\nPossible date columns:")
print(date_columns)


# ------------------------------------------------------------
# FIND LAT/LON COLUMNS
# ------------------------------------------------------------

lat_candidates = [
    c for c in df.columns
    if c.lower() in [
        "latitude",
        "lat"
    ]
]

lon_candidates = [
    c for c in df.columns
    if c.lower() in [
        "longitude",
        "lon",
        "lng"
    ]
]

print("\nLatitude columns:", lat_candidates)
print("Longitude columns:", lon_candidates)


# ------------------------------------------------------------
# USE STANDARD FIRMS NAMES
# ------------------------------------------------------------

if "latitude" not in df.columns:
    raise ValueError("latitude column not found.")

if "longitude" not in df.columns:
    raise ValueError("longitude column not found.")


# ------------------------------------------------------------
# DATE FILTER
# ------------------------------------------------------------

if "acq_date" in df.columns:

    df["test_date"] = (
        pd.to_datetime(
            df["acq_date"],
            errors="coerce"
        )
        .dt.date
    )

    target_date = pd.Timestamp(
        TEST_DATE
    ).date()

    date_df = df[
        df["test_date"] == target_date
    ].copy()

else:

    raise ValueError(
        "acq_date column not found."
    )


print(
    "\nFIRMS detections on test date:",
    len(date_df)
)


# ------------------------------------------------------------
# DISTANCE APPROXIMATION
# ------------------------------------------------------------

# 1 degree latitude ≈ 111 km
# longitude degree corrected by latitude

lat_diff_km = (
    (date_df["latitude"] - TEST_LAT)
    * 111.32
)

lon_diff_km = (
    (date_df["longitude"] - TEST_LON)
    * 111.32
    * np.cos(
        np.radians(TEST_LAT)
    )
)

date_df["distance_km"] = np.sqrt(
    lat_diff_km ** 2 +
    lon_diff_km ** 2
)


# ------------------------------------------------------------
# FIND NEAREST FIRMS DETECTIONS
# ------------------------------------------------------------

nearest = (
    date_df
    .sort_values("distance_km")
    .head(10)
    .copy()
)


print("\n" + "=" * 70)
print("10 NEAREST FIRMS DETECTIONS")
print("=" * 70)

display_columns = [
    c for c in [
        "detection_id",
        "event_id",
        "latitude",
        "longitude",
        "acq_date",
        "acq_time",
        "frp",
        "brightness",
        "satellite",
        "instrument",
        "distance_km"
    ]
    if c in nearest.columns
]

print(
    nearest[display_columns]
    .to_string(index=False)
)


# ------------------------------------------------------------
# CLOSEST POINT
# ------------------------------------------------------------

if len(nearest) == 0:

    print("\nNO FIRMS DETECTION FOUND ON 2025-04-01.")

else:

    closest = nearest.iloc[0]

    print("\n" + "=" * 70)
    print("CLOSEST FIRMS DETECTION")
    print("=" * 70)

    print(
        "Distance:",
        round(
            closest["distance_km"],
            4
        ),
        "km"
    )

    print(
        "Latitude:",
        closest["latitude"]
    )

    print(
        "Longitude:",
        closest["longitude"]
    )

    if "event_id" in closest:
        print(
            "Event ID:",
            closest["event_id"]
        )

    if "detection_id" in closest:
        print(
            "Detection ID:",
            closest["detection_id"]
        )


# ------------------------------------------------------------
# SAVE SEARCH RESULT
# ------------------------------------------------------------

output = (
    "/kaggle/working/"
    "firms_2025_single_test_point_matches.csv"
)

nearest.to_csv(
    output,
    index=False
)

print("\nSaved:")
print(output)

print("\n" + "=" * 70)
print("STEP 10A COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 10B — TEST ONE REAL FIRMS DETECTION
# ============================================================

import pandas as pd
import numpy as np
import joblib

print("=" * 70)
print("STEP 10B — TEST ONE REAL FIRMS DETECTION")
print("=" * 70)


# ------------------------------------------------------------
# 1. TEST FIRMS POINT
# ------------------------------------------------------------

TEST_LAT = 25.67829
TEST_LON = 81.06459
TEST_DATE = "2025-12-06"
TEST_TIME = 830

print("\nTest FIRMS detection:")
print("Latitude :", TEST_LAT)
print("Longitude:", TEST_LON)
print("Date     :", TEST_DATE)
print("Time     :", TEST_TIME)


# ------------------------------------------------------------
# 2. FILES
# ------------------------------------------------------------

event_file = (
    "/kaggle/working/"
    "firms_2025_events.csv"
)

prediction_file = (
    "/kaggle/working/"
    "firms_2025_final_all_event_source_classification.csv"
)

model_file = (
    "/kaggle/working/"
    "firms_2025_final_production_random_forest.joblib"
)

imputer_file = (
    "/kaggle/working/"
    "firms_2025_final_production_imputer.joblib"
)

feature_file = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

feature_columns_file = (
    "/kaggle/working/"
    "firms_2025_final_model_feature_columns.csv"
)


# ------------------------------------------------------------
# 3. LOAD FIRMS EVENT DATA
# ------------------------------------------------------------

events = pd.read_csv(event_file)

print("\nFIRMS event dataset:", events.shape)


# ------------------------------------------------------------
# 4. PREPARE DATE AND TIME
# ------------------------------------------------------------

events["acq_date_check"] = (
    pd.to_datetime(
        events["acq_date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")
)

events["acq_time_check"] = pd.to_numeric(
    events["acq_time"],
    errors="coerce"
)


# ------------------------------------------------------------
# 5. FIND EXACT FIRMS DETECTION
# ------------------------------------------------------------

exact = events[
    (np.isclose(
        events["latitude"],
        TEST_LAT,
        atol=1e-8
    )) &
    (np.isclose(
        events["longitude"],
        TEST_LON,
        atol=1e-8
    )) &
    (events["acq_date_check"] == TEST_DATE) &
    (events["acq_time_check"] == TEST_TIME)
].copy()


# ------------------------------------------------------------
# 6. CHECK MATCH
# ------------------------------------------------------------

if len(exact) == 0:

    print("\n❌ EXACT FIRMS DETECTION NOT FOUND.")

    # Search same date and nearest coordinates
    candidates = events[
        events["acq_date_check"] == TEST_DATE
    ].copy()

    if len(candidates) == 0:

        raise ValueError(
            f"No FIRMS detections found on {TEST_DATE}."
        )

    lat_diff_km = (
        candidates["latitude"] - TEST_LAT
    ) * 111.32

    lon_diff_km = (
        candidates["longitude"] - TEST_LON
    ) * 111.32 * np.cos(
        np.radians(TEST_LAT)
    )

    candidates["distance_km"] = np.sqrt(
        lat_diff_km ** 2 +
        lon_diff_km ** 2
    )

    candidates = (
        candidates
        .sort_values("distance_km")
        .head(10)
    )

    print("\nNearest FIRMS detections on same date:")

    display(
        candidates[
            [
                "detection_id",
                "event_id",
                "latitude",
                "longitude",
                "acq_date",
                "acq_time",
                "distance_km"
            ]
        ]
    )

    raise ValueError(
        "Exact test FIRMS detection was not found."
    )


# ------------------------------------------------------------
# 7. VERIFY UNIQUE MATCH
# ------------------------------------------------------------

if len(exact) > 1:

    print(
        "\nWARNING:",
        len(exact),
        "matching rows found."
    )

    print(
        exact[
            [
                "detection_id",
                "event_id",
                "latitude",
                "longitude",
                "acq_date",
                "acq_time"
            ]
        ]
    )

    raise ValueError(
        "Multiple FIRMS detections matched the test point."
    )


print("\n✅ EXACT FIRMS DETECTION FOUND.")


# ------------------------------------------------------------
# 8. DISPLAY MATCHED FIRMS DETECTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MATCHED FIRMS DETECTION")
print("=" * 70)

display_columns = [
    c for c in [
        "detection_id",
        "event_id",
        "latitude",
        "longitude",
        "frp",
        "brightness",
        "acq_date",
        "acq_time",
        "satellite",
        "instrument"
    ]
    if c in exact.columns
]

print(
    exact[
        display_columns
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 9. GET EVENT ID
# ------------------------------------------------------------

event_id = exact.iloc[0]["event_id"]

print("\nEvent ID:", event_id)


# ------------------------------------------------------------
# 10. LOAD STORED PRODUCTION PREDICTION
# ------------------------------------------------------------

predictions = pd.read_csv(
    prediction_file
)

existing_prediction = predictions[
    predictions["event_id"] == event_id
].copy()

if len(existing_prediction) != 1:

    raise ValueError(
        f"Expected exactly 1 event prediction, "
        f"found {len(existing_prediction)}."
    )


print("\n" + "=" * 70)
print("STORED PRODUCTION PREDICTION")
print("=" * 70)

print(
    existing_prediction[
        [
            "event_id",
            "predicted_label",
            "predicted_class",
            "prediction_confidence",
            "prob_industrial",
            "prob_gas",
            "prob_agriculture",
            "prob_mining",
            "prob_wildfire",
            "status"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 11. LOAD PRODUCTION MODEL
# ------------------------------------------------------------

model = joblib.load(
    model_file
)

imputer = joblib.load(
    imputer_file
)

print(
    "\nProduction model loaded successfully."
)


# ------------------------------------------------------------
# 12. LOAD MODEL FEATURE LIST
# ------------------------------------------------------------

feature_columns = pd.read_csv(
    feature_columns_file
)

if "feature" in feature_columns.columns:

    model_features = (
        feature_columns["feature"]
        .tolist()
    )

else:

    model_features = (
        feature_columns
        .iloc[:, 0]
        .tolist()
    )

print(
    "Model feature count:",
    len(model_features)
)

if len(model_features) != 137:

    raise ValueError(
        f"Expected 137 model features, "
        f"found {len(model_features)}."
    )


# ------------------------------------------------------------
# 13. LOAD EVENT FEATURES
# ------------------------------------------------------------

features = pd.read_csv(
    feature_file
)

test_features = features[
    features["event_id"] == event_id
].copy()

if len(test_features) != 1:

    raise ValueError(
        "Expected exactly one feature row "
        "for this event."
    )


print(
    "Event feature row found."
)


# ------------------------------------------------------------
# 14. CREATE MODEL INPUT
# ------------------------------------------------------------

X_test = test_features[
    model_features
].copy()

X_test = X_test.apply(
    pd.to_numeric,
    errors="coerce"
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# 15. APPLY PRODUCTION IMPUTER
# ------------------------------------------------------------

X_test_imputed = imputer.transform(
    X_test
)


# ------------------------------------------------------------
# 16. FRESH MODEL PREDICTION
# ------------------------------------------------------------

predicted_label = model.predict(
    X_test_imputed
)[0]

probabilities = model.predict_proba(
    X_test_imputed
)[0]

classes = model.classes_

probability_map = dict(
    zip(
        classes,
        probabilities
    )
)

confidence = float(
    probabilities.max()
)


# ------------------------------------------------------------
# 17. CLASS NAMES
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

predicted_class = class_names[
    int(predicted_label)
]

status = (
    "Classified"
    if confidence >= 0.70
    else "Unknown"
)


# ------------------------------------------------------------
# 18. DISPLAY FRESH PREDICTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FRESH PRODUCTION MODEL PREDICTION")
print("=" * 70)

print(
    "\nPredicted source:",
    predicted_class
)

print(
    "Confidence:",
    round(
        confidence,
        6
    )
)

print(
    "Status:",
    status
)

print("\nClass probabilities:")

for class_id in [
    0,
    1,
    2,
    3,
    4
]:

    probability = probability_map.get(
        class_id,
        0.0
    )

    print(
        f"{class_names[class_id]:15s}: "
        f"{probability:.6f}"
    )


# ------------------------------------------------------------
# 19. CONSISTENCY CHECK
# ------------------------------------------------------------

stored = existing_prediction.iloc[0]

stored_source = (
    stored["predicted_class"]
)

stored_confidence = float(
    stored["prediction_confidence"]
)

print("\n" + "=" * 70)
print("PREDICTION CONSISTENCY CHECK")
print("=" * 70)

print(
    "Stored source :",
    stored_source
)

print(
    "Fresh source  :",
    predicted_class
)

print(
    "Stored conf.  :",
    round(
        stored_confidence,
        6
    )
)

print(
    "Fresh conf.   :",
    round(
        confidence,
        6
    )
)


# ------------------------------------------------------------
# SOURCE CHECK
# ------------------------------------------------------------

if stored_source == predicted_class:

    print(
        "\nSOURCE CHECK: PASS"
    )

else:

    print(
        "\nSOURCE CHECK: FAIL"
    )


# ------------------------------------------------------------
# CONFIDENCE CHECK
# ------------------------------------------------------------

if np.isclose(
    stored_confidence,
    confidence,
    atol=1e-6
):

    print(
        "CONFIDENCE CHECK: PASS"
    )

else:

    print(
        "CONFIDENCE CHECK: DIFFERENT"
    )


# ------------------------------------------------------------
# 20. FINAL TEST SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SINGLE-POINT TEST RESULT")
print("=" * 70)

print(
    f"Coordinates : "
    f"{TEST_LAT}, {TEST_LON}"
)

print(
    f"Date        : {TEST_DATE}"
)

print(
    f"Time        : {TEST_TIME}"
)

print(
    f"Event ID    : {event_id}"
)

print(
    f"Source      : {predicted_class}"
)

print(
    f"Confidence  : {confidence:.6f}"
)

print(
    f"Status      : {status}"
)

print("\n" + "=" * 70)
print("STEP 10B COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ============================================================
# STEP 10B — TEST ONE REAL FIRMS DETECTION
# ============================================================

import pandas as pd
import numpy as np
import joblib

print("=" * 70)
print("STEP 10B — TEST ONE REAL FIRMS DETECTION")
print("=" * 70)


# ------------------------------------------------------------
# 1. TEST FIRMS POINT
# ------------------------------------------------------------

TEST_LAT = 23.27387
TEST_LON = 93.24695
TEST_DATE = "2025-03-11"
TEST_TIME = 747

print("\nTest FIRMS detection:")
print("Latitude :", TEST_LAT)
print("Longitude:", TEST_LON)
print("Date     :", TEST_DATE)
print("Time     :", TEST_TIME)


# ------------------------------------------------------------
# 2. FILES
# ------------------------------------------------------------

event_file = (
    "/kaggle/working/"
    "firms_2025_events.csv"
)

prediction_file = (
    "/kaggle/working/"
    "firms_2025_final_all_event_source_classification.csv"
)

model_file = (
    "/kaggle/working/"
    "firms_2025_final_production_random_forest.joblib"
)

imputer_file = (
    "/kaggle/working/"
    "firms_2025_final_production_imputer.joblib"
)

feature_file = (
    "/kaggle/working/"
    "firms_2025_event_features_all_source_evidence.csv"
)

feature_columns_file = (
    "/kaggle/working/"
    "firms_2025_final_model_feature_columns.csv"
)


# ------------------------------------------------------------
# 3. LOAD FIRMS EVENT DATA
# ------------------------------------------------------------

events = pd.read_csv(event_file)

print("\nFIRMS event dataset:", events.shape)


# ------------------------------------------------------------
# 4. PREPARE DATE AND TIME
# ------------------------------------------------------------

events["acq_date_check"] = (
    pd.to_datetime(
        events["acq_date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")
)

events["acq_time_check"] = pd.to_numeric(
    events["acq_time"],
    errors="coerce"
)


# ------------------------------------------------------------
# 5. FIND EXACT FIRMS DETECTION
# ------------------------------------------------------------

exact = events[
    (np.isclose(
        events["latitude"],
        TEST_LAT,
        atol=1e-8
    )) &
    (np.isclose(
        events["longitude"],
        TEST_LON,
        atol=1e-8
    )) &
    (events["acq_date_check"] == TEST_DATE) &
    (events["acq_time_check"] == TEST_TIME)
].copy()


# ------------------------------------------------------------
# 6. CHECK MATCH
# ------------------------------------------------------------

if len(exact) == 0:

    print("\n❌ EXACT FIRMS DETECTION NOT FOUND.")

    candidates = events[
        events["acq_date_check"] == TEST_DATE
    ].copy()

    if len(candidates) == 0:
        raise ValueError(
            f"No FIRMS detections found on {TEST_DATE}."
        )

    lat_diff_km = (
        candidates["latitude"] - TEST_LAT
    ) * 111.32

    lon_diff_km = (
        candidates["longitude"] - TEST_LON
    ) * 111.32 * np.cos(
        np.radians(TEST_LAT)
    )

    candidates["distance_km"] = np.sqrt(
        lat_diff_km ** 2 +
        lon_diff_km ** 2
    )

    candidates = (
        candidates
        .sort_values("distance_km")
        .head(10)
    )

    print("\nNearest FIRMS detections on same date:")

    display(
        candidates[
            [
                "detection_id",
                "event_id",
                "latitude",
                "longitude",
                "acq_date",
                "acq_time",
                "distance_km"
            ]
        ]
    )

    raise ValueError(
        "Exact test FIRMS detection was not found."
    )


# ------------------------------------------------------------
# 7. VERIFY UNIQUE MATCH
# ------------------------------------------------------------

if len(exact) > 1:

    print(
        "\nWARNING:",
        len(exact),
        "matching rows found."
    )

    display(
        exact[
            [
                "detection_id",
                "event_id",
                "latitude",
                "longitude",
                "acq_date",
                "acq_time"
            ]
        ]
    )

    raise ValueError(
        "Multiple FIRMS detections matched the test point."
    )


print("\n✅ EXACT FIRMS DETECTION FOUND.")


# ------------------------------------------------------------
# 8. DISPLAY MATCHED FIRMS DETECTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MATCHED FIRMS DETECTION")
print("=" * 70)

display_columns = [
    c for c in [
        "detection_id",
        "event_id",
        "latitude",
        "longitude",
        "frp",
        "brightness",
        "acq_date",
        "acq_time",
        "satellite",
        "instrument"
    ]
    if c in exact.columns
]

print(
    exact[
        display_columns
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 9. GET EVENT ID
# ------------------------------------------------------------

event_id = exact.iloc[0]["event_id"]

print("\nEvent ID:", event_id)


# ------------------------------------------------------------
# 10. LOAD STORED PRODUCTION PREDICTION
# ------------------------------------------------------------

predictions = pd.read_csv(
    prediction_file
)

existing_prediction = predictions[
    predictions["event_id"] == event_id
].copy()

if len(existing_prediction) != 1:

    raise ValueError(
        f"Expected exactly 1 event prediction, "
        f"found {len(existing_prediction)}."
    )


print("\n" + "=" * 70)
print("STORED PRODUCTION PREDICTION")
print("=" * 70)

print(
    existing_prediction[
        [
            "event_id",
            "predicted_label",
            "predicted_class",
            "prediction_confidence",
            "prob_industrial",
            "prob_gas",
            "prob_agriculture",
            "prob_mining",
            "prob_wildfire",
            "status"
        ]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 11. LOAD PRODUCTION MODEL
# ------------------------------------------------------------

model = joblib.load(
    model_file
)

imputer = joblib.load(
    imputer_file
)

print(
    "\nProduction model loaded successfully."
)


# ------------------------------------------------------------
# 12. LOAD MODEL FEATURE LIST
# ------------------------------------------------------------

feature_columns = pd.read_csv(
    feature_columns_file
)

if "feature" in feature_columns.columns:

    model_features = (
        feature_columns["feature"]
        .tolist()
    )

else:

    model_features = (
        feature_columns
        .iloc[:, 0]
        .tolist()
    )

print(
    "Model feature count:",
    len(model_features)
)

if len(model_features) != 137:

    raise ValueError(
        f"Expected 137 model features, "
        f"found {len(model_features)}."
    )


# ------------------------------------------------------------
# 13. LOAD EVENT FEATURES
# ------------------------------------------------------------

features = pd.read_csv(
    feature_file
)

test_features = features[
    features["event_id"] == event_id
].copy()

if len(test_features) != 1:

    raise ValueError(
        "Expected exactly one feature row "
        "for this event."
    )


print(
    "Event feature row found."
)


# ------------------------------------------------------------
# 14. CREATE MODEL INPUT
# ------------------------------------------------------------

X_test = test_features[
    model_features
].copy()

X_test = X_test.apply(
    pd.to_numeric,
    errors="coerce"
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)


# ------------------------------------------------------------
# 15. APPLY PRODUCTION IMPUTER
# ------------------------------------------------------------

X_test_imputed = imputer.transform(
    X_test
)


# ------------------------------------------------------------
# 16. FRESH MODEL PREDICTION
# ------------------------------------------------------------

predicted_label = model.predict(
    X_test_imputed
)[0]

probabilities = model.predict_proba(
    X_test_imputed
)[0]

classes = model.classes_

probability_map = dict(
    zip(
        classes,
        probabilities
    )
)

confidence = float(
    probabilities.max()
)


# ------------------------------------------------------------
# 17. CLASS NAMES
# ------------------------------------------------------------

class_names = {
    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}

predicted_class = class_names[
    int(predicted_label)
]

status = (
    "Classified"
    if confidence >= 0.70
    else "Unknown"
)


# ------------------------------------------------------------
# 18. DISPLAY FRESH PREDICTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FRESH PRODUCTION MODEL PREDICTION")
print("=" * 70)

print(
    "\nPredicted source:",
    predicted_class
)

print(
    "Confidence:",
    round(
        confidence,
        6
    )
)

print(
    "Status:",
    status
)

print("\nClass probabilities:")

for class_id in [
    0,
    1,
    2,
    3,
    4
]:

    probability = probability_map.get(
        class_id,
        0.0
    )

    print(
        f"{class_names[class_id]:15s}: "
        f"{probability:.6f}"
    )


# ------------------------------------------------------------
# 19. CONSISTENCY CHECK
# ------------------------------------------------------------

stored = existing_prediction.iloc[0]

stored_source = (
    stored["predicted_class"]
)

stored_confidence = float(
    stored["prediction_confidence"]
)

print("\n" + "=" * 70)
print("PREDICTION CONSISTENCY CHECK")
print("=" * 70)

print(
    "Stored source :",
    stored_source
)

print(
    "Fresh source  :",
    predicted_class
)

print(
    "Stored conf.  :",
    round(
        stored_confidence,
        6
    )
)

print(
    "Fresh conf.   :",
    round(
        confidence,
        6
    )
)

if stored_source == predicted_class:

    print(
        "\nSOURCE CHECK: PASS"
    )

else:

    print(
        "\nSOURCE CHECK: FAIL"
    )

if np.isclose(
    stored_confidence,
    confidence,
    atol=1e-6
):

    print(
        "CONFIDENCE CHECK: PASS"
    )

else:

    print(
        "CONFIDENCE CHECK: DIFFERENT"
    )


# ------------------------------------------------------------
# 20. FINAL RESULT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SINGLE-POINT TEST RESULT")
print("=" * 70)

print(
    f"Coordinates : "
    f"{TEST_LAT}, {TEST_LON}"
)

print(
    f"Date        : {TEST_DATE}"
)

print(
    f"Time        : {TEST_TIME}"
)

print(
    f"Event ID    : {event_id}"
)

print(
    f"Source      : {predicted_class}"
)

print(
    f"Confidence  : {confidence:.6f}"
)

print(
    f"Status      : {status}"
)

print("\n" + "=" * 70)
print("STEP 10B COMPLETE")
print("=" * 70)

# %% [code] {"jupyter":{"outputs_hidden":false}}
# ======================================================================
# 2026 FIRE SOURCE CLASSIFICATION
# FINAL SINGLE-CELL KAGGLE PIPELINE
# ======================================================================

import os
import re
import time
import requests
import joblib
import warnings
import numpy as np
import pandas as pd

from io import StringIO
from IPython.display import display

warnings.filterwarnings("ignore")


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

print("=" * 70)
print("2026 FIRE SOURCE CLASSIFICATION PIPELINE - FINAL")
print("=" * 70)


# ----------------------------------------------------------------------
# FIRMS MAP KEY
# ----------------------------------------------------------------------
# IMPORTANT:
# Put your NEW FIRMS MAP KEY here.
# Do NOT use the key that was exposed in the previous code.
# ----------------------------------------------------------------------

MAP_KEY = "bbbbc47df4cd1e089cd78119863f9af2"


# ----------------------------------------------------------------------
# MODEL
# ----------------------------------------------------------------------

MODEL_PATH = (
    "/kaggle/working/"
    "firms_2025_final_production_random_forest.joblib"
)

IMPUTER_PATH = (
    "/kaggle/working/"
    "firms_2025_final_production_imputer.joblib"
)

FEATURE_SCHEMA_PATH = (
    "/kaggle/working/"
    "firms_2025_final_model_feature_columns.csv"
)


# ----------------------------------------------------------------------
# 2025 EVIDENCE
# ----------------------------------------------------------------------

INDUSTRIAL_PATH = (
    "/kaggle/input/datasets/gautamkumar0036/"
    "google-colab-all-real-data/"
    "firms_industrial_evidence_indicators_2025_corrected.csv"
)

MINING_PATH = (
    "/kaggle/input/datasets/gautamkumar0036/"
    "google-colab-all-real-data/"
    "firms_coal_mine_distance_evidence_2025.csv"
)

AGRICULTURE_PATH = (
    "/kaggle/input/datasets/gautamkumar0036/"
    "google-colab-all-real-data/"
    "firms_agriculture_evidence_indicators_2025.csv"
)

WILDFIRE_PATH = (
    "/kaggle/input/datasets/gautamkumar0036/"
    "google-colab-all-real-data/"
    "firms_wildfire_evidence_2025.csv"
)

FLARE_PATH = (
    "/kaggle/input/datasets/gautamkumar0036/"
    "flare-real-correct-data/"
    "Flare-Volume-Estimates-by-individual-Flare-Location-2012-2025.xlsx"
)


# ----------------------------------------------------------------------
# OUTPUT
# ----------------------------------------------------------------------

PREDICTION_OUTPUT = (
    "/kaggle/working/"
    "firms_2026_test_point_prediction.csv"
)

FEATURE_OUTPUT = (
    "/kaggle/working/"
    "firms_2026_test_point_137_features.csv"
)


# ----------------------------------------------------------------------
# FIRMS
# ----------------------------------------------------------------------

FIRMS_SOURCES = [
    "VIIRS_NOAA20_NRT",
    "VIIRS_NOAA21_NRT",
    "VIIRS_SNPP_NRT"
]

FIRMS_SP_SOURCES = [
    "VIIRS_NOAA20_SP",
    "VIIRS_NOAA21_SP",
    "VIIRS_SNPP_SP"
]

CURRENT_DAYS = 5
HISTORICAL_DAYS = 30

EVENT_RADIUS_KM = 1.0
EVENT_TIME_HOURS = 48.0

FIRMS_BBOX_DEG = 0.05


# ----------------------------------------------------------------------
# OSM
# ----------------------------------------------------------------------

OSM_RADIUS_M = 5000

OVERPASS_SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter"
]


# ----------------------------------------------------------------------
# DYNAMIC WORLD
# ----------------------------------------------------------------------
#
# Dynamic World is derived from Sentinel-2 and may not have an image
# exactly on the FIRMS date.
#
# We therefore search a wide window and select the closest usable image.
# ----------------------------------------------------------------------

EE_PROJECT = "possible-haven-507714-f0"

DW_AREA_M = 300

DW_SEARCH_BEFORE_DAYS = 120
DW_SEARCH_AFTER_DAYS = 3

DW_BANDS = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice"
]

DW_CLASS_NAMES = {
    0: "water",
    1: "trees",
    2: "grass",
    3: "flooded_vegetation",
    4: "crops",
    5: "shrub_and_scrub",
    6: "built",
    7: "bare",
    8: "snow_and_ice"
}


# ======================================================================
# 2. HELPERS
# ======================================================================

def safe_float(x):

    try:

        if x is None or pd.isna(x):
            return np.nan

        return float(x)

    except Exception:

        return np.nan


def haversine_m(
    lat1,
    lon1,
    lat2,
    lon2
):

    R = 6371000.0

    lat1 = np.radians(
        np.asarray(lat1, dtype=float)
    )

    lat2 = np.radians(
        np.asarray(lat2, dtype=float)
    )

    dlat = lat2 - lat1

    dlon = np.radians(
        np.asarray(lon2, dtype=float)
        -
        np.asarray(lon1, dtype=float)
    )

    a = (
        np.sin(dlat / 2.0) ** 2
        +
        np.cos(lat1)
        *
        np.cos(lat2)
        *
        np.sin(dlon / 2.0) ** 2
    )

    return (
        2.0
        *
        R
        *
        np.arcsin(
            np.sqrt(a)
        )
    )


def find_column(
    df,
    exact=None,
    contains=None
):

    if df is None:
        return None

    cols = list(df.columns)

    if exact:

        target = str(
            exact
        ).strip().lower()

        for c in cols:

            if (
                str(c)
                .strip()
                .lower()
                ==
                target
            ):

                return c

    if contains:

        for c in cols:

            cl = str(c).lower()

            if all(
                str(x).lower()
                in cl
                for x in contains
            ):

                return c

    return None


def make_acq_datetime(df):

    date = pd.to_datetime(
        df["acq_date"],
        errors="coerce"
    )

    time_numeric = pd.to_numeric(
        df["acq_time"],
        errors="coerce"
    )

    time_string = (
        time_numeric
        .fillna(0)
        .astype(int)
        .astype(str)
        .str.zfill(4)
    )

    return pd.to_datetime(
        date.dt.strftime("%Y-%m-%d")
        + " "
        + time_string,
        format="%Y-%m-%d %H%M",
        errors="coerce"
    )


def safe_stat(
    s,
    function,
    default=np.nan
):

    s = pd.to_numeric(
        s,
        errors="coerce"
    ).dropna()

    if len(s) == 0:
        return default

    if function == "mean":
        return float(s.mean())

    if function == "median":
        return float(s.median())

    if function == "min":
        return float(s.min())

    if function == "max":
        return float(s.max())

    if function == "std":

        if len(s) >= 2:
            return float(s.std())

        return 0.0

    return default


# ======================================================================
# 3. USER INPUT
# ======================================================================

lat = float(
    input(
        "Enter latitude : "
    ).strip()
)

lon = float(
    input(
        "Enter longitude: "
    ).strip()
)


print("\nUser location:")
print(
    "Latitude :",
    lat
)

print(
    "Longitude:",
    lon
)


# ======================================================================
# 4. LOAD MODEL
# ======================================================================

print("\n" + "=" * 70)
print("2025 MODEL")
print("=" * 70)


rf_model = joblib.load(
    MODEL_PATH
)

rf_imputer = joblib.load(
    IMPUTER_PATH
)

schema = pd.read_csv(
    FEATURE_SCHEMA_PATH
)


if "feature" in schema.columns:

    MODEL_FEATURES = (
        schema["feature"]
        .astype(str)
        .tolist()
    )

elif "feature_name" in schema.columns:

    MODEL_FEATURES = (
        schema["feature_name"]
        .astype(str)
        .tolist()
    )

else:

    MODEL_FEATURES = (
        schema.iloc[:, 0]
        .astype(str)
        .tolist()
    )


print(
    "✓ 2025 model loaded"
)

print(
    "Trees   :",
    getattr(
        rf_model,
        "n_estimators",
        "unknown"
    )
)

print(
    "Features:",
    len(MODEL_FEATURES)
)


if len(MODEL_FEATURES) != 137:

    raise RuntimeError(
        "Expected exactly 137 model features."
    )


# ======================================================================
# 5. FIRMS API
# ======================================================================

if (
    not MAP_KEY
    or
    "PASTE_" in MAP_KEY
):

    raise RuntimeError(
        "\nPaste your NEW FIRMS MAP KEY into MAP_KEY."
    )


def firms_request(
    source,
    bbox_string,
    days=5,
    date=None
):

    if date is None:

        url = (
            "https://firms.modaps.eosdis.nasa.gov/"
            "api/area/csv/"
            f"{MAP_KEY}/"
            f"{source}/"
            f"{bbox_string}/"
            f"{days}"
        )

    else:

        url = (
            "https://firms.modaps.eosdis.nasa.gov/"
            "api/area/csv/"
            f"{MAP_KEY}/"
            f"{source}/"
            f"{bbox_string}/"
            f"{days}/"
            f"{date}"
        )

    try:

        response = requests.get(
            url,
            timeout=60
        )

        if response.status_code != 200:

            return None, response.status_code

        if not response.text.strip():

            return (
                pd.DataFrame(),
                200
            )

        return (
            pd.read_csv(
                StringIO(
                    response.text
                )
            ),
            200
        )

    except Exception as e:

        print(
            "FIRMS request error:",
            str(e)[:150]
        )

        return None, None


# ======================================================================
# 6. CURRENT FIRMS SEARCH
# ======================================================================

print("\n" + "=" * 70)
print("NASA FIRMS SEARCH")
print("=" * 70)


bbox = (
    f"{lon - FIRMS_BBOX_DEG},"
    f"{lat - FIRMS_BBOX_DEG},"
    f"{lon + FIRMS_BBOX_DEG},"
    f"{lat + FIRMS_BBOX_DEG}"
)


print(
    "BBOX:",
    bbox
)

print(
    "Searching last",
    CURRENT_DAYS,
    "days..."
)


current_frames = []
successful_current_requests = 0


for source in FIRMS_SOURCES:

    print("\n" + "-" * 60)
    print("Source:", source)

    df_source, status_code = firms_request(
        source,
        bbox,
        CURRENT_DAYS
    )

    print(
        "HTTP status:",
        status_code
    )

    if (
        status_code != 200
        or
        df_source is None
    ):

        print(
            "⚠ Request failed"
        )

        continue

    successful_current_requests += 1

    print(
        "Detections:",
        len(df_source)
    )

    if len(df_source):

        df_source[
            "firms_source_api"
        ] = source

        current_frames.append(
            df_source
        )


if successful_current_requests == 0:

    raise RuntimeError(
        "All current FIRMS requests failed."
    )


if not current_frames:

    raise RuntimeError(
        "No FIRMS detection found within "
        "approximately 5 km during the last 5 days."
    )


firms_current = pd.concat(
    current_frames,
    ignore_index=True
)


# ======================================================================
# 7. CLEAN FIRMS
# ======================================================================

for c in [
    "latitude",
    "longitude",
    "frp",
    "bright_ti4",
    "bright_ti5",
    "scan",
    "track"
]:

    if c in firms_current.columns:

        firms_current[c] = pd.to_numeric(
            firms_current[c],
            errors="coerce"
        )


firms_current["acq_date"] = pd.to_datetime(
    firms_current["acq_date"],
    errors="coerce"
)


firms_current["acq_datetime"] = (
    make_acq_datetime(
        firms_current
    )
)


firms_current = (
    firms_current
    .dropna(
        subset=[
            "latitude",
            "longitude",
            "acq_datetime"
        ]
    )
    .copy()
)


firms_current[
    "distance_m_from_input"
] = haversine_m(
    lat,
    lon,
    firms_current["latitude"].values,
    firms_current["longitude"].values
)


firms_current = (
    firms_current
    .sort_values(
        [
            "distance_m_from_input",
            "acq_datetime"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


# ======================================================================
# 8. SELECT FIRMS DETECTION
# ======================================================================

selected = firms_current.iloc[0]

selected_datetime = selected[
    "acq_datetime"
]


print("\n" + "=" * 70)
print("SELECTED FIRMS DETECTION")
print("=" * 70)


print(
    "Latitude   :",
    selected["latitude"]
)

print(
    "Longitude  :",
    selected["longitude"]
)

print(
    "Date       :",
    selected["acq_date"]
)

print(
    "Time       :",
    selected["acq_time"]
)

print(
    "FRP        :",
    selected.get(
        "frp",
        np.nan
    )
)

print(
    "Satellite  :",
    selected.get(
        "satellite",
        np.nan
    )
)

print(
    "Instrument :",
    selected.get(
        "instrument",
        np.nan
    )
)

print(
    "Confidence :",
    selected.get(
        "confidence",
        np.nan
    )
)

print(
    "Source     :",
    selected[
        "firms_source_api"
    ]
)

print(
    "Distance from input:",
    round(
        selected[
            "distance_m_from_input"
        ],
        2
    ),
    "m"
)


# ======================================================================
# 9. EVENT CONTEXT
# ======================================================================

print("\n" + "=" * 70)
print("EVENT CONTEXT")
print("=" * 70)


firms_current[
    "event_distance_m"
] = haversine_m(
    selected["latitude"],
    selected["longitude"],
    firms_current["latitude"].values,
    firms_current["longitude"].values
)


firms_current[
    "event_time_difference_hours"
] = (
    (
        firms_current[
            "acq_datetime"
        ]
        -
        selected_datetime
    )
    .abs()
    .dt.total_seconds()
    /
    3600.0
)


event_df = firms_current[
    (
        firms_current[
            "event_distance_m"
        ]
        <=
        EVENT_RADIUS_KM * 1000
    )
    &
    (
        firms_current[
            "event_time_difference_hours"
        ]
        <=
        EVENT_TIME_HOURS
    )
].copy()


if len(event_df) == 0:

    event_df = pd.DataFrame(
        [selected]
    )


event_df = (
    event_df
    .drop_duplicates()
    .reset_index(drop=True)
)


print(
    "Candidate observations:",
    len(event_df)
)


display(
    event_df[
        [
            "latitude",
            "longitude",
            "acq_date",
            "acq_datetime",
            "frp",
            "event_distance_m",
            "event_time_difference_hours"
        ]
    ]
)


# ======================================================================
# 10. HISTORICAL FIRMS
# ======================================================================

print("\n" + "=" * 70)
print("FIRMS TEMPORAL HISTORY")
print("=" * 70)


selected_date = (
    pd.Timestamp(
        selected["acq_date"]
    ).normalize()
)


chunk_starts = [

    selected_date - pd.Timedelta(days=29),
    selected_date - pd.Timedelta(days=24),
    selected_date - pd.Timedelta(days=19),
    selected_date - pd.Timedelta(days=14),
    selected_date - pd.Timedelta(days=9),
    selected_date - pd.Timedelta(days=4)
]


historical_frames = []

historical_successful_requests = 0
historical_total_requests = 0


for source in FIRMS_SOURCES:

    for start_date in chunk_starts:

        historical_total_requests += 1

        df_hist, status_code = firms_request(
            source,
            bbox,
            5,
            start_date.strftime("%Y-%m-%d")
        )

        if (
            status_code == 200
            and
            df_hist is not None
        ):

            historical_successful_requests += 1

            if len(df_hist):

                df_hist[
                    "firms_source_api"
                ] = source

                historical_frames.append(
                    df_hist
                )


print(
    "Successful NRT history requests:",
    historical_successful_requests,
    "/",
    historical_total_requests
)


# ----------------------------------------------------------------------
# Standard processing fallback
# ----------------------------------------------------------------------

if historical_successful_requests == 0:

    print(
        "NRT historical data unavailable."
    )

    print(
        "Trying Standard Processing..."
    )

    for source in FIRMS_SP_SOURCES:

        for start_date in chunk_starts:

            historical_total_requests += 1

            df_hist, status_code = firms_request(
                source,
                bbox,
                5,
                start_date.strftime("%Y-%m-%d")
            )

            if (
                status_code == 200
                and
                df_hist is not None
            ):

                historical_successful_requests += 1

                if len(df_hist):

                    df_hist[
                        "firms_source_api"
                    ] = source

                    historical_frames.append(
                        df_hist
                    )


if historical_frames:

    historical_df = pd.concat(
        historical_frames,
        ignore_index=True
    )

else:

    historical_df = pd.DataFrame()


if len(historical_df):

    historical_df["acq_date"] = pd.to_datetime(
        historical_df["acq_date"],
        errors="coerce"
    )

    historical_df["acq_datetime"] = (
        make_acq_datetime(
            historical_df
        )
    )

    historical_df = historical_df.dropna(
        subset=[
            "latitude",
            "longitude",
            "acq_datetime"
        ]
    ).copy()


    history_start = (
        selected_datetime
        -
        pd.Timedelta(days=29)
    )


    historical_df = historical_df[
        (
            historical_df[
                "acq_datetime"
            ]
            >=
            history_start
        )
        &
        (
            historical_df[
                "acq_datetime"
            ]
            <=
            selected_datetime
        )
    ].copy()


    historical_df[
        "distance_to_selected_m"
    ] = haversine_m(
        selected["latitude"],
        selected["longitude"],
        historical_df["latitude"].values,
        historical_df["longitude"].values
    )


    historical_df = historical_df[
        historical_df[
            "distance_to_selected_m"
        ]
        >
        100
    ].copy()


    identity_columns = [
        c
        for c in [
            "latitude",
            "longitude",
            "acq_datetime",
            "frp"
        ]
        if c in historical_df.columns
    ]


    if identity_columns:

        historical_df = (
            historical_df
            .drop_duplicates(
                subset=identity_columns
            )
        )


else:

    historical_df = pd.DataFrame()


print(
    "Total successful history requests:",
    historical_successful_requests
)

print(
    "Historical detections:",
    len(historical_df)
)


if historical_successful_requests == 0:

    print(
        "⚠ HISTORY STATUS: UNAVAILABLE"
    )

elif len(historical_df) == 0:

    print(
        "✓ HISTORY STATUS: QUERY SUCCEEDED, "
        "BUT ZERO HISTORICAL DETECTIONS"
    )

else:

    print(
        "✓ HISTORY STATUS: AVAILABLE"
    )


# ======================================================================
# 11. TEMPORAL FEATURES
# ======================================================================

temporal_features = {}


if historical_successful_requests > 0:

    if len(historical_df):

        historical_df[
            "hours_before_selected"
        ] = (
            (
                selected_datetime
                -
                historical_df[
                    "acq_datetime"
                ]
            )
            .dt.total_seconds()
            /
            3600
        )

        historical_df[
            "days_before_selected"
        ] = (
            historical_df[
                "hours_before_selected"
            ] / 24
        )


    lat_step_500 = (
        0.5 / 111.32
    )


    selected_lat500 = np.floor(
        selected["latitude"]
        /
        lat_step_500
    )


    selected_lon500 = np.floor(
        selected["longitude"]
        /
        (
            0.5
            /
            (
                111.32
                *
                np.cos(
                    np.radians(
                        selected["latitude"]
                    )
                )
            )
        )
    )


    if len(historical_df):

        historical_df[
            "_lat500"
        ] = np.floor(
            historical_df[
                "latitude"
            ] / lat_step_500
        )

        historical_df[
            "_lon500"
        ] = np.floor(
            historical_df[
                "longitude"
            ]
            /
            (
                0.5
                /
                (
                    111.32
                    *
                    np.cos(
                        np.radians(
                            historical_df[
                                "latitude"
                            ]
                        )
                    )
                )
            )
        )


        same_cell = (
            (
                historical_df[
                    "_lat500"
                ]
                ==
                selected_lat500
            )
            &
            (
                historical_df[
                    "_lon500"
                ]
                ==
                selected_lon500
            )
        )


        nearby_1km = (
            historical_df[
                "distance_to_selected_m"
            ]
            <=
            1000
        )


    for days in [3, 7, 30]:

        if len(historical_df):

            mask = (
                historical_df[
                    "days_before_selected"
                ]
                <=
                days
            )

            same_count = int(
                (
                    same_cell
                    &
                    mask
                ).sum()
            )

            nearby_count = int(
                (
                    nearby_1km
                    &
                    mask
                ).sum()
            )

        else:

            same_count = 0
            nearby_count = 0


        temporal_features[
            f"same_cell_fire_count_{days}d_max"
        ] = same_count

        temporal_features[
            f"nearby_fire_count_{days}d_1km_max"
        ] = nearby_count

        temporal_features[
            f"event_max_same_cell_fire_count_{days}d"
        ] = same_count

        temporal_features[
            f"event_max_nearby_fire_count_{days}d_1km"
        ] = nearby_count


else:

    for days in [3, 7, 30]:

        temporal_features[
            f"same_cell_fire_count_{days}d_max"
        ] = np.nan

        temporal_features[
            f"nearby_fire_count_{days}d_1km_max"
        ] = np.nan

        temporal_features[
            f"event_max_same_cell_fire_count_{days}d"
        ] = np.nan

        temporal_features[
            f"event_max_nearby_fire_count_{days}d_1km"
        ] = np.nan


print("\nTemporal features:")

for k, v in temporal_features.items():

    print(
        f"  {k:<45} {v}"
    )


# ======================================================================
# 12. EVENT FEATURES
# ======================================================================

print("\n" + "=" * 70)
print("FIRMS EVENT FEATURES")
print("=" * 70)


ev = event_df.copy()


ev["acq_date"] = pd.to_datetime(
    ev["acq_date"],
    errors="coerce"
)

ev["acq_datetime"] = make_acq_datetime(
    ev
)


for c in [
    "latitude",
    "longitude",
    "frp",
    "bright_ti4",
    "bright_ti5"
]:

    if c in ev.columns:

        ev[c] = pd.to_numeric(
            ev[c],
            errors="coerce"
        )


n = len(ev)


active_days = (
    ev["acq_date"]
    .dt.date
    .nunique()
)

active_months = (
    ev["acq_date"]
    .dt.month
    .nunique()
)


frp = pd.to_numeric(
    ev["frp"],
    errors="coerce"
)

brightness = pd.to_numeric(
    ev.get(
        "bright_ti4",
        pd.Series(
            np.nan,
            index=ev.index
        )
    ),
    errors="coerce"
)


mean_latitude = float(
    ev["latitude"].mean()
)

mean_longitude = float(
    ev["longitude"].mean()
)

min_latitude = float(
    ev["latitude"].min()
)

max_latitude = float(
    ev["latitude"].max()
)

min_longitude = float(
    ev["longitude"].min()
)

max_longitude = float(
    ev["longitude"].max()
)


latitude_span_deg = (
    max_latitude - min_latitude
)

longitude_span_deg = (
    max_longitude - min_longitude
)

latitude_span_km = (
    latitude_span_deg * 111.32
)

longitude_span_km = (
    longitude_span_deg
    *
    111.32
    *
    np.cos(
        np.radians(
            mean_latitude
        )
    )
)


# ----------------------------------------------------------------------
# 500m / 1km blocks
# ----------------------------------------------------------------------

lat_step_500 = 0.5 / 111.32
lat_step_1km = 1.0 / 111.32


ev["_lat500"] = np.floor(
    ev["latitude"] / lat_step_500
)

ev["_lon500"] = np.floor(
    ev["longitude"]
    /
    (
        0.5
        /
        (
            111.32
            *
            np.cos(
                np.radians(
                    ev["latitude"]
                )
            )
        )
    )
)


ev["_lat1km"] = np.floor(
    ev["latitude"] / lat_step_1km
)

ev["_lon1km"] = np.floor(
    ev["longitude"]
    /
    (
        1.0
        /
        (
            111.32
            *
            np.cos(
                np.radians(
                    ev["latitude"]
                )
            )
        )
    )
)


block_500_count = (
    ev[
        [
            "_lat500",
            "_lon500"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


block_1km_count = (
    ev[
        [
            "_lat1km",
            "_lon1km"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


times = (
    ev["acq_datetime"]
    .dropna()
    .sort_values()
)


if len(times) >= 2:

    duration_hours = (
        times.max()
        -
        times.min()
    ).total_seconds() / 3600

else:

    duration_hours = 0.0


duration_days = (
    duration_hours / 24
)


if len(times) >= 2:

    gaps_hours = (
        times
        .diff()
        .dt.total_seconds()
        / 3600
    ).dropna()

else:

    gaps_hours = pd.Series(
        dtype=float
    )


unique_days = (
    ev["acq_date"]
    .dt.normalize()
    .drop_duplicates()
    .sort_values()
)


if len(unique_days) >= 2:

    day_gaps = (
        unique_days
        .diff()
        .dt.total_seconds()
        / 86400
    ).dropna()

else:

    day_gaps = pd.Series(
        dtype=float
    )


daily_counts = (
    ev
    .groupby(
        ev["acq_date"].dt.date
    )
    .size()
)


if len(daily_counts):

    max_daily = int(
        daily_counts.max()
    )

    mean_daily = float(
        daily_counts.mean()
    )

    median_daily = float(
        daily_counts.median()
    )

else:

    max_daily = 0
    mean_daily = 0
    median_daily = 0


monthly_counts = (
    ev["acq_date"]
    .dt.month
    .value_counts()
)


if len(monthly_counts):

    peak_month_detections = int(
        monthly_counts.max()
    )

    peak_month = int(
        monthly_counts.idxmax()
    )

    monthly_concentration = (
        peak_month_detections / n
    )

else:

    peak_month_detections = 0
    peak_month = np.nan
    monthly_concentration = np.nan


hours = (
    pd.to_numeric(
        ev["acq_time"],
        errors="coerce"
    )
    // 100
).dropna()


spatial_extent_km = (
    haversine_m(
        min_latitude,
        min_longitude,
        max_latitude,
        max_longitude
    )
    / 1000
)


frp_valid = frp.dropna()


if (
    len(frp_valid) >= 2
    and
    frp_valid.mean() != 0
):

    frp_cv = float(
        frp_valid.std()
        /
        frp_valid.mean()
    )

else:

    frp_cv = 0.0


event_features = {

    "event_detection_count": n,

    "event_active_days": active_days,

    "event_mean_frp":
        safe_stat(frp, "mean"),

    "event_median_frp":
        safe_stat(frp, "median"),

    "event_max_frp":
        safe_stat(frp, "max"),

    "event_std_frp":
        safe_stat(frp, "std", 0.0),

    "event_min_frp":
        safe_stat(frp, "min"),

    "event_sum_frp":
        (
            float(frp_valid.sum())
            if len(frp_valid)
            else np.nan
        ),

    "event_mean_brightness":
        safe_stat(
            brightness,
            "mean"
        ),

    "event_max_brightness":
        safe_stat(
            brightness,
            "max"
        ),

    "event_median_brightness":
        safe_stat(
            brightness,
            "median"
        ),

    "event_std_brightness":
        safe_stat(
            brightness,
            "std",
            0.0
        ),

    "event_min_brightness":
        safe_stat(
            brightness,
            "min"
        ),

    "event_mean_latitude":
        mean_latitude,

    "event_mean_longitude":
        mean_longitude,

    "event_min_latitude":
        min_latitude,

    "event_max_latitude":
        max_latitude,

    "event_min_longitude":
        min_longitude,

    "event_max_longitude":
        max_longitude,

    "event_500m_block_count":
        block_500_count,

    "event_1km_block_count":
        block_1km_count,

    "event_duration_hours":
        duration_hours,

    "event_duration_days":
        duration_days,

    "event_mean_detection_gap_hours":
        (
            float(gaps_hours.mean())
            if len(gaps_hours)
            else 0
        ),

    "event_median_detection_gap_hours":
        (
            float(gaps_hours.median())
            if len(gaps_hours)
            else 0
        ),

    "event_max_detection_gap_hours":
        (
            float(gaps_hours.max())
            if len(gaps_hours)
            else 0
        ),

    "event_min_detection_gap_hours":
        (
            float(gaps_hours.min())
            if len(gaps_hours)
            else 0
        ),

    "event_mean_active_day_gap":
        (
            float(day_gaps.mean())
            if len(day_gaps)
            else 0
        ),

    "event_median_active_day_gap":
        (
            float(day_gaps.median())
            if len(day_gaps)
            else 0
        ),

    "event_max_active_day_gap":
        (
            float(day_gaps.max())
            if len(day_gaps)
            else 0
        ),

    "event_min_active_day_gap":
        (
            float(day_gaps.min())
            if len(day_gaps)
            else 0
        ),

    "event_latitude_span_deg":
        latitude_span_deg,

    "event_longitude_span_deg":
        longitude_span_deg,

    "event_latitude_span_km":
        latitude_span_km,

    "event_longitude_span_km":
        longitude_span_km,

    "event_spatial_extent_km":
        spatial_extent_km,

    "event_spatial_extent_per_active_day_km":
        (
            spatial_extent_km / active_days
            if active_days
            else 0
        ),

    "event_detections_per_500m_block":
        n / block_500_count
        if block_500_count
        else 0,

    "event_detections_per_1km_block":
        n / block_1km_count
        if block_1km_count
        else 0,

    "event_detections_per_active_day":
        n / active_days
        if active_days
        else 0,

    "event_detections_per_hour":
        n / duration_hours
        if duration_hours > 0
        else float(n),

    "event_peak_month_detections":
        peak_month_detections,

    "event_max_detections_one_day":
        max_daily,

    "event_mean_detections_active_day":
        mean_daily,

    "event_median_detections_active_day":
        median_daily,

    "event_active_months":
        active_months,

    "event_peak_month":
        peak_month,

    "event_monthly_concentration":
        monthly_concentration,

    "event_frp_range":
        (
            float(
                frp_valid.max()
                -
                frp_valid.min()
            )
            if len(frp_valid)
            else np.nan
        ),

    "event_frp_cv":
        frp_cv,

    "event_mean_acquisition_hour":
        (
            float(hours.mean())
            if len(hours)
            else np.nan
        ),

    "event_median_acquisition_hour":
        (
            float(hours.median())
            if len(hours)
            else np.nan
        ),

    "event_min_acquisition_hour":
        (
            float(hours.min())
            if len(hours)
            else np.nan
        ),

    "event_max_acquisition_hour":
        (
            float(hours.max())
            if len(hours)
            else np.nan
        ),

    "event_acquisition_hour_count":
        int(hours.nunique()),

    "event_satellite_count":
        (
            int(
                ev["satellite"].nunique()
            )
            if "satellite" in ev.columns
            else np.nan
        ),

    "event_day_fraction":
        (
            float(
                (
                    ev["daynight"] == "D"
                ).mean()
            )
            if "daynight" in ev.columns
            else np.nan
        ),

    "event_night_fraction":
        (
            float(
                (
                    ev["daynight"] == "N"
                ).mean()
            )
            if "daynight" in ev.columns
            else np.nan
        )
}


# ======================================================================
# 13. EVIDENCE AGGREGATION
# ======================================================================

def aggregate_nearest_evidence(
    df,
    event_df,
    columns_min=None,
    columns_max=None
):

    columns_min = columns_min or []
    columns_max = columns_max or []

    result = {}

    for c in columns_min:
        result[c] = np.nan

    for c in columns_max:
        result[c] = np.nan

    if df is None or len(df) == 0:
        return result

    lat_col = (
        find_column(df, exact="latitude")
        or
        find_column(df, exact="lat")
    )

    lon_col = (
        find_column(df, exact="longitude")
        or
        find_column(df, exact="lon")
    )

    if lat_col is None or lon_col is None:
        return result

    source = df.copy()

    source["_lat"] = pd.to_numeric(
        source[lat_col],
        errors="coerce"
    )

    source["_lon"] = pd.to_numeric(
        source[lon_col],
        errors="coerce"
    )

    source = source.dropna(
        subset=[
            "_lat",
            "_lon"
        ]
    )

    if len(source) == 0:
        return result

    source_lat = source["_lat"].values
    source_lon = source["_lon"].values

    values_min = {
        c: []
        for c in columns_min
    }

    values_max = {
        c: []
        for c in columns_max
    }

    for _, row in event_df.iterrows():

        distances = haversine_m(
            float(row["latitude"]),
            float(row["longitude"]),
            source_lat,
            source_lon
        )

        nearest_idx = int(
            np.argmin(
                distances
            )
        )

        evidence_row = source.iloc[
            nearest_idx
        ]

        for c in columns_min:

            if c in evidence_row.index:

                value = safe_float(
                    evidence_row[c]
                )

                if pd.notna(value):

                    values_min[c].append(
                        value
                    )

        for c in columns_max:

            if c in evidence_row.index:

                value = safe_float(
                    evidence_row[c]
                )

                if pd.notna(value):

                    values_max[c].append(
                        value
                    )

    for c in columns_min:

        if values_min[c]:

            result[c] = min(
                values_min[c]
            )

    for c in columns_max:

        if values_max[c]:

            result[c] = max(
                values_max[c]
            )

    return result


# ======================================================================
# 14. WORLD BANK FLARES
# ======================================================================

print("\n" + "=" * 70)
print("WORLD BANK FLARE EVIDENCE")
print("=" * 70)


flare_features = {

    "nearest_flare_distance_km": np.nan,
    "nearest_flare_2025_activity": np.nan,

    "active_flare_count_within_1km": 0,
    "has_active_flare_within_1km": 0,

    "active_flare_count_within_2km": 0,
    "has_active_flare_within_2km": 0,

    "active_flare_count_within_5km": 0,
    "has_active_flare_within_5km": 0,

    "active_flare_count_within_10km": 0,
    "has_active_flare_within_10km": 0,

    "nearest_gas_flare_distance_km": np.nan,
    "nearest_gas_flare_2025_activity": np.nan,

    "gas_flare_count_within_1km": 0,
    "has_gas_flare_within_1km": 0,

    "gas_flare_count_within_2km": 0,
    "has_gas_flare_within_2km": 0,

    "gas_flare_count_within_5km": 0,
    "has_gas_flare_within_5km": 0,

    "gas_flare_count_within_10km": 0,
    "has_gas_flare_within_10km": 0,

    "nearest_flare_is_gas": 0,
    "nearest_flare_is_oil": 0,
    "nearest_flare_is_unknown": 0
}


try:

    flare_df = pd.read_excel(
        FLARE_PATH,
        sheet_name=(
            "2012-2025-Flare-Volume-Estimate"
        )
    )

    flare_df.columns = [
        str(c).strip()
        for c in flare_df.columns
    ]

    flare_lat_col = (
        find_column(
            flare_df,
            exact="Latitude"
        )
        or
        find_column(
            flare_df,
            contains=["lat"]
        )
    )

    flare_lon_col = (
        find_column(
            flare_df,
            exact="Longitude"
        )
        or
        find_column(
            flare_df,
            contains=["lon"]
        )
    )

    if flare_lat_col and flare_lon_col:

        flare_df["_lat"] = pd.to_numeric(
            flare_df[flare_lat_col],
            errors="coerce"
        )

        flare_df["_lon"] = pd.to_numeric(
            flare_df[flare_lon_col],
            errors="coerce"
        )

        country_col = find_column(
            flare_df,
            exact="Country"
        )

        if country_col:

            india = flare_df[
                flare_df[country_col]
                .astype(str)
                .str.strip()
                .str.lower()
                == "india"
            ].copy()

        else:

            india = flare_df[
                (
                    flare_df["_lat"] >= 6
                )
                &
                (
                    flare_df["_lat"] <= 38
                )
                &
                (
                    flare_df["_lon"] >= 67
                )
                &
                (
                    flare_df["_lon"] <= 98
                )
            ].copy()

        activity_col = None

        for c in india.columns:

            cl = str(c).lower()

            if (
                "2025" in cl
                and
                (
                    "volume" in cl
                    or
                    "flare" in cl
                )
            ):

                activity_col = c
                break

        if activity_col:

            india["_activity"] = pd.to_numeric(
                india[activity_col],
                errors="coerce"
            ).fillna(0)

        else:

            india["_activity"] = 0.0

        india = india[
            india["_activity"] > 0
        ].copy()

        if len(india):

            india["_distance_m"] = haversine_m(
                selected["latitude"],
                selected["longitude"],
                india["_lat"].values,
                india["_lon"].values
            )

            nearest = (
                india
                .sort_values(
                    "_distance_m"
                )
                .iloc[0]
            )

            flare_features[
                "nearest_flare_distance_km"
            ] = (
                nearest["_distance_m"]
                / 1000
            )

            flare_features[
                "nearest_flare_2025_activity"
            ] = nearest["_activity"]

            for radius in [1, 2, 5, 10]:

                mask = (
                    india["_distance_m"]
                    <=
                    radius * 1000
                )

                flare_features[
                    f"active_flare_count_within_{radius}km"
                ] = int(mask.sum())

                flare_features[
                    f"has_active_flare_within_{radius}km"
                ] = int(mask.any())

            field_col = None

            for c in india.columns:

                cl = str(c).lower()

                if (
                    "field" in cl
                    and
                    "type" in cl
                ):

                    field_col = c
                    break

            if field_col:

                nearest_type = str(
                    nearest[field_col]
                ).lower()

            else:

                nearest_type = ""

            flare_features[
                "nearest_flare_is_gas"
            ] = int(
                "gas" in nearest_type
            )

            flare_features[
                "nearest_flare_is_oil"
            ] = int(
                "oil" in nearest_type
            )

            flare_features[
                "nearest_flare_is_unknown"
            ] = int(
                "gas" not in nearest_type
                and
                "oil" not in nearest_type
            )

            if field_col:

                gas = india[
                    india[field_col]
                    .astype(str)
                    .str.lower()
                    .str.contains(
                        "gas",
                        na=False
                    )
                ].copy()

            else:

                gas = pd.DataFrame()

            if len(gas):

                gas["_distance_m"] = haversine_m(
                    selected["latitude"],
                    selected["longitude"],
                    gas["_lat"].values,
                    gas["_lon"].values
                )

                nearest_gas = (
                    gas
                    .sort_values(
                        "_distance_m"
                    )
                    .iloc[0]
                )

                flare_features[
                    "nearest_gas_flare_distance_km"
                ] = (
                    nearest_gas["_distance_m"]
                    / 1000
                )

                flare_features[
                    "nearest_gas_flare_2025_activity"
                ] = nearest_gas[
                    "_activity"
                ]

                for radius in [1, 2, 5, 10]:

                    mask = (
                        gas["_distance_m"]
                        <=
                        radius * 1000
                    )

                    flare_features[
                        f"gas_flare_count_within_{radius}km"
                    ] = int(mask.sum())

                    flare_features[
                        f"has_gas_flare_within_{radius}km"
                    ] = int(mask.any())

            print(
                "Active India 2025 flares:",
                len(india)
            )

        else:

            print(
                "No active India 2025 flares."
            )

    else:

        print(
            "Flare coordinates not found."
        )

except Exception as e:

    print(
        "⚠ Flare evidence error:",
        e
    )


# ======================================================================
# 15. LIVE OSM
# ======================================================================

print("\n" + "=" * 70)
print("LIVE OPENSTREETMAP")
print("=" * 70)


OSM_QUERY_GROUPS = {

    "infrastructure": f"""
    [out:json][timeout:60];
    (
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["man_made"="flare"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["man_made"="petroleum_well"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["man_made"="oil_well"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["man_made"="mineshaft"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["man_made"="adit"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["man_made"="gasometer"];
    );
    out body center;
    """,

    "industry_mining": f"""
    [out:json][timeout:60];
    (
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="industrial"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["industrial"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="quarry"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="mine"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["man_made"="mineshaft"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["man_made"="adit"];
    );
    out body center;
    """,

    "agriculture": f"""
    [out:json][timeout:60];
    (
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="farmland"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="farmyard"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="orchard"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="vineyard"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="plant_nursery"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="greenhouse_horticulture"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="allotments"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["building"="greenhouse"];
    );
    out body center;
    """,

    "natural": f"""
    [out:json][timeout:60];
    (
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["natural"="wood"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["natural"="forest"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["natural"="scrub"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["natural"="grassland"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["natural"="heath"];
      nwr(around:{OSM_RADIUS_M},{lat},{lon})["landuse"="forest"];
    );
    out body center;
    """
}


osm_elements = []


def query_overpass(
    query,
    group_name
):

    for server in OVERPASS_SERVERS:

        print(
            f"\n{group_name} →",
            server
        )

        try:

            response = requests.post(
                server,
                data={
                    "data": query
                },
                headers={
                    "User-Agent":
                    "fire-source-classification-research"
                },
                timeout=90
            )

            print(
                "HTTP status:",
                response.status_code
            )

            if response.status_code == 200:

                data = response.json()

                return data.get(
                    "elements",
                    []
                )

        except Exception as e:

            print(
                "⚠",
                str(e)[:200]
            )

    return []


for group_name, query in OSM_QUERY_GROUPS.items():

    elements = query_overpass(
        query,
        group_name
    )

    print(
        "Objects returned:",
        len(elements)
    )

    osm_elements.extend(
        elements
    )


unique_elements = {}

for element in osm_elements:

    key = (
        element.get("type"),
        element.get("id")
    )

    unique_elements[key] = element


osm_elements = list(
    unique_elements.values()
)


print(
    "\nTotal unique OSM objects:",
    len(osm_elements)
)


OSM_CATEGORIES = [

    "flare",
    "oil_well",
    "mineshaft",
    "adit",
    "gasometer",
    "industrial",
    "quarry",
    "farmland",
    "farmyard",
    "orchard",
    "vineyard",
    "plant_nursery",
    "greenhouse",
    "allotment",
    "forest",
    "scrub",
    "grassland",
    "heath"
]


osm_nearest = {
    c: np.nan
    for c in OSM_CATEGORIES
}


osm_counts = {
    c: 0
    for c in OSM_CATEGORIES
}


def element_coordinates(
    element
):

    if element.get("type") == "node":

        if (
            "lat" in element
            and
            "lon" in element
        ):

            return [
                (
                    float(element["lat"]),
                    float(element["lon"])
                )
            ]

        return []

    if "center" in element:

        center = element["center"]

        if (
            "lat" in center
            and
            "lon" in center
        ):

            return [
                (
                    float(center["lat"]),
                    float(center["lon"])
                )
            ]

    geometry = element.get(
        "geometry",
        []
    )

    return [
        (
            float(p["lat"]),
            float(p["lon"])
        )
        for p in geometry
        if (
            "lat" in p
            and
            "lon" in p
        )
    ]


def classify_osm(
    tags
):

    man_made = tags.get(
        "man_made"
    )

    landuse = tags.get(
        "landuse"
    )

    natural = tags.get(
        "natural"
    )

    building = tags.get(
        "building"
    )

    if man_made == "flare":
        return "flare"

    if man_made in [
        "petroleum_well",
        "oil_well"
    ]:
        return "oil_well"

    if man_made == "mineshaft":
        return "mineshaft"

    if man_made == "adit":
        return "adit"

    if man_made == "gasometer":
        return "gasometer"

    if (
        landuse == "industrial"
        or
        "industrial" in tags
    ):
        return "industrial"

    if landuse == "quarry":
        return "quarry"

    if landuse == "mine":
        return "quarry"

    if landuse == "farmland":
        return "farmland"

    if landuse == "farmyard":
        return "farmyard"

    if landuse == "orchard":
        return "orchard"

    if landuse == "vineyard":
        return "vineyard"

    if landuse == "plant_nursery":
        return "plant_nursery"

    if (
        landuse == "greenhouse_horticulture"
        or
        building == "greenhouse"
    ):
        return "greenhouse"

    if landuse == "allotments":
        return "allotment"

    if (
        natural in [
            "wood",
            "forest"
        ]
        or
        landuse == "forest"
    ):
        return "forest"

    if natural == "scrub":
        return "scrub"

    if natural == "grassland":
        return "grassland"

    if natural == "heath":
        return "heath"

    return None


for element in osm_elements:

    tags = element.get(
        "tags",
        {}
    )

    category = classify_osm(
        tags
    )

    if category is None:
        continue

    coords = element_coordinates(
        element
    )

    if not coords:
        continue

    distance = min(
        haversine_m(
            lat,
            lon,
            xlat,
            xlon
        )
        for xlat, xlon in coords
    )

    osm_counts[category] += 1

    if (
        pd.isna(
            osm_nearest[category]
        )
        or
        distance
        <
        osm_nearest[category]
    ):

        osm_nearest[category] = distance


print("\nOSM category summary:")

for category in OSM_CATEGORIES:

    print(
        f"{category:20s}",
        "count =",
        osm_counts[category],
        "nearest =",
        (
            round(
                osm_nearest[category],
                1
            )
            if pd.notna(
                osm_nearest[category]
            )
            else "NaN"
        )
    )


agriculture_osm_values = [

    osm_nearest[c]

    for c in [
        "farmland",
        "farmyard",
        "orchard",
        "vineyard",
        "plant_nursery",
        "greenhouse",
        "allotment"
    ]

    if pd.notna(
        osm_nearest[c]
    )
]


agri_osm_distance = (
    float(
        min(
            agriculture_osm_values
        )
    )
    if agriculture_osm_values
    else np.nan
)


print(
    "\nAgriculture OSM distance:",
    agri_osm_distance
)


# ======================================================================
# 16. LOAD 2025 EVIDENCE
# ======================================================================

print("\n" + "=" * 70)
print("2025 EVIDENCE DATA")
print("=" * 70)


def load_evidence(
    path,
    name
):

    if not os.path.exists(path):

        print(
            f"⚠ {name} file missing"
        )

        return pd.DataFrame()

    df = pd.read_csv(
        path
    )

    print(
        f"{name} records:",
        len(df)
    )

    return df


industrial_df = load_evidence(
    INDUSTRIAL_PATH,
    "Industrial"
)

mining_df = load_evidence(
    MINING_PATH,
    "Mining"
)

agriculture_df = load_evidence(
    AGRICULTURE_PATH,
    "Agriculture"
)

wildfire_df = load_evidence(
    WILDFIRE_PATH,
    "Wildfire"
)


# ======================================================================
# 17. INDUSTRIAL
# ======================================================================

industrial_columns_min = [

    "steel_distance_m",
    "cement_distance_m",
    "wri_power_distance_m",
    "fertilizer_distance_m",
    "refinery_petro_distance_m"
]


industrial_columns_max = [

    "industrial_source_count_500m",
    "industrial_source_count_1km",
    "industrial_evidence_score"
]


industrial_raw = aggregate_nearest_evidence(
    industrial_df,
    event_df,
    columns_min=industrial_columns_min,
    columns_max=industrial_columns_max
)


industrial_features = {

    "event_min_steel_distance_m":
        industrial_raw[
            "steel_distance_m"
        ],

    "event_min_cement_distance_m":
        industrial_raw[
            "cement_distance_m"
        ],

    "event_min_wri_power_distance_m":
        industrial_raw[
            "wri_power_distance_m"
        ],

    "event_min_fertilizer_distance_m":
        industrial_raw[
            "fertilizer_distance_m"
        ],

    "event_min_refinery_petro_distance_m":
        industrial_raw[
            "refinery_petro_distance_m"
        ],

    "event_max_industrial_source_count_500m":
        industrial_raw[
            "industrial_source_count_500m"
        ],

    "event_max_industrial_source_count_1km":
        industrial_raw[
            "industrial_source_count_1km"
        ],

    "event_max_industrial_evidence_score":
        industrial_raw[
            "industrial_evidence_score"
        ]
}


# ======================================================================
# 18. MINING
# ======================================================================

print("\n" + "=" * 70)
print("MINING EVIDENCE")
print("=" * 70)


mine_distance_column = None


known_mine_columns = [

    "nearest_coal_mine_distance_m",
    "coal_mine_distance_m",
    "event_min_nearest_coal_mine_distance_m",
    "nearest_mine_distance_m"
]


for candidate in known_mine_columns:

    if candidate in mining_df.columns:

        mine_distance_column = candidate
        break


if mine_distance_column is None:

    for c in mining_df.columns:

        cl = str(c).lower()

        if (
            "coal" in cl
            and
            "mine" in cl
            and
            "distance" in cl
        ):

            mine_distance_column = c
            break


print(
    "Mining distance source column:",
    mine_distance_column
)


mine_raw = aggregate_nearest_evidence(
    mining_df,
    event_df,
    columns_min=(
        [mine_distance_column]
        if mine_distance_column
        else []
    )
)


mine_distance_m = (
    mine_raw.get(
        mine_distance_column,
        np.nan
    )
    if mine_distance_column
    else np.nan
)


mining_features = {

    "event_min_nearest_coal_mine_distance_m":
        mine_distance_m,

    "event_min_nearest_coal_mine_distance_km":
        (
            mine_distance_m / 1000
            if pd.notna(
                mine_distance_m
            )
            else np.nan
        )
}


print(
    "Nearest coal mine distance:",
    mine_distance_m,
    "m"
)


# ======================================================================
# 19. AGRICULTURE
# ======================================================================

agri_score_column = None


for c in agriculture_df.columns:

    cl = str(c).lower()

    if (
        "agri" in cl
        and
        "score" in cl
    ):

        agri_score_column = c
        break


agri_raw = aggregate_nearest_evidence(
    agriculture_df,
    event_df,
    columns_max=(
        [agri_score_column]
        if agri_score_column
        else []
    )
)


agriculture_features = {

    "agriculture_evidence_score_max":
        (
            agri_raw.get(
                agri_score_column,
                np.nan
            )
            if agri_score_column
            else np.nan
        ),

    "agri_osm_distance_m_min":
        agri_osm_distance
}


# ======================================================================
# 20. WILDFIRE / NATURAL
# ======================================================================

wildfire_distance_columns = [

    "distance_to_nearest_flare_m",
    "distance_to_nearest_oil_well_m",
    "distance_to_nearest_mineshaft_m",
    "distance_to_nearest_adit_m",
    "distance_to_nearest_gasometer_m",
    "distance_to_nearest_industrial_m",
    "distance_to_nearest_quarry_m",
    "distance_to_nearest_farmland_m",
    "distance_to_nearest_farmyard_m",
    "distance_to_nearest_orchard_m",
    "distance_to_nearest_vineyard_m",
    "distance_to_nearest_plant_nursery_m",
    "distance_to_nearest_greenhouse_m",
    "distance_to_nearest_allotment_m",
    "distance_to_nearest_forest_m",
    "distance_to_nearest_scrub_m",
    "distance_to_nearest_grassland_m",
    "distance_to_nearest_heath_m",
    "distance_to_nearest_agriculture_m",
    "distance_to_nearest_natural_vegetation_m"
]


wildfire_raw = aggregate_nearest_evidence(
    wildfire_df,
    event_df,
    columns_min=wildfire_distance_columns
)


wildfire_features = {}


for c in wildfire_distance_columns:

    wildfire_features[
        c + "_min"
    ] = wildfire_raw.get(
        c,
        np.nan
    )


natural_context_column = find_column(
    wildfire_df,
    exact="natural_landcover_context"
)

natural_strong_column = find_column(
    wildfire_df,
    exact="natural_vegetation_strong"
)

natural_moderate_column = find_column(
    wildfire_df,
    exact="natural_vegetation_moderate"
)


natural_columns = [
    c
    for c in [
        natural_context_column,
        natural_strong_column,
        natural_moderate_column
    ]
    if c is not None
]


natural_raw = aggregate_nearest_evidence(
    wildfire_df,
    event_df,
    columns_max=natural_columns
)


wildfire_features[
    "natural_landcover_context_max"
] = (
    natural_raw.get(
        natural_context_column,
        np.nan
    )
    if natural_context_column
    else np.nan
)


wildfire_features[
    "natural_vegetation_strong_max"
] = (
    natural_raw.get(
        natural_strong_column,
        np.nan
    )
    if natural_strong_column
    else np.nan
)


wildfire_features[
    "natural_vegetation_moderate_max"
] = (
    natural_raw.get(
        natural_moderate_column,
        np.nan
    )
    if natural_moderate_column
    else np.nan
)


# ======================================================================
# 21. DYNAMIC WORLD - FINAL FIX
# ======================================================================

print("\n" + "=" * 70)
print("DYNAMIC WORLD")
print("=" * 70)


dw_rows = []


try:

    import ee


    try:

        ee.Initialize(
            project=EE_PROJECT
        )

    except Exception:

        print(
            "Earth Engine authentication required..."
        )

        ee.Authenticate(
            auth_mode="notebook",
            force=True
        )

        ee.Initialize(
            project=EE_PROJECT
        )


    def get_dw_for_point(
        fire_lat,
        fire_lon,
        fire_date
    ):

        fire_date = pd.Timestamp(
            fire_date
        ).normalize()


        point = ee.Geometry.Point(
            [
                float(fire_lon),
                float(fire_lat)
            ]
        )


        area = (
            point
            .buffer(
                DW_AREA_M / 2
            )
            .bounds()
        )


        # --------------------------------------------------------------
        # Search a large time window.
        #
        # Dynamic World can have no image on the exact FIRMS date.
        # --------------------------------------------------------------

        search_start = (
            fire_date
            -
            pd.Timedelta(
                days=DW_SEARCH_BEFORE_DAYS
            )
        )


        search_end = (
            fire_date
            +
            pd.Timedelta(
                days=DW_SEARCH_AFTER_DAYS + 1
            )
        )


        collection = (
            ee.ImageCollection(
                "GOOGLE/DYNAMICWORLD/V1"
            )
            .filterBounds(
                point
            )
            .filterDate(
                search_start.strftime(
                    "%Y-%m-%d"
                ),
                search_end.strftime(
                    "%Y-%m-%d"
                )
            )
        )


        count = int(
            collection
            .size()
            .getInfo()
        )


        print(
            "DW images in search window:",
            count
        )


        if count == 0:

            return {
                "status":
                    "NO_IMAGE"
            }


        image_list = collection.toList(
            count
        )


        candidates = []


        # --------------------------------------------------------------
        # Get all image dates.
        # --------------------------------------------------------------

        for i in range(count):

            try:

                image = ee.Image(
                    image_list.get(i)
                )


                image_date = (
                    ee.Date(
                        image.get(
                            "system:time_start"
                        )
                    )
                    .format(
                        "YYYY-MM-dd"
                    )
                    .getInfo()
                )


                image_date_ts = pd.Timestamp(
                    image_date
                )


                difference = abs(
                    (
                        image_date_ts
                        -
                        fire_date
                    ).total_seconds()
                ) / 86400.0


                candidates.append(
                    (
                        difference,
                        image_date,
                        image
                    )
                )


            except Exception as e:

                print(
                    "DW date error:",
                    str(e)[:150]
                )


        # --------------------------------------------------------------
        # IMPORTANT:
        # Try closest image first.
        # If cloud masking leaves no valid pixels,
        # try the next closest image.
        # --------------------------------------------------------------

        candidates.sort(
            key=lambda x: x[0]
        )


        for (
            difference,
            image_date,
            image
        ) in candidates:

            print(
                "Trying DW image:",
                image_date,
                "| difference:",
                round(
                    difference,
                    2
                ),
                "days"
            )


            try:

                # ------------------------------------------------------
                # Probability bands
                # ------------------------------------------------------

                means = (
                    image
                    .select(
                        DW_BANDS
                    )
                    .reduceRegion(
                        reducer=ee.Reducer.mean(),
                        geometry=area,
                        scale=10,
                        bestEffort=True,
                        maxPixels=100000
                    )
                    .getInfo()
                )


                if not means:

                    print(
                        "  → no probability result"
                    )

                    continue


                valid_probability_values = [

                    means.get(
                        band
                    )

                    for band in DW_BANDS

                    if means.get(
                        band
                    ) is not None
                ]


                if len(
                    valid_probability_values
                ) == 0:

                    print(
                        "  → all probability pixels masked"
                    )

                    continue


                # ------------------------------------------------------
                # Label histogram
                # ------------------------------------------------------

                histogram_result = (
                    image
                    .select(
                        "label"
                    )
                    .reduceRegion(
                        reducer=(
                            ee.Reducer
                            .frequencyHistogram()
                        ),
                        geometry=area,
                        scale=10,
                        bestEffort=True,
                        maxPixels=100000
                    )
                    .getInfo()
                )


                histogram = (
                    histogram_result.get(
                        "label",
                        {}
                    )
                    if histogram_result
                    else {}
                )


                histogram = {

                    int(float(k)): int(v)

                    for k, v in histogram.items()
                }


                if histogram:

                    dominant_label = max(
                        histogram,
                        key=histogram.get
                    )

                    total_pixels = sum(
                        histogram.values()
                    )

                    dominant_probability = (
                        histogram[
                            dominant_label
                        ]
                        /
                        total_pixels
                    )

                    dominant_class = (
                        DW_CLASS_NAMES.get(
                            dominant_label,
                            "unknown"
                        )
                    )

                else:

                    dominant_label = None

                    dominant_probability = np.nan

                    dominant_class = None


                # ------------------------------------------------------
                # Natural vegetation
                # ------------------------------------------------------

                natural_values = [

                    means.get(
                        "trees"
                    ),

                    means.get(
                        "grass"
                    ),

                    means.get(
                        "shrub_and_scrub"
                    ),

                    means.get(
                        "flooded_vegetation"
                    )
                ]


                if all(
                    x is not None
                    for x in natural_values
                ):

                    natural_probability = sum(
                        float(x)
                        for x in natural_values
                    )

                else:

                    natural_probability = np.nan


                print(
                    "  ✓ VALID DW IMAGE FOUND"
                )


                return {

                    "status":
                        "OK",

                    "dynamic_world_date":
                        image_date,

                    "date_difference_days":
                        float(
                            difference
                        ),

                    "dw_water":
                        means.get(
                            "water"
                        ),

                    "dw_trees":
                        means.get(
                            "trees"
                        ),

                    "dw_grass":
                        means.get(
                            "grass"
                        ),

                    "dw_flooded_vegetation":
                        means.get(
                            "flooded_vegetation"
                        ),

                    "dw_crops":
                        means.get(
                            "crops"
                        ),

                    "dw_shrub_and_scrub":
                        means.get(
                            "shrub_and_scrub"
                        ),

                    "dw_built":
                        means.get(
                            "built"
                        ),

                    "dw_bare":
                        means.get(
                            "bare"
                        ),

                    "dw_snow_and_ice":
                        means.get(
                            "snow_and_ice"
                        ),

                    "dw_dominant_class":
                        dominant_class,

                    "dw_dominant_probability":
                        dominant_probability,

                    "dw_crop_probability":
                        means.get(
                            "crops"
                        ),

                    "dw_top_probability":
                        dominant_probability,

                    "dw_trees_prob":
                        means.get(
                            "trees"
                        ),

                    "dw_grass_prob":
                        means.get(
                            "grass"
                        ),

                    "dw_shrub_prob":
                        means.get(
                            "shrub_and_scrub"
                        ),

                    "dw_flooded_vegetation_prob":
                        means.get(
                            "flooded_vegetation"
                        ),

                    "dw_crop_prob":
                        means.get(
                            "crops"
                        ),

                    "dw_natural_vegetation_prob":
                        natural_probability
                }


            except Exception as e:

                print(
                    "  ⚠ DW image failed:",
                    str(e)[:200]
                )

                continue


        return {
            "status":
                "NO_VALID_PIXELS"
        }


    # ------------------------------------------------------------------
    # Query every event detection
    # ------------------------------------------------------------------

    for i, row in event_df.iterrows():

        try:

            result = get_dw_for_point(
                row["latitude"],
                row["longitude"],
                row["acq_date"]
            )

            result[
                "event_index"
            ] = i

            dw_rows.append(
                result
            )

        except Exception as e:

            print(
                "DW point error:",
                str(e)[:200]
            )


except Exception as e:

    print(
        "⚠ Earth Engine error:",
        e
    )


dw_df = pd.DataFrame(
    dw_rows
)


# ======================================================================
# 22. DYNAMIC WORLD EVENT AGGREGATION
# ======================================================================

dw_event_features = {}


DW_EVENT_FEATURES = [

    "event_max_dw_crop_probability",
    "event_mean_dw_crop_probability",

    "event_max_dw_top_probability",
    "event_mean_dw_top_probability",

    "event_max_dw_trees_prob",
    "event_mean_dw_trees_prob",

    "event_max_dw_grass_prob",
    "event_mean_dw_grass_prob",

    "event_max_dw_shrub_prob",
    "event_mean_dw_shrub_prob",

    "event_max_dw_flooded_vegetation_prob",
    "event_mean_dw_flooded_vegetation_prob",

    "event_max_dw_crop_prob",
    "event_mean_dw_crop_prob",

    "event_max_dw_natural_vegetation_prob",
    "event_mean_dw_natural_vegetation_prob"
]


print(
    "\nDynamic World event records:",
    len(dw_df)
)


if len(dw_df):

    print(
        "Dynamic World status:",
        dw_df[
            "status"
        ]
        .value_counts()
        .to_dict()
    )


def dw_max(column):

    if column not in dw_df.columns:
        return np.nan

    values = pd.to_numeric(
        dw_df[column],
        errors="coerce"
    ).dropna()

    return (
        float(values.max())
        if len(values)
        else np.nan
    )


def dw_mean(column):

    if column not in dw_df.columns:
        return np.nan

    values = pd.to_numeric(
        dw_df[column],
        errors="coerce"
    ).dropna()

    return (
        float(values.mean())
        if len(values)
        else np.nan
    )


dw_event_features[
    "event_max_dw_crop_probability"
] = dw_max(
    "dw_crops"
)

dw_event_features[
    "event_mean_dw_crop_probability"
] = dw_mean(
    "dw_crops"
)

dw_event_features[
    "event_max_dw_top_probability"
] = dw_max(
    "dw_dominant_probability"
)

dw_event_features[
    "event_mean_dw_top_probability"
] = dw_mean(
    "dw_dominant_probability"
)

dw_event_features[
    "event_max_dw_trees_prob"
] = dw_max(
    "dw_trees"
)

dw_event_features[
    "event_mean_dw_trees_prob"
] = dw_mean(
    "dw_trees"
)

dw_event_features[
    "event_max_dw_grass_prob"
] = dw_max(
    "dw_grass"
)

dw_event_features[
    "event_mean_dw_grass_prob"
] = dw_mean(
    "dw_grass"
)

dw_event_features[
    "event_max_dw_shrub_prob"
] = dw_max(
    "dw_shrub_and_scrub"
)

dw_event_features[
    "event_mean_dw_shrub_prob"
] = dw_mean(
    "dw_shrub_and_scrub"
)

dw_event_features[
    "event_max_dw_flooded_vegetation_prob"
] = dw_max(
    "dw_flooded_vegetation"
)

dw_event_features[
    "event_mean_dw_flooded_vegetation_prob"
] = dw_mean(
    "dw_flooded_vegetation"
)

dw_event_features[
    "event_max_dw_crop_prob"
] = dw_max(
    "dw_crops"
)

dw_event_features[
    "event_mean_dw_crop_prob"
] = dw_mean(
    "dw_crops"
)

dw_event_features[
    "event_max_dw_natural_vegetation_prob"
] = dw_max(
    "dw_natural_vegetation_prob"
)

dw_event_features[
    "event_mean_dw_natural_vegetation_prob"
] = dw_mean(
    "dw_natural_vegetation_prob"
)


# Make sure all DW columns exist

for c in DW_EVENT_FEATURES:

    if c not in dw_event_features:

        dw_event_features[c] = np.nan


# ======================================================================
# 23. BASE DYNAMIC WORLD
# ======================================================================

first_valid_dw = None


if len(dw_df):

    valid_mask = (
        dw_df["status"] == "OK"
    )

    if valid_mask.any():

        first_valid_dw = (
            dw_df[
                valid_mask
            ]
            .iloc[0]
        )


dw_base_features = {}


for band in DW_BANDS:

    if (
        first_valid_dw is not None
        and
        f"dw_{band}" in first_valid_dw.index
    ):

        dw_base_features[
            f"dw_{band}"
        ] = first_valid_dw[
            f"dw_{band}"
        ]

    else:

        dw_base_features[
            f"dw_{band}"
        ] = np.nan


dw_base_features[
    "dw_dominant_probability"
] = (
    first_valid_dw[
        "dw_dominant_probability"
    ]
    if first_valid_dw is not None
    else np.nan
)


dw_base_features[
    "dw_dominant_class"
] = (
    first_valid_dw[
        "dw_dominant_class"
    ]
    if first_valid_dw is not None
    else np.nan
)


# ======================================================================
# 24. COMBINE FEATURES
# ======================================================================

print("\n" + "=" * 70)
print("COMBINING FEATURES")
print("=" * 70)


features = {}

features.update(
    event_features
)

features.update(
    flare_features
)

features.update(
    industrial_features
)

features.update(
    temporal_features
)

features.update(
    mining_features
)

features.update(
    agriculture_features
)

features.update(
    wildfire_features
)

features.update(
    dw_event_features
)

features.update(
    dw_base_features
)


# ======================================================================
# 25. OSM FEATURES
# ======================================================================

for category in OSM_CATEGORIES:

    features[
        f"distance_to_nearest_{category}_m"
    ] = osm_nearest[
        category
    ]


features[
    "distance_to_nearest_agriculture_m"
] = agri_osm_distance


# ======================================================================
# 26. FEATURE ALIASES
# ======================================================================

ALIASES = {

    "event_frp_mean":
        "event_mean_frp",

    "event_frp_median":
        "event_median_frp",

    "event_frp_max":
        "event_max_frp",

    "event_frp_std":
        "event_std_frp",

    "event_occupied_blocks_500m":
        "event_500m_block_count",

    "event_occupied_blocks_1km":
        "event_1km_block_count",

    "event_duration":
        "event_duration_hours"
}


for old, new in ALIASES.items():

    if (
        old in features
        and
        new not in features
    ):

        features[new] = features[old]


# ======================================================================
# 27. EXACT 137 FEATURES
# ======================================================================

raw_df = pd.DataFrame(
    [features]
)


print(
    "Generated feature values:",
    len(raw_df.columns)
)


X_test = raw_df.reindex(
    columns=MODEL_FEATURES
)


for c in X_test.columns:

    X_test[c] = pd.to_numeric(
        X_test[c],
        errors="coerce"
    )


print(
    "Final model matrix:",
    X_test.shape
)


if X_test.shape != (1, 137):

    raise RuntimeError(
        f"Expected (1,137), got {X_test.shape}"
    )


# ======================================================================
# 28. FEATURE QUALITY
# ======================================================================

print("\n" + "=" * 70)
print("FEATURE QUALITY CHECK")
print("=" * 70)


missing_features = [

    c

    for c in MODEL_FEATURES

    if pd.isna(
        X_test.iloc[0][c]
    )
]


available_features = (
    137
    -
    len(missing_features)
)


coverage = (
    available_features
    /
    137
)


print(
    "Available before imputation:",
    available_features,
    "/ 137"
)

print(
    "Missing before imputation:",
    len(missing_features)
)

print(
    "Feature coverage:",
    f"{coverage:.2%}"
)


critical_firms = [

    "event_detection_count",
    "event_active_days",
    "event_mean_frp",
    "event_median_frp",
    "event_max_frp",
    "event_std_frp",

    "event_mean_latitude",
    "event_mean_longitude",

    "event_min_latitude",
    "event_max_latitude",

    "event_min_longitude",
    "event_max_longitude",

    "event_500m_block_count",
    "event_1km_block_count",

    "event_duration_hours",
    "event_duration_days",

    "event_detections_per_active_day",
    "event_detections_per_hour"
]


critical_missing = [

    c

    for c in critical_firms

    if (
        c in MODEL_FEATURES
        and
        pd.isna(
            X_test.iloc[0][c]
        )
    )
]


if critical_missing:

    print(
        "\nCRITICAL FEATURES MISSING:"
    )

    for c in critical_missing:

        print(
            " -",
            c
        )

    raise RuntimeError(
        "Critical FIRMS event features missing."
    )


print(
    "✓ Critical FIRMS event features populated."
)


if missing_features:

    print(
        "\nRemaining missing features:"
    )

    for c in missing_features:

        print(
            " -",
            c
        )

else:

    print(
        "✓ No missing features before imputation."
    )


# ======================================================================
# 29. DYNAMIC WORLD DIAGNOSTIC
# ======================================================================

print("\n" + "=" * 70)
print("DYNAMIC WORLD DIAGNOSTIC")
print("=" * 70)


if len(dw_df):

    print(
        "Total DW event records:",
        len(dw_df)
    )

    print(
        "DW status:",
        dw_df[
            "status"
        ]
        .value_counts()
        .to_dict()
    )


    valid_dw = dw_df[
        dw_df["status"] == "OK"
    ]


    if len(valid_dw):

        print(
            "\nSelected Dynamic World:"
        )

        for _, row in valid_dw.iterrows():

            print(
                "  FIRMS event:",
                row.get(
                    "event_index"
                )
            )

            print(
                "  DW date:",
                row.get(
                    "dynamic_world_date"
                )
            )

            print(
                "  Date difference:",
                row.get(
                    "date_difference_days"
                ),
                "days"
            )

            print(
                "  DW dominant class:",
                row.get(
                    "dw_dominant_class"
                )
            )

            print(
                "  DW top probability:",
                row.get(
                    "dw_dominant_probability"
                )
            )

            print(
                "  DW crops:",
                row.get(
                    "dw_crops"
                )
            )

            print(
                "  DW trees:",
                row.get(
                    "dw_trees"
                )
            )

            print(
                "  DW grass:",
                row.get(
                    "dw_grass"
                )
            )

    else:

        print(
            "⚠ No valid Dynamic World pixels."
        )

else:

    print(
        "⚠ Dynamic World returned no records."
    )


# ======================================================================
# 30. IMPORTANT FEATURE VALUES
# ======================================================================

diagnostic_names = [

    "event_detection_count",
    "event_mean_frp",
    "event_max_frp",

    "event_mean_latitude",
    "event_mean_longitude",

    "event_duration_hours",
    "event_duration_days",

    "event_detections_per_active_day",
    "event_detections_per_hour",

    "event_min_steel_distance_m",
    "event_min_cement_distance_m",
    "event_min_wri_power_distance_m",
    "event_min_fertilizer_distance_m",
    "event_min_refinery_petro_distance_m",

    "event_min_nearest_coal_mine_distance_m",
    "event_min_nearest_coal_mine_distance_km",

    "event_max_same_cell_fire_count_3d",
    "event_max_same_cell_fire_count_7d",
    "event_max_same_cell_fire_count_30d",

    "same_cell_fire_count_3d_max",
    "same_cell_fire_count_7d_max",
    "same_cell_fire_count_30d_max",

    "nearby_fire_count_3d_1km_max",
    "nearby_fire_count_7d_1km_max",
    "nearby_fire_count_30d_1km_max",

    "event_max_nearby_fire_count_3d_1km",
    "event_max_nearby_fire_count_7d_1km",
    "event_max_nearby_fire_count_30d_1km",

    "agri_osm_distance_m_min",

    "event_max_dw_crop_probability",
    "event_mean_dw_crop_probability",

    "event_max_dw_top_probability",
    "event_mean_dw_top_probability",

    "event_max_dw_trees_prob",
    "event_mean_dw_trees_prob",

    "event_max_dw_grass_prob",
    "event_mean_dw_grass_prob",

    "event_max_dw_natural_vegetation_prob",
    "event_mean_dw_natural_vegetation_prob"
]


diagnostic_names = [
    c
    for c in diagnostic_names
    if c in X_test.columns
]


print(
    "\nImportant feature values:"
)


if diagnostic_names:

    display(
        X_test[
            diagnostic_names
        ].T.rename(
            columns={
                0: "value"
            }
        )
    )


# ======================================================================
# 31. IMPUTATION
# ======================================================================

print("\n" + "=" * 70)
print("IMPUTATION")
print("=" * 70)


X_imputed = rf_imputer.transform(
    X_test
)


print(
    "✓ 2025 imputer applied"
)


if X_imputed.shape != (1, 137):

    raise RuntimeError(
        "Imputed matrix is not (1,137)."
    )


if missing_features:

    print(
        "\nImputed values:"
    )

    imputed_series = pd.Series(
        X_imputed[0],
        index=MODEL_FEATURES
    )

    for c in missing_features:

        print(
            f"  {c:<55}"
            f"{imputed_series[c]:.6f}"
        )


# ======================================================================
# 32. RANDOM FOREST
# ======================================================================

print("\n" + "=" * 70)
print("RANDOM FOREST PREDICTION")
print("=" * 70)


prediction = rf_model.predict(
    X_imputed
)[0]


probabilities = (
    rf_model
    .predict_proba(
        X_imputed
    )[0]
)


classes = rf_model.classes_


CLASS_NAMES = {

    0: "Industrial",
    1: "Gas",
    2: "Agriculture",
    3: "Mining",
    4: "Wildfire"
}


probability_dict = {}


for cls, probability in zip(
    classes,
    probabilities
):

    probability_dict[
        CLASS_NAMES.get(
            int(cls),
            str(cls)
        )
    ] = float(
        probability
    )


confidence = float(
    np.max(
        probabilities
    )
)


predicted_class = CLASS_NAMES.get(
    int(prediction),
    str(prediction)
)


# ======================================================================
# 33. FINAL DECISION
# ======================================================================

CONFIDENCE_THRESHOLD = 0.70


if confidence >= CONFIDENCE_THRESHOLD:

    final_class = predicted_class
    status = "Classified"

else:

    final_class = "Unknown"
    status = "Unknown"


# ======================================================================
# 34. MODEL DIAGNOSTIC
# ======================================================================

print("\n" + "=" * 70)
print("MODEL DIAGNOSTIC")
print("=" * 70)


if hasattr(
    rf_model,
    "feature_importances_"
):

    importance_df = pd.DataFrame({

        "feature":
            MODEL_FEATURES,

        "importance":
            rf_model.feature_importances_,

        "input_value":
            X_test.iloc[0].values,

        "imputed_value":
            X_imputed[0]

    })


    importance_df = (
        importance_df
        .sort_values(
            "importance",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    print(
        "\nTop 20 model features:"
    )


    display(
        importance_df.head(20)
    )


# ======================================================================
# 35. FINAL OUTPUT
# ======================================================================

print("\n" + "=" * 70)
print("FINAL FIRE SOURCE CLASSIFICATION")
print("=" * 70)


print("\nInput:")

print(
    "  Latitude :",
    lat
)

print(
    "  Longitude:",
    lon
)


print("\nFIRMS:")

print(
    "  Latitude :",
    selected["latitude"]
)

print(
    "  Longitude:",
    selected["longitude"]
)

print(
    "  Date     :",
    selected["acq_date"]
)

print(
    "  Time     :",
    selected["acq_time"]
)

print(
    "  FRP      :",
    selected["frp"]
)

print(
    "  Satellite:",
    selected["satellite"]
)

print(
    "  Distance :",
    round(
        selected[
            "distance_m_from_input"
        ],
        2
    ),
    "m"
)


print("\nModel candidate:")

print(
    "  Class      :",
    predicted_class
)

print(
    "  Probability:",
    f"{confidence:.4f}"
)


print("\nFINAL DECISION:")

print(
    "  Class      :",
    final_class
)

print(
    "  Status     :",
    status
)


print(
    "\nClass probabilities:"
)


for name in [
    "Industrial",
    "Gas",
    "Agriculture",
    "Mining",
    "Wildfire"
]:

    print(
        f"  {name:15s}: "
        f"{probability_dict.get(name, 0):.4f}"
    )


# ======================================================================
# 36. SAVE PREDICTION
# ======================================================================

prediction_row = {

    "input_latitude":
        lat,

    "input_longitude":
        lon,

    "firms_latitude":
        selected["latitude"],

    "firms_longitude":
        selected["longitude"],

    "firms_date":
        selected["acq_date"],

    "firms_time":
        selected["acq_time"],

    "firms_frp":
        selected["frp"],

    "firms_satellite":
        selected["satellite"],

    "firms_source":
        selected[
            "firms_source_api"
        ],

    "firms_distance_m":
        selected[
            "distance_m_from_input"
        ],

    "event_detection_count":
        n,

    "candidate_class_id":
        int(prediction),

    "candidate_class":
        predicted_class,

    "final_class":
        final_class,

    "confidence":
        confidence,

    "status":
        status,

    "probability_industrial":
        probability_dict.get(
            "Industrial",
            np.nan
        ),

    "probability_gas":
        probability_dict.get(
            "Gas",
            np.nan
        ),

    "probability_agriculture":
        probability_dict.get(
            "Agriculture",
            np.nan
        ),

    "probability_mining":
        probability_dict.get(
            "Mining",
            np.nan
        ),

    "probability_wildfire":
        probability_dict.get(
            "Wildfire",
            np.nan
        ),

    "osm_object_count":
        len(osm_elements),

    "osm_agriculture_distance_m":
        agri_osm_distance,

    "dynamic_world_event_records":
        len(dw_df),

    "dynamic_world_valid_records":
        (
            int(
                (
                    dw_df["status"] == "OK"
                ).sum()
            )
            if len(dw_df)
            else 0
        ),

    "historical_firms_detections":
        len(historical_df),

    "historical_firms_requests_successful":
        historical_successful_requests,

    "missing_before_imputation":
        len(missing_features),

    "feature_coverage":
        coverage
}


prediction_df = pd.DataFrame(
    [prediction_row]
)


prediction_df.to_csv(
    PREDICTION_OUTPUT,
    index=False
)


# ======================================================================
# 37. SAVE 137 FEATURES
# ======================================================================

X_test.to_csv(
    FEATURE_OUTPUT,
    index=False
)


# ======================================================================
# 38. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)


print(
    "✓ NASA FIRMS N20/N21/SNPP"
)

print(
    "✓ Exact 30-day FIRMS history"
)

print(
    "✓ FIRMS event context"
)

print(
    "✓ FIRMS event statistics"
)

print(
    "✓ World Bank flare evidence"
)

print(
    "✓ Live OpenStreetMap"
)

print(
    "✓ Industrial evidence"
)

print(
    "✓ Mining evidence"
)

print(
    "✓ Agriculture evidence"
)

print(
    "✓ Wildfire/natural evidence"
)

print(
    "✓ Dynamic World wide-window search"
)

print(
    "✓ Dynamic World valid-pixel fallback"
)

print(
    "✓ Exact 137-feature schema"
)

print(
    "✓ 2025 imputer"
)

print(
    "✓ 700-tree Random Forest"
)


print(
    "\nCANDIDATE CLASS:",
    predicted_class
)

print(
    "CANDIDATE PROBABILITY:",
    f"{confidence:.2%}"
)

print(
    "FINAL CLASS:",
    final_class
)

print(
    "STATUS:",
    status
)

print(
    "FEATURE COVERAGE:",
    f"{coverage:.2%}"
)

print(
    "MISSING BEFORE IMPUTATION:",
    len(missing_features)
)

print(
    "HISTORICAL DETECTIONS:",
    len(historical_df)
)

print(
    "DYNAMIC WORLD RECORDS:",
    len(dw_df)
)

print(
    "DYNAMIC WORLD VALID RECORDS:",
    (
        int(
            (
                dw_df["status"] == "OK"
            ).sum()
        )
        if len(dw_df)
        else 0
    )
)


print(
    "\nPrediction file:"
)

print(
    PREDICTION_OUTPUT
)


print(
    "\n137-feature file:"
)

print(
    FEATURE_OUTPUT
)


print("=" * 70)